# Sharm Image 2.1 — VL 4B FP8

**Un notebook complet, exécuté localement dans le kernel de votre machine distante.**

Parcours : installation → corpus RGBA → enseignant 8B → caches 4B → projecteur spatial → comparaisons → export → inférence → publication et rapports.

Le VL 4B est gelé en FP8. Le projecteur utilise un calcul BF16 et des poids maîtres FP32. Le 8B est uniquement un enseignant hors ligne. Ce profil n’entraîne aucun LoRA du diffuseur.

Tous les lancements sont à `False` dans Git. Les mettre à `True` constitue un lancement manuel réel, pas un aperçu. Les définitions complètes figurent dans l’annexe de ce même fichier ; aucun script ou notebook supplémentaire n’est requis.

**Sommaire :** 1 environnement · 2 paramètres · 3 données · 4 caches et entraînement · 5 résultats · 6 revue · 7 export et inférence · 8 publication · 9 versionnement · annexe code.

## Résultat archivé du 19 octobre 2026

**Candidat refusé après revue par agent.** Les 1 000 pas de warmup, 1 500 pas de flow et 288 générations sont terminés. Les 144 images du candidat et leurs 144 homologues 8B ont été inspectés. Le candidat répète des formes et déforme les sujets. Les 24 images de diagnostic warmup ne valident pas non plus un remplacement. Aucune nouvelle Base n’a été publiée.

Le [compte rendu et les galeries complètes](reports/README.md) conservent les résultats réels, leurs paramètres et SHA-256. Le profil ci-dessous reproduit une expérience refusée ; sa qualité n’est pas validée. Les pertes numériques ne mesurent pas une erreur de position. Aucun nouvel entraînement Monster n’a été lancé.

L’étape 9 peut archiver un rejet avec `ARCHIVE_OUTCOME = "rejected"`. Les actions restent désactivées par défaut.


In [ ]:
from pathlib import Path
import json, types, sys, textwrap

NOTEBOOK_FILE = next(
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent / 'Sharm_Image_2_1_VL4B.ipynb', parent / 'Sharm-Image-2.1-VL4B/Sharm_Image_2_1_VL4B.ipynb', parent / "2-LoRA-qwen" / 'Sharm-Image-2.1-VL4B/Sharm_Image_2_1_VL4B.ipynb')
    if candidate.is_file()
)
book = json.loads(NOTEBOOK_FILE.read_text(encoding="utf-8"))
engine = types.ModuleType("sharm_notebook_runtime")
engine.__file__ = str(NOTEBOOK_FILE)
engine.SOURCE_MODULES = book["metadata"]["sharm"]["modules"]
engine.SOURCE_ALIASES = book["metadata"]["sharm"]["aliases"]
sys.modules["sharm_notebook_runtime"] = engine
for cell in book["cells"]:
    if "runtime" in cell.get("metadata", {}).get("tags", []):
        lines = "".join(cell["source"]).splitlines(keepends=True)
        exec(compile(textwrap.dedent("".join(lines[1:])), str(NOTEBOOK_FILE), "exec"), engine.__dict__)
from sharm_notebook_runtime import (
    PROJECT, REPO, ARTIFACTS, read_json, read_jsonl, sha256, contained,
    current_dataset, archived_report, result_file, show_json, table, show_image,
    load_notebook, execute_step, run_stage,
)
print("Notebook prêt. Les calculs se lancent dans ce kernel uniquement via leurs drapeaux RUN_.")


## 1. Installer et contrôler l’environnement

Utiliser un kernel Python 3.12 sur la machine Linux CUDA. L’installation vise **le Python du kernel**, avec les versions figées. Un compilateur C est requis par Triton. Après installation ou changement de Base, redémarrer le kernel puis reprendre au début. Aucun entraînement n'est lancé par cette cellule.

In [ ]:
import os, shutil, subprocess, platform
RUN_INSTALL = False
RUN_DOWNLOAD_FP8_KERNEL = False
print("Python :", platform.python_version())
print("Compilateur C :", shutil.which(os.environ.get("CC", "cc")))
if RUN_INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT / "Sharm-Image-2.1-VL4B/requirements-sharm.txt"),
                    "-r", str(PROJECT / "Sharm-Image-2.1-VL4B/requirements-inference.txt")], check=True)
if RUN_DOWNLOAD_FP8_KERNEL:
    from kernels import get_kernel
    from huggingface_hub import snapshot_download
    get_kernel("kernels-community/finegrained-fp8", version=4)
    snapshot_download(repo_id="kernels-community/finegrained-fp8", repo_type="kernel", revision="v4", max_workers=2)


## 2. Paramètres de la Base

Le profil reste à 3 600 compositions, 256 px, 1 000 pas de warmup et 1 500 pas de flow maximum avec l'arrêt anticipé défini. Les entrées, cibles et identités des caches sont signées. Une modification incompatible demande un nouveau dossier de run.

~~~text
Texte → VL 4B FP8 → projecteur entraînable → diffuseur → VAE → PNG RGBA
                         ↑
              8B BF16 et compositions mesurées
~~~

In [ ]:
import os
os.environ.pop("SHARM_BASE_DIR", None)  # La Base se construit à partir des composants publics figés.
PROFILE = read_json(PROJECT / "Sharm-Image-2.1-VL4B/configs/remote16gb-256.json")
DATASET = ARTIFACTS / "spatial/dataset"
CACHE = PROJECT / "cache/spatial"
RUN = ARTIFACTS / "spatial/run"
RELEASE = ARTIFACTS / "spatial/release"
show_json(PROJECT / "Sharm-Image-2.1-VL4B/configs/remote16gb-256.json")


## 3. Préparer les compositions RGBA

Les assets ont leurs identités et empreintes ; le split se fait par paire d'espèces. Les images sont composées avec des boîtes normalisées mesurées. Un dataset existant n'est pas remplacé silencieusement.

In [ ]:
RUN_PREPARE_DATA = False
ASSETS = CACHE / "assets-pokeapi/assets.jsonl"
run_stage("base.prepare_assets", {"output": ASSETS.parent}, enabled=RUN_PREPARE_DATA)
run_stage("base.prepare_dataset", {"assets": ASSETS, "output": DATASET, **PROFILE["dataset"]}, enabled=RUN_PREPARE_DATA)


## 4. Caches, FP8 et entraînement

L'enseignant est téléchargé et extrait d'abord. Ses shards temporaires ne sont retirés qu'après contrôle du cache complet dans un workspace déclaré comme propriétaire. Le 4B et le diffuseur sont ensuite préparés. Le runner en mémoire reprend les caches et checkpoints compatibles et exécute les étapes dans ce kernel.

Pour un workspace neuf, activer l'initialisation du marqueur seulement si les poids téléchargés dans ce dossier sont bien temporaires. Le run historique déjà actif doit finir avant tout nouveau calcul GPU.

In [ ]:
RUN_INITIALIZE_WORKSPACE = False
if RUN_INITIALIZE_WORKSPACE:
    marker = ARTIFACTS / "spatial/remote-workspace.json"
    marker.parent.mkdir(parents=True, exist_ok=True)
    if marker.exists():
        raise FileExistsError("Le marqueur existe déjà ; conserver son identité.")
    marker.write_text(json.dumps({"root": str(REPO.resolve()), "temporary_teacher_weights": True}, indent=2) + "\n", encoding="utf-8")

RUN_BASE_TRAINING = False
run_stage("base.run_remote", enabled=RUN_BASE_TRAINING, gpu=True)


### Comment lire la MSE

MSE = moyenne des carrés des différences. La MSE relative divise cette valeur par la moyenne des carrés de la cible, avec un plancher de 10⁻⁸. L’alignement ajoute 0,1 × (1 − cosinus), puis la phase flow apprend sur les images et les trajectoires.

Une perte de 0,10 n’est pas une erreur de position de 10 %. Les positions, identités, séparations et transparences doivent être examinées sur les générations.

In [ ]:
history = read_jsonl(result_file("run/flow/history.jsonl"))
table([{"pas": r["step"], "validation": r.get("numerical_validation_loss"), "temps s": round(r.get("seconds", 0), 1)}
       for r in history], ["pas", "validation", "temps s"])
show_json(RUN / "run-status.json")
show_json(RUN / "flow/status.json")


In [ ]:
import importlib.util
if history and importlib.util.find_spec("matplotlib"):
    import matplotlib.pyplot as plt
    points = [r for r in history if "numerical_validation_loss" in r]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot([r["step"] for r in points], [r["numerical_validation_loss"] for r in points], marker="o")
    ax.set(xlabel="Pas", ylabel="Perte de validation numérique", title="Mesures enregistrées")
    ax.grid(alpha=0.25)
    plt.show()


## 5. Comparaisons appariées et PNG RGBA

24 scènes × 3 graines × 4 méthodes donnent 288 PNG, dont 144 candidats. Les prompts naturels et les modes avec coordonnées sont séparés. Le damier est ajouté à l'affichage ; les PNG d'origine ne sont pas retouchés.

In [ ]:
RUN_GENERATE_COMPARISONS = False
run_stage("base.generate", {"candidate": RUN / "flow/best", "dataset": DATASET, "cache": CACHE,
    "output": RUN / "validation", "include_teacher": True, **PROFILE["generation"]}, enabled=RUN_GENERATE_COMPARISONS, gpu=True)
PREDICTIONS = result_file("run/validation/predictions.json")
predictions = read_json(PREDICTIONS, {"rows": []})
PAGE = 0
GROUPS_PER_PAGE = 4
groups = sorted({(r["scene_id"], r["seed"]) for r in predictions["rows"]})
print("Groupes :", len(groups), "| page :", PAGE + 1)
for scene, seed in groups[PAGE * GROUPS_PER_PAGE:(PAGE + 1) * GROUPS_PER_PAGE]:
    for method in ("teacher_text", "candidate_text", "teacher_layout", "candidate_layout"):
        row = next((r for r in predictions["rows"] if (r["scene_id"], r["seed"], r["method"]) == (scene, seed, method)), None)
        if row:
            path = contained(PREDICTIONS.parent, row["file_name"])
            if sha256(path) != row["image_sha256"]: raise ValueError("PNG modifié.")
            print(row["prompt"], "| graine :", seed)
            show_image(path, method)


## 6. Revue effective et évaluation spatiale

Parcourir toutes les pages. La revue de publication doit identifier son auteur, sa méthode, les empreintes et **tous les IDs candidats examinés**. Ne renseigner `passed=True` qu'après inspection réelle ; conserver les défauts.

Le protocole statistique distinct exige des annotations humaines `human_identity_and_boxes`, mesure IoU, erreur de centre, relation, séparation et alpha, et calcule Wilson à 95 % par groupe de paires. Une revue d'agent reste `agent_visual_review` et ne remplace pas ce protocole. Ce petit benchmark ne satisfait pas à lui seul les 300 groupes indépendants requis.

In [ ]:
# Renseigner ces observations après inspection, pas avant.
REVIEWER = ""
REVIEW_METHOD = "human_visual_review"  # ou agent_visual_review pour une revue par agent
REVIEWED_IDS = []
OBSERVATIONS = []
REVIEW_PASSED = False
RUN_SAVE_VISUAL_REVIEW = False
if RUN_SAVE_VISUAL_REVIEW:
    candidate = read_json(RUN / "flow/best/candidate.json")
    required = {r["id"] for r in predictions["rows"] if r["method"].startswith("candidate_")}
    if not REVIEWER or not required or not required <= set(REVIEWED_IDS):
        raise ValueError("Identifier l'auteur et couvrir chaque candidat réellement inspecté.")
    report = {"reviewer": REVIEWER, "method": REVIEW_METHOD, "reviewed_ids": REVIEWED_IDS,
              "prediction_signature": predictions["signature"], "projector_sha256": candidate["projector_sha256"],
              "observations": OBSERVATIONS, "passed": REVIEW_PASSED}
    (ARTIFACTS / "spatial/visual-review.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

RUN_HUMAN_EVALUATION = False
run_stage("base.evaluate", {"predictions": PREDICTIONS,
    "annotations": ARTIFACTS / "spatial/human-annotations.jsonl", "output": ARTIFACTS / "spatial/human-evaluation.json"},
    enabled=RUN_HUMAN_EVALUATION)


## 7. Export et inférence réelle

L'export relie les gros poids sur le même volume. Le code Python exigé par le format HF est généré dans l'artefact d'export, hors Git. Le vérificateur charge ce code exact, vérifie le FP8 et le projecteur, et produit deux vraies images RGBA. Examiner aussi ces deux images.

In [ ]:
RUN_PREPARE_EXPORT = False
RUN_VERIFY_EXPORT = False
run_stage("publish.publish_spatial", {"action": "prepare"}, enabled=RUN_PREPARE_EXPORT)
run_stage("publish.verify_spatial_export", enabled=RUN_VERIFY_EXPORT, gpu=True)
verification = read_json(result_file("release/export-validation.json"))
if verification:
    for method in ("candidate_text", "candidate_layout"):
        path = result_file("release/validation/" + method + ".png")
        if path.exists(): show_image(path, method)


### Générer avec un prompt personnel

Le candidat doit être terminé. Des boîtes optionnelles peuvent être fournies au notebook d’inférence ; leur convention est identique à celle du corpus.

In [ ]:
RUN_PERSONAL_GENERATION = False
PROMPT = "A red cube to the left of a blue sphere, separated, transparent background"
run_stage("base.infer", {"candidate": RUN / "flow/best", "prompt": PROMPT,
    "resolution": 256, "steps": 30, "seed": 1000, "output": ARTIFACTS / "notebooks/base-example.png"},
    enabled=RUN_PERSONAL_GENERATION, gpu=True)


## 8. Publication vers la destination autorisée

La connexion utilise le gestionnaire HF local de la machine qui publie. Le jeton est saisi avec getpass et ne doit jamais être stocké dans les cellules ou sorties. Pour le suivi automatique existant, l'authentification canonique Windows reste utilisée.

Le publisher contrôle la revue, le loader, le compte, la visibilité, la révision de départ puis chaque fichier sur la révision distante immuable.

In [ ]:
RUN_HF_LOGIN = False
RUN_HF_PUSH = False
if RUN_HF_LOGIN:
    from getpass import getpass
    from huggingface_hub import login
    login(token=getpass("Jeton HF : "), add_to_git_credential=False)
run_stage("publish.publish_spatial", {"action": "push"}, enabled=RUN_HF_PUSH)
show_json(result_file("publication-status.json"))


## 9. Conserver les vrais résultats et versionner

Choisir `ARCHIVE_OUTCOME = "published"` après une publication HF vérifiée, ou `"rejected"` après la revue complète d’un candidat refusé. Le mode rejet conserve toute la galerie, les historiques, les paramètres, les observations et les SHA, sans inventer une publication ni exiger un export destiné à HF. Les deux modes refusent une galerie incomplète ou modifiée.

Les rapports sont dans Base/reports. Les poids, caches, journaux, environnements et secrets restent hors Git. Nettoyer les sorties des deux notebooks, inspecter le diff, puis créer un commit Angular et pousser sur origin/main.


In [ ]:
RUN_ARCHIVE_RESULTS = False
ARCHIVE_OUTCOME = "published"  # "rejected" pour conserver un candidat refusé
RUN_CLEAN_NOTEBOOK_OUTPUTS = False
run_stage("tools.archive_spatial_results", {"outcome": ARCHIVE_OUTCOME}, enabled=RUN_ARCHIVE_RESULTS)
run_stage("tools.clean_notebooks", enabled=RUN_CLEAN_NOTEBOOK_OUTPUTS)
archive = archived_report()
if archive: show_json(archive / "summary.json")


# Annexe — Code intégré au notebook

Chaque cellule suivante contient un module lisible. Son garde de nom évite un lancement accidentel lors de « Tout exécuter ». Le chargeur local lit ces mêmes cellules et crée des modules en mémoire à la demande ; aucune source .py ni autre notebook n'est écrit pour les calculs. Les cellules de parcours appellent directement leurs fonctions dans le kernel. L'export HF matérialise uniquement le code imposé par le format du modèle, dans le dossier d'artefact.

Les tests existants sont conservés dans cette annexe ; ils n'ont pas été exécutés pendant cette migration.

## Moteur local : chargement des cellules, chemins, verrous et empreintes

In [ ]:
if __name__ == "sharm_notebook_runtime":
    """Notebook source loader, typed parameters, paths and explicit local execution."""
    import contextlib
    import gc
    import hashlib
    import html
    import importlib.abc
    import importlib.util
    import json
    import os
    from pathlib import Path
    import subprocess
    import sys
    import types


    PROJECT = Path(__file__).resolve().parents[1]
    REPO = PROJECT.parent
    ARTIFACTS = PROJECT / "artifacts"


    def read_json(path, default=None):
        path = Path(path)
        return json.loads(path.read_text(encoding="utf-8-sig")) if path.exists() else default


    def read_jsonl(path):
        path = Path(path)
        return [json.loads(s) for s in path.read_text(encoding="utf-8-sig").splitlines() if s.strip()] if path.exists() else []


    def contained(root, name):
        root = Path(root).resolve()
        relative = Path(name)
        path = (root / relative).resolve()
        if relative.is_absolute() or not path.is_relative_to(root):
            raise ValueError("Chemin extérieur au dossier demandé.")
        return path


    def source_key(path):
        value = str(path).replace("\\", "/")
        if value in SOURCE_ALIASES:
            return SOURCE_ALIASES[value]
        candidate = Path(path)
        if candidate.is_absolute():
            try:
                value = candidate.relative_to(PROJECT).as_posix()
            except ValueError:
                return None
        return value if value in SOURCE_MODULES else None


    def notebook_source(path):
        key = source_key(path)
        if key is None:
            raise FileNotFoundError("Module absent du notebook : " + str(path))
        book = json.loads(Path(__file__).read_text(encoding="utf-8"))
        chunks = []
        import textwrap
        for cell in book["cells"]:
            if cell.get("metadata", {}).get("sharm_source") == key:
                lines = "".join(cell["source"]).splitlines(keepends=True)
                chunks.append(textwrap.dedent("".join(lines[1:])))
        return "\n\n".join(chunks) + "\n"


    def sha256(path):
        path = Path(path)
        if path.suffix == ".ipynb":
            return hashlib.sha256(notebook_source(path).encode("utf-8")).hexdigest()
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()


    class NotebookLoader(importlib.abc.Loader):
        def __init__(self, path):
            self.path = Path(path).resolve()

        def create_module(self, spec):
            return None

        def exec_module(self, module):
            module.__file__ = str(self.path)
            if self.path.stem == "__init__":
                module.__path__ = [str(self.path.parent)]
            # Only definition cells execute. Parameter and action cells are excluded.
            source = notebook_source(self.path)
            import __future__
            with local_imports(self.path.parent):
                exec(compile(source, str(self.path), "exec", flags=__future__.annotations.compiler_flag, dont_inherit=True), module.__dict__)


    class NotebookFinder(importlib.abc.MetaPathFinder):
        def find_spec(self, fullname, path=None, target=None):
            if globals().get("_SHIPPED_ROOT"):
                return None
            for key, data in SOURCE_MODULES.items():
                if data["name"] == fullname or (fullname in ("prepare_monster_dataset", "publish_spatial") and Path(key).stem == fullname):
                    return importlib.util.spec_from_loader(fullname, NotebookLoader(PROJECT / key), is_package=Path(key).stem == "__init__")
            return None


    @contextlib.contextmanager
    def local_imports(directory):
        value = str(Path(directory).resolve())
        original = list(sys.path)
        sys.path.insert(0, value)
        try:
            yield
        finally:
            sys.path[:] = original


    def load_notebook(path, name=None, fresh=False):
        key = source_key(path)
        if key is None:
            raise FileNotFoundError("Module absent du notebook : " + str(path))
        path = PROJECT / key
        name = name or "sharm_step_" + hashlib.sha256(str(path.resolve()).encode()).hexdigest()[:16]
        if name in sys.modules and not fresh:
            return sys.modules[name]
        spec = importlib.util.spec_from_loader(name, NotebookLoader(path), is_package=path.stem == "__init__")
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        try:
            spec.loader.exec_module(module)
        except BaseException:
            sys.modules.pop(name, None)
            raise
        return module


    def parameter_namespace(defaults, supplied=None, *, required=(), converters=None, choices=None, groups=()):
        supplied = {k.replace("-", "_"): v for k, v in dict(supplied or {}).items()}
        unknown = set(supplied) - set(defaults)
        if unknown:
            raise ValueError("Paramètres inconnus : " + ", ".join(sorted(unknown)))
        values = {**defaults, **supplied}
        for key, converter in (converters or {}).items():
            if key in supplied and values[key] is not None:
                value = values[key]
                values[key] = [converter(v) for v in value] if isinstance(value, list) else converter(value)
        for key in required:
            if values[key] is None:
                raise ValueError("Renseigner le paramètre " + key)
        for key, allowed in (choices or {}).items():
            if values[key] is not None and values[key] not in allowed:
                raise ValueError(f"Valeur de {key} hors des choix : {allowed}")
        for keys, mandatory in groups:
            count = sum(values.get(key) is not None and values.get(key) is not False for key in keys)
            if count > 1 or (mandatory and count != 1):
                raise ValueError("Choisir une source parmi : " + ", ".join(keys))
        return types.SimpleNamespace(**values)


    @contextlib.contextmanager
    def shipped_package(directory):
        global _SHIPPED_ROOT
        previous_root = globals().get("_SHIPPED_ROOT")
        _SHIPPED_ROOT = str(directory)
        saved = {k: v for k, v in sys.modules.items() if k == "sharm_image21" or k.startswith("sharm_image21.")}
        for name in saved:
            sys.modules.pop(name, None)
        try:
            with local_imports(directory):
                yield
        finally:
            for name in list(sys.modules):
                if name == "sharm_image21" or name.startswith("sharm_image21."):
                    sys.modules.pop(name, None)
            sys.modules.update(saved)
            _SHIPPED_ROOT = previous_root


    def require_free_gpu():
        if sys.platform == "win32":
            raise RuntimeError("Choisir le kernel Linux CUDA de la machine distante.")
        ps = subprocess.check_output(["ps", "-eo", "pid=,args="], text=True)
        for line in ps.splitlines():
            pieces = line.strip().split(None, 1)
            if len(pieces) == 2 and int(pieces[0]) != os.getpid() and any(s in pieces[1] for s in (
                    "/run_remote.py", "/03-spatial-distillation/run.py", "/03-spatial-distillation/train.py", "/train_lora.py")):
                raise RuntimeError("La chaîne distante actuelle est encore active ; attendre sa fin.")
        pids = subprocess.check_output(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader,nounits"], text=True)
        if any(s.strip().isdigit() and int(s.strip()) != os.getpid() for s in pids.splitlines()):
            raise RuntimeError("Un autre processus utilise le GPU ; attendre sa libération.")


    def execute_step(path, parameters=None, *, enabled=False, gpu=False):
        print("Notebook :", path)
        if not enabled:
            print("Prêt. Activer le drapeau de cette cellule pour exécuter dans ce kernel.")
            return None
        if gpu:
            require_free_gpu()
        ARTIFACTS.mkdir(parents=True, exist_ok=True)
        with (ARTIFACTS / "notebook-kernel.lock").open("a") as lock:
            if sys.platform != "win32":
                import fcntl
                fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
            values = dict(parameters or {})
            package = None
            if Path(path).stem == "verify_spatial_export":
                package = Path(values.get("release", ARTIFACTS / "spatial/release"))
            elif Path(path).stem == "verify_published":
                package = values.get("base")
            context = shipped_package(package) if package else contextlib.nullcontext()
            try:
                with context:
                    module = load_notebook(path, fresh=True)
                    return module.run(values)
            finally:
                gc.collect()
                if "torch" in sys.modules and sys.modules["torch"].cuda.is_available():
                    sys.modules["torch"].cuda.empty_cache()


    def call_with_flags(path, arguments=()):
        """Internal migration adapter for existing reproducible stage recipes."""
        module = load_notebook(Path(path).with_suffix(".ipynb"))
        schema = SOURCE_MODULES[source_key(module.__file__)]["parameters"]
        positional = [p["name"] for p in schema if p.get("positional")]
        boolean = {p["name"] for p in schema if p.get("boolean")}
        values = {}
        args = list(arguments)
        index = 0
        while index < len(args):
            item = str(args[index])
            if item.startswith("--"):
                key = item[2:].replace("-", "_")
                if key in boolean:
                    values[key] = True
                    index += 1
                    continue
                index += 1
                collected = []
                while index < len(args) and not str(args[index]).startswith("--"):
                    collected.append(args[index])
                    index += 1
                if not collected:
                    raise ValueError("Valeur manquante pour " + key)
                values[key] = collected if len(collected) > 1 else collected[0]
            else:
                if not positional:
                    raise ValueError("Paramètre positionnel inattendu.")
                values[positional.pop(0)] = args[index]
                index += 1
        try:
            return module.run(values)
        finally:
            gc.collect()
            if "torch" in sys.modules and sys.modules["torch"].cuda.is_available():
                sys.modules["torch"].cuda.empty_cache()


    def export_runtime(destination):
        """Write the package required by the HF model format into the export artifact."""
        destination = Path(destination)
        destination.mkdir(parents=True, exist_ok=True)
        for key in SOURCE_MODULES:
            if key.startswith("common/sharm_image21/"):
                source = notebook_source(key)
                (destination / (Path(key).stem + ".py")).write_text("from __future__ import annotations\n" + source, encoding="utf-8", newline="\n")


    def current_dataset():
        root = PROJECT / "Sharm-Image-2.1-VL4B-LoRA-Monster/dataset"
        pointer = read_json(root / "current.json")
        if not pointer:
            raise FileNotFoundError("Sélectionner une version du dataset Monster.")
        return contained(root, pointer["dataset"]), pointer


    def archived_report():
        pointer = read_json(PROJECT / "Sharm-Image-2.1-VL4B/reports/current.json")
        if not pointer:
            return None
        root = contained(PROJECT / "Sharm-Image-2.1-VL4B/reports", pointer["report"])
        if sha256(root / "manifest.json") != pointer["manifest_sha256"]:
            raise ValueError("Manifeste d'archive modifié.")
        return root


    def result_file(name):
        archive = archived_report()
        if archive and (archive / name).is_file():
            path = contained(archive, name)
            if sha256(path) != read_json(archive / "manifest.json")["files"].get(name):
                raise ValueError("Rapport archivé modifié.")
            return path
        return ARTIFACTS / "spatial" / name


    def show_json(path):
        value = read_json(path)
        print(json.dumps(value, ensure_ascii=False, indent=2) if value is not None else "Rapport non disponible ; aucune mesure supposée.")
        return value


    def table(rows, columns):
        from IPython.display import HTML, display
        head = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
        body = "".join("<tr>" + "".join(f"<td>{html.escape(str(row.get(c, '')))}</td>" for c in columns) + "</tr>" for row in rows)
        display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))


    def show_image(path, label=""):
        from PIL import Image, ImageDraw
        from IPython.display import display
        with Image.open(path) as image:
            rgba = image.convert("RGBA")
            background = Image.new("RGBA", rgba.size, "#eeeeee")
            draw = ImageDraw.Draw(background)
            for y in range(0, rgba.height, 8):
                for x in range(0, rgba.width, 8):
                    if (x // 8 + y // 8) % 2:
                        draw.rectangle((x, y, x + 7, y + 7), fill="#cccccc")
            background.alpha_composite(rgba)
            print(label or Path(path).name, image.mode, image.size)
            display(background.convert("RGB"))


    sys.meta_path[:] = [finder for finder in sys.meta_path if not getattr(finder, "sharm_notebooks", False)]
    finder = NotebookFinder()
    finder.sharm_notebooks = True
    sys.meta_path.insert(0, finder)
    runtime = types.ModuleType("sharm_notebook_runtime")
    runtime.__dict__.update(globals())
    sys.modules["sharm_notebook_runtime"] = runtime
    for directory in (PROJECT / "common", PROJECT / "notebooks"):
        if str(directory) not in sys.path:
            sys.path.insert(0, str(directory))


    def notebook_completed(command, capture_output=False, text=False, env=None, check=False, **kwargs):
        import io
        out, err = io.StringIO(), io.StringIO()
        status = 0
        previous = dict(os.environ)
        try:
            if env is not None:
                os.environ.clear(); os.environ.update(env)
            with contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):
                try:
                    call_with_flags(command[1], command[2:])
                except SystemExit as error:
                    status = error.code if isinstance(error.code, int) else 1
                    if error.code: print(error, file=sys.stderr)
                except Exception as error:
                    status = 1
                    print(type(error).__name__ + ": " + str(error), file=sys.stderr)
        finally:
            os.environ.clear(); os.environ.update(previous)
        result = subprocess.CompletedProcess(command, status, out.getvalue(), err.getvalue())
        if check and status:
            raise subprocess.CalledProcessError(status, command, result.stdout, result.stderr)
        return result


    def run_stage(name, parameters=None, *, enabled=False, gpu=False):
        return execute_step(SOURCE_ALIASES.get(name, name), parameters, enabled=enabled, gpu=gpu)


    runtime.__dict__.update(globals())


## tools.patch_trainer

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_31e3eceecaf7af78':
    """Patch Diffusers' Qwen DreamBooth dataset for JSONL captions and lazy image loading."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    from pathlib import Path


    ORIGINAL = '''        else:
                self.instance_data_root = Path(instance_data_root)
                if not self.instance_data_root.exists():
                    raise ValueError("Instance images root doesn't exists.")

                instance_images = [Image.open(path) for path in list(Path(instance_data_root).iterdir())]
                self.custom_instance_prompts = None

            self.instance_images = []
    '''


    PATCHED = '''        else:
                self.instance_data_root = Path(instance_data_root)
                if not self.instance_data_root.exists():
                    raise ValueError("Instance images root doesn't exists.")

                metadata_path = self.instance_data_root / "metadata.jsonl"
                if metadata_path.is_file():
                    metadata = [json.loads(line) for line in metadata_path.read_text(encoding="utf-8").splitlines() if line]
                    image_paths = [self.instance_data_root / entry["file_name"] for entry in metadata]
                    if not image_paths:
                        raise ValueError(f"No image captions found in {metadata_path}.")
                    missing_images = [str(path) for path in image_paths if not path.is_file()]
                    if missing_images:
                        raise ValueError(f"Images referenced by {metadata_path} are missing: {missing_images[:5]}")
                    if any(not entry.get("text") for entry in metadata):
                        raise ValueError(f"Every metadata.jsonl row must contain a non-empty 'text' caption: {metadata_path}")
                    instance_images = image_paths
                    self.custom_instance_prompts = []
                    for entry in metadata:
                        self.custom_instance_prompts.extend(itertools.repeat(entry["text"], repeats))
                else:
                    image_paths = sorted(
                        path for path in self.instance_data_root.iterdir()
                        if path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
                    )
                    instance_images = image_paths
                    self.custom_instance_prompts = None

            self.instance_images = []
    '''


    PATCHED_PREVIOUS = PATCHED.replace(
        "instance_images = image_paths",
        "instance_images = [Image.open(path) for path in image_paths]",
    )


    PIXEL_VALUES_ORIGINAL = '''        self.pixel_values = []
            self.buckets = []
            bucket_to_idx = {}
            for image in self.instance_images:
                image = exif_transpose(image)
                # RGBA, not RGB: this VAE takes and returns four channels.
                if not image.mode == "RGBA":
                    image = image.convert("RGBA")

                width, height = image.size

                # Assign the image to a bucket.
                target = self._bucket_for_image(height, width)
                if target not in bucket_to_idx:
                    bucket_to_idx[target] = len(self.buckets)
                    self.buckets.append(target)
                bucket_idx = bucket_to_idx[target]

                # based on the bucket assignment, define the transformations
                image = self.train_transform(
                    image,
                    size=target,
                    center_crop=args.center_crop,
                    random_flip=args.random_flip,
                )
                self.pixel_values.append((image, bucket_idx))
    '''


    PIXEL_VALUES_PATCHED = '''        self.pixel_values = []
            self.buckets = []
            bucket_to_idx = {}
            self.random_flip = args.random_flip
            for image in self.instance_images:
                if isinstance(image, (str, Path)):
                    image = Path(image)
                    with Image.open(image) as source:
                        width, height = exif_transpose(source).size
                else:
                    image = exif_transpose(image)
                    width, height = image.size

                # Assign the image to a bucket without decoding every image into a training tensor.
                target = self._bucket_for_image(height, width)
                if target not in bucket_to_idx:
                    bucket_to_idx[target] = len(self.buckets)
                    self.buckets.append(target)
                bucket_idx = bucket_to_idx[target]
                self.pixel_values.append((image, bucket_idx))
    '''


    GETITEM_ORIGINAL = '''        instance_image, bucket_idx = self.pixel_values[index % self.num_instance_images]
            example["index"] = index
            example["instance_images"] = instance_image
    '''


    GETITEM_PATCHED = '''        image_source, bucket_idx = self.pixel_values[index % self.num_instance_images]
            if isinstance(image_source, (str, Path)):
                with Image.open(image_source) as source:
                    image = exif_transpose(source)
                    if image.mode != "RGBA":
                        image = image.convert("RGBA")
                    instance_image = self.train_transform(
                        image,
                        size=self.buckets[bucket_idx],
                        center_crop=self.center_crop,
                        random_flip=self.random_flip,
                    )
            else:
                image = exif_transpose(image_source)
                if image.mode != "RGBA":
                    image = image.convert("RGBA")
                instance_image = self.train_transform(
                    image,
                    size=self.buckets[bucket_idx],
                    center_crop=self.center_crop,
                    random_flip=self.random_flip,
                )
            example["index"] = index
            example["instance_images"] = instance_image
    '''


    CACHE_SAMPLER_ORIGINAL = '''        cache_batch_sampler = BucketBatchSampler(
                train_dataset, batch_size=args.train_batch_size, drop_last=False, seed=args.seed
            )
    '''


    CACHE_SAMPLER_PATCHED = '''        cache_batch_size = max(args.train_batch_size, 2)
            cache_batch_sampler = BucketBatchSampler(
                train_dataset, batch_size=cache_batch_size, drop_last=False, seed=args.seed
            )
    '''


    def run(parameters=None) -> None:
        args = parameter_namespace({'trainer': None}, parameters, required=['trainer'], converters={'trainer': Path}, choices={}, groups=[])
        source = trainer_path.read_text(encoding="utf-8")
        changes: list[str] = []
        if PATCHED not in source:
            if source.count(PATCHED_PREVIOUS) == 1:
                source = source.replace(PATCHED_PREVIOUS, PATCHED, 1)
                changes.append("chargement paresseux des fichiers dataset")
            elif source.count(ORIGINAL) == 1:
                source = source.replace(ORIGINAL, PATCHED, 1)
                changes.append("légendes metadata.jsonl")
            else:
                raise SystemExit(
                    "La version Diffusers ne correspond pas au bloc de légendes attendu ; aucun changement appliqué. "
                    "Vérifie le trainer avant de mettre à jour patch_trainer.py."
                )

        if PIXEL_VALUES_PATCHED not in source:
            if source.count(PIXEL_VALUES_ORIGINAL) != 1:
                raise SystemExit(
                    "La version Diffusers ne correspond pas au bloc de chargement des images attendu ; aucun changement appliqué. "
                    "Vérifie le trainer avant de mettre à jour patch_trainer.py."
                )
            source = source.replace(PIXEL_VALUES_ORIGINAL, PIXEL_VALUES_PATCHED, 1)
            changes.append("chargement des images à la demande")

        if GETITEM_PATCHED not in source:
            if source.count(GETITEM_ORIGINAL) != 1:
                raise SystemExit(
                    "La version Diffusers ne correspond pas au bloc __getitem__ attendu ; aucun changement appliqué. "
                    "Vérifie le trainer avant de mettre à jour patch_trainer.py."
                )
            source = source.replace(GETITEM_ORIGINAL, GETITEM_PATCHED, 1)
            changes.append("transformation par lot")

        if CACHE_SAMPLER_PATCHED not in source:
            if source.count(CACHE_SAMPLER_ORIGINAL) != 1:
                raise SystemExit(
                    "La version Diffusers ne correspond pas au bloc de pré-calcul attendu ; aucun changement appliqué. "
                    "Vérifie le trainer avant de mettre à jour patch_trainer.py."
                )
            source = source.replace(CACHE_SAMPLER_ORIGINAL, CACHE_SAMPLER_PATCHED, 1)
            changes.append("pré-calcul par lots de deux")

        if changes:
            trainer_path.write_text(source, encoding="utf-8")
            print(f"Trainer configuré pour {', '.join(changes)} : {trainer_path}")
        else:
            print(f"Trainer déjà configuré pour les légendes et le chargement à la demande : {trainer_path}")



## tools.verify_training

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_d43c9de1937951d5':
    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    """Small real-GPU training checks before dispatching the long jobs."""


    from pathlib import Path


    import argparse


    import gc


    import json


    import sys


    import tempfile


    sys.path.insert(0, str(Path(__file__).resolve().parent / "common"))


    import torch


    from sharm_image21.core import ARTIFACTS, FeatureCache, Projector, atomic_json, seed_all


    def verify_projector():
        seed_all()
        with tempfile.TemporaryDirectory() as folder:
            folder = Path(folder)
            model = Projector(8, 16, 4).cuda()
            source = torch.randn(256, 8, device="cuda")
            target = source @ torch.randn(8, 16, device="cuda")
            optimizer = torch.optim.AdamW(model.parameters(), lr=0.03)
            initial = torch.nn.functional.mse_loss(model(source), target).item()
            for _ in range(150):
                optimizer.zero_grad()
                loss = torch.nn.functional.mse_loss(model(source), target)
                loss.backward()
                optimizer.step()
            final = torch.nn.functional.mse_loss(model(source), target).item()
            if final > initial*0.02:
                raise RuntimeError(f"Le projecteur n'apprend pas la correspondance synthétique : {initial} -> {final}")
            model.save(folder / "projector")
            reloaded = Projector.load(folder / "projector", "cuda")
            torch.testing.assert_close(model(source), reloaded(source), rtol=0, atol=0)
            model.save(folder / "projector-bf16", storage_dtype=torch.bfloat16)
            reloaded_bf16 = Projector.load(folder / "projector-bf16", "cuda")
            reference_bf16 = model.to(torch.bfloat16)
            torch.testing.assert_close(reference_bf16(source.to(torch.bfloat16)),
                                       reloaded_bf16(source.to(torch.bfloat16)), rtol=0, atol=0)
            cache = FeatureCache(folder / "cache", {"model": "one"})
            cache.write([("sample", source[:4].cpu(), [1, 2, 3, 4])])
            torch.testing.assert_close(cache.get("sample"), source[:4].cpu())
            try:
                FeatureCache(folder / "cache", {"model": "two"})
            except ValueError:
                pass
            else:
                raise RuntimeError("Le cache accepte une identité incompatible.")
            print(json.dumps({"projector_smoke": True, "initial_loss": initial, "final_loss": final}), flush=True)


    def verify_diffusion_training():
        from sharm_image21.diffusion import load_transformer, predict_flow
        from peft import LoraConfig
        model = load_transformer(training=True)
        model.add_adapter(LoraConfig(r=8, lora_alpha=8, init_lora_weights="gaussian",
                                     target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
        model.train()
        params = [param for param in model.parameters() if param.requires_grad]
        import bitsandbytes as bnb
        optimizer = bnb.optim.AdamW8bit(params, lr=1e-4)
        seed_all()
        clean = torch.randn(1, 64, 1, 32, 32, device="cuda", dtype=torch.bfloat16)
        embeddings = torch.randn(1, 32, 4096, device="cuda", dtype=torch.bfloat16)
        losses = []
        torch.cuda.reset_peak_memory_stats()
        for _ in range(2):
            optimizer.zero_grad(set_to_none=True)
            noise = torch.randn_like(clean)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                prediction = predict_flow(model, (noise+clean)/2, embeddings,
                                          torch.tensor([0.5], device="cuda", dtype=torch.bfloat16))
            if prediction.shape != clean.shape:
                raise RuntimeError("Forme de prédiction incorrecte.")
            loss = torch.nn.functional.mse_loss(prediction.float(), (noise-clean).float())
            if not torch.isfinite(loss):
                raise RuntimeError("Perte du diffuseur non finie.")
            loss.backward()
            norm = torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=True)
            if norm == 0:
                raise RuntimeError("Aucun gradient LoRA.")
            optimizer.step()
            losses.append(loss.item())
        report = {"passed": True, "lora_backward": True, "losses": losses,
                  "trainable_parameters": sum(param.numel() for param in params),
                  "gpu_peak_gib": torch.cuda.max_memory_allocated()/2**30}
        with tempfile.TemporaryDirectory() as folder:
            from diffusers import QwenImage21Pipeline
            from diffusers.training_utils import _collate_lora_metadata
            from peft.utils import get_peft_model_state_dict
            from peft import set_peft_model_state_dict
            folder = Path(folder)
            QwenImage21Pipeline.save_lora_weights(folder, transformer_lora_layers=get_peft_model_state_dict(model),
                                                  **_collate_lora_metadata({"transformer": model}))
            if not (folder / "pytorch_lora_weights.safetensors").exists():
                raise RuntimeError("Adaptateur non sérialisé.")
            torch.save({"adapter": get_peft_model_state_dict(model), "optimizer": optimizer.state_dict()}, folder / "state.pt")
            state = torch.load(folder / "state.pt", weights_only=True, map_location="cpu")
            set_peft_model_state_dict(model, state["adapter"], adapter_name="default")
            optimizer.load_state_dict(state["optimizer"])
            report["checkpoint_roundtrip"] = True
        atomic_json(ARTIFACTS / "diffusion-training-audit.json", report)
        print(json.dumps(report, indent=2), flush=True)


    def run(parameters=None):
        args = parameter_namespace({'diffusion': False}, parameters, required=[], converters={}, choices={}, groups=[])
        verify_projector()
        gc.collect()
        torch.cuda.empty_cache()
        if args.diffusion:
            verify_diffusion_training()



## encoder.audit_runtime

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_8bd6443b4aafe641':
    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    from pathlib import Path


    import json


    import os


    import sys


    import time


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    import torch


    from sharm_image21.core import ARTIFACTS, ORIGINAL, STUDENT, FrozenEncoder, atomic_json, digest


    @torch.inference_mode()
    def verify_fp8_kernel():
        """Compile/execute a small real FP8 operation before loading the 4B."""
        from transformers.integrations.finegrained_fp8 import FP8Linear
        os.environ.setdefault("TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR", "1")
        torch.manual_seed(42)
        layer = FP8Linear(256, 256, block_size=(128, 128)).cuda().eval().requires_grad_(False)
        layer.weight.copy_((torch.randn(256, 256, device="cuda")/16).to(torch.float8_e4m3fn))
        layer.weight_scale_inv.copy_(torch.rand(2, 2, device="cuda")*0.02+0.005)
        source = torch.randn(17, 256, device="cuda", dtype=torch.bfloat16)
        result = layer(source)
        scale = layer.weight_scale_inv.repeat_interleave(128, 0).repeat_interleave(128, 1)
        reference = torch.nn.functional.linear(source.float(), layer.weight.float()*scale)
        relative_mse = ((result.float()-reference).square().mean()/reference.square().mean()).item()
        if not torch.isfinite(result).all() or not relative_mse < 0.02:
            raise RuntimeError(f"Calcul FP8 du kernel incohérent : MSE relative synthétique {relative_mse}")
        report = {"passed": True, "synthetic_relative_mse": relative_mse, "shape": [17, 256, 256],
                  "scope": "FP8 arithmetic versus dequantized reference; not 4B versus 8B quality"}
        print(json.dumps({"kernel_smoke": report}), flush=True)
        return report


    def run(parameters=None):
        args = parameter_namespace({}, parameters, required=[], converters={}, choices={}, groups=[])
        teacher = json.loads((ORIGINAL / "processor/tokenizer.json").read_text())
        student = json.loads((STUDENT / "tokenizer.json").read_text())
        teacher_vocab = teacher["model"]["vocab"]
        student_vocab = student["model"]["vocab"]
        def normalized_merges(config):
            return [tuple(pair.split(" ", 1)) if isinstance(pair, str) else tuple(pair)
                    for pair in config["model"].get("merges", [])]
        if (teacher_vocab != student_vocab or normalized_merges(teacher) != normalized_merges(student)
                or teacher["added_tokens"] != student["added_tokens"]):
            raise RuntimeError("Tokenizers non alignés : arrêter avant la distillation token par token.")
        kernel_check = verify_fp8_kernel()
        encoder = FrozenEncoder("student")
        prompts = ["A red cube to the left of a blue sphere on a white table.",
                   "This is an RGBA image with transparency. A small green dragon pixel-art sprite. The background is transparent."]
        torch.cuda.reset_peak_memory_stats()
        started = time.monotonic()
        outputs = encoder.encode(prompts)
        single = encoder.encode(prompts[:1])[0][0]
        batch_delta = (outputs[0][0].float()-single.float()).norm()/single.float().norm()
        if batch_delta > 0.05:
            raise RuntimeError(f"Incohérence entre encodage seul et batch : {batch_delta.item():.4f}")
        report = {"passed": True, "encoder_identity": encoder.identity,
                  "kernel_smoke": kernel_check,
                  "vocab_sha256": digest(teacher_vocab), "fp8_linears": encoder.quantized_linears,
                  "feature_shapes": [list(value.shape) for value, _ in outputs],
                  "batch_relative_delta": batch_delta.item(), "seconds": time.monotonic()-started,
                  "fp8_prefill_policy": "single-prompt-fp8",
                  "gpu_peak_gib": torch.cuda.max_memory_allocated()/2**30,
                  "capability": list(torch.cuda.get_device_capability()),
                  "scope": "text-to-image; image editing not yet validated"}
        atomic_json(ARTIFACTS / "runtime-audit.json", report)
        print(json.dumps(report, indent=2), flush=True)



## encoder.cache_features

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_bd3a6fdd31cba443':


    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import fcntl


    import sys


    import time


    from pathlib import Path


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    from sharm_image21.core import (ARTIFACTS, CACHE, FeatureCache, FrozenEncoder,
                                    atomic_json, digest, model_identity, read_jsonl)


    def run(parameters=None):
        args = parameter_namespace({'kind': None, 'corpus': ARTIFACTS / 'corpus.jsonl', 'batch_size': 4, 'gpu_memory': 'auto', 'limit': None, 'cache_root': CACHE}, parameters, required=['kind'], converters={'corpus': Path, 'batch_size': int, 'limit': int, 'cache_root': Path}, choices={'kind': ['teacher', 'student']}, groups=[])
        lock_dir = args.cache_root / args.kind
        lock_dir.mkdir(parents=True, exist_ok=True)
        writer_lock = (lock_dir / ".writer.lock").open("a")
        fcntl.flock(writer_lock.fileno(), fcntl.LOCK_EX)
        corpus = read_jsonl(args.corpus)
        rows = corpus[:args.limit] if args.limit else corpus
        identity = {"encoder": model_identity(args.kind), "corpus": digest(corpus)}
        cache = FeatureCache(lock_dir, identity)
        pending = [row for row in rows if row["id"] not in cache]
        if not pending:
            print(f"Cache {args.kind} complet pour {len(rows)} exemples.", flush=True)
            return
        encoder = FrozenEncoder(args.kind, args.gpu_memory)
        pending.sort(key=lambda row: row.get("tokens_unwrapped", len(row["text"])))
        started, buffered, completed = time.monotonic(), [], len(rows)-len(pending)
        for start in range(0, len(pending), args.batch_size):
            batch = pending[start:start+args.batch_size]
            output = encoder.encode([row["text"] for row in batch])
            buffered.extend((row["id"], values, token_ids) for row, (values, token_ids) in zip(batch, output))
            completed += len(batch)
            if len(buffered) >= 32 or start+args.batch_size >= len(pending):
                cache.write(buffered)
                buffered = []
                elapsed = time.monotonic()-started
                report = {"stage": f"cache_{args.kind}", "completed": completed, "total": len(rows),
                          "seconds": elapsed, "updated_at": time.time(),
                          "examples_per_second": (start+len(batch))/max(elapsed, 1e-6)}
                status_root = ARTIFACTS if args.cache_root.resolve() == CACHE.resolve() else args.cache_root
                atomic_json(status_root / f"cache-{args.kind}-status.json", report)
                print(f"{args.kind}: {completed}/{len(rows)} — {elapsed:.1f}s", flush=True)
        cache.require(rows)



## encoder.prepare_corpus

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_78a611dd69adfa54':


    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import json


    import random


    import re


    import sys


    from pathlib import Path


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    from sharm_image21.core import ARTIFACTS, CACHE, ORIGINAL, ROOT, atomic_json, digest, read_jsonl


    DATASET_REVISION = "fb620fbe49fa4420e0734bd9c0df11f51176b61f"


    def synthetic_prompts(count, seed):
        rng = random.Random(seed)
        objects = ["dragon", "robot", "cat", "castle", "tree", "bicycle", "glass bottle", "owl",
                   "fox", "wooden chair", "train", "sailing boat", "mushroom", "flower", "book"]
        colors = ["red", "blue", "green", "yellow", "purple", "orange", "white", "black", "silver"]
        styles = ["pixel art", "watercolor illustration", "studio photograph", "pencil drawing",
                  "clay sculpture", "flat vector illustration", "oil painting", "game sprite"]
        texts = set()
        while len(texts) < count:
            obj, other = rng.sample(objects, 2)
            color, other_color = rng.sample(colors, 2)
            style = rng.choice(styles)
            n = rng.choice(["two", "three", "four"])
            templates = [
                f"A {color} {obj} to the left of a {other_color} {other}, {style}, clear composition.",
                f"Exactly {n} objects: {color} {obj}s arranged in a row on a white table, {style}.",
                f"This is an RGBA image with transparency. A single {color} {obj}, {style}. The image has alpha channel and the background is transparent.",
                f'A {color} sign above a {other}, with the exact text "{rng.choice(["HELLO", "SHARM", "OPEN", "BONJOUR", "2026"])}", legible typography, {style}.',
                f"Une illustration de {obj}, {style}, au premier plan devant {other}, lumière douce, composition nette.",
                f"A close view of a {color} {obj} beside a {other_color} {other}, warm sunset light, detailed texture, {style}.",
            ]
            texts.add(rng.choice(templates))
        return sorted(texts)


    def run(parameters=None):
        args = parameter_namespace({'limit': 20000, 'seed': 42, 'max_tokens': 192, 'output': ARTIFACTS / 'corpus.jsonl'}, parameters, required=[], converters={'limit': int, 'seed': int, 'max_tokens': int, 'output': Path}, choices={}, groups=[])
        if args.output.exists():
            print(f"Corpus déjà préparé : {args.output}", flush=True)
            return
        from huggingface_hub import hf_hub_download
        import pyarrow.parquet as pq
        from transformers import AutoTokenizer

        source = Path(hf_hub_download("poloclub/diffusiondb", "metadata.parquet", repo_type="dataset",
                                      revision=DATASET_REVISION, local_dir=CACHE / "sources"))
        tokenizer = AutoTokenizer.from_pretrained(ORIGINAL / "processor", local_files_only=True)
        rng = random.Random(args.seed)
        sample, seen, accepted = [], set(), 0
        capacity = args.limit * 2
        for batch in pq.ParquetFile(source).iter_batches(batch_size=8192, columns=["prompt", "prompt_nsfw"]):
            values = batch.to_pydict()
            for text, score in zip(values["prompt"], values["prompt_nsfw"]):
                text = re.sub(r"\s+", " ", text or "").strip()
                key = text.casefold()
                if not 35 <= len(text) <= 1000 or (score is not None and score > 0.3) or key in seen:
                    continue
                seen.add(key)
                accepted += 1
                if len(sample) < capacity:
                    sample.append(text)
                else:
                    index = rng.randrange(accepted)
                    if index < capacity:
                        sample[index] = text
        rng.shuffle(sample)
        local = synthetic_prompts(min(1500, args.limit // 10), args.seed)
        sprite_metadata = ROOT / "dataset/train_512/metadata.jsonl"
        sprites = sorted({row["text"] for row in read_jsonl(sprite_metadata)}) if sprite_metadata.exists() else []
        rows, used = [], set()
        candidates = [(text, "sharm-authored-templates") for text in local]
        candidates += [(text, "local-sprite-captions") for text in sprites]
        candidates += [(text, "poloclub/diffusiondb") for text in sample]
        for start in range(0, len(candidates), 256):
            selected = candidates[start:start+256]
            ids = tokenizer([text for text, _ in selected], add_special_tokens=False)["input_ids"]
            for (text, source_name), tokens in zip(selected, ids):
                if not 8 <= len(tokens) <= args.max_tokens or text.casefold() in used:
                    continue
                used.add(text.casefold())
                sample_id = digest(text)
                # Group close caption prefixes together, instead of splitting near-identical variants.
                group = re.sub(r"\W+", " ", text.casefold()).strip()[:96]
                split = "validation" if int(digest(group)[:8], 16) % 10 == 0 else "train"
                rows.append({"id": sample_id, "text": text, "split": split, "source": source_name,
                             "tokens_unwrapped": len(tokens)})
                if len(rows) == args.limit:
                    break
            if len(rows) == args.limit:
                break
        if len(rows) != args.limit:
            raise RuntimeError(f"Corpus insuffisant : {len(rows)} / {args.limit}")
        args.output.parent.mkdir(parents=True, exist_ok=True)
        args.output.write_text("".join(json.dumps(row, ensure_ascii=False)+"\n" for row in rows), encoding="utf-8")
        report = {"count": len(rows), "sha256": digest(rows), "seed": args.seed,
                  "max_tokens": args.max_tokens, "dataset_revision": DATASET_REVISION,
                  "dataset_license": "CC0-1.0", "source_url": "https://huggingface.co/datasets/poloclub/diffusiondb",
                  "splits": {split: sum(row["split"] == split for row in rows) for split in ("train", "validation")},
                  "sources": {name: sum(row["source"] == name for row in rows) for name in sorted({r["source"] for r in rows})}}
        atomic_json(args.output.with_suffix(".manifest.json"), report)
        print(json.dumps(report, indent=2), flush=True)



## encoder.prepare_models

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_b346df92f5d243d2':
    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    from pathlib import Path


    import sys


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    from huggingface_hub import snapshot_download


    from sharm_image21.core import ORIGINAL, STUDENT, STUDENT_REVISION, TEACHER_REVISION


    def run(parameters=None):
        args = parameter_namespace({'stage': 'all'}, parameters, required=[], converters={}, choices={'stage': ('all', 'teacher', 'inference')}, groups=[])
        original_patterns = ["model_index.json", "processor/*", "LICENSE", "README.md",
                             "text_encoder/*.json", "transformer/*.json", "vae/*.json", "scheduler/*"]
        if args.stage in ("all", "teacher"):
            original_patterns.append("text_encoder/*")
        if args.stage in ("all", "inference"):
            original_patterns.extend(["transformer/*", "vae/*"])
        snapshot_download("Qwen/Qwen-Image-2.1", revision=TEACHER_REVISION, local_dir=ORIGINAL, max_workers=2,
                           allow_patterns=original_patterns)
        student_patterns = ["*.json", "*.txt", "*.model", "*.jinja"]
        if args.stage in ("all", "inference"):
            student_patterns.append("*.safetensors")
        snapshot_download("Qwen/Qwen3-VL-4B-Instruct-FP8", revision=STUDENT_REVISION, local_dir=STUDENT, max_workers=2,
                           allow_patterns=student_patterns)
        print("Poids source vérifiés aux révisions figées.", flush=True)



## encoder.train_projector

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_fca12e9af7fac690':


    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import json


    import random


    import sys


    import time


    from pathlib import Path


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    import torch


    from torch.nn import functional as F


    from sharm_image21.core import (ARTIFACTS, CACHE, FeatureCache, Projector, alignment_metrics,
                                    atomic_json, digest, read_jsonl, seed_all)


    def paired_features(row, student, teacher):
        key = row["id"]
        if student.index["entries"][key]["token_ids"] != teacher.index["entries"][key]["token_ids"]:
            raise ValueError(f"Tokens désalignés pour {key}")
        source, target = student.get(key), teacher.get(key)
        if source.shape[0] != target.shape[0]:
            raise ValueError(f"Longueurs désalignées pour {key}")
        return source, target


    @torch.no_grad()
    def evaluate(model, rows, student, teacher):
        model.eval()
        scores = []
        for row in rows:
            source, target = paired_features(row, student, teacher)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                prediction = model(source.cuda())
            scores.append(alignment_metrics(prediction, target.cuda()))
        return {name: sum(score[name] for score in scores)/len(scores) for name in scores[0]}


    def train_batch(rows, student, teacher, token_limit):
        inputs, targets = [], []
        for row in rows:
            source, target = paired_features(row, student, teacher)
            if len(source) > token_limit:
                selected = torch.randperm(len(source))[:token_limit]
                source, target = source[selected], target[selected]
            inputs.append(source)
            targets.append(target)
        return torch.cat(inputs).cuda(), torch.cat(targets).cuda()


    def run(parameters=None):
        args = parameter_namespace({'corpus': ARTIFACTS / 'corpus.jsonl', 'linear_steps': 1000, 'mlp_steps': 5000, 'batch_size': 16, 'tokens_per_caption': 96, 'eval_every': 250, 'seed': 42, 'limit': None, 'output_root': ARTIFACTS}, parameters, required=[], converters={'corpus': Path, 'linear_steps': int, 'mlp_steps': int, 'batch_size': int, 'tokens_per_caption': int, 'eval_every': int, 'seed': int, 'limit': int, 'output_root': Path}, choices={}, groups=[])
        if min(args.linear_steps, args.mlp_steps, args.batch_size, args.tokens_per_caption, args.eval_every) < 1:
            raise ValueError("Les tailles de lots et durées d'entraînement doivent être positives.")
        seed_all(args.seed)
        torch.set_num_threads(4)
        corpus = read_jsonl(args.corpus)
        rows = corpus[:args.limit] if args.limit else corpus
        args.output_root.mkdir(parents=True, exist_ok=True)
        student, teacher = FeatureCache(CACHE / "student"), FeatureCache(CACHE / "teacher")
        student.require(rows)
        teacher.require(rows)
        for cache in (student, teacher):
            if cache.index["identity"]["corpus"] != digest(corpus):
                raise ValueError("Le cache ne correspond pas au corpus d'entraînement.")
        train = [row for row in rows if row["split"] == "train"]
        valid = [row for row in rows if row["split"] == "validation"]
        if not train or not valid:
            raise ValueError("Ensembles entraînement et validation requis.")
        random.Random(args.seed).shuffle(valid)
        monitor = valid[:256]
        metadata = {"teacher": teacher.index["identity"], "student": student.index["identity"],
                    "corpus_sha256": digest(corpus), "training_subset_sha256": digest(rows),
                    "scope": "experimental text-to-image alignment; editing not validated"}
        destination = args.output_root / "projector"
        linear_destination = args.output_root / "projector-linear"
        if (destination / "training-complete.json").is_file():
            previous = json.loads((destination / "training-complete.json").read_text())
            if (previous["corpus_sha256"] != metadata["corpus_sha256"]
                    or previous.get("training_subset_sha256") != metadata["training_subset_sha256"]):
                raise ValueError("Le projecteur existant correspond à un autre corpus.")
            print("Entraînement du projecteur déjà terminé.", flush=True)
            return
        model = Projector().cuda()
        checkpoint = args.output_root / "projector-checkpoint.pt"
        state = torch.load(checkpoint, weights_only=True, map_location="cpu") if checkpoint.exists() else None
        if state and state["metadata"] != metadata:
            raise ValueError("Checkpoint projecteur incompatible.")
        if state:
            model.load_state_dict(state["model"])
        global_best = state["global_best"] if state else float("inf")
        history = args.output_root / "projector-history.jsonl"
        started = time.monotonic()
        for phase, steps in (("linear", args.linear_steps), ("mlp", args.mlp_steps)):
            if state and state["phase"] == "mlp" and phase == "linear":
                continue
            if phase == "mlp" and (linear_destination / "model.safetensors").exists() and not (state and state["phase"] == "mlp"):
                model = Projector.load(linear_destination, "cuda")
            for name, param in model.named_parameters():
                param.requires_grad_(phase == "mlp" or name.startswith("skip."))
            optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.01)
            start_step = 0
            if state and state["phase"] == phase:
                optimizer.load_state_dict(state["optimizer"])
                start_step = state["step"]
                torch.set_rng_state(state["torch_rng"])
                random.setstate(state["python_rng"])
            phase_best = state.get("phase_best", float("inf")) if state and state["phase"] == phase else float("inf")
            for step in range(start_step, steps):
                model.train()
                source, target = train_batch(random.sample(train, min(args.batch_size, len(train))), student, teacher,
                                             args.tokens_per_caption)
                lr_scale = min(1.0, (step+1)/max(1, int(steps*0.03)))
                optimizer.param_groups[0]["lr"] = 3e-4*lr_scale
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    predicted = model(source)
                pred, gold = predicted.float(), target.float()
                relative_mse = (pred-gold).square().mean()/gold.square().mean().clamp_min(1e-8)
                cosine_loss = 1-F.cosine_similarity(pred, gold, dim=-1).mean()
                loss = relative_mse + 0.1*cosine_loss
                if not torch.isfinite(loss):
                    raise RuntimeError("Perte projecteur non finie ; arrêt avant le LoRA.")
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
                optimizer.step()
                if (step+1) % 25 == 0:
                    print(f"projector {phase} {step+1}/{steps}: loss={loss.item():.5f}", flush=True)
                if (step+1) % args.eval_every == 0 or step+1 == steps:
                    metrics = evaluate(model, monitor, student, teacher)
                    report = {"stage": "train_projector", "phase": phase, "step": step+1, "steps": steps,
                              "metrics": metrics, "seconds": time.monotonic()-started, "updated_at": time.time()}
                    with history.open("a", encoding="utf-8") as output:
                        output.write(json.dumps(report)+"\n")
                    atomic_json(args.output_root / "projector-status.json", report)
                    if metrics["relative_mse"] < phase_best:
                        phase_best = metrics["relative_mse"]
                        model.save(linear_destination if phase == "linear" else args.output_root / "projector-mlp", metadata)
                    if metrics["relative_mse"] < global_best:
                        global_best = metrics["relative_mse"]
                        model.save(destination, {**metadata, "selected_phase": phase, "validation": metrics})
                    temporary = checkpoint.with_suffix(".tmp")
                    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                                "phase": phase, "step": step+1, "global_best": global_best, "phase_best": phase_best,
                                "metadata": metadata, "torch_rng": torch.get_rng_state(), "python_rng": random.getstate()}, temporary)
                    temporary.replace(checkpoint)
                    print(json.dumps(report), flush=True)
            state = None
        best = Projector.load(destination, "cuda")
        selected_config = json.loads((destination / "config.json").read_text())
        best.save(destination, selected_config, storage_dtype=torch.bfloat16)
        best = Projector.load(destination, "cuda")
        final_scores = evaluate(best, valid, student, teacher)
        linear = Projector.load(linear_destination, "cuda")
        baseline_scores = evaluate(linear, valid, student, teacher)
        complete = {**metadata, "validation": final_scores, "linear_baseline": baseline_scores,
                    "finished_at": time.time(), "config": {key: str(value) if isinstance(value, Path) else value
                                                             for key, value in vars(args).items()}}
        atomic_json(destination / "training-complete.json", complete)
        print(json.dumps(complete, indent=2), flush=True)



## encoder.validate_base

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_ec65d40c350810fe':
    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    from pathlib import Path


    import argparse


    import json


    import sys


    import time


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    import numpy as np


    import torch


    from sharm_image21.core import (ARTIFACTS, CACHE, PROJECT, FeatureCache, Projector, atomic_json,
                                    file_hash, model_identity, read_jsonl)


    from sharm_image21.diffusion import load_transformer, load_diffuser_pipeline, predict_flow


    from sharm_image21.image_io import require_transparent_background


    from sharm_image21.model_cards import (LABELS, comparison_manifest, generation_kwargs,
                                           load_comparisons, save_member)


    @torch.no_grad()
    def run(parameters=None):
        args = parameter_namespace({'samples': 4, 'steps': 20}, parameters, required=[], converters={'samples': int, 'steps': int}, choices={}, groups=[])
        if not 1 <= args.samples <= 6 or args.steps < 1:
            raise ValueError("Prévoir de 1 à 6 comparaisons et un nombre de pas positif.")
        projector_path = ARTIFACTS / "projector"
        fingerprint = file_hash(projector_path / "model.safetensors")
        report_path = ARTIFACTS / "base-validation.json"
        output = ARTIFACTS / "base-comparisons"
        if report_path.exists():
            previous = json.loads(report_path.read_text())
            if previous.get("projector_sha256") == fingerprint and previous.get("passed"):
                try:
                    existing = load_comparisons(output, "encoder", fingerprint)
                except (OSError, ValueError, KeyError):
                    existing = None
                if (existing and len(existing["pairs"]) == args.samples
                        and existing["settings"]["num_inference_steps"] == args.steps):
                    print("Base déjà validée, comparaisons appariées présentes à cette révision.", flush=True)
                    return
        training = json.loads((projector_path / "training-complete.json").read_text())
        metrics = training["validation"]
        if not (metrics["relative_mse"] < 0.5 and metrics["cosine"] > 0.65 and 0.75 < metrics["norm_ratio"] < 1.25):
            atomic_json(report_path, {"passed": False, "reason": "embedding_alignment_below_gate", "metrics": metrics,
                                      "projector_sha256": fingerprint})
            raise RuntimeError("Alignement insuffisant ; le LoRA ne sera pas lancé sur cette base.")
        rows = [row for row in read_jsonl(ARTIFACTS / "corpus.jsonl") if row["split"] == "validation"]
        chosen = []
        for marker in ("alpha channel", "exact text", "to the left", "Une illustration", "Exactly", "photograph"):
            match = next((row for row in rows if marker in row["text"] and row not in chosen), None)
            if match:
                chosen.append(match)
        chosen += [row for row in rows if row not in chosen][:max(0, 6-len(chosen))]
        student, teacher = FeatureCache(CACHE / "student"), FeatureCache(CACHE / "teacher")
        projector = Projector.load(projector_path, "cuda").eval()
        pairs = []
        for row in chosen:
            if student.index["entries"][row["id"]]["token_ids"] != teacher.index["entries"][row["id"]]["token_ids"]:
                raise ValueError("Tokens de validation désalignés.")
            with torch.autocast("cuda", dtype=torch.bfloat16):
                projected = projector(student.get(row["id"]).cuda()).to(torch.bfloat16)
            pairs.append((teacher.get(row["id"]).unsqueeze(0).cuda(), projected.unsqueeze(0)))
        del projector
        transformer = load_transformer()
        errors = []
        for index, (reference, adapted) in enumerate(pairs):
            generator = torch.Generator(device="cuda").manual_seed(700+index)
            latents = torch.randn((1, 64, 1, 32, 32), generator=generator, device="cuda", dtype=torch.bfloat16)
            for timestep in (0.2, 0.5, 0.8):
                t = torch.tensor([timestep], device="cuda", dtype=torch.bfloat16)
                reference_prediction = predict_flow(transformer, latents, reference, t).float()
                adapted_prediction = predict_flow(transformer, latents, adapted, t).float()
                error = ((reference_prediction-adapted_prediction).square().mean()/reference_prediction.square().mean().clamp_min(1e-8)).item()
                errors.append(error)
                if not np.isfinite(error):
                    raise RuntimeError("Prédiction du diffuseur non finie.")
        report = {"projector_sha256": fingerprint, "embedding_metrics": metrics,
                  "denoiser_relative_mse": float(np.mean(errors)), "denoiser_worst_mse": max(errors),
                  "scope": "experimental text-to-image; image editing not validated", "updated_at": time.time()}
        if report["denoiser_relative_mse"] > 0.25 or report["denoiser_worst_mse"] > 0.5:
            atomic_json(report_path, {**report, "passed": False, "reason": "denoiser_deviation_above_gate"})
            raise RuntimeError("Déviation du diffuseur trop forte ; arrêt avant le LoRA.")
        pipeline = load_diffuser_pipeline(transformer=transformer)
        output.mkdir(exist_ok=True)
        comparison = comparison_manifest(
            "encoder", pipeline, steps=args.steps, projector_sha256=fingerprint,
            controls={"base_revision": model_identity("teacher")["revision"],
                      "teacher_identity": model_identity("teacher"), "student_identity": model_identity("student"),
                      "teacher_cache_index_sha256": file_hash(teacher.path),
                      "student_cache_index_sha256": file_hash(student.path),
                      "lora": "absent on both sides", "shared_diffusion_and_vae": True},
            selection={"split": "projector validation", "method": "first matching prompt per fixed category, then corpus order",
                       "markers": ["alpha channel", "exact text", "to the left", "Une illustration", "Exactly", "photograph"],
                       "selected_before_generation": True, "count": args.samples},
            script=Path(__file__).resolve(), project=PROJECT)
        samples = []
        for index, (row, pair) in enumerate(zip(chosen[:args.samples], pairs[:args.samples])):
            record = {"id": row["id"], "prompt": row["text"], "seed": 1000+index}
            for side, label, embeddings in zip(("left", "right"), ("teacher", "sharm"), pair):
                image = pipeline(prompt_embeds=embeddings, prompt_embeds_mask=None, **generation_kwargs(args.steps),
                                 generator=torch.Generator(device="cuda").manual_seed(1000+index)).images[0]
                if image.mode != "RGBA" or np.asarray(image)[..., :3].std() < 1 or np.asarray(image)[..., 3].max() == 0:
                    raise RuntimeError(f"Échantillon {label} vide ou format incorrect.")
                if any(marker in row["text"].lower() for marker in
                       ("alpha channel", "transparent background", "with transparency")):
                    require_transparent_background(image)
                member = save_member(image, output, f"{index:02d}-{label}", LABELS["encoder"][0 if side == "left" else 1])
                record[side] = member
                samples.append({"file": member["image"]["file"], "prompt": row["text"],
                                "seed": 1000+index, "conditioner": label})
            comparison["pairs"].append(record)
        atomic_json(output / "comparison.json", comparison)
        load_comparisons(output, "encoder", fingerprint)
        atomic_json(output / "samples.json", samples)
        atomic_json(report_path, {**report, "passed": True, "samples": samples,
                                 "comparison_sha256": file_hash(output / "comparison.json")})
        print(json.dumps(report, indent=2), flush=True)



## monster.audit_monster_dataset

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_43b172f5d26b1ef2':
    """Audit prepared data, source identities, RGBA pixels and species holdout."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import json


    from collections import Counter


    from concurrent.futures import ThreadPoolExecutor


    from pathlib import Path


    import numpy as np


    from PIL import Image


    from prepare_monster_dataset import HERE, Reference, bbox, decode_label, dump, legacy_cells, sha


    def read_jsonl(path):
        return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


    def run(parameters=None):
        args = parameter_namespace({'dataset': None, 'raw_dir': HERE / 'dataset/raw', 'reference_dir': HERE / 'dataset/reference'}, parameters, required=['dataset'], converters={'dataset': Path, 'raw_dir': Path, 'reference_dir': Path}, choices={}, groups=[])
        root = args.dataset
        report = json.loads((root / "dataset.json").read_text())
        rows = read_jsonl(root / "manifest.jsonl")
        sources = json.loads((root / "sources.json").read_text())
        catalog = sources["source_catalog"]
        unresolved = json.loads((root / "unresolved.json").read_text())
        errors = []

        def require(condition, message):
            if not condition:
                errors.append(message)

        require(sha((root / "manifest.jsonl").read_bytes()) == report["manifest_sha256"], "Manifest hash")
        require(len(rows) == report["unique_images"], "Manifest count")
        require(len({r["pixel_sha256"] for r in rows}) == len(rows), "Duplicate target RGBA pixels")
        require(len({r["sample_id"] for r in rows}) == len(rows), "Duplicate sample IDs")
        train = {r["species_id"] for r in rows if r["split"] == "train"}
        validation = {r["species_id"] for r in rows if r["split"] == "validation"}
        require(not train & validation and bool(train) and bool(validation), "Species leakage / empty split")
        raw = {}
        for entry in sources["sources"]:
            path = args.raw_dir / entry["file"]
            require(sha(path.read_bytes()) == entry["sha256"], f"Changed raw file: {path.name}")
            if "duplicate_of" not in entry:
                with Image.open(path) as image:
                    raw[path.name] = np.array(image.convert("RGBA"))

        refs = Reference(args.reference_dir)
        accounted = set()
        expected = set()
        for group in catalog["battle_groups"]:
            for name in group["sheets"]:
                a = raw[name]
                for r in range(a.shape[0] // 103):
                    for c in range(8):
                        if a[r*103:r*103+96,c*96:c*96+96,3].any():
                            expected.add((name,r,c))
        for name, (generation, _) in catalog["legacy_sheets"].items():
            for i, _ in enumerate(legacy_cells(generation)):
                r,c = divmod(i,10)
                expected.add((name,r,c))
        for row in rows:
            d = row["definition"]
            require(d["species_id"] == row["species_id"], f"Definition species: {row['sample_id']}")
            require(d["name_en"] == refs.names[row["species_id"], "9"]["name"], f"Canonical name: {row['sample_id']}")
            require(d["form_identifier"].split("-")[0] == refs.species[row["species_id"]]["identifier"].split("-")[0], f"Form identity: {row['sample_id']}")
            if d["form_name_en"].lower() in ("male", "female"):
                require(d["form_name_en"].lower() == d["gender"], f"Contradictory sex/form: {row['sample_id']}")
            for origin in row["source_cells"]:
                key = (origin["sheet"],origin["row"],origin["column"])
                require(key not in accounted, f"Cell used twice: {key}")
                accounted.add(key)
                if origin["sheet"] in catalog["legacy_sheets"]:
                    generation = catalog["legacy_sheets"][origin["sheet"]][0]
                    species, _ = legacy_cells(generation)[origin["row"]*10+origin["column"]]
                    require(species == row["species_id"], f"Legacy cell species: {key}")
                else:
                    r,c = origin["row"],origin["column"]
                    label = decode_label(raw[origin["sheet"]][r*103+96:r*103+103,c*96:c*96+96,3])
                    require(int(label[:3]) == row["species_id"], f"Printed species: {key}")
        for record in unresolved:
            key = (record["sheet"],record["row"],record["column"])
            require(key not in accounted, f"Quarantined and trained: {key}")
            accounted.add(key)
            require((root / record["original_cell_file"]).is_file(), f"Missing quarantine cell: {key}")
        require(accounted == expected, f"Source-cell accounting: missing={len(expected-accounted)}, extra={len(accounted-expected)}")
        require(len(expected) == report["verified_source_cells"] + report["unresolved_cells"], "Cell totals")
        inventory_path = root / "source_cells.jsonl"
        if inventory_path.exists():
            inventory = read_jsonl(inventory_path)
            require(len(inventory) == len(expected), "Source inventory coverage")
            form_numbers = {}
            for item in inventory:
                label = item.get("printed_label")
                if not label or item["status"] == "quarantine":
                    continue
                d = item["definition"]
                require(int(label[:3]) == d["species_id"], f"Inventory printed species: {label}")
                suffix = label.split("-")[-1]
                if suffix in ("F","M"):
                    require(d["gender"] == {"F":"female","M":"male"}[suffix], f"Printed sex: {label}")
                elif "-" in label and suffix.isdigit():
                    key = (d["species_id"],int(suffix))
                    if key in form_numbers:
                        require(form_numbers[key] == d["form_id"], f"Cross-palette/view form conflict: {label}")
                    form_numbers[key] = d["form_id"]

        def audit_image(row):
            name = row["file_name"]
            path = root / "images_128" / name
            findings = []
            with Image.open(path) as image:
                if image.mode != "RGBA" or image.size != (128,128):
                    return [f"Invalid RGBA/size: {name}"]
                a = np.array(image)
            if sha(path.read_bytes()) != row["image_sha256"] or sha(a.tobytes()) != row["pixel_sha256"]:
                findings.append(f"Changed image: {name}")
            if set(np.unique(a[...,3])) != {0,255} or np.any(a[a[...,3]==0,:3]):
                findings.append(f"Alpha/hidden RGB: {name}")
            box = bbox(a)
            if list(box or []) != row["target_bbox_xyxy"] or min(box[0],box[1],128-box[2],128-box[3]) < 4:
                findings.append(f"Clipping/margin: {name}")
            x0,y0,x1,y1 = box
            scale = row["integer_scale_from_native"]
            crop = a[y0:y1,x0:x1]
            native = crop[::scale,::scale]
            if not np.array_equal(crop,np.repeat(np.repeat(native,scale,0),scale,1)):
                findings.append(f"Fractional resize/smoothing: {name}")
            origin = row["source_cells"][0]
            cx,cy,cw,ch = origin["crop_xywh"]
            source = raw[origin["sheet"]][cy:cy+ch,cx:cx+cw]
            nx0,ny0,nx1,ny1 = row["native_bbox_xyxy"]
            original = source[ny0:ny1,nx0:nx1]
            mask = native[...,3] > 0
            if original.shape != native.shape or not np.array_equal(original[mask,:3],native[mask,:3]):
                findings.append(f"Source foreground pixels changed: {name}")
            for size in (256,512):
                with Image.open(root / f"train_{size}" / name) as im:
                    enlarged = np.array(im)
                    if im.mode != "RGBA" or not np.array_equal(enlarged,np.repeat(np.repeat(a,size//128,0),size//128,1)):
                        findings.append(f"Training resize {size}: {name}")
            return findings

        with ThreadPoolExecutor(max_workers=8) as pool:
            for count, findings in enumerate(pool.map(audit_image,rows),1):
                errors.extend(findings)
                if count % 1000 == 0:
                    print(f"Audited pixels {count}/{len(rows)}",flush=True)
        for size in (128,256,512):
            folder = root / ("images_128" if size==128 else f"train_{size}")
            require({p.name for p in folder.glob("*.png")} == {r["file_name"] for r in rows}, f"Unlisted or missing PNG files {size}")
            metadata = read_jsonl(folder / "metadata.jsonl")
            require(len(metadata)==len(rows), f"Metadata count {size}")
            require({r["file_name"] for r in metadata} == {r["file_name"] for r in rows}, f"Metadata files {size}")
            by_name = {r["file_name"]:r for r in rows}
            for item in metadata:
                original = by_name[item["file_name"]]
                require(item["text"]==original["text"] and item["split"]==original["split"] and item["group_id"]==original["group_id"], f"Metadata disagreement {size}: {item['file_name']}")
            for split in ("train","validation"):
                require({r["file_name"] for r in read_jsonl(folder / f"{split}.jsonl")} == {r["file_name"] for r in rows if r["split"]==split}, f"Split list {size}/{split}")
        corpus = read_jsonl(root / "caption-corpus.jsonl")
        require({r["text"] for r in corpus} == {r["text"] for r in rows}, "Caption corpus coverage")
        require(all(r["id"]==sha(json.dumps(r["text"],sort_keys=True,ensure_ascii=False).encode()) for r in corpus), "Caption cache digest")
        result = {"passed": not errors, "errors": errors, "unique_images_checked":len(rows),
                  "png_files_checked":len(rows)*3,"source_cells_accounted":len(expected),
                  "species":len(train|validation),"train_species":len(train),"validation_species":len(validation),
                  "verification_levels":dict(Counter(r["verification"] for r in rows)),
                  "manifest_sha256":report["manifest_sha256"],"human_review":False,
                  "note":"Integrity/geometry/metadata audit, not a guarantee of generated image quality."}
        dump(root / "audit.json",result)
        print(json.dumps(result,ensure_ascii=False,indent=2),flush=True)
        if errors:
            raise SystemExit(1)



## monster.evaluate_sprites

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_ca801ff00ff0f159':
    import re

    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    from pathlib import Path


    import argparse


    import json


    import math


    import sys


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    import numpy as np


    import torch


    from sharm_image21.core import ARTIFACTS, PROJECT, WEIGHTS, FeatureCache, atomic_json, file_hash


    from sharm_image21.diffusion import load_diffuser_pipeline


    from sharm_image21.image_io import require_transparent_background


    from sharm_image21.model_cards import (LABELS, comparison_manifest, generation_kwargs, load_comparisons,
                                           lora_card, package_comparisons, save_member)


    @torch.inference_mode()
    def run(parameters=None):
        args = parameter_namespace({'lora_weight': 1.0, 'lora_dir': None, 'training_cache': None, 'base': None, 'output': None, 'base_hf_revision': None}, parameters, required=[], converters={'lora_weight': float}, choices={}, groups=[])
        if not math.isfinite(args.lora_weight) or not 0 < args.lora_weight <= 2:
            raise ValueError("Choisir un poids LoRA fini entre 0 et 2.")
        lora_dir = Path(args.lora_dir) if args.lora_dir else WEIGHTS / "sharm-image-2.1-sprites-128-lora"
        if not (lora_dir / "training-complete.json").exists():
            raise RuntimeError("Le LoRA n'est pas terminé.")
        data = json.loads((Path(args.training_cache) if args.training_cache else ARTIFACTS / "sprite-training-512.json").read_text())
        base = Path(args.base) if args.base else WEIGHTS / "sharm-image-2.1-vl4b-fp8"
        base_config = json.loads((base / "sharm_config.json").read_text())
        training = json.loads((lora_dir / "training-complete.json").read_text())
        fingerprint = data["identity"]["projector_sha256"]
        if (base_config["projector_sha256"] != fingerprint
                or file_hash(base / "projector/model.safetensors") != fingerprint
                or training["config"]["cache_identity"] != data["identity"]):
            raise ValueError("Les comparaisons doivent utiliser la base et le cache exacts du LoRA.")
        spatial = base_config.get("pipeline_format") == "spatial-v1"
        if not spatial:
            load_comparisons(base / "comparisons/encoder", "encoder", fingerprint)
        elif not re.fullmatch(r"[a-f0-9]{40}", args.base_hf_revision or ""):
            raise ValueError("Renseigner la révision HF immuable de la Base spatiale retenue.")
        lora_fingerprint = file_hash(lora_dir / "pytorch_lora_weights.safetensors")
        rows = [row for row in data["rows"] if row["split"] == "validation"]
        samples, species = [], set()
        for row in rows:
            if row["species_id"] not in species:
                samples.append(row)
                species.add(row["species_id"])
            if len(samples) == 6:
                break
        if len(samples) != 6:
            raise ValueError("Six espèces de validation distinctes sont requises pour la galerie.")
        cache = FeatureCache(Path(data["text_cache"]))
        if cache.index["identity"]["projector_sha256"] != fingerprint:
            raise ValueError("Les embeddings ne correspondent pas au projecteur de la base.")
        pipeline = load_diffuser_pipeline(base=base)
        output = Path(args.output) if args.output else ARTIFACTS / "sprite-samples"
        output.mkdir(parents=True, exist_ok=True)
        existing = None
        if (output / "comparison.json").exists():
            candidate = load_comparisons(output, "lora", fingerprint, lora_fingerprint)
            if (candidate["settings"]["num_inference_steps"] == 30
                    and [pair["species_id"] for pair in candidate["pairs"]] == [row["species_id"] for row in samples]):
                existing = candidate
        previous_report = json.loads((output / "evaluation.json").read_text()) if existing else None
        comparison = comparison_manifest(
            "lora", pipeline, steps=30, projector_sha256=fingerprint,
            controls={"base_model": base_config["hf_repo_intent"], "base_revision": base_config["base_revision"],
                      "student_identity": base_config["student_identity"], "same_conditioning_embeddings": True,
                      "text_cache_index_sha256": file_hash(cache.path), "dataset_identity": data["identity"],
                      "lora_sha256": lora_fingerprint, "lora_weight": args.lora_weight,
                      "training": {key: training["config"][key] for key in
                                   ("steps", "resolution", "rank", "learning_rate", "accumulation")},
                      "training_image_count": sum(row["split"] == "train" for row in data["rows"]),
                      "validation_image_count": len(rows),
                      "left_adapter": "absent", "right_adapter": "sharm_sprites"},
            selection={"split": "LoRA validation by species", "method": "first six distinct species in metadata order",
                       "selected_before_generation": True, "count": len(samples)},
            script=Path(__file__).resolve(), project=PROJECT)
        comparison["pairs"] = [{"id": f"species-{row['species_id']}", "species_id": row["species_id"],
                                "caption_id": row["caption_id"],
                                "prompt": row["text"], "seed": 2000+index}
                               for index, row in enumerate(samples)]
        report = [{"species_id": row["species_id"], "prompt": row["text"], "seed": 2000+index}
                  for index, row in enumerate(samples)]
        # Complete the whole baseline pass before any adapter is loaded.
        for side, name, label in zip(("left", "right"), ("before", "after"), LABELS["lora"]):
            if side == "left" and existing:
                for index, pair in enumerate(existing["pairs"]):
                    comparison["pairs"][index]["left"] = pair["left"]
                    report[index]["before"] = previous_report["samples"][index]["before"]
                continue
            if side == "right":
                pipeline.load_lora_weights(lora_dir, adapter_name="sharm_sprites")
                pipeline.set_adapters("sharm_sprites", adapter_weights=args.lora_weight)
            for index, row in enumerate(samples):
                embeds = cache.get(row["caption_id"]).unsqueeze(0).cuda()
                image = pipeline(prompt_embeds=embeds, prompt_embeds_mask=None, **generation_kwargs(30),
                                 generator=torch.Generator(device="cuda").manual_seed(2000+index)).images[0]
                pixels = np.asarray(image)
                if image.mode != "RGBA" or pixels[..., :3].std() < 1 or pixels[..., 3].max() == 0:
                    raise RuntimeError(f"Échantillon sprite {name} vide ou mauvais format.")
                alpha_fraction = require_transparent_background(image)
                comparison["pairs"][index][side] = save_member(
                    image, output, f"{row['species_id']}-{name}", label, final_size=128)
                report[index][name] = {"transparent_fraction": float((pixels[..., 3] == 0).mean()),
                                       "near_transparent_fraction": alpha_fraction,
                                       "image": comparison["pairs"][index][side]["image"]["file"]}
        atomic_json(output / "comparison.json", comparison)
        load_comparisons(output, "lora", fingerprint, lora_fingerprint)
        atomic_json(output / "evaluation.json", {"samples": report, "manual_visual_review_required": True,
                                                "projector_sha256": fingerprint, "lora_sha256": lora_fingerprint,
                                                "comparison_sha256": file_hash(output / "comparison.json"),
                                                "scope": "automatic checks verify file and numerical integrity, not identity quality"})
        atomic_json(lora_dir / "sharm_lora_config.json", {"projector_sha256": fingerprint,
            "base_model": "ethanzxv/Sharm-Image-2.1-VL4B-FP8", "private": True, "published": False,
            "recommended_weight": args.lora_weight,
            "base_diffusion_revision": base_config["base_revision"],
            "base_hf_revision": args.base_hf_revision,
            "lora_sha256": lora_fingerprint, "comparison_sha256": file_hash(output / "comparison.json")})
        packaged_lora = package_comparisons(output, lora_dir / "comparisons/lora", "lora", fingerprint, lora_fingerprint)
        if spatial:
            atomic_json(lora_dir / "base-reference.json", {"repo_id": base_config["hf_repo_intent"],
                "revision": args.base_hf_revision, "projector_sha256": fingerprint,
                "dataset_identity": data["identity"]})
            card = "# Sharm Image Monster Pixel Art\n\nBase : " + base_config["hf_repo_intent"] + " @ " + args.base_hf_revision
            card += "\n\nComparaisons sur six espèces réservées ; revue visuelle requise.\n"
            for pair in packaged_lora["pairs"]:
                card += "\n## " + pair["prompt"] + "\n"
                for side in ("left", "right"):
                    card += "\n![" + side + "](comparisons/lora/" + pair[side]["preview"]["file"] + ")\n"
            (lora_dir / "README.md").write_text(card, encoding="utf-8")
            for filename in ("LICENSE", "LICENSE.encoder", "NOTICE", "requirements-inference.txt"):
                (lora_dir / filename).write_bytes((base / filename).read_bytes())
            atomic_json(lora_dir / "evaluation.json", json.loads((output / "evaluation.json").read_text()))
            print(json.dumps(report, indent=2), flush=True)
            return
        packaged_encoder = package_comparisons(base / "comparisons/encoder", lora_dir / "comparisons/encoder", "encoder", fingerprint)
        footprint = json.loads((base / "footprint.json").read_text())
        atomic_json(lora_dir / "footprint.json", footprint)
        atomic_json(lora_dir / "evaluation.json", json.loads((output / "evaluation.json").read_text()))
        (lora_dir / "README.md").write_text(lora_card(packaged_lora, packaged_encoder, footprint), encoding="utf-8")
        for filename in ("LICENSE", "LICENSE.encoder", "NOTICE", "requirements-inference.txt"):
            (lora_dir / filename).write_bytes((base / filename).read_bytes())
        notice = lora_dir / "NOTICE"
        notice.write_text(notice.read_text(encoding="utf-8") +
                          "Sharm Sprites 128 adds trained LoRA adapters to the diffusion transformer.\n", encoding="utf-8")
        print(json.dumps(report, indent=2), flush=True)



## monster.fetch_monster_references

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_b43b447f3ee9bd55':
    """Download a pinned PokéAPI reference, never training images from this cache."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import csv


    import hashlib


    import json


    import time


    from concurrent.futures import ThreadPoolExecutor, as_completed


    from pathlib import Path


    from urllib.request import Request, urlopen


    SPRITES_REVISION = "7a7d66b1620e646f69afd48dc1273da9033ede9c"


    DATA_REVISION = "6fd69dfe90465ba96534f6816fc817737d1f97f6"


    SPRITE_PREFIX = "sprites/pokemon/versions/generation-v/black-white"


    CSV_FILES = ("pokemon_species", "pokemon_species_names", "pokemon", "pokemon_forms",
                 "pokemon_form_names", "pokemon_shapes", "pokemon_shape_prose", "pokemon_colors",
                 "pokemon_types", "types", "type_names")


    def download(url: str) -> bytes:
        for attempt in range(4):
            try:
                with urlopen(Request(url, headers={"User-Agent": "SharmMonsterDataset/1.0"}), timeout=60) as response:
                    return response.read()
            except Exception:
                if attempt == 3:
                    raise
                time.sleep(attempt + 1)
        raise AssertionError("unreachable")


    def git_blob_hash(data: bytes) -> str:
        return hashlib.sha1(b"blob " + str(len(data)).encode() + b"\0" + data).hexdigest()


    def run(parameters=None) -> None:
        args = parameter_namespace({'reference_dir': Path(__file__).parent / 'dataset/reference', 'workers': 12}, parameters, required=[], converters={'reference_dir': Path, 'workers': int}, choices={}, groups=[])
        root = args.reference_dir
        root.mkdir(parents=True, exist_ok=True)
        lock = {"sprites_revision": SPRITES_REVISION, "pokeapi_revision": DATA_REVISION}
        for name in CSV_FILES:
            url = f"https://raw.githubusercontent.com/PokeAPI/pokeapi/{DATA_REVISION}/data/v2/csv/{name}.csv"
            data = download(url)
            (root / f"{name}.csv").write_bytes(data)
            lock.setdefault("csv_sha256", {})[name] = hashlib.sha256(data).hexdigest()
        pokemon = {row["id"]: row for row in csv.DictReader((root / "pokemon.csv").open(encoding="utf-8"))}
        allowed = {str(i) for i in range(1, 650)}
        for row in csv.DictReader((root / "pokemon_forms.csv").open(encoding="utf-8")):
            mon = pokemon[row["pokemon_id"]]
            if int(mon["species_id"]) <= 649 and int(row["introduced_in_version_group_id"] or 999) <= 14:
                allowed.add(row["pokemon_id"])
        entries = []
        for prefix, folder, tree_name in ((SPRITE_PREFIX, "sprites", "sprite-tree.json"),
                ("sprites/pokemon/versions/generation-iii/emerald", "emerald", "emerald-tree.json")):
            tree = json.loads(download(f"https://api.github.com/repos/PokeAPI/sprites/git/trees/"
                                       f"{SPRITES_REVISION}:{prefix}?recursive=1"))
            if tree.get("truncated"):
                raise RuntimeError("Incomplete GitHub reference tree")
            (root / tree_name).write_text(json.dumps(tree), encoding="utf-8")
            for entry in tree["tree"]:
                path = entry["path"]
                stem = Path(path).stem.split("-", 1)[0]
                if entry["type"] != "blob" or not path.endswith(".png"):
                    continue
                if folder == "sprites":
                    keep = not path.startswith("animated/") and stem in allowed
                else:
                    keep = path.startswith("animated/") and stem.isdigit() and (int(stem) <= 386 or stem in {"10013", "10014", "10015"})
                if keep:
                    entries.append({**entry, "prefix": prefix, "folder": folder})

        def fetch(entry):
            target = root / entry["folder"] / entry["path"]
            if target.is_file() and git_blob_hash(target.read_bytes()) == entry["sha"]:
                return
            data = download(f"https://raw.githubusercontent.com/PokeAPI/sprites/{SPRITES_REVISION}/"
                            f"{entry['prefix']}/{entry['path']}")
            if git_blob_hash(data) != entry["sha"]:
                raise RuntimeError(f"Reference checksum mismatch: {entry['path']}")
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)

        with ThreadPoolExecutor(max_workers=args.workers) as pool:
            futures = [pool.submit(fetch, entry) for entry in entries]
            for index, future in enumerate(as_completed(futures), 1):
                future.result()
                if index % 250 == 0 or index == len(futures):
                    print(f"References: {index}/{len(futures)}", flush=True)
        lock["sprite_files"] = len(entries)
        (root / "reference-lock.json").write_text(json.dumps(lock, indent=2) + "\n", encoding="utf-8")



## monster.prepare_dataset

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_c82e7d6e33c0e8a3':
    """Cut the local Pokémon overworld sheets into 64px RGBA training images."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import csv


    import hashlib


    import json


    import re


    import time


    from concurrent.futures import ThreadPoolExecutor, as_completed


    from pathlib import Path


    from urllib.error import URLError


    from urllib.request import Request, urlopen


    from PIL import Image


    SCRIPT_DIR = Path(__file__).resolve().parents[1]


    SHEETS = (
        ("21228.png", "Hoenn / Gen 3", False),
        ("21229.png", "Johto / Gen 2", False),
        ("21230.png", "Kanto / Gen 1", False),
        ("21231.png", "Kanto / Gen 1", True),
        ("21301.png", "Johto / Gen 2", True),
        ("21650.png", "Hoenn / Gen 3", True),
    )


    CELL_SIZE = 64


    PITCH = 69


    ORIGIN = 5


    BACKGROUND = (165, 235, 255)  # #A5EBFF, the flat blue used behind every cell.


    POKEAPI_BASE = "https://pokeapi.co/api/v2/pokemon-species/"


    DISPLAY_NAME_OVERRIDES = {
        "farfetchd": "Farfetch'd",
        "ho-oh": "Ho-Oh",
        "mr-mime": "Mr. Mime",
        "mime-jr": "Mime Jr.",
        "nidoran-f": "Nidoran♀",
        "nidoran-m": "Nidoran♂",
    }


    def sheet_cells(generation: str) -> list[tuple[int, str]]:
        """Species/form per 64px cell, audited against these six source sheets.

        Johto omits Unown (#201); Hoenn omits Spinda (#327) and gives Castform
        four cells instead of two. Footer artwork is outside this cell list.
        """
        start, stop = {"Kanto / Gen 1": (1, 152), "Johto / Gen 2": (152, 252),
                       "Hoenn / Gen 3": (252, 387)}[generation]
        cells = []
        for species_id in range(start, stop):
            if species_id in (201, 327):
                continue
            forms = ("normal form", "sunny form", "rainy form", "snowy form") if species_id == 351 else ("", "")
            cells.extend((species_id, form) for form in forms)
        return cells


    def parse_color(value: str) -> tuple[int, int, int]:
        value = value.removeprefix("#")
        if len(value) != 6:
            raise argparse.ArgumentTypeError("La couleur doit être au format RRGGBB ou #RRGGBB.")
        try:
            return tuple(int(value[index : index + 2], 16) for index in (0, 2, 4))  # type: ignore[return-value]
        except ValueError as error:
            raise argparse.ArgumentTypeError("La couleur contient un chiffre hexadécimal invalide.") from error


    def remove_background_color(tile: Image.Image, color: tuple[int, int, int]) -> int:
        """Make every exact match of the flat sheet background transparent."""
        # `Image.convert("RGBA")` returns a separate image even when `tile` is
        # already RGBA. Keep the caller's image in that case so the alpha edits
        # are present when the caller saves the tile.
        if tile.mode != "RGBA":
            tile = tile.convert("RGBA")
        pixels = tile.load()
        width, height = tile.size
        for y in range(height):
            for x in range(width):
                if pixels[x, y][:3] == color:
                    pixels[x, y] = (0, 0, 0, 0)

        return tile.getchannel("A").tobytes().count(255)


    def normalize_flavor_text(value: str) -> str:
        value = re.sub(r"\s+", " ", value.replace("POKéMON", "Pokémon")).strip()
        if len(value) > 200:
            value = value[:197].rsplit(" ", 1)[0] + "…"
        return value


    def fetch_species_trait(species_id: int) -> dict[str, str]:
        request = Request(
            f"{POKEAPI_BASE}{species_id}/",
            headers={"User-Agent": "pokemon-overworld-lora-dataset/1.0"},
        )
        last_error: Exception | None = None
        for attempt in range(3):
            try:
                with urlopen(request, timeout=30) as response:
                    payload = json.load(response)
                genus = next(
                    entry["genus"]
                    for entry in payload["genera"]
                    if entry["language"]["name"] == "en"
                )
                flavor_text = next(
                    entry["flavor_text"]
                    for entry in payload["flavor_text_entries"]
                    if entry["language"]["name"] == "en"
                )
                identifier = payload["name"]
                return {
                    "name": DISPLAY_NAME_OVERRIDES.get(identifier, identifier.replace("-", " ").title()),
                    "genus": genus,
                    "color": payload["color"]["name"],
                    "shape": payload["shape"]["name"].replace("-", " "),
                    "pokedex_note": normalize_flavor_text(flavor_text),
                }
            except (URLError, TimeoutError, KeyError, StopIteration, json.JSONDecodeError) as error:
                last_error = error
                if attempt < 2:
                    time.sleep(attempt + 1)
        raise RuntimeError(f"Impossible de récupérer les traits PokéAPI du Pokémon #{species_id}: {last_error}")


    def load_species_traits(output_dir: Path) -> dict[str, dict[str, str]]:
        """Cache English Pokédex genus/color/shape and one short Pokédex note."""
        cache_path = output_dir / "pokemon_traits.json"
        if cache_path.is_file():
            try:
                cached = json.loads(cache_path.read_text(encoding="utf-8"))
                if all(str(species_id) in cached for species_id in range(1, 387)):
                    return cached
            except (OSError, json.JSONDecodeError):
                pass

        traits: dict[str, dict[str, str]] = {}
        with ThreadPoolExecutor(max_workers=8) as executor:
            futures = {
                executor.submit(fetch_species_trait, species_id): species_id
                for species_id in range(1, 387)
            }
            for future in as_completed(futures):
                species_id = futures[future]
                traits[str(species_id)] = future.result()
        cache_path.write_text(json.dumps(traits, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        return traits


    def asset_label(species: dict[str, str], shiny: bool = False, form: str = "") -> str:
        return f"{'shiny ' if shiny else ''}{species['name']}" + (f", {form}" if form else "")


    def make_caption(species: dict[str, str], shiny: bool = False, form: str = "") -> str:
        # PokéAPI's species color describes the normal palette, not shiny/weather forms.
        coloration = "shiny coloration" if shiny else f"{species['color']} coloration"
        if form:
            coloration = "shiny coloration" if shiny else "weather-form coloration"
        return (
            f"pkm_ow_sprite, {asset_label(species, shiny, form)}, the {species['genus']}; "
            f"{coloration}, {species['shape']} body. "
            f"Pokédex note: {species['pokedex_note']} "
            "Classic handheld-game overworld pixel-art sprite, single character, transparent background."
        )


    def run(parameters=None) -> None:
        args = parameter_namespace({'output_dir': SCRIPT_DIR / 'dataset', 'training_size': 1024, 'background': BACKGROUND, 'sheet_dir': SCRIPT_DIR}, parameters, required=[], converters={'output_dir': Path, 'training_size': int, 'background': parse_color, 'sheet_dir': Path}, choices={}, groups=[])
        if args.training_size < CELL_SIZE or args.training_size % 32:
            parser.error("--training-size doit être au moins 64 et être un multiple de 32.")

        sheet_dir = args.sheet_dir.expanduser().resolve()
        missing = [str(sheet_dir / name) for name, *_ in SHEETS if not (sheet_dir / name).is_file()]
        if missing:
            raise FileNotFoundError("Planches source absentes (le script les découpe mais ne les télécharge pas) : "
                                    + ", ".join(missing) + ". Fournir --sheet-dir ou utiliser 03-spatial-distillation/run.py --pokeapi.")
        output_dir = args.output_dir.expanduser().resolve()
        tiles_dir = output_dir / "sprites_64"
        training_dir = output_dir / f"train_{args.training_size}"
        tiles_dir.mkdir(parents=True, exist_ok=True)
        training_dir.mkdir(parents=True, exist_ok=True)

        manifest_path = output_dir / "manifest.csv"
        traits = load_species_traits(output_dir)
        records: list[dict[str, object]] = []
        total_cells = 0
        total_expected_sprites = sum(len(sheet_cells(generation)) for _, generation, _ in SHEETS)
        sources = []
        empty_cells = 0
        for filename, generation, shiny in SHEETS:
            sheet_path = sheet_dir / filename
            if not sheet_path.is_file():
                raise FileNotFoundError(f"Planche locale introuvable : {sheet_path}")
            with Image.open(sheet_path) as source:
                source = source.convert("RGB")
                if source.getpixel((ORIGIN, ORIGIN)) != args.background:
                    raise ValueError(
                        f"Le fond attendu {args.background} n’est pas présent en ({ORIGIN}, {ORIGIN}) dans {filename}. "
                        "Vérifie la planche avant de changer la couleur avec --background."
                    )
                columns = (source.width - ORIGIN - CELL_SIZE) // PITCH + 1
                rows = (source.height - ORIGIN - CELL_SIZE) // PITCH + 1
                if columns != 10:
                    raise ValueError(f"La grille auditée requiert dix colonnes : {filename}.")
                cells = sheet_cells(generation)
                if rows * columns < len(cells):
                    raise ValueError(f"Planche tronquée : {filename}.")
                source_sha256 = hashlib.sha256(sheet_path.read_bytes()).hexdigest()
                sources.append({"file_name": filename, "sha256": source_sha256, "generation": generation,
                                "shiny": shiny, "size": list(source.size), "expected_cells": len(cells)})
                total_cells += rows * columns
                kept = 0
                for row in range(rows):
                    for column in range(columns):
                        cell_index = row * columns + column
                        if cell_index >= len(cells):
                            continue  # Ignore white placeholders and footer artwork/credits.
                        left = ORIGIN + column * PITCH
                        top = ORIGIN + row * PITCH
                        tile = source.crop((left, top, left + CELL_SIZE, top + CELL_SIZE)).convert("RGBA")
                        colors = tile.getcolors(maxcolors=2)
                        if colors and len(colors) == 1 and colors[0][1] == (255, 255, 255, 255):
                            empty_cells += 1
                            continue  # Some source-sheet slots are blank white placeholders.
                        occupied_pixels = remove_background_color(tile, args.background)
                        if occupied_pixels < 12:
                            continue

                        species_id, form = cells[cell_index]
                        species = traits[str(species_id)]
                        caption = make_caption(species, shiny, form)
                        stem = f"{Path(filename).stem}_r{row:02d}_c{column:02d}"
                        tile_path = tiles_dir / f"{stem}.png"
                        training_path = training_dir / f"{stem}.png"
                        tile.save(tile_path)
                        tile.resize((args.training_size, args.training_size), Image.Resampling.NEAREST).save(training_path)
                        records.append(
                            {
                                "file": training_path.relative_to(output_dir).as_posix(),
                                "preview_64": tile_path.relative_to(output_dir).as_posix(),
                                "sheet": filename,
                                "generation": generation,
                                "species_id": species_id,
                                "species": species["name"],
                                "shiny": shiny,
                                "form": form,
                                "label": asset_label(species, shiny, form),
                                "source_sha256": source_sha256,
                                "caption": caption,
                                "row": row,
                                "column": column,
                                "occupied_pixels": occupied_pixels,
                            }
                        )
                        kept += 1
                print(
                    f"{filename}: {source.width}×{source.height}, grille {columns}×{rows}, "
                    f"cellules Pokédex utilisées {len(cells)}, sprites extraits {kept}."
                )

        if not records:
            raise RuntimeError("Aucune tuile non vide détectée ; aucun dataset n’a été produit.")

        # Remove only stale files bearing this script's generated sheet/row/column
        # naming pattern (for example footer cells left by an older run).
        expected_names = {
            Path(str(record[key])).name
            for record in records
            for key in ("file", "preview_64")
        }
        generated_prefixes = tuple(f"{Path(filename).stem}_r" for filename, *_ in SHEETS)
        removed_stale = 0
        for folder in (tiles_dir, training_dir):
            for image_path in folder.glob("*.png"):
                if image_path.name.startswith(generated_prefixes) and image_path.name not in expected_names:
                    image_path.unlink()
                    removed_stale += 1

        with manifest_path.open("w", newline="", encoding="utf-8") as manifest_file:
            writer = csv.DictWriter(manifest_file, fieldnames=list(records[0]))
            writer.writeheader()
            writer.writerows(records)

        metadata_path = training_dir / "metadata.jsonl"
        with metadata_path.open("w", encoding="utf-8") as metadata_file:
            for record in records:
                metadata_file.write(
                    json.dumps(
                        {
                            "file_name": Path(str(record["file"])).name,
                            "text": record["caption"],
                            "group_id": str(record["species_id"]),
                            "label": record["label"],
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )

        (output_dir / "sources.json").write_text(json.dumps({
            "schema": 1, "layout": "six-sheets-missing-unown-spinda-castform-four-cells-v1",
            "sheets": sources, "sprite_count": len(records),
            "species_count": len({r["species_id"] for r in records}),
            "missing_species": [201, 327], "background_removed": list(args.background),
            "training_size": args.training_size,
        }, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

        print(
            f"Cellules examinées : {total_cells} ; sprites Pokédex attendus : "
            f"{total_expected_sprites} ; images conservées : {len(records)}."
        )
        if empty_cells:
            print(f"Cases blanches vides ignorées : {empty_cells}.")
        if removed_stale:
            print(f"Anciennes images générées supprimées : {removed_stale}.")
        print(f"Aperçus 64×64 : {tiles_dir}")
        print(f"Images d’entraînement {args.training_size}×{args.training_size} : {training_dir}")
        print(f"Légendes image par image : {metadata_path}")
        print(f"Inventaire : {manifest_path}")



## monster.prepare_monster_dataset

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_fdcf4ee35b88ed38':
    """Reconcile supplied battle sheets with pinned references and build RGBA sprites."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import colorsys


    import csv


    import hashlib


    import html


    import importlib.util


    import io


    import json


    from collections import Counter, defaultdict


    from concurrent.futures import ThreadPoolExecutor


    from functools import cache


    from pathlib import Path


    import numpy as np


    from PIL import Image, ImageDraw


    from fetch_monster_references import SPRITES_REVISION, download, git_blob_hash


    HERE = Path(__file__).resolve().parents[1] / "Sharm-Image-2.1-VL4B-LoRA-Monster"


    VERSION = "monster-rgba-v1"


    def sha(data: bytes) -> str:
        return hashlib.sha256(data).hexdigest()


    def dump(path: Path, value) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


    def jsonl(path: Path, rows) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows), encoding="utf-8")


    def decode_label(alpha):
        """Exact bitmap decoding of the sheet's font; no probabilistic OCR."""
        glyphs = json.loads((HERE / "monster_label_font.json").read_text())["glyphs"]
        occupied = np.where((alpha > 0).any(axis=0))[0]
        chunks = np.split(occupied, np.where(np.diff(occupied) > 1)[0] + 1)
        text = ""
        for chunk in chunks:
            if not len(chunk):
                continue
            key = "/".join("".join("1" if v else "0" for v in line)
                           for line in alpha[:, chunk.min():chunk.max() + 1] > 0)
            if key not in glyphs:
                raise ValueError("Unknown source-label glyph")
            text += glyphs[key]
        return text


    def bbox(a):
        y, x = np.nonzero(a[..., 3])
        return (int(x.min()), int(y.min()), int(x.max()) + 1, int(y.max()) + 1) if len(y) else None


    def signature(a):
        box = bbox(a)
        if box is None:
            return None
        x0, y0, x1, y1 = box
        v = a[y0:y1, x0:x1].copy()
        v[v[..., 3] == 0] = 0
        v[..., :3] >>= 3  # 5-bit channels: sheet v<<3 versus reference round(v*255/31).
        return f"{v.shape[1]}x{v.shape[0]}:" + sha(v.tobytes())


    def read_csv(root, name):
        with (root / f"{name}.csv").open(encoding="utf-8", newline="") as stream:
            return list(csv.DictReader(stream))


    class Reference:
        def __init__(self, root: Path):
            self.root = root
            self.lock = json.loads((root / "reference-lock.json").read_text())
            if self.lock["sprites_revision"] != SPRITES_REVISION:
                raise RuntimeError("Unexpected reference revision")
            for name, expected in self.lock["csv_sha256"].items():
                if sha((root / f"{name}.csv").read_bytes()) != expected:
                    raise RuntimeError(f"Modified reference CSV: {name}")
            self.pokemon = {int(r["id"]): r for r in read_csv(root, "pokemon")}
            self.species = {int(r["id"]): r for r in read_csv(root, "pokemon_species")}
            self.names = {(int(r["pokemon_species_id"]), r["local_language_id"]): r
                          for r in read_csv(root, "pokemon_species_names")}
            self.forms = {r["identifier"]: r for r in read_csv(root, "pokemon_forms")}
            self.default_forms = {int(r["pokemon_id"]): r for r in self.forms.values() if r["is_default"] == "1"}
            self.form_names = {int(r["pokemon_form_id"]): r for r in read_csv(root, "pokemon_form_names")
                               if r["local_language_id"] == "9"}
            self.form_names_fr = {int(r["pokemon_form_id"]): r for r in read_csv(root, "pokemon_form_names")
                                  if r["local_language_id"] == "5"}
            self.shapes = {int(r["id"]): r["identifier"] for r in read_csv(root, "pokemon_shapes")}
            self.shape_descriptions = {int(r["pokemon_shape_id"]): r["description"]
                                      for r in read_csv(root, "pokemon_shape_prose") if r["local_language_id"] == "9"}
            self.types = {int(r["id"]): r["identifier"] for r in read_csv(root, "types")}
            self.mon_types = defaultdict(list)
            for r in read_csv(root, "pokemon_types"):
                self.mon_types[int(r["pokemon_id"])].append(self.types[int(r["type_id"])])
            self.tree = json.loads((root / "sprite-tree.json").read_text())
            self.tree_entries = {r["path"]: r for r in self.tree["tree"] if r["type"] == "blob"}
            self.frames_cache = {}
            self.local_paths = {p.relative_to(root / "sprites").as_posix() for p in (root / "sprites").rglob("*") if p.is_file()}
            self.emerald_entries = {r["path"]: r for r in json.loads((root / "emerald-tree.json").read_text())["tree"]
                                    if r["type"] == "blob"}

        @cache
        def entity(self, path):
            stem = Path(path).stem
            identifier, _, suffix = stem.partition("-")
            mon = self.pokemon[int(identifier)]
            species_id = int(mon["species_id"])
            form = self.forms.get(f"{self.species[species_id]['identifier']}-{suffix}") if suffix else None
            form = form or self.default_forms.get(int(identifier))
            form_id = int(form["id"]) if form else species_id
            sex = "female" if "female" in Path(path).parts or suffix == "female" else None
            if sex is None and self.species[species_id]["has_gender_differences"] == "1":
                sex = "male"
            # Frillish/Jellicent have explicit form IDs in addition to female/ paths.
            sex_form = self.forms.get(f"{self.species[species_id]['identifier']}-{sex}") if sex else None
            if sex_form:
                form_id = int(sex_form["id"])
            return (species_id, form_id, sex)

        def frames(self, family, path):
            key = (family, path)
            if key not in self.frames_cache:
                entries = self.tree_entries if family == "sprites" else self.emerald_entries
                if git_blob_hash((self.root / family / path).read_bytes()) != entries[path]["sha"]:
                    raise RuntimeError(f"Modified reference sprite: {family}/{path}")
                with Image.open(self.root / family / path) as image:
                    seen, frames = set(), []
                    for index in range(getattr(image, "n_frames", 1)):
                        image.seek(index)
                        a = np.array(image.convert("RGBA"))
                        key_hash = sha(a.tobytes())
                        if key_hash not in seen:
                            frames.append((index, a))
                            seen.add(key_hash)
                    self.frames_cache[key] = frames
            return self.frames_cache[key]

        def index(self):
            cache = self.root / "match-index-v1.json"
            identity = sha(json.dumps(self.lock, sort_keys=True).encode())
            if cache.is_file():
                data = json.loads(cache.read_text())
                if data.get("identity") == identity:
                    return data["index"]
            jobs = []
            for family in ("sprites", "emerald"):
                for f in sorted((self.root / family).rglob("*.png")):
                    rel = f.relative_to(self.root / family).as_posix()
                    if family == "emerald" and "back" in f.parts:
                        continue
                    jobs.append((family, rel))

            def process(job):
                family, path = job
                # All files must match their Git blob, including warm-cache builds.
                tree = self.tree_entries if family == "sprites" else self.emerald_entries
                if git_blob_hash((self.root / family / path).read_bytes()) != tree[path]["sha"]:
                    raise RuntimeError(f"Modified reference sprite: {family}/{path}")
                results = []
                for frame, a in self.frames(family, path):
                    sig = signature(a)
                    if sig:
                        results.append((sig, {"family": family, "path": path, "frame": frame}))
                return results

            self.emerald_entries = {r["path"]: r for r in json.loads((self.root / "emerald-tree.json").read_text())["tree"]
                                    if r["type"] == "blob"}
            index = defaultdict(list)
            with ThreadPoolExecutor(max_workers=8) as pool:
                for count, result in enumerate(pool.map(process, jobs), 1):
                    for key, match in result:
                        index[key].append(match)
                    if count % 500 == 0:
                        print(f"Index references {count}/{len(jobs)}", flush=True)
            dump(cache, {"identity": identity, "index": dict(index)})
            return dict(index)

        def animated_candidates(self, entities, view, shiny):
            candidates = []
            for path, entry in self.tree_entries.items():
                if not path.startswith("animated/") or not path.endswith(".gif"):
                    continue
                if not Path(path).stem.split("-", 1)[0].isdigit():
                    continue
                if ("back" in Path(path).parts) != (view == "back") or ("shiny" in Path(path).parts) != shiny:
                    continue
                if self.entity(path) not in entities:
                    continue
                target = self.root / "sprites" / path
                if not target.is_file() or git_blob_hash(target.read_bytes()) != entry["sha"]:
                    data = download(f"https://raw.githubusercontent.com/PokeAPI/sprites/{SPRITES_REVISION}/"
                                    f"sprites/pokemon/versions/generation-v/black-white/{path}")
                    if git_blob_hash(data) != entry["sha"]:
                        raise RuntimeError(f"Bad animated reference: {path}")
                    target.parent.mkdir(parents=True, exist_ok=True)
                    target.write_bytes(data)
                candidates.extend({"family": "sprites", "path": path, "frame": frame}
                                  for frame, _ in self.frames("sprites", path))
            return candidates


    def match_alpha(tile, reference, matte):
        """Require every composited RGB5 pixel to agree; restore opaque bg-colored details."""
        bg = np.array(matte, dtype=np.uint8)
        cut = tile.copy()
        cut[np.all(cut[..., :3] == bg, axis=-1)] = 0
        candidate = reference.copy()
        candidate[np.all((candidate[..., :3] >> 3) == (bg >> 3), axis=-1)] = 0
        if signature(cut) != signature(candidate) or bbox(cut) is None:
            return None
        x0, y0, _, _ = bbox(cut)
        x1, y1, _, _ = bbox(candidate)
        dx, dy = x0 - x1, y0 - y1
        aligned = np.zeros_like(tile)
        yy, xx = np.nonzero(reference[..., 3])
        ty, tx = yy + dy, xx + dx
        if (ty < 0).any() or (tx < 0).any() or (ty >= tile.shape[0]).any() or (tx >= tile.shape[1]).any():
            return None
        aligned[ty, tx] = reference[yy, xx]
        composite = aligned[..., :3].copy()
        composite[aligned[..., 3] == 0] = bg
        if not np.array_equal(composite >> 3, tile[..., :3] >> 3):
            return None
        result = tile.copy()
        result[..., 3] = aligned[..., 3]
        result[result[..., 3] == 0] = 0
        return result


    def legacy_cells(generation):
        module = load_notebook("02-sprite-lora/prepare_dataset.ipynb", name="legacy_sprite_cutter")
        return module.sheet_cells(generation)


    def get_tiles(raw, catalog):
        tiles, sources = [], []
        known = set(catalog["duplicate_files"])
        for group in catalog["battle_groups"]:
            for name, (view, shiny) in group["sheets"].items():
                known.add(name)
                with Image.open(raw / name) as image:
                    a = np.array(image.convert("RGBA"))
                if a.shape[1] != 768 or a.shape[0] % 103:
                    raise ValueError(f"Unexpected grid: {name}")
                for r in range(a.shape[0] // 103):
                    # The following seven pixels contain tiny dex-number/form labels;
                    # they are outside every sprite crop (black text on transparency).
                    for c in range(8):
                        tile = a[r * 103:r * 103 + 96, c * 96:c * 96 + 96].copy()
                        if not tile[..., 3].any():
                            continue
                        source_label = decode_label(a[r * 103 + 96:r * 103 + 103, c * 96:c * 96 + 96, 3])
                        if not source_label[:3].isdigit():
                            raise ValueError(f"Invalid source label: {name}:{r}:{c}: {source_label}")
                        tiles.append({"sheet": name, "row": r, "column": c, "view": view, "shiny": shiny,
                                      "region": group["region"], "family": "sprites", "tile": tile,
                                      "matte_rgb": list(Counter(map(tuple, np.concatenate((tile[0,:,:3], tile[-1,:,:3], tile[:,0,:3], tile[:,-1,:3])).tolist())).most_common(1)[0][0]),
                                      "source_label": source_label, "source_species_id": int(source_label[:3]),
                                      "species_range": group["species_range"], "crop_xywh": [c * 96, r * 103, 96, 96]})
                sources.append({"file": name, "sha256": sha((raw / name).read_bytes()), "size": [a.shape[1], a.shape[0]],
                                "view": view, "shiny": shiny, "region": group["region"]})
        for name, (generation, shiny) in catalog["legacy_sheets"].items():
            known.add(name)
            with Image.open(raw / name) as image:
                a = np.array(image.convert("RGBA"))
            for i, (species_id, form) in enumerate(legacy_cells(generation)):
                r, c = divmod(i, 10)
                tile = a[5 + r * 69:69 + r * 69, 5 + c * 69:69 + c * 69].copy()
                tiles.append({"sheet": name, "row": r, "column": c, "view": "front", "shiny": shiny,
                              "region": generation.split(" /")[0], "family": "emerald", "tile": tile,
                              "matte_rgb": [165, 235, 255],
                              "expected_species": species_id, "expected_form": form, "pose_variant": i % 2,
                              "crop_xywh": [5 + c * 69, 5 + r * 69, 64, 64]})
            sources.append({"file": name, "sha256": sha((raw / name).read_bytes()), "size": [a.shape[1], a.shape[0]],
                            "view": "front", "shiny": shiny, "region": generation.split(" /")[0]})
        for duplicate, canonical in catalog["duplicate_files"].items():
            if sha((raw / duplicate).read_bytes()) != sha((raw / canonical).read_bytes()):
                raise ValueError(f"Declared duplicate changed: {duplicate}")
            sources.append({"file": duplicate, "duplicate_of": canonical, "sha256": sha((raw / duplicate).read_bytes())})
        unexpected = sorted(p.name for p in raw.glob("*") if p.is_file() and p.name not in known and not p.name.endswith(":Zone.Identifier"))
        if unexpected:
            raise ValueError(f"Uncatalogued raw files: {unexpected}")
        return tiles, sources


    def palette(a):
        counts = Counter(map(tuple, a[a[..., 3] > 0, :3].tolist()))
        colors = Counter()
        for rgb, count in counts.items():
            h, s, v = colorsys.rgb_to_hsv(*(c / 255 for c in rgb))
            if v < 0.18:
                color = "black"
            elif s < 0.16:
                color = "white" if v > 0.85 else "gray"
            else:
                hue = h * 360
                if hue < 15 or hue >= 345:
                    color = "pink" if v > 0.7 and s < 0.65 else "red"
                elif hue < 45:
                    color = "brown" if v < 0.72 else "orange"
                elif hue < 70:
                    color = "yellow"
                elif hue < 165:
                    color = "green"
                elif hue < 195:
                    color = "cyan"
                elif hue < 260:
                    color = "blue"
                elif hue < 290:
                    color = "purple"
                else:
                    color = "pink"
            colors[color] += count
        total = sum(counts.values())
        return [{"rgb": list(rgb), "hex": "#" + "".join(f"{c:02X}" for c in rgb), "pixels": n,
                 "fraction": round(n / total, 6)} for rgb, n in counts.most_common()], [c for c, n in colors.most_common() if n / total >= .08][:3]


    def definition(ref, entity, tile):
        species_id, form_id, gender = entity
        species = ref.species[species_id]
        name = ref.names[species_id, "9"]["name"]
        french = ref.names[species_id, "5"]["name"]
        form = next((f for f in ref.forms.values() if int(f["id"]) == form_id), None)
        form_name = ref.form_names.get(form_id, {}).get("form_name") or (form["form_identifier"].replace("-", " ") if form else "")
        if species_id == 351 and not form_name:
            form_name = "Normal Form"
        types = ref.mon_types[int(form["pokemon_id"]) if form else species_id]
        if species_id == 493 and form:
            types = [form["form_identifier"]]
        return {"species_id": species_id, "name_en": name, "name_fr": french,
                "genus_en": ref.names[species_id, "9"]["genus"], "form_id": form_id,
                "form_identifier": form["identifier"] if form else species["identifier"], "form_name_en": form_name,
                "form_name_fr": ref.form_names_fr.get(form_id, {}).get("form_name") or form_name,
                "gender": gender or ("genderless" if species["gender_rate"] == "-1" else "unspecified"),
                "palette": "shiny" if tile["shiny"] else "normal", "view": tile["view"],
                "sprite_style": "emerald" if tile["family"] == "emerald" else "black-white",
                "body_shape_taxonomy": ref.shapes[int(species["shape_id"])],
                "body_shape_taxonomy_description": ref.shape_descriptions[int(species["shape_id"])],
                "types_taxonomy": types,
                "taxonomy_is_not_a_visual_annotation": True}


    def caption(d):
        parts = ["sharm_monster, pixel art monster sprite", d["name_en"]]
        if d["form_name_en"] and d["form_name_en"].lower() not in ("male", "female"):
            parts.append(d["form_name_en"])
        if d["gender"] in ("male", "female"):
            parts.append(d["gender"])
        parts.extend([f"{d['palette']} palette", "front view" if d["view"] == "front" else "back view",
                      "Emerald-style battle sprite" if d["sprite_style"] == "emerald" else "Black-and-White-style battle sprite"])
        return ", ".join(parts) + ". Single complete character, crisp pixel edges, transparent background."


    def run(parameters=None):
        args = parameter_namespace({'raw_dir': HERE / 'dataset/raw', 'reference_dir': HERE / 'dataset/reference', 'output_dir': HERE / 'dataset/prepared/monster-clean-v2', 'catalog': HERE / 'monster_sources.json'}, parameters, required=[], converters={'raw_dir': Path, 'reference_dir': Path, 'output_dir': Path, 'catalog': Path}, choices={}, groups=[])
        output = args.output_dir
        if (output / "dataset.json").exists():
            raise FileExistsError("Use a new version directory; existing prepared datasets are immutable.")
        if output.resolve() == args.raw_dir.resolve() or args.raw_dir.resolve() in output.resolve().parents:
            raise ValueError("Output must be outside raw sources")
        catalog = json.loads(args.catalog.read_text())
        reference = Reference(args.reference_dir)
        index = reference.index()
        tiles, sources = get_tiles(args.raw_dir, catalog)
        print(f"Source cells: {len(tiles)}", flush=True)
        for tile in tiles:
            cut = tile["tile"].copy()
            cut[np.all(cut[..., :3] == tile["matte_rgb"], axis=-1)] = 0
            matches = index.get(signature(cut), [])
            matches = [m for m in matches if m["family"] == tile["family"]
                       and ("shiny" in Path(m["path"]).parts) == tile["shiny"]
                       and ("back" in Path(m["path"]).parts) == (tile["view"] == "back")]
            if "source_species_id" in tile:
                if any(reference.entity(m["path"])[0] != tile["source_species_id"] for m in matches):
                    raise ValueError(f"Pixel match contradicts printed label: {tile['sheet']}:{tile['source_label']}")
                suffix = tile["source_label"].split("-")[-1]
                if suffix in ("F", "M"):
                    gender = {"F": "female", "M": "male"}[suffix]
                    matches = [m for m in matches if reference.entity(m["path"])[2] == gender]
            if "expected_species" in tile:
                matches = [m for m in matches if reference.entity(m["path"])[0] == tile["expected_species"]]
            else:
                lo, hi = tile["species_range"]
                matches = [m for m in matches if lo <= reference.entity(m["path"])[0] <= hi]
            tile["matches"] = matches

        # Concordance between front/back and normal/shiny at the same source cell.
        groups = defaultdict(list)
        for tile in tiles:
            if tile["family"] == "sprites":
                groups[tile["region"], tile["row"], tile["column"]].append(tile)
        for key, group in groups.items():
            sets = [{reference.entity(m["path"]) for m in t["matches"]} for t in group if t["matches"]]
            entities = set.intersection(*sets) if sets else set()
            if not entities and sets:
                raise ValueError(f"Conflicting cross-sheet identities: {key}: {sets}")
            fallback = catalog["fallback_identities"].get(":".join(map(str, key)))
            labels = {t["source_label"] for t in group}
            if len(labels) != 1:
                raise ValueError(f"Different printed labels in a paired group: {key}: {labels}")
            label = next(iter(labels))
            suffix = label.split("-")[-1]
            type_order = catalog.get("form_label_type_order", {}).get(label[:3].lstrip("0"))
            if type_order is not None and suffix.isdigit() and "-" in label:
                type_name = reference.types[type_order[int(suffix)]]
                canonical_form = reference.forms[f"{reference.species[int(label[:3])]['identifier']}-{type_name}"]
                entities = {e for e in entities if e[1] == int(canonical_form["id"])}
                if sets and not entities:
                    raise ValueError(f"Printed type form contradicts reference matches: {key}: {label}")
            if not entities and ("-" not in label or suffix in ("F", "M")):
                sid = int(label[:3])
                entities = {(sid, int(reference.default_forms[sid]["id"]), {"F":"female", "M":"male"}.get(suffix))}
            if not entities and fallback:
                sid = fallback["species_id"]
                forms = reference.default_forms[sid]
                sex = fallback.get("gender")
                entities = {(sid, int(forms["id"]), sex)}
            if len(entities) != 1:
                raise ValueError(f"Unresolved source identity: {key}: {label}: {entities}")
            for tile in group:
                tile["entities"] = entities

        accepted, deferred = [], []
        for count, tile in enumerate(tiles, 1):
            candidates = tile["matches"]
            entities = tile.get("entities") or {reference.entity(m["path"]) for m in candidates}
            if entities:
                candidates = [m for m in candidates if reference.entity(m["path"]) in entities]
            if not candidates and tile["family"] == "sprites" and entities:
                # First recheck static references: a foreground color can equal the cell background.
                for path in reference.tree_entries:
                    if path.startswith("animated/") or not path.endswith(".png"):
                        continue
                    if ("back" in Path(path).parts) != (tile["view"] == "back") or ("shiny" in Path(path).parts) != tile["shiny"]:
                        continue
                    if path in reference.local_paths and reference.entity(path) in entities:
                        candidates.append({"family": "sprites", "path": path, "frame": 0})
            match, clean = None, None

            def attempt(possible):
                for item in possible:
                    for frame, a in reference.frames(item["family"], item["path"]):
                        if frame != item["frame"]:
                            continue
                        result = match_alpha(tile["tile"], a, tile["matte_rgb"])
                        if result is not None:
                            return item, result
                return None, None

            match, clean = attempt(candidates)
            if match is None and tile["family"] == "sprites" and entities:
                match, clean = attempt(reference.animated_candidates(entities, tile["view"], tile["shiny"]))
            # Unverified cells are visible in the review, never silently accepted as clean.
            if match is None:
                tile["entities"] = entities
                deferred.append(tile)
            else:
                entity = reference.entity(match["path"])
                if tile["family"] == "emerald":
                    entity = (entity[0], entity[1], None)  # Gen III artwork has no male/female sprite pair.
                tile.update({"clean": clean, "match": match, "entity": entity, "verification": "exact_rgb5_composite"})
                accepted.append(tile)
            if count % 250 == 0 or count == len(tiles):
                print(f"Reconciled {count}/{len(tiles)}; verified {len(accepted)}; pending {len(deferred)}", flush=True)

        # A second palette can verify the exact opacity mask of the same pose. If
        # that is unavailable, use a key color only when absent from the reference
        # foreground palette. Never erase a color that may be part of the monster.
        peers = defaultdict(list)
        for tile in accepted:
            peers[tile["family"], tile["region"], tile["row"], tile["column"], tile["view"]].append(tile)
        rejected = []
        for tile in deferred:
            raw = tile["tile"]
            bg = np.array(tile["matte_rgb"], dtype=np.uint8)
            observed = np.any(raw[..., :3] != bg, axis=-1) & (raw[..., 3] > 0)
            clean, entity, proof = None, None, None
            key = (tile["family"], tile["region"], tile["row"], tile["column"], tile["view"])
            for peer in peers[key]:
                mask = peer["clean"][..., 3] > 0
                if not (observed & ~mask).any():
                    clean = raw.copy()
                    clean[..., 3] = peer["clean"][..., 3]
                    entity = peer["entity"]
                    proof = {"paired_source": peer["sheet"], "paired_verification": peer["verification"],
                             "paired_reference": peer["match"]}
                    tile["verification"] = "paired_palette_exact_alpha"
                    break
            if clean is None and tile["family"] == "emerald":
                # The cyan matte is not a representable v<<3 color; all foreground
                # channels must be multiples of eight. The audited legacy table
                # supplies the identity, independently of modern sprite animation.
                if np.all(raw[observed, :3] % 8 == 0) and np.any(bg % 8 != 0):
                    sid = tile["expected_species"]
                    form_word = tile["expected_form"].split(" ")[0]
                    identifier = f"castform-{form_word}" if sid == 351 and form_word not in ("", "normal") else None
                    form = reference.forms[identifier] if identifier else reference.default_forms[sid]
                    entity = (sid, int(form["id"]), None)
                    clean = raw.copy()
                    clean[~observed] = 0
                    proof = {"identity": "audited legacy layout", "alpha": "non-15bit cyan matte; all foreground channels v<<3"}
                    tile["verification"] = "audited_legacy_grid_unique_matte"
            if clean is None and tile["family"] == "sprites":
                entities = tile.get("entities", set())
                palette_colors = set()
                for (family, path), frames in list(reference.frames_cache.items()):
                    if family != "sprites":
                        continue
                    if ("back" in Path(path).parts) != (tile["view"] == "back") or ("shiny" in Path(path).parts) != tile["shiny"]:
                        continue
                    if reference.entity(path) not in entities:
                        continue
                    for _, a in frames:
                        palette_colors.update(map(tuple, (a[a[..., 3] > 0, :3] >> 3).tolist()))
                actual = set(map(tuple, (raw[observed, :3] >> 3).tolist()))
                if entities and palette_colors and tuple(bg >> 3) not in palette_colors:
                    clean = raw.copy()
                    clean[~observed] = 0
                    entity = sorted(entities, key=str)[0]
                    proof = {"identity": "printed dex/form label + canonical taxonomy; cross-sheet concordance when available", "alpha": "matte absent from canonical foreground palette",
                             "foreground_palette_rgb5": sorted(palette_colors),
                             "source_colors_not_in_reference": sorted(actual - palette_colors),
                             "pixel_geometry_is_not_an_exact_reference_match": True}
                    tile["verification"] = "source_label_reference_palette_unique_matte"
            if clean is not None:
                clean[clean[..., 3] == 0] = 0
                tile.update({"clean": clean, "entity": entity, "match": proof})
                accepted.append(tile)
            else:
                record = {k: v for k, v in tile.items() if k not in ("tile", "matches", "entities")}
                record["reason"] = "opacity not sufficiently verified; never used for training"
                if tile.get("entities"):
                    record["proposed_definition"] = definition(reference, sorted(tile["entities"], key=str)[0], tile)
                record["_tile"] = raw
                rejected.append(record)

        dump(args.reference_dir / "reconciliation.json", {"accepted": len(accepted), "rejected": [{k:v for k,v in r.items() if k != "_tile"} for r in rejected]})
        build_output(output, args, reference, sources, accepted, rejected)


    def build_output(output, args, ref, sources, accepted, rejected):
        output.mkdir(parents=True, exist_ok=True)
        for folder in ("images_128", "train_256", "train_512"):
            (output / folder).mkdir(exist_ok=True)
        pool = ThreadPoolExecutor(max_workers=8)
        writes = []

        def save_images(image, filename, data):
            (output / "images_128" / filename).write_bytes(data)
            for size in (256, 512):
                image.resize((size, size), Image.Resampling.NEAREST).save(output / f"train_{size}" / filename)
        source_hashes = {r["file"]: r["sha256"] for r in sources}
        manifest, duplicates, by_pixels, inventory = [], [], {}, []
        for tile in accepted:
            clean = tile["clean"]
            # Integer fitting with a four-pixel margin preserves the source grid and
            # aspect ratio, without fractional resizing or smoothing.
            box = bbox(clean)
            x0, y0, x1, y1 = box
            content = clean[y0:y1, x0:x1].copy()
            scale = min(4, 120 // content.shape[1], 120 // content.shape[0])
            content = np.repeat(np.repeat(content, scale, axis=0), scale, axis=1)
            normalized = np.zeros((128, 128, 4), dtype=np.uint8)
            ox, oy = (128 - content.shape[1]) // 2, (128 - content.shape[0]) // 2
            normalized[oy:oy + content.shape[0], ox:ox + content.shape[1]] = content
            pixel_sha = sha(normalized.tobytes())
            d = definition(ref, tile["entity"], tile)
            variant = {k:d[k] for k in ("species_id","form_id","form_identifier","form_name_en","form_name_fr","gender","palette","view","sprite_style")}
            origin = {"sheet": tile["sheet"], "row": tile["row"], "column": tile["column"],
                      "crop_xywh": tile["crop_xywh"], "source_sha256": source_hashes[tile["sheet"]],
                      "printed_label": tile.get("source_label"), "matte_rgb": tile["matte_rgb"],
                      "verification":tile["verification"], "reference": tile["match"]}
            item = {**origin, "species_id":d["species_id"],"definition":variant,"status":"included"}
            if pixel_sha in by_pixels:
                existing = by_pixels[pixel_sha]
                if existing["species_id"] != d["species_id"]:
                    raise ValueError("Cross-species pixel duplicate: requires explicit grouping review")
                existing["source_cells"].append(origin)
                if variant not in existing["equivalent_variants"]:
                    existing["equivalent_variants"].append(variant)
                inventory.append({**item,"sample_id":existing["sample_id"],"status":"deduplicated"})
                duplicates.append({"sheet": tile["sheet"], "row": tile["row"], "column": tile["column"],
                                   "duplicate_of": existing["sample_id"], "equivalent_variant":variant,
                                   "reason": "identical centered RGBA pixels"})
                continue
            sid = f"{d['species_id']:04d}-{d['form_identifier']}-{d['gender']}-{d['palette']}-{d['view']}-{d['sprite_style']}-{pixel_sha[:10]}"
            image = Image.fromarray(normalized)
            folder = output / "images_128"
            folder.mkdir(exist_ok=True)
            filename = sid + ".png"
            buffer = io.BytesIO()
            image.save(buffer, format="PNG")
            data = buffer.getvalue()
            writes.append(pool.submit(save_images, image, filename, data))
            colors, color_names = palette(clean)
            d.update({"palette_rgb": colors, "palette_color_families_measured": color_names,
                      "description_en": caption(d), "description_fr":
                      f"Sprite de {d['name_fr']}, vue {'de face' if d['view']=='front' else 'de dos'}, "
                      f"forme {d['form_name_fr'] or 'standard'}, "
                      f"palette {'shiny' if d['palette']=='shiny' else 'normale'}, "
                      f"sexe {dict(male='mâle', female='femelle', genderless='asexué', unspecified='non spécifié')[d['gender']]}, "
                      f"style {d['sprite_style']}, personnage entier sur fond transparent.",
                      "description_provenance": "canonical identity + exact image/reference reconciliation; no invented lore"})
            split_group = str(d["species_id"])
            # Same existing species split rule, with the canonical JSON-string digest.
            group_hash = sha(json.dumps(split_group, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode())
            split = "validation" if int(group_hash[:8], 16) % 10 == 0 else "train"
            row = {"sample_id": sid, "file_name": filename, "file": f"images_128/{filename}",
                   "text": d["description_en"], "species_id": d["species_id"], "species": d["name_en"],
                   "group_id": split_group, "split": split, "label": d["name_en"], "definition": d,
                   "pixel_sha256": pixel_sha, "image_sha256": sha(data),
                   "native_cell_size": tile["tile"].shape[0], "native_bbox_xyxy": list(box),
                   "target_bbox_xyxy": [ox, oy, ox + content.shape[1], oy + content.shape[0]],
                   "integer_scale_from_native": scale, "foreground_pixels": int((normalized[..., 3] > 0).sum()),
                   "verification": tile["verification"], "source_cells": [origin],"equivalent_variants":[variant]}
            manifest.append(row)
            inventory.append({**item,"sample_id":sid})
            by_pixels[pixel_sha] = row
            if len(manifest) % 500 == 0:
                print(f"Prepared unique images: {len(manifest)}", flush=True)
        for future in writes:
            future.result()
        pool.shutdown()
        jsonl(output / "manifest.jsonl", manifest)
        for size in (128, 256, 512):
            folder = output / ("images_128" if size == 128 else f"train_{size}")
            jsonl(folder / "metadata.jsonl", [{"file_name": r["file_name"], "text": r["text"],
                  "group_id": r["group_id"], "species_id": r["species_id"], "split": r["split"],
                  "sample_id": r["sample_id"], "label": r["label"]} for r in manifest])
            for split in ("train", "validation"):
                jsonl(folder / f"{split}.jsonl", [{"file_name": r["file_name"], "text": r["text"],
                      "group_id": r["group_id"], "species_id": r["species_id"]} for r in manifest if r["split"] == split])
        with (output / "manifest.csv").open("w", encoding="utf-8", newline="") as stream:
            fields = ["file", "species_id", "species", "split", "caption", "shiny", "form", "view", "style"]
            writer = csv.DictWriter(stream, fields)
            writer.writeheader()
            for row in manifest:
                d = row["definition"]
                writer.writerow({"file": row["file"], "species_id": row["species_id"], "species": row["species"],
                                 "split": row["split"], "caption": row["text"], "shiny": d["palette"] == "shiny",
                                 "form": d["form_identifier"], "view": d["view"], "style": d["sprite_style"]})
        captions = sorted(set(r["text"] for r in manifest))
        jsonl(output / "caption-corpus.jsonl", [{"id": sha(json.dumps(text, ensure_ascii=False, sort_keys=True,
              separators=(",", ":")).encode()), "text": text} for text in captions])
        definitions = {}
        for row in manifest:
            d = row["definition"]
            species = definitions.setdefault(str(row["species_id"]), {"name_en": d["name_en"],
                        "name_fr": d["name_fr"], "genus_en":d["genus_en"],"species_id":row["species_id"],
                        "body_shape_taxonomy":d["body_shape_taxonomy"],
                        "taxonomy_is_not_a_visual_annotation":True,"forms":{},"samples":[]})
            species["samples"].append(row["sample_id"])
            for variant in row["equivalent_variants"]:
                form = species["forms"].setdefault(variant["form_identifier"],
                    {"id":variant["form_id"],"name_en":variant["form_name_en"],"name_fr":variant["form_name_fr"],"samples":[]})
                form["samples"].append(row["sample_id"])
        dump(output / "definitions.json", definitions)
        dump(output / "sources.json", {"sources": sources, "references": ref.lock, "source_catalog": json.loads(args.catalog.read_text())})
        dump(output / "duplicates.json", duplicates)
        quarantine = output / "quarantine"
        quarantine.mkdir(exist_ok=True)
        for record in rejected:
            pixels = record.pop("_tile")
            name = f"{Path(record['sheet']).stem}-r{record['row']:02d}-c{record['column']:02d}.png"
            Image.fromarray(pixels).save(quarantine / name)
            record["original_cell_file"] = f"quarantine/{name}"
            inventory.append({**record,"status":"quarantine"})
        dump(output / "unresolved.json", rejected)
        jsonl(output / "source_cells.jsonl", inventory)
        report = {"schema_version": 1, "dataset_version": output.name, "preparation_version":VERSION,"raw_files": len(sources),
                  "unique_source_sheets": len(sources) - sum("duplicate_of" in r for r in sources),
                  "verified_source_cells": len(accepted), "unique_images": len(manifest),
                  "duplicate_cells": len(duplicates), "unresolved_cells": len(rejected),
                  "species": len(definitions), "by_style": dict(Counter(r["definition"]["sprite_style"] for r in manifest)),
                  "by_view": dict(Counter(r["definition"]["view"] for r in manifest)),
                  "by_palette": dict(Counter(r["definition"]["palette"] for r in manifest)),
                  "by_split": dict(Counter(r["split"] for r in manifest)),
                  "by_verification": dict(Counter(r["verification"] for r in manifest)),
                  "manifest_sha256": sha((output / "manifest.jsonl").read_bytes()),
                  "preparation_script_sha256": notebook_hash(Path(__file__)),
                  "preparation_notebook": "02-sprite-lora/prepare_monster_dataset.ipynb",
                  "recipe_hash_convention": "library-cells-v1",
                  "training_started": False, "human_review": False, "reference_matching_is_not_human_review": True}
        make_review(output, manifest)
        dump(output / "dataset.json", report)
        print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)


    def make_review(output, manifest):
        review = output / "review"
        review.mkdir(exist_ok=True)
        links = []
        for start in range(0, len(manifest), 80):
            page = start // 80 + 1
            rows = manifest[start:start + 80]
            cards = []
            canvas = Image.new("RGB", (8 * 160, 10 * 160), "#DDDDE2")
            draw = ImageDraw.Draw(canvas)
            for i, r in enumerate(rows):
                x, y = (i % 8) * 160, (i // 8) * 160
                for gy in range(0, 128, 8):
                    for gx in range(0, 128, 8):
                        draw.rectangle((x + 16 + gx, y + gy, x + 23 + gx, y + gy + 7),
                                       fill="#CCCCCC" if (gx // 8 + gy // 8) % 2 else "#EEEEEE")
                with Image.open(output / r["file"]) as im:
                    canvas.paste(im, (x + 16, y), im)
                d = r["definition"]
                draw.text((x + 3, y + 129), f"{r['species_id']} {r['species']}", fill="black")
                draw.text((x + 3, y + 143), f"{d['view']} {d['palette']} {d['gender'][:1]}", fill="black")
                cards.append(f"<article><img loading='lazy' src='../{html.escape(r['file'])}'>"
                             f"<b>{html.escape(r['sample_id'])}</b><p>{html.escape(r['text'])}</p>"
                             f"<details><summary>Definition / provenance</summary><pre>{html.escape(json.dumps(r, ensure_ascii=False, indent=2))}</pre></details></article>")
            canvas.save(review / f"contact-{page:03d}.png")
            (review / f"page-{page:03d}.html").write_text("<!doctype html><meta charset='utf-8'><style>"
                "body{font:14px system-ui;background:#eee}main{display:grid;grid-template-columns:repeat(5,1fr);gap:12px}"
                "article{background:white;padding:12px;overflow-wrap:anywhere}img{width:256px;height:256px;image-rendering:pixelated;"
                "background:repeating-conic-gradient(#ddd 0% 25%,#fff 0% 50%) 0/16px 16px}pre{white-space:pre-wrap}</style>"
                f"<h1>Monster dataset — page {page}</h1><a href='index.html'>Index</a><main>{''.join(cards)}</main>", encoding="utf-8")
            links.append(f"<li><a href='page-{page:03d}.html'>Page {page}</a> — <a href='contact-{page:03d}.png'>Planche contact</a></li>")
        (review / "index.html").write_text("<!doctype html><meta charset='utf-8'><h1>Sharm Image Monster Pixel Art</h1>"
            "<p>Galerie de contrôle, sans annotation humaine inventée. Images à 128px natifs, affichage nearest-neighbor.</p>"
            f"<ul>{''.join(links)}</ul>", encoding="utf-8")



## monster.prepare_training_cache

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_bdf488a3cc80be92':


    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import csv


    import json


    import sys


    from pathlib import Path


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    import numpy as np


    import torch


    from PIL import Image


    from sharm_image21.core import (ARTIFACTS, CACHE, ORIGINAL, ROOT, TEACHER_REVISION,
                                    FeatureCache, Projector, atomic_json, digest, file_hash, read_jsonl)


    @torch.no_grad()
    def run(parameters=None):
        args = parameter_namespace({'resolution': 512, 'dataset_root': ROOT / 'dataset', 'cache_root': CACHE, 'projector_dir': ARTIFACTS / 'projector', 'base_validation': ARTIFACTS / 'base-validation.json', 'output_report': None}, parameters, required=[], converters={'resolution': int, 'dataset_root': Path, 'cache_root': Path, 'projector_dir': Path, 'base_validation': Path, 'output_report': Path}, choices={}, groups=[])
        resolution = args.resolution
        if resolution < 64 or resolution % 32:
            raise ValueError("Résolution multiple de 32 requise.")
        projector_dir = args.projector_dir
        fingerprint = file_hash(projector_dir / "model.safetensors")
        gate = json.loads(args.base_validation.read_text())
        if not gate.get("passed") or gate["projector_sha256"] != fingerprint:
            raise RuntimeError("La base Sharm doit être validée avant le cache LoRA.")
        dataset = args.dataset_root / f"train_{resolution}"
        if not (dataset / "metadata.jsonl").is_file():
            raise FileNotFoundError(f"Préparer le dataset à {resolution}px avant l'entraînement.")
        with (args.dataset_root / "manifest.csv").open(encoding="utf-8") as stream:
            species = {Path(row["file"]).name: row["species_id"] for row in csv.DictReader(stream)}
        rows = []
        for row in read_jsonl(dataset / "metadata.jsonl"):
            image_path = dataset / row["file_name"]
            image_sha = file_hash(image_path)
            species_id = species[row["file_name"]]
            split = "validation" if int(digest(species_id)[:8], 16) % 10 == 0 else "train"
            if str(row.get("group_id", species_id)) != str(species_id) or row.get("split", split) != split:
                raise ValueError(f"Split ou groupe incompatible avec le manifeste : {row['file_name']}")
            rows.append({**row, "id": digest({"name": row["file_name"], "sha256": image_sha}),
                         "image_sha256": image_sha, "species_id": species_id, "caption_id": digest(row["text"]),
                         "split": split})
        identity = {"dataset_sha256": digest(rows), "resolution": resolution,
                    "projector_sha256": fingerprint, "teacher_revision": TEACHER_REVISION}
        student = FeatureCache(args.cache_root / "student")
        caption_rows = list({row["caption_id"]: row for row in rows}.values())
        missing = [row["caption_id"] for row in caption_rows if row["caption_id"] not in student]
        if missing:
            raise RuntimeError(f"{len(missing)} légendes sprites sont absentes du cache du 4B ; compléter le corpus.")
        projected = FeatureCache(args.cache_root / f"sprites-{resolution}-text", {**identity, "student": student.index["identity"]})
        model = Projector.load(projector_dir, "cuda").eval()
        buffer = []
        for row in caption_rows:
            key = row["caption_id"]
            if key in projected:
                continue
            with torch.autocast("cuda", dtype=torch.bfloat16):
                values = model(student.get(key).cuda()).to(torch.bfloat16).cpu()
            if not torch.isfinite(values).all():
                raise RuntimeError("Embeddings projetés non finis.")
            buffer.append((key, values, student.index["entries"][key]["token_ids"]))
            if len(buffer) == 32:
                projected.write(buffer)
                buffer = []
        projected.write(buffer)
        del model
        torch.cuda.empty_cache()
        latents = FeatureCache(args.cache_root / f"sprites-{resolution}-latents",
                               {**identity, "vae_config": file_hash(ORIGINAL / "vae/config.json"),
                                "sampling": "per-image-seed-42-bf16-rgba-normalized-v1"})
        pending = [row for row in rows if row["id"] not in latents]
        if pending:
            from diffusers import AutoencoderKLQwenImage21
            vae = AutoencoderKLQwenImage21.from_pretrained(ORIGINAL, subfolder="vae", torch_dtype=torch.bfloat16,
                                                          local_files_only=True).to("cuda").eval()
            vae.requires_grad_(False)
            mean = torch.tensor(vae.config.latents_mean, device="cuda", dtype=torch.bfloat16).view(1, -1, 1, 1, 1)
            std = torch.tensor(vae.config.latents_std, device="cuda", dtype=torch.bfloat16).view(1, -1, 1, 1, 1)
            buffer = []
            for index, row in enumerate(pending):
                with Image.open(dataset / row["file_name"]) as image:
                    if image.mode != "RGBA" or image.size != (resolution, resolution):
                        raise ValueError(f"Image RGBA à la taille attendue requise : {row['file_name']}")
                    pixels = np.array(image, copy=True)
                blue = (pixels[..., :3] == (165, 235, 255)).all(axis=-1) & (pixels[..., 3] != 0)
                if blue.any() or not (pixels[..., 3] == 0).any():
                    raise ValueError(f"Fond bleu opaque ou alpha absent : {row['file_name']}")
                tensor = torch.from_numpy(pixels).permute(2, 0, 1).unsqueeze(0).unsqueeze(2)
                tensor = tensor.to("cuda", dtype=torch.bfloat16)/127.5-1
                generator = torch.Generator(device="cuda").manual_seed(42+int(row["id"][:8], 16))
                encoded = vae.encode(tensor).latent_dist.sample(generator=generator)
                normalized = ((encoded-mean)/std)[0].cpu().contiguous()
                if not torch.isfinite(normalized).all():
                    raise RuntimeError("Latents non finis.")
                buffer.append((row["id"], normalized, []))
                if len(buffer) == 16 or index+1 == len(pending):
                    latents.write(buffer)
                    buffer = []
                    print(f"Latents sprites : {len(latents.index['entries'])}/{len(rows)}", flush=True)
        latents.require(rows)
        result = {"identity": identity, "rows": rows, "text_cache": str(projected.directory),
                  "latent_cache": str(latents.directory), "dataset_dir": str(dataset)}
        default_report = ARTIFACTS / ("monster" if args.dataset_root.resolve() != (ROOT / "dataset").resolve() else "") / f"sprite-training-{resolution}.json"
        atomic_json(args.output_report or default_report, result)
        print(json.dumps({"images": len(rows), "captions": len(caption_rows), "identity": identity}), flush=True)



## monster.train_lora

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_d9e577f6f19e3c79':


    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import json


    import random


    import sys


    import time


    import uuid


    from pathlib import Path


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    import torch


    from torch.nn import functional as F


    from sharm_image21.core import ARTIFACTS, ORIGINAL, WEIGHTS, FeatureCache, atomic_json, digest, seed_all


    from sharm_image21.diffusion import load_transformer, predict_flow


    def save_adapter(model, folder):
        from diffusers import QwenImage21Pipeline
        from diffusers.training_utils import _collate_lora_metadata
        from peft.utils import get_peft_model_state_dict
        QwenImage21Pipeline.save_lora_weights(save_directory=folder,
            transformer_lora_layers=get_peft_model_state_dict(model), **_collate_lora_metadata({"transformer": model}))


    def get_pair(row, texts, latents):
        return latents.get(row["id"]).unsqueeze(0).cuda(), texts.get(row["caption_id"]).unsqueeze(0).cuda()


    @torch.no_grad()
    def evaluate(model, rows, texts, latents):
        model.eval()
        losses = []
        for index, row in enumerate(rows[:16]):
            clean, embeddings = get_pair(row, texts, latents)
            noise = torch.randn(clean.shape, device="cuda", dtype=clean.dtype,
                                generator=torch.Generator(device="cuda").manual_seed(8400+index))
            noisy = 0.5*clean+0.5*noise
            with torch.autocast("cuda", dtype=torch.bfloat16):
                predicted = predict_flow(model, noisy, embeddings, torch.tensor([0.5], device="cuda", dtype=torch.bfloat16))
            losses.append(F.mse_loss(predicted.float(), (noise-clean).float()).item())
        return sum(losses)/len(losses)


    def run(parameters=None):
        args = parameter_namespace({'resolution': 512, 'training_cache': None, 'base_validation': ARTIFACTS / 'base-validation.json', 'steps': 500, 'accumulation': 4, 'rank': 8, 'learning_rate': 0.0001, 'checkpoint_every': 100, 'gradient_checkpointing': False, 'checkpointing_mode': 'alternating', 'output': WEIGHTS / 'sharm-image-2.1-sprites-128-lora'}, parameters, required=[], converters={'resolution': int, 'training_cache': Path, 'base_validation': Path, 'steps': int, 'accumulation': int, 'rank': int, 'learning_rate': float, 'checkpoint_every': int, 'output': Path}, choices={'checkpointing_mode': ('all', 'alternating', 'none')}, groups=[])
        if args.gradient_checkpointing:
            args.checkpointing_mode = "all"
        if min(args.steps, args.accumulation, args.rank, args.checkpoint_every) < 1 or args.learning_rate <= 0:
            raise ValueError("Pas, accumulation, rang et fréquence de checkpoint doivent être positifs.")
        seed_all(42)
        from diffusers import FlowMatchEulerDiscreteScheduler
        from peft import LoraConfig
        from peft import set_peft_model_state_dict
        import bitsandbytes as bnb
        from safetensors.torch import load_file

        data = json.loads((args.training_cache or ARTIFACTS / f"sprite-training-{args.resolution}.json").read_text())
        validation = json.loads(args.base_validation.read_text())
        if not validation.get("passed") or validation["projector_sha256"] != data["identity"]["projector_sha256"]:
            raise RuntimeError("Base validée et cache sprites de révisions différentes.")
        rows = data["rows"]
        training_rows = [row for row in rows if row["split"] == "train"]
        validation_rows = [row for row in rows if row["split"] == "validation"]
        if not training_rows or not validation_rows:
            raise ValueError("Un holdout par espèce est requis.")
        assert not ({r['species_id'] for r in training_rows} & {r['species_id'] for r in validation_rows})
        texts, latents = FeatureCache(Path(data["text_cache"])), FeatureCache(Path(data["latent_cache"]))
        run_config = {**{key: str(value) if isinstance(value, Path) else value for key,value in vars(args).items()},
                      "output": str(args.output), "cache_identity": data["identity"]}
        # Activation recomputation changes memory/time, not the training examples or
        # objective. Keep checkpoints compatible when toggling this runtime option.
        run_signature = digest({key: value for key, value in run_config.items()
                                if key not in ("gradient_checkpointing", "checkpointing_mode",
                                               "training_cache", "base_validation")})
        complete_path = args.output / "training-complete.json"
        if complete_path.exists():
            previous = json.loads(complete_path.read_text())
            if previous["run_signature"] != run_signature:
                raise RuntimeError("Un LoRA différent existe ; choisir un autre dossier de sortie.")
            print("LoRA déjà terminé.", flush=True)
            return
        args.output.mkdir(mode=0o700, parents=True, exist_ok=True)
        model = load_transformer(training=True, gradient_checkpointing=args.checkpointing_mode)
        model.add_adapter(LoraConfig(r=args.rank, lora_alpha=args.rank, init_lora_weights="gaussian",
                                     target_modules=["to_k", "to_q", "to_v", "to_out.0"]))
        parameters = [p for p in model.parameters() if p.requires_grad]
        optimizer = bnb.optim.AdamW8bit(parameters, lr=args.learning_rate, betas=(0.9, 0.999), weight_decay=0.01)
        scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(ORIGINAL, subfolder="scheduler", local_files_only=True)
        timesteps = scheduler.timesteps.to("cuda")
        sigmas = scheduler.sigmas.to(device="cuda", dtype=torch.bfloat16)
        checkpoint_pointer = args.output / "latest-checkpoint.json"
        first_step = 0
        if checkpoint_pointer.exists():
            folder = args.output / json.loads(checkpoint_pointer.read_text())["folder"]
            state = torch.load(folder / "training-state.pt", weights_only=True, map_location="cpu")
            if state["run_signature"] != run_signature:
                raise ValueError("Checkpoint LoRA incompatible avec cette configuration.")
            set_peft_model_state_dict(model, load_file(folder / "adapter-state.safetensors"), adapter_name="default")
            optimizer.load_state_dict(state["optimizer"])
            first_step = state["step"]
            torch.set_rng_state(state["torch_rng"])
            torch.cuda.set_rng_state(state["cuda_rng"])
            random.setstate(state["python_rng"])
        baseline = evaluate(model, validation_rows, texts, latents)
        started = time.monotonic()
        for step in range(first_step, args.steps):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            total_loss = 0.0
            for micro in range(args.accumulation):
                row = random.choice(training_rows)
                clean, embeddings = get_pair(row, texts, latents)
                noise = torch.randn_like(clean)
                index = torch.randint(len(timesteps), (1,), device="cuda")
                sigma = sigmas[index].view(1, 1, 1, 1, 1)
                noisy = ((1-sigma)*clean+sigma*noise).to(torch.bfloat16)
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    predicted = predict_flow(model, noisy, embeddings, timesteps[index]/1000)
                loss = F.mse_loss(predicted.float(), (noise-clean).float())
                if not torch.isfinite(loss):
                    raise RuntimeError("Perte LoRA non finie.")
                (loss/args.accumulation).backward()
                total_loss += loss.item()/args.accumulation
            grad_norm = torch.nn.utils.clip_grad_norm_(parameters, 1.0, error_if_nonfinite=True)
            if grad_norm == 0:
                raise RuntimeError("Aucun gradient LoRA ; vérifier le checkpointing et les adaptateurs.")
            optimizer.step()
            report = {"stage": "train_lora", "step": step+1, "steps": args.steps,
                      "loss": total_loss, "seconds": time.monotonic()-started,
                      "updated_at": time.time(), "gpu_peak_gib": torch.cuda.max_memory_allocated()/2**30}
            if (step+1) % 5 == 0 or step == first_step:
                atomic_json(ARTIFACTS / "lora-status.json", report)
                print(json.dumps(report), flush=True)
            if (step+1) % args.checkpoint_every == 0 or step+1 == args.steps:
                report["validation_flow_mse"] = evaluate(model, validation_rows, texts, latents)
                folder = args.output / f"checkpoint-{step+1}-{uuid.uuid4().hex[:8]}"
                staging = folder.with_name(folder.name + ".staging")
                save_adapter(model, staging)
                from peft.utils import get_peft_model_state_dict
                from safetensors.torch import save_file
                save_file({name: value.detach().cpu().contiguous() for name, value in get_peft_model_state_dict(model).items()},
                           staging / "adapter-state.safetensors")
                torch.save({"run_signature": run_signature, "step": step+1, "optimizer": optimizer.state_dict(),
                            "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state(),
                            "python_rng": random.getstate()}, staging / "training-state.pt")
                staging.replace(folder)
                atomic_json(checkpoint_pointer, {"folder": folder.name, "step": step+1, "run_signature": run_signature})
                with (args.output / "history.jsonl").open("a") as stream:
                    stream.write(json.dumps(report)+"\n")
        save_adapter(model, args.output)
        atomic_json(complete_path, {"run_signature": run_signature, "config": run_config,
                                    "baseline_validation_flow_mse": baseline, "finished_at": time.time(),
                                    "hf_repo_intent": "ethanzxv/Sharm-Image-2.1-Sprites-128-LoRA",
                                    "private": True, "published": False})
        print(f"LoRA terminé : {args.output}", flush=True)



## monster.validate_monster_base

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_fa14afbc712afbf9':
    import json
    from pathlib import Path
    from sharm_notebook_runtime import PROJECT, ARTIFACTS, read_json, sha256

    def run(parameters=None):
        p = dict(parameters or {})
        base = Path(p.get("base", ARTIFACTS / "spatial/release"))
        output = Path(p.get("output", ARTIFACTS / "monster/base-validation.json"))
        config = read_json(base / "sharm_config.json", {})
        manifest = read_json(base / "release_manifest.json", {})
        verification = read_json(base / "export-validation.json", {})
        review = read_json(base / "visual-review.json", {})
        predictions = read_json(base / "comparisons/spatial/predictions.json", {"rows": []})
        fingerprint = sha256(base / "projector/model.safetensors")
        required = {r["id"] for r in predictions["rows"] if r["method"].startswith("candidate_")}
        conditions = {
            "spatial_base": config.get("pipeline_format") == "spatial-v1",
            "matching_projector": fingerprint == config.get("projector_sha256") == manifest.get("projector_sha256") == verification.get("projector_sha256") == review.get("projector_sha256"),
            "exact_export": verification.get("passed") is True and verification.get("loader_from_release") is True and verification.get("projector_roundtrip_exact") is True,
            "bound_export": verification.get("release_manifest_sha256") == sha256(base / "release_manifest.json"),
            "reviewed_gallery": bool(required) and required <= set(review.get("reviewed_ids", [])) and review.get("passed") is True and bool(review.get("reviewer")),
            "identified_review": review.get("method") in ("agent_visual_review", "human_visual_review") and review.get("prediction_signature") == predictions.get("signature"),
            "no_competing_spatial_adapter": not config.get("spatial_lora_sha256"),
        }
        for name, expected in {**manifest.get("files", {}), **verification.get("images", {})}.items():
            path = (base / name).resolve()
            if not path.is_relative_to(base.resolve()) or sha256(path) != expected:
                raise ValueError("Export de Base modifié ou chemin externe.")
        if not all(conditions.values()):
            raise ValueError("La Base n'est pas retenue pour Monster : " + str(conditions))
        report = {"passed": True, "projector_sha256": fingerprint, "checks": conditions,
                  "visual_review_method": review["method"], "spatial_quality_validated": False,
                  "scope": "Training compatibility of the exact reviewed export; not human statistical spatial validation"}
        output.parent.mkdir(parents=True, exist_ok=True)
        output.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print(json.dumps(report, indent=2))
        return report



## base.compose_layout

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_cbb6cc9cf8f2cf2f':
    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    """Place existing PNG assets deterministically inside explicit boxes."""


    import argparse


    import json


    import sys


    from pathlib import Path


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    from sharm_image21.spatial_data import compose, write_json


    from sharm_image21.image_io import save_png


    def run(parameters=None):
        args = parameter_namespace({'layout': None, 'output': None}, parameters, required=['layout', 'output'], converters={'layout': Path, 'output': Path}, choices={}, groups=[])
        layout = json.loads(args.layout.read_text(encoding="utf-8"))
        image, records = compose(layout["resolution"], layout["objects"], args.layout.parent)
        save_png(image, args.output, preview_path=args.output.with_name(args.output.stem+"-preview.png"), require_transparency=True)
        write_json(args.output.with_suffix(".layout.json"), {"resolution": layout["resolution"], "objects": records})



## base.evaluate

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_28b34161c4b04eba':
    """Score identity-checked object annotations; missing reviews never disappear."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import json


    import itertools


    import sys


    from collections import defaultdict


    from pathlib import Path


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    from PIL import Image


    from sharm_image21.spatial_data import box_iou, center_error, fingerprint, relation_holds, sha256, wilson, write_json


    def score_record(record, annotation, root, min_iou, max_center_error, require_placement=True):
        path = root / record["file_name"]
        if sha256(path) != record["image_sha256"]:
            raise ValueError("Génération modifiée après son manifeste.")
        if not annotation or annotation.get("image_sha256") != record["image_sha256"] or not annotation.get("reviewed"):
            return {"passed": False, "reason": "missing_or_stale_review"}
        if not annotation.get("reviewer") or annotation.get("method") != "human_identity_and_boxes":
            raise ValueError("Revue humaine identifiée requise pour ce protocole.")
        found = annotation.get("objects", [])
        expected_ids = {o["object_id"] for o in record["objects"]}
        if len(found) != len(expected_ids) or {o.get("object_id") for o in found} != expected_ids:
            return {"passed": False, "reason": "object_count_or_identity"}
        if annotation.get("extra_objects", 0) != 0 or not annotation.get("distinct") or not all(o.get("identity_ok") for o in found):
            return {"passed": False, "reason": "fusion_extra_or_wrong_identity"}
        boxes = {o["object_id"]: o["box"] for o in found}
        ious = [box_iou(o["box"], boxes[o["object_id"]]) for o in record["objects"]]
        errors = [center_error(o["box"], boxes[o["object_id"]]) for o in record["objects"]]
        relation_ok = relation_holds(boxes["1"], boxes["2"], record["relation"])
        separation_ok = all(box_iou(a, b) == 0 for a, b in itertools.combinations(boxes.values(), 2))
        with Image.open(path) as image:
            alpha_ok = False
            if image.mode == "RGBA":
                histogram = image.getchannel("A").histogram()
                alpha_ok = (sum(histogram[:17])/(image.width*image.height) >= 0.005
                            and sum(histogram[240:])/(image.width*image.height) >= 0.01)
        placement_ok = min(ious) >= min_iou and max(errors) <= max_center_error
        passed = relation_ok and separation_ok and alpha_ok and (placement_ok or not require_placement)
        return {"passed": passed, "reason": "ok" if passed else "geometry_or_alpha", "relation_ok": relation_ok,
                "alpha_ok": alpha_ok, "separation_ok": separation_ok, "placement_ok": placement_ok,
                "min_iou": min(ious), "max_center_error": max(errors)}


    def evaluate(manifest, annotations, root, methods, min_iou=0.5, max_center_error=0.05, min_groups=300, lower_bound=0.95):
        if len({a["id"] for a in annotations}) != len(annotations):
            raise ValueError("Annotations dupliquées.")
        index = {a["id"]: a for a in annotations}
        if len({r["id"] for r in manifest["rows"]}) != len(manifest["rows"]):
            raise ValueError("Générations dupliquées.")
        reports = {}
        for method in methods:
            records = [r for r in manifest["rows"] if r["method"] == method]
            if not records:
                raise ValueError(f"Aucune génération pour {method}.")
            expected = len(manifest["config"]["rows"])*len(manifest["config"]["seeds"])
            if len(records) != expected:
                raise ValueError("Génération incomplète : tous les prompts et graines sont requis.")
            expected_pairs = {(scene, seed) for scene in manifest["config"]["rows"] for seed in manifest["config"]["seeds"]}
            if {(r["scene_id"], r["seed"]) for r in records} != expected_pairs:
                raise ValueError("Les scènes/graines ne correspondent pas au protocole.")
            groups, results, relations = defaultdict(list), [], defaultdict(list)
            for record in records:
                outcome = score_record(record, index.get(record["id"]), root, min_iou, max_center_error,
                                       require_placement=method.endswith("layout"))
                groups[fingerprint(record["pair_group"])].append(outcome["passed"])
                relations[record["relation"]].append(outcome["passed"])
                results.append({"id": record["id"], **outcome})
            successes = sum(all(values) for values in groups.values())
            interval = wilson(successes, len(groups))
            reports[method] = {"images": len(records), "success_rate_per_image": sum(r["passed"] for r in results)/len(results),
                "pair_groups": len(groups), "successful_pair_groups": successes, "success_rate": successes/len(groups),
                "require_exact_placement": method.endswith("layout"),
                "wilson_95": interval, "statistical_unit": "object pair group; every scene and seed must pass",
                "per_relation": {k: sum(v)/len(v) for k, v in relations.items()}, "results": results,
                "passed": len(groups) >= min_groups and interval[0] >= lower_bound}
        return {"prediction_signature": manifest["signature"], "methods": reports,
                "criteria": {"min_iou": min_iou, "max_center_error": max_center_error, "min_groups": min_groups,
                             "required_wilson_lower_bound": lower_bound},
                "scope": "Two/three RGBA objects from the asset vocabulary; held-out pairs. Wilson assumes independent pair groups."}


    def run(parameters=None):
        args = parameter_namespace({'predictions': None, 'annotations': None, 'methods': ['candidate_text', 'candidate_layout'], 'min_iou': 0.5, 'max_center_error': 0.05, 'min_groups': 300, 'lower_bound': 0.95, 'output': None}, parameters, required=['predictions', 'annotations', 'output'], converters={'predictions': Path, 'annotations': Path, 'min_iou': float, 'max_center_error': float, 'min_groups': int, 'lower_bound': float, 'output': Path}, choices={}, groups=[])
        if not 0 < args.min_iou <= 1 or not 0 <= args.max_center_error <= 1 or args.min_groups < 1 or not 0 <= args.lower_bound <= 1:
            raise ValueError("Seuils d'évaluation invalides.")
        manifest = json.loads(args.predictions.read_text(encoding="utf-8"))
        annotations = json.loads(args.annotations.read_text(encoding="utf-8"))
        if annotations["prediction_signature"] != manifest["signature"]:
            raise ValueError("Annotations d'une autre génération.")
        report = evaluate(manifest, annotations["rows"], args.predictions.parent, args.methods,
                          args.min_iou, args.max_center_error, args.min_groups, args.lower_bound)
        report["annotations_sha256"] = sha256(args.annotations)
        write_json(args.output, report)
        print(json.dumps({k: {a: b for a, b in v.items() if a != "results"} for k, v in report["methods"].items()}, indent=2))
        if not all(r["passed"] for r in report["methods"].values()):
            raise SystemExit(1)



## base.generate

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_1d0c7eee4c637b48':
    """Generate an auditable spatial benchmark using paired seeds and shared diffusion."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import gc


    import json


    import sys


    from pathlib import Path


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    import torch


    from sharm_image21.core import FeatureCache, Projector, atomic_json, digest, file_hash, read_jsonl


    from sharm_image21.diffusion import load_diffuser_pipeline


    from sharm_image21.image_io import save_png


    @torch.inference_mode()
    def run(parameters=None):
        args = parameter_namespace({'candidate': None, 'dataset': ROOT / 'artifacts/spatial/dataset', 'cache': ROOT / 'cache/spatial', 'output': None, 'split': 'validation', 'limit': 24, 'seeds': [1000, 1001, 1002], 'steps': 30, 'baseline_projector': None, 'include_teacher': False}, parameters, required=['candidate', 'output'], converters={'candidate': Path, 'dataset': Path, 'cache': Path, 'output': Path, 'limit': int, 'seeds': int, 'steps': int, 'baseline_projector': Path}, choices={'split': ('validation', 'test')}, groups=[])
        if min(args.limit, args.steps) < 1 or len(set(args.seeds)) != len(args.seeds):
            raise ValueError("Limite/pas positifs et graines distinctes requis.")
        candidate = json.loads((args.candidate / "candidate.json").read_text())
        if candidate["projector_sha256"] != file_hash(args.candidate / "projector/model.safetensors"):
            raise ValueError("Projecteur candidat modifié.")
        if candidate["lora_sha256"] and candidate["lora_sha256"] != file_hash(args.candidate / "lora/pytorch_lora_weights.safetensors"):
            raise ValueError("LoRA candidat modifié.")
        all_rows = read_jsonl(args.dataset / "scenes.jsonl")
        if candidate["identity"]["dataset_sha256"] != digest(all_rows):
            raise ValueError("Candidat entraîné sur un autre dataset.")
        rows = [r for r in all_rows if r["split"] == args.split][:args.limit]
        if not rows:
            raise ValueError("Aucun prompt dans cet ensemble.")
        student, teacher = (FeatureCache(args.cache / k) for k in ("student", "teacher"))
        for name, cache in (("student", student), ("teacher", teacher)):
            if file_hash(cache.path) != candidate["identity"][f"{name}_index"]:
                raise ValueError("Features différentes de celles de l'entraînement.")
        dataset = json.loads((args.dataset / "manifest.json").read_text())
        methods = (["baseline"] if args.baseline_projector else []) + (["teacher_text", "teacher_layout"] if args.include_teacher else [])
        methods += ["candidate_text", "candidate_layout"]
        config = {"candidate_sha256": file_hash(args.candidate / "candidate.json"), "dataset_sha256": digest(all_rows),
                  "baseline_sha256": file_hash(args.baseline_projector / "model.safetensors") if args.baseline_projector else None,
                  "rows": [r["id"] for r in rows], "seeds": args.seeds, "steps": args.steps, "split": args.split,
                  "resolution": dataset["resolution"], "methods": methods, "cfg": 1.0, "kv_cache": False,
                  "generator_sha256": file_hash(Path(__file__)), "core_sha256": file_hash(ROOT / "common/sharm_image21/core.ipynb")}
        signature = digest(config)
        args.output.mkdir(parents=True, exist_ok=True)
        path = args.output / "predictions.json"
        previous = json.loads(path.read_text()) if path.exists() else {"signature": signature, "rows": []}
        if previous["signature"] != signature:
            raise ValueError("Générations incompatibles dans ce dossier.")
        predictions = {r["id"]: r for r in previous["rows"]}
        embeddings = FeatureCache(args.output / "embeddings", {"signature": signature})
        # Prepare embeddings before loading the diffuser: no encoder resides on GPU.
        for method in methods:
            projector = None if method.startswith("teacher") else Projector.load(
                args.baseline_projector if method == "baseline" else args.candidate / "projector", "cuda").eval()
            for row in rows:
                key = digest({"method": method, "scene": row["id"]})
                if key in embeddings:
                    continue
                caption = row["caption_id"] if method.endswith("layout") else row.get("caption_plain_id", row["caption_id"])
                if method.startswith("teacher"):
                    values = teacher.get(caption)
                else:
                    boxes = torch.tensor([o["box"] for o in row["objects"]], device="cuda", dtype=torch.bfloat16) if method == "candidate_layout" else None
                    with torch.autocast("cuda", dtype=torch.bfloat16):
                        values = projector(student.get(caption).cuda(), layout=boxes).to(torch.bfloat16).cpu()
                embeddings.write([(key, values.contiguous(), [])])
            del projector
            gc.collect()
            torch.cuda.empty_cache()
        pipeline = load_diffuser_pipeline()
        lora_loaded = False
        resolution = dataset["resolution"]
        for method in methods:
            if method.startswith("candidate") and candidate["lora_sha256"] and not lora_loaded:
                pipeline.load_lora_weights(args.candidate / "lora", adapter_name="spatial")
                lora_loaded = True
            for row in rows:
                values = embeddings.get(digest({"method": method, "scene": row["id"]}))[None].cuda()
                for seed in args.seeds:
                    key = digest({"method": method, "scene": row["id"], "seed": seed})
                    if key in predictions and file_hash(args.output / predictions[key]["file_name"]) == predictions[key]["image_sha256"]:
                        continue
                    image = pipeline(prompt_embeds=values, width=resolution, height=resolution, output_resolution=resolution,
                                     num_inference_steps=args.steps, true_cfg_scale=1.0, use_kv_cache=False,
                                     generator=torch.Generator(device="cuda").manual_seed(seed)).images[0]
                    if image.mode != "RGBA":
                        raise ValueError("Le générateur n'a pas retourné un RGBA natif.")
                    name = f"images/{key}.png"
                    # Keep failed opaque examples: spatial evaluation must count them.
                    save_png(image, args.output / name, preview_path=args.output / f"previews/{key}.png")
                    predictions[key] = {"id": key, "scene_id": row["id"], "pair_group": row["pair_group"],
                        "method": method, "seed": seed, "file_name": name, "preview": f"previews/{key}.png",
                        "image_sha256": file_hash(args.output / name), "objects": row["objects"],
                        "relation": row["relation"], "prompt": row["text"] if method.endswith("layout") else row.get("text_plain", row["text"])}
                    atomic_json(path, {"signature": signature, "config": config, "rows": list(predictions.values())})
                    print(f"{method} : {len(predictions)}/{len(methods)*len(rows)*len(args.seeds)}", flush=True)



## base.infer

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_8620494b1db80431':
    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    """Use a local spatial candidate with the actual frozen FP8 encoder."""


    import argparse


    import gc


    import json


    import sys


    from pathlib import Path


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    import torch


    from sharm_image21.core import FrozenEncoder, Projector, file_hash


    from sharm_image21.diffusion import load_diffuser_pipeline


    from sharm_image21.image_io import save_png


    from sharm_image21.spatial_data import layout_prompt


    @torch.inference_mode()
    def run(parameters=None):
        args = parameter_namespace({'candidate': None, 'prompt': None, 'layout': None, 'output': None, 'resolution': 512, 'steps': 30, 'seed': 1000}, parameters, required=['candidate', 'prompt', 'output'], converters={'candidate': Path, 'layout': Path, 'output': Path, 'resolution': int, 'steps': int, 'seed': int}, choices={}, groups=[])
        if args.resolution < 64 or args.resolution % 32 or args.steps < 1:
            raise ValueError("Résolution multiple de 32 et nombre de pas positif requis.")
        config = json.loads((args.candidate / "candidate.json").read_text())
        if file_hash(args.candidate / "projector/model.safetensors") != config["projector_sha256"]:
            raise ValueError("Poids du projecteur modifiés.")
        prompt, boxes = args.prompt, None
        if args.layout:
            objects = json.loads(args.layout.read_text(encoding="utf-8"))["objects"]
            prompt, coordinates = layout_prompt(prompt, objects)
            boxes = torch.tensor(coordinates, device="cuda", dtype=torch.bfloat16)
        encoder = FrozenEncoder("student")
        if encoder.identity != config["identity"]["student"]["encoder"]:
            raise ValueError("Encodeur différent de celui de l'entraînement.")
        source, _ = encoder.encode([prompt])[0]
        del encoder
        gc.collect()
        torch.cuda.empty_cache()
        projector = Projector.load(args.candidate / "projector", "cuda").eval()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            embeddings = projector(source.cuda(), layout=boxes)[None].to(torch.bfloat16)
        del projector
        gc.collect()
        torch.cuda.empty_cache()
        pipeline = load_diffuser_pipeline()
        if config["lora_sha256"]:
            if file_hash(args.candidate / "lora/pytorch_lora_weights.safetensors") != config["lora_sha256"]:
                raise ValueError("Poids du LoRA modifiés.")
            pipeline.load_lora_weights(args.candidate / "lora", adapter_name="spatial")
        image = pipeline(prompt_embeds=embeddings, width=args.resolution, height=args.resolution, output_resolution=args.resolution,
                         num_inference_steps=args.steps, true_cfg_scale=1.0, use_kv_cache=False,
                         generator=torch.Generator(device="cuda").manual_seed(args.seed)).images[0]
        if image.mode != "RGBA":
            raise ValueError("Sortie RGBA native requise.")
        save_png(image, args.output, preview_path=args.output.with_name(args.output.stem+"-preview.png"), require_transparency=True)



## base.prepare_assets

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_923331f7dcfd732f':
    """Download labelled, reproducible RGBA assets from PokeAPI's sprite repository."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    from concurrent.futures import ThreadPoolExecutor, as_completed


    import hashlib


    import importlib.util


    import io


    import json


    import re


    import sys


    import time


    from pathlib import Path


    from urllib.request import Request, urlopen


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    from PIL import Image


    from sharm_image21.spatial_data import sha256, write_json


    def fetch(url):
        for attempt in range(3):
            try:
                with urlopen(Request(url, headers={"User-Agent": "Sharm-spatial-assets/1.0"}), timeout=30) as response:
                    return response.read()
            except OSError:
                if attempt == 2:
                    raise
                time.sleep(2**attempt)


    def run(parameters=None):
        args = parameter_namespace({'output': ROOT / 'cache/spatial/assets-pokeapi', 'species': 386, 'workers': 6}, parameters, required=[], converters={'output': Path, 'species': int, 'workers': int}, choices={}, groups=[])
        if not 3 <= args.species <= 386 or not 1 <= args.workers <= 8:
            raise ValueError("De 3 à 386 espèces et de 1 à 8 téléchargements parallèles requis.")
        args.output.mkdir(parents=True, exist_ok=True)
        provenance = args.output / "source.json"
        if provenance.exists():
            source = json.loads(provenance.read_text(encoding="utf-8"))
            if source["species"] != args.species:
                raise ValueError("Un autre corpus existe ici ; choisir un autre dossier.")
        else:
            commit = json.loads(fetch("https://api.github.com/repos/PokeAPI/sprites/commits/master"))["sha"]
            if not re.fullmatch(r"[a-f0-9]{40}", commit):
                raise ValueError("Révision GitHub invalide.")
            source = {"repository": "https://github.com/PokeAPI/sprites", "revision": commit, "species": args.species,
                      "variants": ["default", "shiny"], "style": "front battle sprites; distinct from the original overworld sheets"}
            write_json(provenance, source)
        module = load_notebook(ROOT / "02-sprite-lora/prepare_dataset.ipynb", name="sprite_traits")

        def prepare(species_id):
            trait_path = args.output / f"species-{species_id}.json"
            traits = json.loads(trait_path.read_text(encoding="utf-8")) if trait_path.exists() else module.fetch_species_trait(species_id)
            if not trait_path.exists():
                write_json(trait_path, traits)
            records = []
            for variant in source["variants"]:
                name = f"pokemon-{species_id}-{variant}.png"
                target = args.output / name
                stamp = target.with_suffix(".source.json")
                url = (f"https://raw.githubusercontent.com/PokeAPI/sprites/{source['revision']}/sprites/pokemon/"
                       f"{'shiny/' if variant == 'shiny' else ''}{species_id}.png")
                if target.exists() and stamp.exists():
                    metadata = json.loads(stamp.read_text())
                    if metadata["url"] != url or metadata["image_sha256"] != sha256(target):
                        raise ValueError(f"Asset modifié : {target}")
                else:
                    content = fetch(url)
                    with Image.open(io.BytesIO(content)) as raw:
                        image = raw.convert("RGBA")
                    alpha = image.getchannel("A")
                    if alpha.getbbox() is None or alpha.getextrema()[0] != 0:
                        raise ValueError(f"Sprite vide ou sans alpha : {url}")
                    image.save(target)
                    write_json(stamp, {"url": url, "source_sha256": hashlib.sha256(content).hexdigest(),
                                       "image_sha256": sha256(target)})
                # Shiny coloration may differ from the canonical species color.
                label = f"{traits['name']}, the {traits['genus']}, {variant} pixel-art sprite"
                records.append({"id": f"pokemon-{species_id}-{variant}", "group_id": f"pokemon-{species_id}",
                                "label": label, "file_name": name, "source_url": url, "image_sha256": sha256(target)})
            return records

        rows = []
        with ThreadPoolExecutor(max_workers=args.workers) as pool:
            futures = [pool.submit(prepare, species_id) for species_id in range(1, args.species+1)]
            for index, future in enumerate(as_completed(futures)):
                rows.extend(future.result())
                if (index+1) % 20 == 0 or index+1 == args.species:
                    print(f"Assets PokéAPI : {index+1}/{args.species} espèces", flush=True)
        rows.sort(key=lambda r: r["id"])
        path = args.output / "assets.jsonl"
        path.write_text("".join(json.dumps(row, ensure_ascii=False)+"\n" for row in rows), encoding="utf-8")
        print(f"Manifeste RGBA : {path}", flush=True)



## base.prepare_cache

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_580d330e13802912':
    """Cache RGBA VAE targets or teacher flows along actual Euler trajectories."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import json


    import random


    import sys


    from pathlib import Path


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    import numpy as np


    import torch


    from PIL import Image


    from sharm_image21.core import ORIGINAL, TEACHER_REVISION, FeatureCache, atomic_json, digest, file_hash, read_jsonl


    from sharm_image21.diffusion import load_transformer, predict_flow


    @torch.no_grad()
    def run(parameters=None):
        args = parameter_namespace({'kind': None, 'dataset': ROOT / 'artifacts/spatial/dataset', 'cache': ROOT / 'cache/spatial', 'steps': 20, 'points': 3, 'max_scenes': 512}, parameters, required=['kind'], converters={'dataset': Path, 'cache': Path, 'steps': int, 'points': int, 'max_scenes': int}, choices={'kind': ('latents', 'trajectories')}, groups=[])
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA et les poids Qwen locaux sont requis pour ce cache.")
        rows = read_jsonl(args.dataset / "scenes.jsonl")
        manifest = json.loads((args.dataset / "manifest.json").read_text())
        if digest(rows) != manifest["dataset_sha256"]:
            raise ValueError("Scènes modifiées depuis le manifeste.")
        identity = {"dataset_sha256": digest(rows), "teacher_revision": TEACHER_REVISION,
                    "resolution": manifest["resolution"]}
        resolution = manifest["resolution"]
        if args.kind == "latents":
            from diffusers import AutoencoderKLQwenImage21
            identity.update(vae_config=file_hash(ORIGINAL / "vae/config.json"), sampling="mode-normalized-rgba-v1")
            cache = FeatureCache(args.cache / "latents", identity)
            pending = [r for r in rows if r["id"] not in cache]
            if not pending:
                return
            vae = AutoencoderKLQwenImage21.from_pretrained(ORIGINAL, subfolder="vae", torch_dtype=torch.bfloat16,
                                                          local_files_only=True).cuda().eval().requires_grad_(False)
            mean = torch.tensor(vae.config.latents_mean, device="cuda", dtype=torch.bfloat16).view(1, -1, 1, 1, 1)
            std = torch.tensor(vae.config.latents_std, device="cuda", dtype=torch.bfloat16).view(1, -1, 1, 1, 1)
            for index, row in enumerate(pending):
                path = args.dataset / row["file_name"]
                if file_hash(path) != row["image_sha256"]:
                    raise ValueError(f"Image modifiée : {path}")
                with Image.open(path) as image:
                    if image.mode != "RGBA" or image.size != (resolution, resolution):
                        raise ValueError(f"Image RGBA à {resolution}px requise : {path}")
                    pixels = torch.from_numpy(np.array(image, copy=True)).permute(2, 0, 1)[None, :, None]
                tensor = pixels.cuda().to(torch.bfloat16)/127.5-1
                values = ((vae.encode(tensor).latent_dist.mode()-mean)/std)[0].cpu().contiguous()
                if not torch.isfinite(values).all():
                    raise RuntimeError("Latents VAE non finis.")
                cache.write([(row["id"], values, [])])
                if index % 25 == 0:
                    print(f"Latents RGBA : {len(cache.index['entries'])}/{len(rows)}", flush=True)
            return
        if not 1 <= args.points <= args.steps or args.steps < 2 or args.max_scenes < 2:
            raise ValueError("De 1 à steps points ; au moins deux pas de trajectoire.")
        from diffusers import FlowMatchEulerDiscreteScheduler
        from diffusers.pipelines.qwenimage21.pipeline_qwenimage21 import calculate_shift, retrieve_timesteps
        teacher = FeatureCache(args.cache / "teacher")
        teacher.require([{"id": r.get("caption_plain_id", r["caption_id"])} for r in rows])
        # Cache teacher trajectories on a fixed subset rather than regenerating the
        # entire corpus. All compositions still contribute supervised image targets.
        train_rows = sorted((r for r in rows if r["split"] == "train"), key=lambda r: r["id"])
        valid_rows = sorted((r for r in rows if r["split"] == "validation"), key=lambda r: r["id"])
        train_count = max(1, int(args.max_scenes*0.8))
        scenes = train_rows[:train_count]+valid_rows[:max(1, args.max_scenes-train_count)]
        identity.update(teacher_index_sha256=file_hash(teacher.path), steps=args.steps, points=args.points,
                        selected_scenes=[r["id"] for r in scenes],
                        teacher_prompt_variant="natural-text-without-coordinates",
                        transformer_config=file_hash(ORIGINAL / "transformer/config.json"),
                        precision="NF4-double-BF16", sampler="Euler-CFG1-no-KV-cache-v2-bf16-output", seed_policy="700+scene-id",
                        diffusion_sha256=file_hash(ROOT / "common/sharm_image21/diffusion.ipynb"))
        noisy = FeatureCache(args.cache / "trajectory-latents", identity)
        flows = FeatureCache(args.cache / "trajectory-flows", identity)
        scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(ORIGINAL, subfolder="scheduler", local_files_only=True)
        mu = calculate_shift((resolution//16)**2, scheduler.config.get("base_image_seq_len", 256),
                             scheduler.config.get("max_image_seq_len", 4096), scheduler.config.get("base_shift", 0.5),
                             scheduler.config.get("max_shift", 1.15))
        def reset_schedule():
            return retrieve_timesteps(scheduler, args.steps, "cuda", sigmas=np.linspace(1, 1/args.steps, args.steps), mu=mu)[0]
        schedule = reset_schedule()
        # Test prompts are never used to prepare training targets or select checkpoints.
        records = []
        for row in scenes:
            selected = sorted(random.Random(int(row["id"][:8], 16)).sample(range(args.steps), args.points))
            records.extend({"id": f"{row['id']}-{step}", "scene_id": row["id"], "step": step,
                            "split": row["split"], "timestep": float((schedule[step].to(torch.bfloat16)/1000).item())}
                           for step in selected)
        if all(r["id"] in noisy and r["id"] in flows for r in records):
            atomic_json(args.cache / "trajectories.json", {"identity": identity, "rows": records})
            return
        model = load_transformer()
        by_scene = {}
        for record in records:
            by_scene.setdefault(record["scene_id"], {})[record["step"]] = record
        for index, row in enumerate(scenes):
            wanted = by_scene[row["id"]]
            if all(r["id"] in noisy and r["id"] in flows for r in wanted.values()):
                continue
            reference = teacher.get(row.get("caption_plain_id", row["caption_id"]))[None].cuda()
            generator = torch.Generator(device="cuda").manual_seed(700+int(row["id"][:8], 16))
            latent = torch.randn((1, 64, 1, resolution//16, resolution//16), device="cuda", dtype=torch.bfloat16, generator=generator)
            timesteps = reset_schedule()
            for step, timestep in enumerate(timesteps):
                t = timestep.expand(1).to(device="cuda", dtype=torch.bfloat16)/1000
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    target = predict_flow(model, latent, reference, t)
                if not torch.isfinite(target).all():
                    raise RuntimeError("Prédiction enseignant non finie.")
                if step in wanted:
                    record = wanted[step]
                    record["timestep"] = float(t.item())
                    noisy.write([(record["id"], latent[0].cpu().contiguous(), [])])
                    flows.write([(record["id"], target[0].cpu().contiguous(), [])])
                latent = scheduler.step(target, timestep, latent, return_dict=False)[0]
            atomic_json(args.cache / "trajectories.json", {"identity": identity, "rows": records})
            print(f"Trajectoires enseignant : {index+1}/{len(scenes)}", flush=True)
        noisy.require(records)
        flows.require(records)



## base.prepare_dataset

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_1bfc2f6a04378de4':
    """Build paired RGBA compositions with measured boxes and held-out object pairs."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import csv


    import itertools


    import json


    import random


    import sys


    from pathlib import Path


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    from sharm_image21.spatial_data import (RELATIONS, compose, fingerprint, read_rows,
                                           relation_holds, sha256, write_json)


    from sharm_image21.image_io import checkerboard_preview


    def run(parameters=None):
        args = parameter_namespace({'assets': None, 'sprites': None, 'output': ROOT / 'artifacts/spatial/dataset', 'count': 3000, 'resolution': 512, 'seed': 42}, parameters, required=[], converters={'assets': Path, 'sprites': Path, 'output': Path, 'count': int, 'resolution': int, 'seed': int}, choices={}, groups=[(['assets', 'sprites'], True)])
        if args.count < 30 or args.resolution < 64 or args.resolution % 32:
            raise ValueError("Au moins 30 scènes ; résolution multiple de 32, au moins 64.")
        if args.output.exists():
            raise FileExistsError(f"Choisir un nouveau dossier de dataset : {args.output}")
        source_path = args.assets or args.sprites
        asset_root = source_path.resolve().parent
        assets = read_rows(source_path)
        if args.sprites:
            with (asset_root.parent / "manifest.csv").open(encoding="utf-8") as stream:
                species = {Path(r["file"]).name: r["species_id"] for r in csv.DictReader(stream)}
            assets = [{"id": r["file_name"], "group_id": species[r["file_name"]],
                       "label": r.get("label") or r["text"].split(";")[0].replace("pkm_ow_sprite, ", ""),
                       "file_name": r["file_name"]} for r in assets]
        by_group = {}
        for asset in assets:
            if not all(asset.get(k) for k in ("id", "group_id", "label", "file_name")):
                raise ValueError("Chaque asset requiert id, group_id, label et file_name.")
            asset["sha256"] = sha256(asset_root / asset["file_name"])
            by_group.setdefault(str(asset["group_id"]), []).append(asset)
        pairs = list(itertools.combinations(sorted(by_group), 2))
        if len(pairs) < 3 or len({a["id"] for a in assets}) != len(assets):
            raise ValueError("Au moins trois groupes distincts et des identifiants uniques sont requis.")
        rng = random.Random(args.seed)
        rng.shuffle(pairs)
        holdout = max(1, len(pairs)//10)
        groups = {"test": pairs[:holdout], "validation": pairs[holdout:2*holdout], "train": pairs[2*holdout:]}
        totals = {"test": max(1, args.count//10), "validation": max(1, args.count//10)}
        totals["train"] = args.count-totals["test"]-totals["validation"]
        args.output.mkdir(parents=True)
        (args.output / "images").mkdir()
        (args.output / "previews").mkdir()
        rows = []
        for split, total in totals.items():
            for index in range(total):
                group_a, group_b = rng.choice(groups[split])
                a, b = (rng.choice(by_group[g]) for g in (group_a, group_b))
                relation = RELATIONS[index % len(RELATIONS)]
                small, large = rng.choice((0.23, 0.28)), rng.choice((0.30, 0.34))
                widths = (small, large) if index % 2 else (large, small)
                centers = [(0.25, 0.5), (0.75, 0.5)] if relation in ("left", "right") else [(0.5, 0.25), (0.5, 0.75)]
                if relation in ("right", "below"):
                    centers.reverse()
                scene_assets = (a, b)
                if index % 5 == 4:
                    centers = ([(0.17, 0.5), (0.5, 0.5), (0.83, 0.5)] if relation in ("left", "right")
                               else [(0.5, 0.17), (0.5, 0.5), (0.5, 0.83)])
                    if relation in ("right", "below"):
                        centers.reverse()
                    scene_assets, widths = (a, b, a), (0.20, 0.24, 0.20)
                # Vary exact coordinates while retaining the requested relation.
                jitter = 0.015 if len(scene_assets) == 3 else 0.05
                centers = [(cx+rng.uniform(-jitter, jitter), cy+rng.uniform(-jitter, jitter)) for cx, cy in centers]
                objects = [{**asset, "box": [cx-w/2, cy-w/2, cx+w/2, cy+w/2]}
                           for asset, (cx, cy), w in zip(scene_assets, centers, widths)]
                image, objects = compose(args.resolution, objects, asset_root)
                if not relation_holds(objects[0]["box"], objects[1]["box"], relation):
                    raise RuntimeError("Composition incompatible avec sa relation.")
                en = {"left": "to the left of", "right": "to the right of", "above": "above", "below": "below"}
                fr = {"left": "à gauche de", "right": "à droite de", "above": "au-dessus de", "below": "en dessous de"}
                french = index % 3 == 0
                text = (f"Exactement deux objets distincts. Objet 1 : {a['label']}. Objet 2 : {b['label']}. "
                        f"L'objet 1 est {fr[relation]} l'objet 2. Aucun chevauchement. Fond transparent."
                        if french else
                        f"Exactly two distinct objects. Object 1: {a['label']}. Object 2: {b['label']}. "
                        f"Object 1 is {en[relation]} object 2. No overlap. Transparent background.")
                if len(objects) == 3:
                    text = (text.replace("Exactement deux", "Exactement trois") + f" Objet 3 : {a['label']}."
                            if french else text.replace("Exactly two", "Exactly three") + f" Object 3: {a['label']}.")
                plain_text = text
                # Include the requested allocations in text too, with matching order.
                text += " Normalized object boxes xyxy: " + json.dumps([o["box"] for o in objects], separators=(",", ":")) + "."
                scene_id = fingerprint({"split": split, "index": index, "seed": args.seed, "objects": objects})
                name = f"images/{scene_id}.png"
                image.save(args.output / name)
                if index < 4:
                    checkerboard_preview(image).save(args.output / f"previews/{split}-{index}.png")
                rows.append({"id": scene_id, "caption_id": fingerprint(text), "text": text,
                             "caption_plain_id": fingerprint(plain_text), "text_plain": plain_text, "split": split,
                             "pair_group": sorted((group_a, group_b)), "relation": relation,
                             "language": "fr" if french else "en", "objects": objects,
                             "file_name": name, "image_sha256": sha256(args.output / name)})
        captions = [{"id": r[key], "text": r[text_key], "split": r["split"]}
                    for r in rows for key, text_key in (("caption_id", "text"), ("caption_plain_id", "text_plain"))]
        corpus = list({r["id"]: r for r in captions}.values())
        for name, values in (("scenes.jsonl", rows), ("corpus.jsonl", corpus)):
            (args.output / name).write_text("".join(json.dumps(r, ensure_ascii=False)+"\n" for r in values), encoding="utf-8")
        write_json(args.output / "manifest.json", {"schema": 1, "seed": args.seed, "resolution": args.resolution,
                   "dataset_sha256": fingerprint(rows), "corpus_sha256": fingerprint(corpus), "counts": totals,
                   "split_unit": "unordered object group pair; variants and inverse relations remain together",
                   "asset_manifest_sha256": sha256(source_path), "assets": assets})
        print(json.dumps({"dataset": str(args.output), "scenes": len(rows), "splits": totals}), flush=True)



## base.review

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_9804d97f64622fb9':
    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    """Build an offline annotation page with identity checks and drawable object boxes."""


    import argparse


    import json


    from pathlib import Path


    def run(parameters=None):
        args = parameter_namespace({'predictions': None}, parameters, required=['predictions'], converters={'predictions': Path}, choices={}, groups=[])
        data = json.loads(args.predictions.read_text(encoding="utf-8"))
        payload = json.dumps(data, ensure_ascii=False).replace("<", "\\u003c")
        document = r'''<!doctype html><html lang="fr"><meta charset="utf-8">
    <title>Revue spatiale Sharm</title><style>
    body{font:16px system-ui;margin:24px;max-width:1150px;background:#f6f7f9;color:#19232d}button,input,select{font:inherit;padding:7px;margin:4px}
    main{display:flex;gap:24px;flex-wrap:wrap}canvas{width:512px;max-width:100%;touch-action:none;background:repeating-conic-gradient(#bbb 0% 25%,#eee 0% 50%) 50%/24px 24px}
    aside{flex:1;min-width:300px}label{display:block;margin:10px 0}pre{white-space:pre-wrap}small{display:block;color:#45515c}
    </style><h1>Revue des positions</h1>
    <p>Identifiez chaque objet puis tracez sa boîte visible. Laissez sans boîte un objet absent ou impossible à identifier.
    Les rectangles cibles sont affichables pour comparer les positions. Chaque image doit recevoir une revue explicite.</p>
    <label>Votre nom <input id="reviewer" placeholder="Identifiant du relecteur"></label>
    <button id="prev">← Précédente</button><button id="next">Suivante →</button><span id="progress"></span>
    <main><section><canvas id="canvas" width="512" height="512"></canvas><label><input id="targets" type="checkbox">Afficher les boîtes cibles</label></section>
    <aside><p id="prompt"></p><label>Objet à annoter <select id="object"></select></label><div id="identities"></div>
    <button id="remove">Effacer la boîte de cet objet</button>
    <label><input id="distinct" type="checkbox">Les objets attendus sont distincts et séparés</label>
    <label>Objets supplémentaires <input id="extra" type="number" min="0" value="0"></label>
    <label><input id="reviewed" type="checkbox">J’ai vérifié cette image, y compris les objets manquants</label>
    <small>Le numéro de méthode est masqué pour limiter le biais. Le protocole reste une revue individuelle ; utilisez plusieurs relecteurs pour estimer leur accord.</small>
    </aside></main><button id="export">Exporter les annotations JSON</button><label>Reprendre un fichier <input id="import" type="file" accept="application/json"></label>
    <script>const data=__DATA__;const rows=[...data.rows].sort((a,b)=>a.id.localeCompare(b.id));let index=0,start=null;
    const $=id=>document.getElementById(id),annotations={},image=new Image();const ctx=$('canvas').getContext('2d');
    function current(){const r=rows[index];return annotations[r.id]??={id:r.id,image_sha256:r.image_sha256,objects:[],reviewed:false,distinct:false,extra_objects:0,method:'human_identity_and_boxes',reviewer:''};}
    function save(){const a=current();a.reviewer=$('reviewer').value.trim();a.distinct=$('distinct').checked;a.extra_objects=Number($('extra').value);a.reviewed=$('reviewed').checked;}
    function draw(){ctx.clearRect(0,0,512,512);ctx.drawImage(image,0,0,512,512);const a=current();function box(b,color,text){ctx.strokeStyle=color;ctx.lineWidth=3;ctx.strokeRect(b[0]*512,b[1]*512,(b[2]-b[0])*512,(b[3]-b[1])*512);ctx.fillStyle=color;ctx.font='15px sans-serif';ctx.fillText(text,b[0]*512,Math.max(16,b[1]*512-3));}
    if($('targets').checked)for(const o of rows[index].objects)box(o.box,'#00bbcc','cible '+o.object_id);for(const o of a.objects)box(o.box,'#ff6633','tracé '+o.object_id);}
    function show(){const r=rows[index],a=current();$('progress').textContent=`${index+1}/${rows.length} — ${Object.values(annotations).filter(a=>a.reviewed).length} revues`;$('prompt').textContent=r.prompt;
    $('object').replaceChildren();$('identities').replaceChildren();for(const o of r.objects){const opt=document.createElement('option');opt.value=o.object_id;opt.textContent=`${o.object_id} : ${o.label}`;$('object').append(opt);
    const label=document.createElement('label'),check=document.createElement('input');check.type='checkbox';check.checked=!!a.objects.find(x=>x.object_id===o.object_id)?.identity_ok;check.onchange=()=>{const found=current().objects.find(x=>x.object_id===o.object_id);if(found)found.identity_ok=check.checked;else{check.checked=false;alert('Tracez d’abord la boîte de cet objet.');}};label.append(check,document.createTextNode(' Identité correcte : '+o.label));$('identities').append(label);}
    $('distinct').checked=a.distinct;$('extra').value=a.extra_objects;$('reviewed').checked=a.reviewed;image.src=r.file_name;}
    function point(e){const r=$('canvas').getBoundingClientRect();return [Math.max(0,Math.min(1,(e.clientX-r.left)/r.width)),Math.max(0,Math.min(1,(e.clientY-r.top)/r.height))];}
    $('canvas').onpointerdown=e=>{start=point(e);$('canvas').setPointerCapture(e.pointerId);};$('canvas').onpointerup=e=>{if(!start)return;const p=point(e),box=[Math.min(start[0],p[0]),Math.min(start[1],p[1]),Math.max(start[0],p[0]),Math.max(start[1],p[1])];start=null;if(box[2]-box[0]<.005||box[3]-box[1]<.005)return;save();const a=current(),id=$('object').value;a.objects=a.objects.filter(o=>o.object_id!==id);a.objects.push({object_id:id,box,identity_ok:false});show();draw();};
    image.onload=draw;$('targets').onchange=draw;$('remove').onclick=()=>{save();current().objects=current().objects.filter(o=>o.object_id!==$('object').value);show();draw();};
    $('prev').onclick=()=>{save();index=Math.max(0,index-1);show();};$('next').onclick=()=>{save();index=Math.min(rows.length-1,index+1);show();};
    $('export').onclick=()=>{save();if(!$('reviewer').value.trim()){alert('Renseignez votre identifiant.');return;}const blob=new Blob([JSON.stringify({prediction_signature:data.signature,rows:Object.values(annotations)},null,2)],{type:'application/json'});const a=document.createElement('a');a.href=URL.createObjectURL(blob);a.download='annotations.json';a.click();URL.revokeObjectURL(a.href);};
    $('import').onchange=async e=>{const restored=JSON.parse(await e.target.files[0].text());if(restored.prediction_signature!==data.signature){alert('Ce fichier concerne d’autres générations.');return;}for(const a of restored.rows)annotations[a.id]=a;$('reviewer').value=restored.rows.find(a=>a.reviewer)?.reviewer??'';show();};show();</script></html>'''
        path = args.predictions.parent / "review.html"
        path.write_text(document.replace("__DATA__", payload), encoding="utf-8")
        print(path)



## base.run

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_0ff4b54ab6a0722d':
    """Run the spatial experiment in isolated folders; never publish a candidate."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import importlib.util


    import json


    import os


    import shutil


    import signal


    import subprocess


    import sys


    from pathlib import Path


    HERE = Path(__file__).resolve().parent


    ROOT = HERE.parent


    sys.path.insert(0, str(ROOT / "common"))


    from sharm_image21.spatial_data import sha256, write_json


    def preflight():
        modules = {k: importlib.util.find_spec(k) is not None for k in
                   ("torch", "torchvision", "diffusers", "transformers", "bitsandbytes", "peft", "safetensors", "PIL", "numpy")}
        weights = ROOT.parent / "1-local-generation/weights"
        paths = [weights / "qwen-image-2.1/transformer/config.json", weights / "qwen-image-2.1/vae/config.json",
                 weights / "qwen-image-2.1/text_encoder/model.safetensors.index.json",
                 weights / "sharm-vl4b-fp8/model.safetensors.index.json"]
        gpu = None
        compiler = shutil.which(os.environ.get("CC", "cc"))
        if modules["torch"]:
            import torch
            if torch.cuda.is_available():
                gpu = {"name": torch.cuda.get_device_name(), "total_gib": torch.cuda.get_device_properties(0).total_memory/2**30}
        return {"ready": all(modules.values()) and all(p.exists() for p in paths) and gpu is not None and compiler is not None,
                "modules": modules, "gpu": gpu, "c_compiler": compiler,
                "missing_model_paths": [str(p) for p in paths if not p.exists()]}


    def flags(config):
        return [item for key, value in config.items() for item in ("--"+key, str(value))]


    def run(parameters=None):
        args = parameter_namespace({'check': False, 'assets': None, 'sprites': None, 'pokeapi': False, 'profile': HERE / 'configs/8gb.json', 'dataset': ROOT / 'artifacts/spatial/dataset', 'cache': ROOT / 'cache/spatial', 'output': ROOT / 'artifacts/spatial/run', 'stage': 'all'}, parameters, required=[], converters={'assets': Path, 'sprites': Path, 'profile': Path, 'dataset': Path, 'cache': Path, 'output': Path}, choices={'stage': ('all', 'data', 'features', 'latents', 'trajectories', 'warmup', 'flow', 'generation')}, groups=[(['assets', 'sprites', 'pokeapi'], False)])
        if args.check:
            report = preflight()
            print(json.dumps(report, indent=2))
            raise SystemExit(0 if report["ready"] else 2)
        if args.stage != "data":
            report = preflight()
            if not report["ready"]:
                print(json.dumps(report, indent=2))
                raise SystemExit("Préparer le runtime avec setup_sharm.sh et les modèles avec prepare_models.py.")
        profile = json.loads(args.profile.read_text())
        args.output.mkdir(parents=True, exist_ok=True)
        # Linux/WSL is the supported GPU runtime. Data generation also works on Windows.
        lock = (args.output / "run.lock").open("a")
        if sys.platform != "win32":
            import fcntl
            fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        stages = []
        if args.pokeapi:
            args.assets = args.cache / "assets-pokeapi/assets.jsonl"
            if args.stage in ("all", "data"):
                stages.append(("assets", HERE / "prepare_assets.ipynb", ["--output", str(args.assets.parent)]))
        if args.stage in ("all", "data"):
            if args.dataset.exists():
                manifest = json.loads((args.dataset / "manifest.json").read_text())
                if any(manifest[k] != profile["dataset"][k] for k in ("resolution", "seed")) or sum(manifest["counts"].values()) != profile["dataset"]["count"]:
                    raise ValueError("Le profil ne correspond pas au dataset existant ; choisir un autre dossier.")
                if (args.assets or args.sprites) and sha256(args.assets or args.sprites) != manifest["asset_manifest_sha256"]:
                    raise ValueError("Le manifeste d'assets a changé.")
            else:
                if not (args.assets or args.sprites):
                    raise ValueError("Fournir --assets ou --sprites pour préparer les compositions.")
                stages.append(("data", HERE / "prepare_dataset.ipynb", ["--assets" if args.assets else "--sprites", str(args.assets or args.sprites), "--output", str(args.dataset), *flags(profile["dataset"])]))
        if args.stage in ("all", "features"):
            stages.append(("audit_fp8", ROOT / "01-encoder-distillation/audit_runtime.ipynb", []))
            for kind, batch in (("teacher", "4"), ("student", "1")):
                stages.append((f"features-{kind}", ROOT / "01-encoder-distillation/cache_features.ipynb",
                               [kind, "--corpus", str(args.dataset / "corpus.jsonl"), "--cache-root", str(args.cache), "--batch-size", batch]))
        if args.stage in ("all", "flow"):
            stages.append(("audit_backward", HERE / "verify_runtime.ipynb", []))
        for kind in ("latents", "trajectories"):
            if args.stage in ("all", kind):
                stages.append((kind, HERE / "prepare_cache.ipynb", [kind, "--dataset", str(args.dataset), "--cache", str(args.cache),
                               *(flags(profile["trajectories"]) if kind == "trajectories" else [])]))
        for phase in ("warmup", "flow"):
            if args.stage in ("all", phase):
                init = ROOT / "artifacts/projector" if phase == "warmup" else args.output / "warmup/best/projector"
                initialize = ["--init-projector", str(init)] if init.exists() or phase == "flow" else []
                stages.append((phase, HERE / "train.ipynb", [phase, "--dataset", str(args.dataset), "--cache", str(args.cache),
                               "--output", str(args.output / phase), *initialize, *flags(profile[phase])]))
        if args.stage in ("all", "generation"):
            predictions = args.output / "validation/predictions.json"
            stages.append(("generation", HERE / "generate.ipynb", ["--candidate", str(args.output / "flow/best"),
                           "--dataset", str(args.dataset), "--cache", str(args.cache), "--output", str(predictions.parent),
                           "--include-teacher", *flags(profile["generation"])]))
            stages.append(("review", HERE / "review.ipynb", ["--predictions", str(predictions)]))
        def interrupted(signum, frame):
            raise KeyboardInterrupt(f"Signal {signum}")
        for name, script, arguments in stages:
            write_json(args.output / "run-status.json", {"stage": name, "state": "running"})
            print(f"Démarrage : {name}", flush=True)
            child = None
            try:
                code = call_with_flags(script, arguments)
                if code:
                    raise subprocess.CalledProcessError(code, child.args)
            except BaseException as error:
                write_json(args.output / "run-status.json", {"stage": name, "state": "failed", "error": str(error)})
                raise
            finally:
                if child is not None and child.poll() is None:
                    child.terminate()
                    try:
                        child.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        child.kill()
                        child.wait()
        write_json(args.output / "run-status.json", {"state": "complete", "stage": args.stage,
                   "quality": "candidate_requires_spatial_review", "profile_sha256": sha256(args.profile)})



## base.run_remote

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_8e37aea114694883':
    """Resume the spatial experiment remotely with a staged model download budget."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import fcntl


    import json


    import os


    from pathlib import Path


    import shutil


    import subprocess


    import sys


    import time


    ROOT = Path(__file__).resolve().parents[1]


    PROJECT = ROOT.parent


    ARTIFACTS = ROOT / "artifacts/spatial"


    sys.path.insert(0, str(ROOT / "common"))


    from sharm_image21.core import (FeatureCache, ORIGINAL, atomic_json, digest, file_hash,
                                    model_identity, read_jsonl)


    def status(stage, state="running", **extra):
        atomic_json(ARTIFACTS / "remote-status.json", {"stage": stage, "state": state, "pid": os.getpid(),
                    "updated_at": time.time(), "free_disk_gib": shutil.disk_usage(PROJECT).free/2**30, **extra})


    def execute(stage, script, *arguments, offline=False):
        status(stage)
        environment = dict(os.environ)
        if offline:
            environment["HF_HUB_OFFLINE"] = "1"
        else:
            environment.pop("HF_HUB_OFFLINE", None)
        call_with_flags(script, arguments)


    def release_teacher_weights():
        """Delete only this new workspace's temporary teacher shards after a full cache."""
        marker = json.loads((ARTIFACTS / "remote-workspace.json").read_text(encoding="utf-8"))
        if marker.get("root") != str(PROJECT.resolve()) or marker.get("temporary_teacher_weights") is not True:
            raise ValueError("Le nettoyage du professeur exige le marqueur de ce workspace isolé.")
        corpus = read_jsonl(ARTIFACTS / "dataset/corpus.jsonl")
        teacher = FeatureCache(ROOT / "cache/spatial/teacher",
                               {"encoder": model_identity("teacher"), "corpus": digest(corpus)})
        teacher.require(corpus)
        # Validate every indexed shard before deleting any regenerable source weights.
        from safetensors import safe_open
        for name in {r["shard"] for r in teacher.index["entries"].values()}:
            path = teacher.directory / name
            if path.parent.resolve() != teacher.directory.resolve() or path.is_symlink() or not path.is_file():
                raise ValueError("Shard du professeur manquant ou externe.")
            with safe_open(path, framework="pt", device="cpu") as values:
                for key, entry in teacher.index["entries"].items():
                    if entry["shard"] == name and list(values.get_slice(key).get_shape()) != entry["shape"]:
                        raise ValueError("Dimensions du cache du professeur incorrectes.")
        directory = ORIGINAL / "text_encoder"
        if directory.is_symlink() or not directory.resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError("Le professeur doit appartenir au workspace isolé.")
        directory = directory.resolve()
        index = json.loads((directory / "model.safetensors.index.json").read_text(encoding="utf-8"))
        targets = []
        for name in set(index["weight_map"].values()):
            path = directory / name
            if Path(name).name != name or not name.endswith(".safetensors") or path.is_symlink():
                raise ValueError("Chemin de poids temporaire inattendu.")
            targets.append(path)
        report = {"teacher_index_sha256": file_hash(teacher.path), "captions": len(corpus),
                  "released_bytes": sum(p.stat().st_size for p in targets if p.exists()),
                  "files": [p.name for p in targets], "scope": "Owned temporary teacher shards; configs retained"}
        for path in targets:
            path.unlink(missing_ok=True)
        report_path = ARTIFACTS / "temporary-teacher-release.json"
        if report["released_bytes"] or not report_path.exists():
            atomic_json(report_path, report)


    def run(parameters=None):
        args = parameter_namespace({}, parameters, required=[], converters={}, choices={}, groups=[])
        ARTIFACTS.mkdir(parents=True, exist_ok=True)
        lock = (ARTIFACTS / "remote.lock").open("a")
        fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        stage = "initializing"
        try:
            corpus = read_jsonl(ARTIFACTS / "dataset/corpus.jsonl")
            directory = ROOT / "cache/spatial/teacher"
            cache = FeatureCache(directory) if (directory / "index.json").exists() else None
            teacher_complete = cache is not None and all(r["id"] in cache for r in corpus)
            if not teacher_complete:
                stage = "download_teacher"
                execute(stage, ROOT / "01-encoder-distillation/prepare_models.ipynb", "--stage", "teacher")
                stage = "features-teacher"
                execute(stage, ROOT / "01-encoder-distillation/cache_features.ipynb", "teacher", "--corpus",
                        ARTIFACTS / "dataset/corpus.jsonl", "--cache-root", ROOT / "cache/spatial", "--batch-size", 4,
                        offline=True)
            stage = "release_temporary_teacher"
            status(stage)
            release_teacher_weights()
            stage = "download_inference"
            execute(stage, ROOT / "01-encoder-distillation/prepare_models.ipynb", "--stage", "inference")
            stage = "training_chain"
            execute(stage, ROOT / "03-spatial-distillation/run.ipynb", "--profile",
                    ROOT / "Sharm-Image-2.1-VL4B/configs/remote16gb-256.json", offline=True)
            status("finished", "complete", quality="candidate_requires_spatial_review")
        except BaseException as error:
            status(stage, "failed", error=str(error))
            raise



## base.train

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_7d03ca7400966d26':
    """Train a contextual projector, then optional diffuser LoRA, against spatial targets."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import json


    import math


    import random


    import sys


    import time


    from pathlib import Path


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    import torch


    from torch.nn import functional as F


    from sharm_image21.core import (FeatureCache, Projector, atomic_json, digest, file_hash, read_jsonl, seed_all)


    from sharm_image21.diffusion import load_transformer, predict_flow


    def relative_mse(predicted, target):
        return (predicted.float()-target.float()).square().mean()/target.float().square().mean().clamp_min(1e-8)


    def embedding_loss(predicted, target):
        return relative_mse(predicted, target)+0.1*(1-F.cosine_similarity(predicted.float(), target.float(), dim=-1).mean())


    def load_adapter_state(model, state):
        from peft import set_peft_model_state_dict
        set_peft_model_state_dict(model, state, adapter_name="default")


    def save_candidate(projector, diffuser, destination, metadata):
        projector.save(destination / "projector", metadata, storage_dtype=torch.bfloat16)
        config = {**metadata, "projector_sha256": file_hash(destination / "projector/model.safetensors"),
                  "status": "candidate_requires_spatial_review", "lora_sha256": None}
        if diffuser is not None and any(p.requires_grad for p in diffuser.parameters()):
            from diffusers import QwenImage21Pipeline
            from diffusers.training_utils import _collate_lora_metadata
            from peft.utils import get_peft_model_state_dict
            folder = destination / "lora"
            QwenImage21Pipeline.save_lora_weights(folder, transformer_lora_layers=get_peft_model_state_dict(diffuser),
                                                  **_collate_lora_metadata({"transformer": diffuser}))
            config["lora_sha256"] = file_hash(folder / "pytorch_lora_weights.safetensors")
        atomic_json(destination / "candidate.json", config)


    def run(parameters=None):
        args = parameter_namespace({'phase': None, 'dataset': ROOT / 'artifacts/spatial/dataset', 'cache': ROOT / 'cache/spatial', 'output': None, 'init_projector': None, 'steps': 1000, 'accumulation': 8, 'learning_rate': 0.0001, 'context_layers': 2, 'context_heads': 8, 'lora_rank': 0, 'lora_learning_rate': 5e-05, 'distillation_weight': 0.25, 'embedding_weight': 0.05, 'layout_dropout': 0.5, 'eval_every': 100, 'validation_limit': 64, 'patience': 6, 'checkpointing_mode': 'all', 'seed': 42, 'device': 'cuda'}, parameters, required=['phase', 'output'], converters={'dataset': Path, 'cache': Path, 'output': Path, 'init_projector': Path, 'steps': int, 'accumulation': int, 'learning_rate': float, 'context_layers': int, 'context_heads': int, 'lora_rank': int, 'lora_learning_rate': float, 'distillation_weight': float, 'embedding_weight': float, 'layout_dropout': float, 'eval_every': int, 'validation_limit': int, 'patience': int, 'seed': int}, choices={'phase': ('warmup', 'flow'), 'checkpointing_mode': ('all', 'alternating', 'none'), 'device': ('cuda', 'cpu')}, groups=[])
        if min(args.steps, args.accumulation, args.eval_every, args.validation_limit, args.patience) < 1:
            raise ValueError("Durées, accumulation et tailles de validation positives requises.")
        if not 0 <= args.layout_dropout <= 1 or min(args.distillation_weight, args.embedding_weight, args.lora_rank) < 0:
            raise ValueError("Poids de pertes/rang positifs et dropout entre 0 et 1 requis.")
        if min(args.learning_rate, args.lora_learning_rate) <= 0 or (args.phase == "warmup" and args.lora_rank):
            raise ValueError("LR positifs ; le LoRA est réservé à la phase flow.")
        if args.device == "cpu" and args.phase == "flow":
            raise ValueError("Le flow NF4 exige CUDA.")
        if args.device == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA requis ; les contrôles CPU sont dans tests/test_spatial.py.")
        device = torch.device(args.device)
        def amp():
            return torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == "cuda")
        seed_all(args.seed)
        rows = read_jsonl(args.dataset / "scenes.jsonl")
        dataset_manifest = json.loads((args.dataset / "manifest.json").read_text())
        if digest(rows) != dataset_manifest["dataset_sha256"]:
            raise ValueError("Dataset modifié après préparation.")
        train = [r for r in rows if r["split"] == "train"]
        valid = [r for r in rows if r["split"] == "validation"]
        random.Random(args.seed).shuffle(valid)
        valid = valid[:args.validation_limit]
        if not train or not valid:
            raise ValueError("Entraînement et validation distincts requis.")
        student, teacher = (FeatureCache(args.cache / k) for k in ("student", "teacher"))
        corpus_hash = digest(read_jsonl(args.dataset / "corpus.jsonl"))
        for cache in (student, teacher):
            if cache.index["identity"]["corpus"] != corpus_hash:
                raise ValueError("Features d'un autre corpus.")
            cache.require([{"id": key} for r in train+valid for key in (r["caption_id"], r.get("caption_plain_id", r["caption_id"]))])
        for row in train+valid:
            for key in (row["caption_id"], row.get("caption_plain_id", row["caption_id"])):
                if student.index["entries"][key]["token_ids"] != teacher.index["entries"][key]["token_ids"]:
                    raise ValueError(f"Tokens désalignés : {key}")
        config = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}
        identity = {"dataset_sha256": digest(rows), "student": student.index["identity"], "teacher": teacher.index["identity"],
                    "student_index": file_hash(student.path), "teacher_index": file_hash(teacher.path),
                    "init_projector": file_hash(args.init_projector / "model.safetensors") if args.init_projector else None,
                    "trainer_sha256": file_hash(Path(__file__)),
                    "core_sha256": file_hash(ROOT / "common/sharm_image21/core.ipynb"),
                    "diffusion_sha256": file_hash(ROOT / "common/sharm_image21/diffusion.ipynb")}
        diffuser, latents, trajectory_noisy, trajectory_flow = None, None, None, None
        trajectory_by_scene = {}
        trajectory_training_rows = []
        if args.phase == "flow":
            latents = FeatureCache(args.cache / "latents")
            if latents.index["identity"]["dataset_sha256"] != digest(rows):
                raise ValueError("Latents d'un autre dataset.")
            latents.require(train+valid)
            if args.distillation_weight:
                trajectory = json.loads((args.cache / "trajectories.json").read_text())
                if trajectory["identity"]["dataset_sha256"] != digest(rows) or trajectory["identity"]["teacher_index_sha256"] != file_hash(teacher.path):
                    raise ValueError("Trajectoires d'une autre révision.")
                trajectory_noisy, trajectory_flow = (FeatureCache(args.cache / k) for k in ("trajectory-latents", "trajectory-flows"))
                for cache in (trajectory_noisy, trajectory_flow):
                    if cache.index["identity"] != trajectory["identity"]:
                        raise ValueError("Identité des trajectoires incompatible.")
                    cache.require(trajectory["rows"])
                for record in trajectory["rows"]:
                    if "timestep" not in record:
                        raise ValueError("Cache de trajectoires incomplet.")
                    trajectory_by_scene.setdefault(record["scene_id"], []).append(record)
                trajectory_training_rows = [r for r in train if r["id"] in trajectory_by_scene]
                if not trajectory_training_rows:
                    raise ValueError("Aucune trajectoire d'entraînement ; les cibles de validation sont exclues des gradients.")
                identity["trajectory_identity"] = trajectory["identity"]
            identity["latent_identity"] = latents.index["identity"]
        signature = digest({"config": config, "identity": identity})
        args.output.mkdir(parents=True, exist_ok=True)
        complete_path = args.output / "training-complete.json"
        if complete_path.exists():
            if json.loads(complete_path.read_text())["signature"] != signature:
                raise ValueError("Une autre expérience occupe ce dossier.")
            print("Expérience déjà terminée.", flush=True)
            return
        if args.init_projector:
            initial = Projector.load(args.init_projector)
            sizes = {k: initial.settings[k] for k in ("input_dim", "output_dim", "bottleneck")}
            projector = Projector(**sizes, context_layers=args.context_layers, context_heads=args.context_heads, layout_enabled=True)
            result = projector.load_state_dict(initial.float().state_dict(), strict=False)
            if result.unexpected_keys:
                raise ValueError(f"Architecture initiale incompatible : {result.unexpected_keys}")
            del initial
        else:
            projector = Projector(context_layers=args.context_layers, context_heads=args.context_heads, layout_enabled=True)
        projector = projector.float().to(device)
        groups = [{"params": list(projector.parameters()), "lr": args.learning_rate, "initial_lr": args.learning_rate}]
        if args.phase == "flow":
            diffuser = load_transformer(training=True, gradient_checkpointing=args.checkpointing_mode)
            if args.lora_rank:
                from peft import LoraConfig
                diffuser.add_adapter(LoraConfig(r=args.lora_rank, lora_alpha=args.lora_rank,
                                                target_modules=["to_q", "to_k", "to_v", "to_out.0"], init_lora_weights="gaussian"))
                groups.append({"params": [p for p in diffuser.parameters() if p.requires_grad],
                               "lr": args.lora_learning_rate, "initial_lr": args.lora_learning_rate})
        if device.type == "cuda":
            import bitsandbytes as bnb
            optimizer = bnb.optim.AdamW8bit(groups, weight_decay=0.01)
        else:
            optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
        parameters = [p for group in groups for p in group["params"]]
        checkpoint = args.output / "checkpoint.pt"
        first_step, best, stale = 0, float("inf"), 0
        if checkpoint.exists():
            state = torch.load(checkpoint, map_location="cpu", weights_only=True)
            if state["signature"] != signature:
                raise ValueError("Checkpoint incompatible : choisir un autre dossier.")
            projector.load_state_dict(state["projector"])
            if state["lora"]:
                load_adapter_state(diffuser, state["lora"])
            optimizer.load_state_dict(state["optimizer"])
            first_step, best, stale = state["step"], state["best"], state["stale"]
            torch.set_rng_state(state["torch_rng"])
            if device.type == "cuda":
                torch.cuda.set_rng_state(state["cuda_rng"])
            random.setstate(state["python_rng"])
            del state

        def features(row, use_layout):
            dtype = torch.bfloat16 if device.type == "cuda" else torch.float32
            key = row["caption_id"] if use_layout else row.get("caption_plain_id", row["caption_id"])
            source = student.get(key).to(device=device, dtype=dtype)
            gold = teacher.get(key).to(device=device, dtype=dtype)
            boxes = torch.tensor([o["box"] for o in row["objects"]], device=device, dtype=dtype) if use_layout else None
            predicted = projector(source, layout=boxes)
            return predicted, gold, len(source)

        @torch.no_grad()
        def evaluate():
            projector.eval()
            if diffuser is not None:
                diffuser.eval()
            scores = []
            for index, row in enumerate(valid):
                with amp():
                    if args.phase == "warmup":
                        predicted, gold, length = features(row, False)
                        scores.append(embedding_loss(predicted[:length], gold).item())
                    else:
                        # Both inference modes must be monitored; hold noise/t fixed.
                        clean = latents.get(row["id"])[None].cuda()
                        generator = torch.Generator(device="cuda").manual_seed(8000+index)
                        noise = torch.randn(clean.shape, device="cuda", dtype=clean.dtype, generator=generator)
                        t = torch.tensor([0.15+0.7*((index % 7)/6)], device="cuda", dtype=torch.bfloat16)
                        losses = []
                        for use_layout in (False, True):
                            predicted, _, _ = features(row, use_layout)
                            flow = predict_flow(diffuser, (1-t.view(1, 1, 1, 1, 1))*clean+t.view(1, 1, 1, 1, 1)*noise, predicted[None], t)
                            losses.append(F.mse_loss(flow.float(), (noise-clean).float()).item())
                        scores.append(sum(losses)/len(losses))
            return sum(scores)/len(scores)

        started = time.monotonic()
        last_step = first_step
        for step in range(first_step, args.steps if stale < args.patience else first_step):
            projector.train()
            if diffuser is not None:
                diffuser.train()
            optimizer.zero_grad(set_to_none=True)
            scale = min(1, (step+1)/max(1, int(args.steps*0.05)))*0.5*(1+math.cos(math.pi*step/args.steps))
            for group in optimizer.param_groups:
                group["lr"] = group["initial_lr"]*scale
            totals = {"embedding": 0.0, "supervised": 0.0, "distillation": 0.0}
            for micro in range(args.accumulation):
                row = random.choice(train)
                with amp():
                    projected, gold, length = features(row, args.phase == "flow" and random.random() >= args.layout_dropout)
                    alignment = embedding_loss(projected[:length], gold)
                    if args.phase == "warmup":
                        loss = alignment
                    else:
                        clean = latents.get(row["id"])[None].cuda()
                        noise = torch.randn_like(clean)
                        t = torch.rand(1, device="cuda", dtype=torch.float32).clamp(0.01, 0.99).to(torch.bfloat16)
                        sigma = t.view(1, 1, 1, 1, 1)
                        flow = predict_flow(diffuser, (1-sigma)*clean+sigma*noise, projected[None], t)
                        supervised = F.mse_loss(flow.float(), (noise-clean).float())
                        loss = supervised+args.embedding_weight*alignment
                        totals["supervised"] += supervised.item()/args.accumulation
                if not torch.isfinite(loss):
                    raise RuntimeError("Perte non finie.")
                (loss/args.accumulation).backward()
                totals["embedding"] += alignment.item()/args.accumulation
                # Separate backwards free the first diffuser graph before distillation.
                if args.phase == "flow" and args.distillation_weight:
                    distill_row = random.choice(trajectory_training_rows)
                    record = random.choice(trajectory_by_scene[distill_row["id"]])
                    with amp():
                        projected, _, _ = features(distill_row, random.random() >= args.layout_dropout)
                        noisy = trajectory_noisy.get(record["id"])[None].cuda()
                        target = trajectory_flow.get(record["id"])[None].cuda()
                        t = torch.tensor([record["timestep"]], device="cuda", dtype=torch.bfloat16)
                        distillation = relative_mse(predict_flow(diffuser, noisy, projected[None], t), target)
                    if not torch.isfinite(distillation):
                        raise RuntimeError("Distillation non finie.")
                    (args.distillation_weight*distillation/args.accumulation).backward()
                    totals["distillation"] += distillation.item()/args.accumulation
            norm = torch.nn.utils.clip_grad_norm_(parameters, 1.0, error_if_nonfinite=True)
            if norm == 0:
                raise RuntimeError("Aucun gradient : vérifier le passage arrière jusqu'au projecteur.")
            optimizer.step()
            last_step = step+1
            report = {"phase": args.phase, "step": last_step, "steps": args.steps, "losses": totals,
                      "seconds": time.monotonic()-started, "gpu_peak_gib": torch.cuda.max_memory_allocated()/2**30 if device.type == "cuda" else 0}
            if last_step % 5 == 0 or step == first_step:
                atomic_json(args.output / "status.json", report)
                print(json.dumps(report), flush=True)
            if last_step % args.eval_every == 0 or last_step == args.steps:
                score = evaluate()
                if not math.isfinite(score):
                    raise RuntimeError("Perte de validation non finie.")
                report["numerical_validation_loss"] = score
                if score < best:
                    best, stale = score, 0
                    save_candidate(projector, diffuser, args.output / "best", {"signature": signature, "identity": identity,
                                   "config": config, "step": last_step, "numerical_validation_loss": score})
                else:
                    stale += 1
                lora_state = {}
                if args.lora_rank:
                    from peft.utils import get_peft_model_state_dict
                    lora_state = {k: v.detach().cpu() for k, v in get_peft_model_state_dict(diffuser).items()}
                state = {"signature": signature, "step": last_step, "best": best, "stale": stale,
                         "projector": {k: v.detach().cpu() for k, v in projector.state_dict().items()},
                         "lora": lora_state,
                         "optimizer": optimizer.state_dict(), "torch_rng": torch.get_rng_state(),
                         "cuda_rng": torch.cuda.get_rng_state() if device.type == "cuda" else None, "python_rng": random.getstate()}
                temporary = checkpoint.with_suffix(".tmp")
                torch.save(state, temporary)
                temporary.replace(checkpoint)
                with (args.output / "history.jsonl").open("a", encoding="utf-8") as stream:
                    stream.write(json.dumps(report)+"\n")
                if stale >= args.patience:
                    break
        atomic_json(complete_path, {"signature": signature, "config": config, "identity": identity,
                    "steps_completed": last_step, "best_numerical_validation_loss": best,
                    "spatial_quality_validated": False, "reason": "early_stopping" if stale >= args.patience else "steps_completed"})
        print(f"Candidat enregistré : {args.output / 'best'}. Évaluation spatiale requise.", flush=True)



## base.verify_runtime

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_54e8e19bf7723539':
    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    """Exercise the real NF4 backward into contextual and layout projector weights."""


    import argparse


    import json


    import sys


    from pathlib import Path


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    import torch


    from sharm_image21.core import Projector, atomic_json


    from sharm_image21.diffusion import load_transformer, predict_flow


    def run(parameters=None):
        args = parameter_namespace({'resolution': 128, 'tokens': 32, 'context_layers': 2}, parameters, required=[], converters={'resolution': int, 'tokens': int, 'context_layers': int}, choices={}, groups=[])
        if args.resolution < 64 or args.resolution % 32:
            raise ValueError("Résolution multiple de 32 requise.")
        if not 1 <= args.tokens <= 4096 or args.context_layers < 1:
            raise ValueError("De 1 à 4096 tokens et au moins un bloc contextuel requis.")
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requis.")
        import bitsandbytes as bnb
        torch.cuda.reset_peak_memory_stats()
        diffuser = load_transformer(training=True, gradient_checkpointing="all").train()
        projector = Projector(context_layers=args.context_layers, layout_enabled=True).cuda().train()
        optimizer = bnb.optim.AdamW8bit(projector.parameters(), lr=1e-4)
        hidden = torch.randn(args.tokens, 2560, device="cuda", dtype=torch.bfloat16)
        boxes = torch.tensor([[0.1, 0.3, 0.35, 0.7], [0.65, 0.3, 0.9, 0.7]], device="cuda", dtype=torch.bfloat16)
        latent = torch.randn(1, 64, 1, args.resolution//16, args.resolution//16, device="cuda", dtype=torch.bfloat16)
        losses = []
        for step in range(2):
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                predicted = predict_flow(diffuser, latent, projector(hidden, layout=boxes)[None],
                                         torch.tensor([0.5], device="cuda", dtype=torch.bfloat16))
            loss = predicted.float().square().mean()
            if not torch.isfinite(loss):
                raise RuntimeError("Perte NF4 non finie.")
            loss.backward()
            norm = torch.nn.utils.clip_grad_norm_(projector.parameters(), 1, error_if_nonfinite=True)
            if not norm or any(p.grad is not None for p in diffuser.parameters()):
                raise RuntimeError("Gradient absent ou poids du diffuseur dégelés.")
            if step == 1:
                for prefix in ("context.", "layout_box.", "layout_object."):
                    if not any(p.grad is not None and p.grad.abs().sum() > 0 for n, p in projector.named_parameters() if n.startswith(prefix)):
                        raise RuntimeError(f"Aucun gradient dans {prefix}")
            optimizer.step()
            losses.append(loss.item())
        report = {"passed": True, "backward_into_projector": True, "diffuser_frozen": True,
                  "resolution": args.resolution, "tokens": args.tokens, "context_layers": args.context_layers,
                  "optimizer": "AdamW8bit", "losses": losses,
                  "peak_gib": torch.cuda.max_memory_allocated()/2**30,
                  "scope": "two synthetic NF4 backward steps; generated spatial quality is not evaluated"}
        atomic_json(ROOT / "artifacts/spatial/runtime-backward-audit.json", report)
        print(json.dumps(report), flush=True)



## base.verify_trajectory_runtime

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_9f593c1bbf628422':
    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    """Execute a complete 20-step real NF4 teacher trajectory across Euler updates."""


    import json


    from pathlib import Path


    import sys


    import time


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    import numpy as np


    import torch


    from sharm_image21.core import FeatureCache, ORIGINAL, atomic_json, file_hash, read_jsonl


    from sharm_image21.diffusion import load_transformer, predict_flow


    @torch.no_grad()
    def run(parameters=None):
        args = parameter_namespace({}, parameters, required=[], converters={}, choices={}, groups=[])
        from diffusers import FlowMatchEulerDiscreteScheduler
        from diffusers.pipelines.qwenimage21.pipeline_qwenimage21 import calculate_shift, retrieve_timesteps
        rows = read_jsonl(ROOT / "artifacts/spatial/dataset/scenes.jsonl")
        row = sorted((r for r in rows if r["split"] == "train"), key=lambda r: r["id"])[0]
        resolution = json.loads((ROOT / "artifacts/spatial/dataset/manifest.json").read_text())["resolution"]
        reference = FeatureCache(ROOT / "cache/spatial/teacher").get(row["caption_plain_id"])[None].cuda()
        scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(ORIGINAL, subfolder="scheduler", local_files_only=True)
        mu = calculate_shift((resolution//16)**2, scheduler.config.get("base_image_seq_len", 256),
                             scheduler.config.get("max_image_seq_len", 4096), scheduler.config.get("base_shift", 0.5),
                             scheduler.config.get("max_shift", 1.15))
        timesteps = retrieve_timesteps(scheduler, 20, "cuda", sigmas=np.linspace(1, 1/20, 20), mu=mu)[0]
        model = load_transformer()
        torch.cuda.reset_peak_memory_stats()
        started = time.monotonic()
        generator = torch.Generator(device="cuda").manual_seed(700+int(row["id"][:8], 16))
        latent = torch.randn((1, 64, 1, resolution//16, resolution//16), device="cuda", dtype=torch.bfloat16, generator=generator)
        for step, timestep in enumerate(timesteps):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                target = predict_flow(model, latent, reference, timestep[None].to(torch.bfloat16)/1000)
            if target.dtype != reference.dtype or not torch.isfinite(target).all():
                raise ValueError("Prédiction NF4 incompatible avec la précision des embeddings.")
            latent = scheduler.step(target, timestep, latent, return_dict=False)[0]
            if latent.dtype != reference.dtype or not torch.isfinite(latent).all():
                raise ValueError("Le scheduler a changé la précision ou produit des latents non finis.")
            print(f"Euler BF16 : {step+1}/20", flush=True)
        report = {"passed": True, "steps": 20, "resolution": resolution, "scene_id": row["id"],
                  "latent_dtype": str(latent.dtype), "prediction_dtype": str(target.dtype),
                  "seconds": time.monotonic()-started, "peak_gib": torch.cuda.max_memory_allocated()/2**30,
                  "diffusion_sha256": file_hash(ROOT / "common/sharm_image21/diffusion.ipynb"),
                  "scope": "Full real NF4 Euler trajectory regression; not generated spatial quality"}
        atomic_json(ROOT / "artifacts/spatial/trajectory-runtime-audit.json", report)
        print(json.dumps(report), flush=True)



## tools.archive_spatial_results

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_fdeb9ac65efb4c4a':
    """Archive reviewed spatial results for Git after verified HF publication; no weights."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    from collections import Counter


    import datetime


    import importlib.util


    import json


    from pathlib import Path


    import re


    import shutil


    import subprocess


    from sharm_notebook_runtime import PROJECT, contained, read_json, sha256


    def write(path, value):
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


    def select(value, keys):
        return {k: value[k] for k in keys if k in value}


    def archive_rejected(source):
        """Preserve a completed, rejected candidate without claiming HF publication."""
        run = source / "run"
        if read_json(run / "run-status.json", {}).get("state") != "complete":
            raise ValueError("La chaîne doit être terminée avant son archivage de rejet.")
        predictions_path = run / "validation/predictions.json"
        predictions = read_json(predictions_path)
        candidate_path = run / "flow/best/candidate.json"
        candidate = read_json(candidate_path)
        review = read_json(source / "visual-review.json")
        if sha256(candidate_path) != predictions["config"]["candidate_sha256"]:
            raise ValueError("La galerie ne correspond pas au candidat archivé.")
        rows = predictions["rows"]
        candidate_ids = {r["id"] for r in rows if r["method"].startswith("candidate_")}
        if (review.get("passed") is not False or not review.get("reviewer")
                or review.get("method") not in ("agent_visual_review", "human_visual_review")
                or review.get("projector_sha256") != candidate["projector_sha256"]
                or review.get("prediction_signature") != predictions["signature"]
                or not candidate_ids or not candidate_ids <= set(review.get("reviewed_ids", []))):
            raise ValueError("Une revue de rejet couvrant exactement ce candidat est requise.")
        expected_keys = {(scene, seed, method) for scene in predictions["config"]["rows"]
                         for seed in predictions["config"]["seeds"] for method in predictions["config"]["methods"]}
        actual_keys = [(r["scene_id"], r["seed"], r["method"]) for r in rows]
        if set(actual_keys) != expected_keys or len(actual_keys) != len(expected_keys) or len({r["id"] for r in rows}) != len(rows):
            raise ValueError("La galerie appariée doit être complète et sans doublon.")
        completions = {}
        for phase in ("warmup", "flow"):
            complete = read_json(run / phase / "training-complete.json")
            if not complete or complete.get("steps_completed", 0) < 1:
                raise ValueError("Rapports des deux phases terminées requis.")
            completions[phase] = complete
        flow = completions["flow"]
        if flow["signature"] != candidate["signature"] or flow["steps_completed"] < candidate["step"]:
            raise ValueError("Le candidat ne correspond pas à la phase flow terminée.")
        profile = read_json(source / "run/profile.json")
        if not profile:
            raise ValueError("Le profil exact du run est requis.")
        for phase, complete in completions.items():
            for key, expected in profile[phase].items():
                if complete["config"].get(key.replace("-", "_")) != expected:
                    raise ValueError("Le profil archivé ne correspond pas aux calculs terminés.")
        copies = {"run/validation/predictions.json": predictions_path,
                  "candidate.json": candidate_path, "visual-review.json": source / "visual-review.json",
                  "profile.json": source / "run/profile.json"}
        for phase in completions:
            copies[f"run/{phase}/history.jsonl"] = run / phase / "history.jsonl"
        for row in rows:
            path = contained(run / "validation", row["file_name"])
            if path.suffix != ".png" or sha256(path) != row["image_sha256"]:
                raise ValueError("PNG modifié depuis le manifeste des générations.")
            copies["run/validation/" + row["file_name"]] = path
            preview = contained(run / "validation", row["preview"])
            if preview.suffix != ".png":
                raise ValueError("Un aperçu PNG est requis.")
            copies["run/validation/" + row["preview"]] = preview
        for name in ("post-training-diagnostics.json", "warmup-diagnostic-review.json"):
            if (source / name).is_file():
                copies[name] = source / name
        if (run / "validation/review.html").is_file():
            copies["run/validation/review.html"] = run / "validation/review.html"
        # A diagnostic gallery is optional and explicitly separate from the full flow evaluation.
        diagnostic = source / "warmup-diagnostic/predictions.json"
        if diagnostic.is_file():
            data = read_json(diagnostic)
            copies["diagnostics/warmup/predictions.json"] = diagnostic
            for row in data["rows"]:
                path = contained(diagnostic.parent, row["file_name"])
                if path.suffix != ".png" or sha256(path) != row["image_sha256"]:
                    raise ValueError("PNG de diagnostic modifié.")
                for key in ("file_name", "preview"):
                    copies["diagnostics/warmup/" + row[key]] = contained(diagnostic.parent, row[key])
        for name, path in copies.items():
            if not path.is_file() or path.stat().st_size > 10 * 1024 * 1024:
                raise ValueError(f"Rapport absent ou trop volumineux : {name}")
        report_id = "rejected-" + sha256(candidate_path)[:12]
        destination = PROJECT / "Sharm-Image-2.1-VL4B/reports" / report_id
        staging = source / "notebook-report-staging" / report_id
        if destination.exists() or staging.exists():
            raise FileExistsError("Archive ou brouillon déjà présent ; inspecter avant reprise.")
        staging.mkdir(parents=True)
        for name, path in copies.items():
            target = contained(staging, name)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path, target)
        completion_keys = ("signature", "steps_completed", "best_numerical_validation_loss", "spatial_quality_validated", "reason")
        for phase, complete in completions.items():
            write(staging / f"run/{phase}/training-complete.json", select(complete, completion_keys))
            write(staging / f"run/{phase}/parameters.json", complete["config"])
        write(staging / "summary.json", {
            "schema": 1, "model": "Sharm Image Base", "outcome": "rejected",
            "published": False, "hf_repo": "ethanzxv/Sharm-Image-2.1-VL4B-FP8", "hf_revision": None,
            "candidate_sha256": sha256(candidate_path), "projector_sha256": candidate["projector_sha256"],
            "dataset_sha256": candidate["identity"]["dataset_sha256"],
            "generation_counts": dict(Counter(r["method"] for r in rows)),
            "visual_review_method": review["method"], "spatial_quality_validated": False,
            "scope": "Completed flow candidate rejected by agent visual review; no human accuracy estimate",
            "archived_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "archive_code_revision": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip(),
            "training_identity": flow.get("identity", {}),
        })
        files = {p.relative_to(staging).as_posix(): sha256(p) for p in sorted(staging.rglob("*")) if p.is_file()}
        write(staging / "manifest.json", {"schema": 1, "files": files})
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(staging, destination)
        write(destination.parent / "current.json", {"schema": 1, "report": report_id, "outcome": "rejected", "hf_revision": None,
                                                    "manifest_sha256": sha256(destination / "manifest.json")})
        print(json.dumps({"report": report_id, "outcome": "rejected", "files": len(files), "published": False}))


    def run(parameters=None):
        args = parameter_namespace({'artifacts': PROJECT / 'artifacts/spatial', 'outcome': 'published'}, parameters, required=[], converters={'artifacts': Path}, choices={'outcome': ('published', 'rejected')}, groups=[])
        source = args.artifacts.resolve()
        if args.outcome == "rejected":
            return archive_rejected(source)
        publication = read_json(source / "publication-status.json", {})
        if publication.get("state") != "complete" or not re.fullmatch(r"[a-f0-9]{40}", publication.get("revision", "")):
            raise ValueError("Publication HF vérifiée et révision immuable requises avant archivage final.")
        if publication.get("repo_id") != "ethanzxv/Sharm-Image-2.1-VL4B-FP8":
            raise ValueError("La publication ne concerne pas Sharm Image Base.")
        run = source / "run"
        if read_json(run / "run-status.json", {}).get("state") != "complete":
            raise ValueError("La chaîne de génération doit être terminée.")
        warmup = read_json(run / "warmup/training-complete.json")
        flow = read_json(run / "flow/training-complete.json")
        if not warmup or not flow:
            raise ValueError("Rapports des deux phases requis.")
        predictions_path = run / "validation/predictions.json"
        predictions = read_json(predictions_path)
        release = source / "release"
        # Reuse the publisher's integrity checks, including the shipped loader report.
        publisher = load_notebook(PROJECT / "publish/publish_spatial.ipynb", name="spatial_publisher")
        manifest, _ = publisher.verified_payload(release)
        if publication.get("projector_sha256") != manifest["projector_sha256"]:
            raise ValueError("Publication et export ne décrivent pas le même projecteur.")
        review = read_json(release / "visual-review.json")
        candidate_ids = {r["id"] for r in predictions["rows"] if r["method"].startswith("candidate_")}
        if (review.get("passed") is not True or review.get("method") not in ("agent_visual_review", "human_visual_review")
                or not review.get("reviewer") or review.get("projector_sha256") != manifest["projector_sha256"]
                or review.get("prediction_signature") != predictions["signature"]
                or not candidate_ids <= set(review.get("reviewed_ids", []))):
            raise ValueError("Revue complète liée à cette galerie et à ce projecteur requise.")
        if predictions != read_json(release / "comparisons/spatial/predictions.json"):
            raise ValueError("La galerie locale diffère de la galerie publiée.")
        methods = Counter(r["method"] for r in predictions["rows"])
        expected = len(predictions["config"]["rows"]) * len(predictions["config"]["seeds"])
        if not candidate_ids or set(methods) != set(predictions["config"]["methods"]) or any(n != expected for n in methods.values()):
            raise ValueError("Galerie incomplète.")
        copies = {}
        for row in predictions["rows"]:
            image = contained(run / "validation", row["file_name"])
            if sha256(image) != row["image_sha256"]:
                raise ValueError("Une génération a changé depuis son manifeste.")
            copies["run/validation/" + row["file_name"]] = image
            preview = contained(run / "validation", row["preview"])
            if not preview.is_file():
                raise FileNotFoundError(preview)
            if sha256(preview) != sha256(contained(release / "comparisons/spatial", row["preview"])):
                raise ValueError("Un aperçu diffère de la galerie publiée.")
            copies["run/validation/" + row["preview"]] = preview
        verification = read_json(release / "export-validation.json")
        for name in verification["images"]:
            copies["release/" + name] = contained(release, name)
        # Only named JSON reports and PNGs enter Git. No recursive artifact copy.
        copies.update({"run/validation/predictions.json": predictions_path,
                       "visual-review.json": release / "visual-review.json",
                       "release/export-validation.json": release / "export-validation.json",
                       "release/release_manifest.json": release / "release_manifest.json"})
        for phase in ("warmup", "flow"):
            copies[f"run/{phase}/history.jsonl"] = run / phase / "history.jsonl"
        for name, path in copies.items():
            if not path.is_file() or path.stat().st_size > 10 * 1024 * 1024:
                raise ValueError(f"Fichier absent ou trop volumineux pour le rapport : {name}")
        profile = read_json(PROJECT / "Sharm-Image-2.1-VL4B/configs/remote16gb-256.json")
        config = flow.get("config", {})
        # Fail on a changed local recipe instead of attributing it to an older run.
        for key, expected_value in profile["flow"].items():
            if config.get(key.replace("-", "_")) != expected_value:
                raise ValueError("Le profil local ne correspond plus au run ; archiver son profil exact.")
        report_id = "spatial-" + publication["revision"][:12]
        destination = PROJECT / "Sharm-Image-2.1-VL4B/reports" / report_id
        if destination.exists():
            raise FileExistsError("Le rapport est immuable ; consulter son manifeste, ne pas l'écraser.")
        staging = source / "notebook-report-staging" / report_id
        if staging.exists():
            raise FileExistsError("Un brouillon d'archivage existe déjà ; l'inspecter avant reprise.")
        staging.mkdir(parents=True)
        for name, path in copies.items():
            target = contained(staging, name)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path, target)
        completion_keys = ("signature", "steps_completed", "best_numerical_validation_loss", "spatial_quality_validated", "reason")
        for phase, value in (("warmup", warmup), ("flow", flow)):
            write(staging / f"run/{phase}/training-complete.json", select(value, completion_keys))
        write(staging / "publication-status.json", select(publication, ("state", "repo_id", "revision", "private", "files", "projector_sha256")))
        write(staging / "profile.json", profile)
        code_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip()
        write(staging / "summary.json", {
            "schema": 1, "model": "Sharm Image Base", "hf_repo": publication["repo_id"],
            "hf_revision": publication["revision"], "projector_sha256": manifest["projector_sha256"],
            "dataset_sha256": manifest["dataset_sha256"], "generation_counts": dict(methods),
            "visual_review_method": review["method"], "spatial_quality_validated": False,
            "scope": "Reviewed experimental candidate; no statistical human spatial validation claimed",
            "archived_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "archive_code_revision": code_revision,
            "training_identity": flow.get("identity", {}),
        })
        files = {path.relative_to(staging).as_posix(): sha256(path) for path in sorted(staging.rglob("*")) if path.is_file()}
        write(staging / "manifest.json", {"schema": 1, "files": files})
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(staging, destination)
        write(destination.parent / "current.json", {"schema": 1, "report": report_id, "hf_revision": publication["revision"],
                                                    "manifest_sha256": sha256(destination / "manifest.json")})
        print(json.dumps({"report": report_id, "files": len(files), "next": "Review the report, commit with Angular convention and push origin/main"}))



## tools.clean_notebooks

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_fe9131e26a12630d':
    from sharm_notebook_runtime import PROJECT, parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    """Remove execution output from the versioned notebooks without running cells."""


    import json


    from pathlib import Path


    def run(parameters=None):
        args = parameter_namespace({}, parameters, required=[], converters={}, choices={}, groups=[])
        changed = []
        for path in sorted([PROJECT / "Sharm-Image-2.1-VL4B/Sharm_Image_2_1_VL4B.ipynb", PROJECT / "Sharm-Image-2.1-VL4B-LoRA-Monster/Sharm_Image_2_1_VL4B_LoRA_Monster.ipynb"]):
            notebook = json.loads(path.read_text(encoding="utf-8"))
            notebook.get("metadata", {}).pop("widgets", None)
            for cell in notebook["cells"]:
                if cell["cell_type"] == "code":
                    cell["outputs"] = []
                    cell["execution_count"] = None
                    cell.get("metadata", {}).pop("execution", None)
            content = json.dumps(notebook, ensure_ascii=False, indent=1) + "\n"
            if path.read_text(encoding="utf-8") != content:
                path.write_text(content, encoding="utf-8", newline="\n")
                changed.append(path.name)
        print(json.dumps({"cleaned": changed}, ensure_ascii=False))



## publish.collect_remote_release

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_36e5d5812e5d6337':
    """Assemble the exact remotely verified release using identical local base weights."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    from pathlib import Path


    from publish_spatial import (ARTIFACTS, PROJECT, REPO, contained, link_file,
                                 payload_path_allowed, read, sha, verified_payload, write)


    def base_source(name, project=PROJECT):
        """Only public, frozen weight components can be reused from the local machine."""
        contained(project, name)
        parts = name.split("/")
        if len(parts) != 2 or not name.endswith(".safetensors"):
            return None
        if parts[0] == "text_encoder":
            return contained(project / "1-local-generation/weights/sharm-vl4b-fp8", parts[1], ".safetensors")
        if parts[0] in ("transformer", "vae"):
            return contained(project / "1-local-generation/weights/qwen-image-2.1", name, ".safetensors")
        return None


    def collect(incoming, output, project=PROJECT):
        incoming, output = Path(incoming).resolve(), Path(output).resolve()
        if incoming == output or incoming.is_relative_to(output) or output.is_relative_to(incoming):
            raise ValueError("Les dossiers de réception et d'export doivent être distincts.")
        manifest = read(incoming / "release_manifest.json")
        verification = read(incoming / "export-validation.json")
        if (manifest.get("repo_id") != REPO or verification.get("passed") is not True
                or verification.get("release_manifest_sha256") != sha(incoming / "release_manifest.json")
                or verification.get("projector_sha256") != manifest["projector_sha256"]):
            raise ValueError("Un export vérifié sur le serveur pour ce dépôt précis est requis.")
        expected_images = {f"validation/{method}{suffix}.png" for method in ("candidate_text", "candidate_layout")
                           for suffix in ("", "-preview")}
        if set(verification.get("images", {})) != expected_images:
            raise ValueError("Les deux générations RGBA vérifiées et leurs aperçus sont requis.")
        files = {}
        for name, expected in manifest["files"].items():
            if not payload_path_allowed(name):
                raise ValueError(f"Fichier hors périmètre de publication : {name}")
            source = base_source(name, project) or contained(incoming, name)
            if sha(source) != expected:
                raise ValueError(f"Fichier local incompatible avec l'export distant : {name}")
            files[name] = source
        for name, expected in verification["images"].items():
            source = contained(incoming, name, ".png")
            if sha(source) != expected:
                raise ValueError(f"Image de vérification modifiée : {name}")
            files[name] = source
        for name in ("release_manifest.json", "export-validation.json"):
            files[name] = contained(incoming, name)
        # Validate the complete source set before populating the final release.
        for name, source in files.items():
            link_file(source, contained(output, name))
        _, payload = verified_payload(output)
        report = {"passed": True, "files": len(payload), "release_manifest_sha256": sha(output / "release_manifest.json"),
                  "projector_sha256": manifest["projector_sha256"], "reused_base_files": sum(base_source(n, project) is not None for n in files),
                  "scope": "Exact remote release assembled locally; no inference or remote publication performed"}
        write(output.parent / "remote-collection.json", report)
        return report


    def run(parameters=None):
        args = parameter_namespace({'incoming': ARTIFACTS / 'remote-release', 'output': ARTIFACTS / 'release'}, parameters, required=[], converters={'incoming': Path, 'output': Path}, choices={}, groups=[])
        import json
        print(json.dumps(collect(args.incoming, args.output)))



## publish.export_base

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_8d9cad92937be2f1':


    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import json


    import os


    import shutil


    import sys


    from pathlib import Path


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    from sharm_image21.core import (ARTIFACTS, ORIGINAL, ROOT, STUDENT, WEIGHTS,
                                    atomic_json, file_hash, model_identity)


    from sharm_image21.model_cards import base_card, measure_footprint, package_comparisons


    def link_or_copy(source, destination):
        try:
            os.link(source, destination)
        except OSError:
            shutil.copy2(source, destination)


    def write_presentation(destination, validation, fingerprint):
        source = ARTIFACTS / "base-comparisons"
        if validation.get("comparison_sha256") != file_hash(source / "comparison.json"):
            raise ValueError("La galerie VL ne correspond pas au rapport de validation.")
        comparison = package_comparisons(source, destination / "comparisons/encoder", "encoder", fingerprint)
        footprint = measure_footprint(ORIGINAL, STUDENT, ARTIFACTS / "projector")
        atomic_json(destination / "footprint.json", footprint)
        atomic_json(destination / "base-validation.json", validation)
        shutil.copy2(ROOT / "Sharm-Image-2.1-VL4B/requirements-inference.txt", destination / "requirements-inference.txt")
        (destination / "README.md").write_text(base_card(comparison, footprint, validation), encoding="utf-8")


    def write_checksums(destination):
        checksums = {str(path.relative_to(destination)): file_hash(path) for path in destination.rglob("*")
                     if path.is_file() and path != destination / "checksums.json"}
        atomic_json(destination / "checksums.json", checksums)


    def run(parameters=None):
        args = parameter_namespace({}, parameters, required=[], converters={}, choices={}, groups=[])
        destination = WEIGHTS / "sharm-image-2.1-vl4b-fp8"
        projector_dir = ARTIFACTS / "projector"
        validation = json.loads((ARTIFACTS / "base-validation.json").read_text())
        fingerprint = file_hash(projector_dir / "model.safetensors")
        if not validation.get("passed") or validation["projector_sha256"] != fingerprint:
            raise ValueError("Export d'une base non validée refusé.")
        if (destination / "sharm_config.json").exists():
            previous = json.loads((destination / "sharm_config.json").read_text())
            if previous["projector_sha256"] != fingerprint:
                raise ValueError("Une base d'une autre révision existe dans le dossier de sortie.")
            write_presentation(destination, validation, fingerprint)
            write_checksums(destination)
            print(f"Base déjà exportée, fiche et comparaisons actualisées : {destination}", flush=True)
            return
        staging = destination.with_name(destination.name + ".staging")
        staging.mkdir(mode=0o700, parents=True, exist_ok=True)
        for component in ("transformer", "vae", "scheduler"):
            shutil.copytree(ORIGINAL / component, staging / component, copy_function=link_or_copy, dirs_exist_ok=True)
        for path in STUDENT.iterdir():
            if not path.is_file() or path.name == "README.md":
                continue
            if path.suffix == ".safetensors" or path.name in ("config.json", "generation_config.json", "model.safetensors.index.json"):
                target = staging / "text_encoder" / path.name
            elif path.name.startswith(("tokenizer", "preprocessor", "video_preprocessor", "chat_template", "vocab", "merges")):
                target = staging / "processor" / path.name
            else:
                continue
            target.parent.mkdir(exist_ok=True)
            if not target.exists():
                link_or_copy(path, target)
        shutil.copytree(projector_dir, staging / "projector", copy_function=shutil.copy2, dirs_exist_ok=True)
        export_runtime(staging / "sharm_image21")
        shutil.copy2(ORIGINAL / "LICENSE", staging / "LICENSE")
        (staging / "NOTICE").write_text(
            "Built with Qwen.\nQwen is licensed under the Qwen RESEARCH LICENSE AGREEMENT, "
            "Copyright (c) 2026 Hangzhou Tongyi Laboratory Technology Co., Ltd. All Rights Reserved.\n"
            "Sharm modifies the conditioning pipeline and adds a trained projector.\n"
            "Qwen3-VL-4B-Instruct-FP8: Qwen Team, Apache-2.0.\n", encoding="utf-8")
        shutil.copy2(ROOT / "Sharm-Image-2.1-VL4B/licenses/Apache-2.0.txt", staging / "LICENSE.encoder")
        config = {"name": "Sharm Image 2.1 VL4B FP8", "encoder_format": "fp8-e4m3", "encoder_frozen": True,
                  "student_identity": model_identity("student"), "projector_sha256": fingerprint,
                  "base_validation_passed": True, "base_revision": model_identity("teacher")["revision"],
                  "scope": "experimental-text-to-image", "image_editing_validated": False,
                  "hf_repo_intent": "ethanzxv/Sharm-Image-2.1-VL4B-FP8", "private": True, "published": False}
        atomic_json(staging / "sharm_config.json", config)
        atomic_json(staging / "model_index.json", {"_class_name": "SharmImage21Pipeline", "_module": "sharm_image21.pipeline"})
        atomic_json(staging / "training_manifest.json", json.loads((projector_dir / "training-complete.json").read_text()))
        write_presentation(staging, validation, fingerprint)
        write_checksums(staging)
        staging.replace(destination)
        print(f"Base Sharm exportée localement, publication différée : {destination}", flush=True)



## publish.publish_monster

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_11fedeb32c91aa2f':
    import json
    import re
    from pathlib import Path
    from sharm_notebook_runtime import PROJECT, ARTIFACTS, read_json, sha256, contained, load_notebook

    REPO_ID = "ethanzxv/Sharm-Image-2.1-Sprites-128-LoRA"

    def run(parameters=None):
        p = dict(parameters or {})
        action = p.get("action", "prepare")
        output = Path(p.get("output", ARTIFACTS / "monster/release"))
        publisher = load_notebook(PROJECT / "publish/publish_spatial.ipynb")
        if action == "prepare":
            lora = Path(p["lora"])
            review = read_json(Path(p.get("review", ARTIFACTS / "monster/visual-review.json")), {})
            validation = Path(p.get("validation", ARTIFACTS / "monster/loader-validation"))
            verification = read_json(validation / "export-validation.json", {})
            config = read_json(lora / "sharm_lora_config.json")
            base = read_json(lora / "base-reference.json")
            comparison = read_json(lora / "comparisons/lora/comparison.json")
            if not re.fullmatch(r"[a-f0-9]{40}", base.get("revision", "")):
                raise ValueError("Révision HF immuable de la Base requise.")
            if (review.get("passed") is not True or not review.get("reviewer") or
                review.get("method") not in ("agent_visual_review", "human_visual_review") or
                review.get("comparison_sha256") != sha256(lora / "comparisons/lora/comparison.json") or
                not {r["id"] for r in comparison["pairs"]} <= set(review.get("reviewed_ids", []))):
                raise ValueError("Revue effective de toutes les paires Monster requise.")
            if (verification.get("passed") is not True or verification.get("lora_sha256") != config["lora_sha256"] or
                verification.get("projector_sha256") != base["projector_sha256"] or
                verification.get("comparison_sha256") != sha256(lora / "comparisons/lora/comparison.json") or
                sha256(lora / "pytorch_lora_weights.safetensors") != config["lora_sha256"]):
                raise ValueError("Vérification RGBA du même adaptateur et projecteur requise.")
            names = ["pytorch_lora_weights.safetensors", "sharm_lora_config.json", "base-reference.json", "README.md", "LICENSE", "LICENSE.encoder", "NOTICE", "requirements-inference.txt", "evaluation.json", "comparisons/lora/comparison.json"]
            for pair in comparison["pairs"]:
                for side in ("left", "right"):
                    for asset in ("source", "image", "preview"):
                        if asset in pair[side]:
                            item = pair[side][asset]
                            name = "comparisons/lora/" + item["file"]
                            if sha256(contained(lora, name)) != item["sha256"]:
                                raise ValueError("Galerie Monster modifiée.")
                            names.append(name)
            for name in set(names):
                publisher.link_file(contained(lora, name), contained(output, name))
            for name, checksum in verification["images"].items():
                source = contained(validation, name)
                if sha256(source) != checksum:raise ValueError("PNG de vérification modifié.")
                publisher.link_file(source, contained(output, "validation/" + name))
            publisher.write(output / "visual-review.json", review)
            publisher.write(output / "export-validation.json", verification)
            files = {f.relative_to(output).as_posix(): sha256(f) for f in output.rglob("*") if f.is_file() and f.name != "release_manifest.json"}
            publisher.write(output / "release_manifest.json", {"repo_id": REPO_ID, "base_reference": base, "files": files})
            print("Export Monster prêt :", len(files), "fichiers")
        elif action == "push":
            from huggingface_hub import HfApi, hf_hub_download
            api = HfApi()
            if api.whoami()["name"] != "ethanzxv":raise ValueError("Compte HF incorrect.")
            info = api.model_info(REPO_ID)
            if not info.private:raise ValueError("La destination autorisée est privée.")
            manifest = read_json(output / "release_manifest.json")
            if manifest.get("repo_id") != REPO_ID:raise ValueError("Destination incorrecte.")
            for name, checksum in manifest["files"].items():
                if sha256(contained(output, name)) != checksum:raise ValueError("Export modifié.")
            reference = manifest["base_reference"]
            remote_base = read_json(hf_hub_download(reference["repo_id"], "release_manifest.json", revision=reference["revision"]))
            if remote_base["projector_sha256"] != reference["projector_sha256"]:raise ValueError("Base HF incompatible.")
            commit = api.upload_folder(repo_id=REPO_ID, folder_path=output, parent_commit=info.sha,
                allow_patterns=[*manifest["files"], "release_manifest.json"], commit_message="Publish reviewed Monster LoRA bound to the spatial Base")
            remote = api.model_info(REPO_ID, revision=commit.oid, files_metadata=True)
            index = {f.rfilename: f for f in remote.siblings}
            for name in [*manifest["files"], "release_manifest.json"]:
                expected = sha256(output / name)
                item = index.get(name)
                if item is None or item.size != (output / name).stat().st_size:raise ValueError("Fichier HF absent ou tronqué.")
                actual = item.lfs.sha256 if item.lfs else sha256(hf_hub_download(REPO_ID, name, revision=commit.oid))
                if actual != expected:raise ValueError("Empreinte HF incorrecte.")
            if remote.private != info.private:raise ValueError("La visibilité HF a changé.")
            publisher.write(ARTIFACTS / "monster/publication-status.json", {"state": "complete", "repo_id": REPO_ID, "revision": commit.oid, "private": remote.private, "base_reference": reference})
            print("Publication vérifiée :", commit.oid)
        else:
            raise ValueError("action attend prepare ou push")



## publish.publish_private

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_7e2a94541d791432':
    """Publish validated Sharm releases privately and verify the exact HF revisions."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import hashlib


    import json


    import os


    from pathlib import Path


    import shutil


    import subprocess


    import sys


    import time


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    from huggingface_hub import HfApi, hf_hub_download, snapshot_download


    from huggingface_hub.errors import HfHubHTTPError


    from sharm_image21.core import ARTIFACTS, ROOT, WEIGHTS, atomic_json, file_hash


    from sharm_image21.model_cards import load_comparisons


    REPOS = {"base": "ethanzxv/Sharm-Image-2.1-VL4B-FP8",
             "lora": "ethanzxv/Sharm-Image-2.1-Sprites-128-LoRA"}


    SOURCES = {"base": WEIGHTS / "sharm-image-2.1-vl4b-fp8",
               "lora": WEIGHTS / "sharm-image-2.1-sprites-128-lora"}


    def status(state, **fields):
        path = ARTIFACTS / "publication-status.json"
        previous = json.loads(path.read_text()) if path.exists() else {}
        atomic_json(path, {**previous, "state": state, "updated_at": time.time(), **fields})
        print(json.dumps({"state": state, **fields}, ensure_ascii=False), flush=True)


    def hardlink_or_copy(source, destination):
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists():
            destination.unlink()
        try:
            os.link(source, destination)
        except OSError:
            shutil.copy2(source, destination)


    def prerequisites(component):
        request = json.loads((ARTIFACTS / "publication-request.json").read_text())
        if request.get("authorized") is not True or request.get("private") is not True or request.get("account") != "ethanzxv":
            raise RuntimeError("Une demande de publication privée pour ethanzxv est requise.")
        run = json.loads((ARTIFACTS / "run-status.json").read_text())
        if component != "base" and run.get("state") != "complete":
            raise RuntimeError("Les deux entraînements et leurs évaluations doivent être terminés.")
        base, lora = SOURCES["base"], SOURCES["lora"]
        config = json.loads((base / "sharm_config.json").read_text())
        fingerprint = config["projector_sha256"]
        if file_hash(base / "projector/model.safetensors") != fingerprint:
            raise ValueError("Le projecteur de l’export a changé.")
        for report in ("base-validation.json", "export-validation.json"):
            value = json.loads((ARTIFACTS / report).read_text())
            if value.get("passed") is not True or value["projector_sha256"] != fingerprint:
                raise ValueError(f"Validation incompatible ou manquante : {report}")
        if not (base / "projector/training-complete.json").is_file():
            raise RuntimeError("L’entraînement du projecteur n’est pas terminé.")
        load_comparisons(base / "comparisons/encoder", "encoder", fingerprint)
        review = json.loads((ARTIFACTS / "visual-review.json").read_text())
        if (review.get("encoder_passed") is not True
                or review.get("encoder_comparison_sha256") != file_hash(base / "comparisons/encoder/comparison.json")):
            raise ValueError("Une revue visuelle des images exactes du VL est requise.")
        if component == "base":
            return config, None, review
        if not (lora / "training-complete.json").is_file():
            raise RuntimeError("L’entraînement LoRA n’est pas terminé.")
        lora_config = json.loads((lora / "sharm_lora_config.json").read_text())
        lora_sha = file_hash(lora / "pytorch_lora_weights.safetensors")
        if lora_config["projector_sha256"] != fingerprint or lora_config["lora_sha256"] != lora_sha:
            raise ValueError("Les poids LoRA et leur déclaration sont incompatibles.")
        load_comparisons(lora / "comparisons/encoder", "encoder", fingerprint)
        lora_comparison = load_comparisons(lora / "comparisons/lora", "lora", fingerprint, lora_sha)
        if lora_config.get("recommended_weight", 1.0) != lora_comparison["controls"]["lora_weight"]:
            raise ValueError("La force recommandée et celle des comparaisons sont différentes.")
        if file_hash(base / "comparisons/encoder/comparison.json") != file_hash(lora / "comparisons/encoder/comparison.json"):
            raise ValueError("Les deux fiches ne montrent pas la même référence VL.")
        expected = {"encoder_comparison_sha256": file_hash(base / "comparisons/encoder/comparison.json"),
                    "lora_comparison_sha256": file_hash(lora / "comparisons/lora/comparison.json")}
        if (review.get("passed") is not True or review.get("lora_passed") is not True
                or any(review.get(key) != value for key, value in expected.items())):
            raise ValueError("La revue visuelle doit porter sur les images exactes de cette publication.")
        return config, lora_sha, review


    def release_files(kind):
        root = SOURCES[kind]
        names = ["README.md", "LICENSE", "LICENSE.encoder", "NOTICE", "footprint.json"]
        if kind == "base":
            names += ["sharm_config.json", "model_index.json", "training_manifest.json", "base-validation.json"]
            patterns = ("transformer/*.json", "transformer/*.safetensors", "vae/*.json", "vae/*.safetensors",
                        "scheduler/*.json", "text_encoder/*.json", "text_encoder/*.safetensors", "processor/*",
                        "projector/config.json", "projector/model.safetensors", "projector/training-complete.json",
                        "sharm_image21/*.py", "comparisons/encoder/*.png", "comparisons/encoder/comparison.json")
        else:
            names += ["pytorch_lora_weights.safetensors", "sharm_lora_config.json", "training-complete.json", "evaluation.json"]
            patterns = ("comparisons/encoder/*.png", "comparisons/encoder/comparison.json",
                        "comparisons/lora/*.png", "comparisons/lora/comparison.json")
        paths = {root / name for name in names}
        for pattern in patterns:
            paths.update(path for path in root.glob(pattern) if path.is_file())
        for path in paths:
            if not path.is_file() or not path.resolve().is_relative_to(root.resolve()):
                raise ValueError(f"Fichier de publication manquant ou externe : {path.name}")
            if path.suffix in (".pt", ".pth", ".log") or path.name.startswith("."):
                raise ValueError(f"Fichier privé hors export : {path.name}")
        return sorted(paths)


    def build_release(kind, config, lora_sha, review, base_hf_revision=None):
        source = SOURCES[kind]
        destination = ARTIFACTS / "hf-releases" / kind
        if not destination.resolve().is_relative_to(ARTIFACTS.resolve()):
            raise ValueError("Le dossier temporaire HF doit rester dans artifacts.")
        if destination.exists():
            shutil.rmtree(destination)
        destination.mkdir(mode=0o700, parents=True, exist_ok=True)
        for path in release_files(kind):
            hardlink_or_copy(path, destination / path.relative_to(source))
        # Configs are replaced atomically so the hardlinked local export stays intact.
        filename = "sharm_config.json" if kind == "base" else "sharm_lora_config.json"
        metadata = json.loads((destination / filename).read_text())
        metadata.pop("published", None)
        metadata.update(private=True, hf_repo_id=REPOS[kind])
        if kind == "lora":
            if not base_hf_revision:
                raise ValueError("La révision HF exacte de la base est requise pour publier le LoRA.")
            metadata.update(base_hf_revision=base_hf_revision, base_diffusion_revision=config["base_revision"])
        atomic_json(destination / filename, metadata)
        shutil.copy2(ROOT / "Sharm-Image-2.1-VL4B/requirements-inference.txt", destination / "requirements-inference.txt")
        atomic_json(destination / "visual-review.json", review)
        atomic_json(destination / "release_manifest.json", {
            "target_repo": REPOS[kind], "private": True, "projector_sha256": config["projector_sha256"],
            "original_base_revision": config["base_revision"], "student_identity": config["student_identity"],
            "lora_sha256": lora_sha if kind == "lora" else None, "base_hf_revision": base_hf_revision,
            "scope": "experimental text-to-image, image editing not validated", "prepared_at": time.time(),
            "source_export": source.name, "source_config_sha256": file_hash(source / filename),
        })
        card_path = destination / "README.md"
        # Replace a hardlink before editing the release-specific card.
        card = card_path.read_text(encoding="utf-8")
        notes = list(review.get("encoder_notes", []))
        if kind == "lora":
            card += f"\nRévision HF exacte de la base : `{base_hf_revision}`.\n"
            notes += review.get("lora_notes", [])
        card += "\n## Observations sur ce premier candidat\n\n"
        card += "Inspection qualitative des images par Codex ; la portée reste limitée aux paires présentées.\n\n"
        card += "\n".join(f"- {note}" for note in notes) + "\n\n"
        card += "Le rapport lié aux empreintes des images est disponible dans [visual-review.json](visual-review.json).\n"
        card_path.unlink()
        card_path.write_text(card, encoding="utf-8")
        checksums = {path.relative_to(destination).as_posix(): file_hash(path) for path in destination.rglob("*")
                     if path.is_file() and ".cache" not in path.relative_to(destination).parts
                     and path.name != "checksums.json"}
        atomic_json(destination / "checksums.json", checksums)
        return destination


    def ensure_private(api, repo):
        api.create_repo(repo, repo_type="model", private=True, exist_ok=True)
        info = api.model_info(repo)
        if info.private is not True:
            api.update_repo_settings(repo, private=True, repo_type="model")
        if api.model_info(repo).private is not True:
            raise RuntimeError(f"Confidentialité non confirmée pour {repo} ; envoi refusé.")


    def git_blob_hash(path):
        result = hashlib.sha1(f"blob {path.stat().st_size}\0".encode())
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
                result.update(block)
        return result.hexdigest()


    def verify_remote(api, repo, revision, directory):
        info = api.model_info(repo, revision=revision, files_metadata=True)
        if info.private is not True or info.sha != revision:
            raise RuntimeError("La révision distante ou sa confidentialité ne correspond pas.")
        local = json.loads((directory / "checksums.json").read_text())
        local["checksums.json"] = file_hash(directory / "checksums.json")
        remote = {entry.rfilename: entry for entry in info.siblings}
        unexpected = set(remote) - set(local) - {".gitattributes"}
        if unexpected:
            raise RuntimeError(f"Fichiers inattendus dans le dépôt HF : {sorted(unexpected)}")
        for filename, checksum in local.items():
            entry = remote.get(filename)
            path = directory / filename
            if entry is None or entry.size != path.stat().st_size:
                raise RuntimeError(f"Taille distante incorrecte : {filename}")
            if entry.lfs is not None:
                if entry.lfs.sha256 != checksum:
                    raise RuntimeError(f"Empreinte distante incorrecte : {filename}")
            elif entry.blob_id != git_blob_hash(path):
                raise RuntimeError(f"Contenu Git distant incorrect : {filename}")
        # Fetch metadata from the authenticated endpoint at the immutable revision.
        for filename in ("README.md", "checksums.json", "release_manifest.json"):
            fetched = Path(hf_hub_download(repo, filename, revision=revision, token=True))
            if file_hash(fetched) != local[filename]:
                raise RuntimeError(f"Relecture HF incohérente : {filename}")
        try:
            HfApi(token=False).model_info(repo)
        except HfHubHTTPError as error:
            if error.response.status_code not in (401, 403, 404):
                raise
        else:
            raise RuntimeError("Le dépôt est accessible sans authentification ; publication interrompue.")
        return {"repo_id": repo, "revision": revision, "private": True, "verified_files": len(local),
                "bytes": sum((directory / name).stat().st_size for name in local), "anonymous_access": False}


    def upload(api, kind, directory):
        repo = REPOS[kind]
        ensure_private(api, repo)
        status(f"uploading_{kind}", current_repo=repo)
        commit = api.upload_folder(repo_id=repo, repo_type="model", folder_path=directory,
                                   ignore_patterns=[".cache/**", "**/__pycache__/**", "*.pyc"],
                                   commit_message=f"Publish validated private Sharm {kind} with paired comparisons")
        status(f"verifying_{kind}", current_repo=repo, current_revision=commit.oid)
        return verify_remote(api, repo, commit.oid, directory)


    def run(parameters=None):
        args = parameter_namespace({'component': 'both', 'verify_inference': False}, parameters, required=[], converters={}, choices={'component': ('base', 'lora', 'both')}, groups=[])
        if args.component == "base" and args.verify_inference:
            parser.error("La vérification d’inférence combinée suit la publication du LoRA.")
        os.umask(0o077)
        api = HfApi()
        if api.whoami().get("name") != "ethanzxv":
            raise RuntimeError("Authentification HF du compte ethanzxv requise.")
        config, lora_sha, review = prerequisites(args.component)
        status("preparing", private=True, account="ethanzxv")
        if args.component != "lora":
            base_dir = build_release("base", config, lora_sha, review)
            base = upload(api, "base", base_dir)
            status("base_published", base=base)
            if args.component == "base":
                return
        else:
            previous = json.loads((ARTIFACTS / "publication-status.json").read_text())
            published = previous["base"]
            base_dir = ARTIFACTS / "hf-releases/base"
            manifest = json.loads((base_dir / "release_manifest.json").read_text())
            if manifest["projector_sha256"] != config["projector_sha256"]:
                raise ValueError("La base déjà publiée utilise un autre projecteur.")
            ensure_private(api, REPOS["base"])
            base = verify_remote(api, REPOS["base"], published["revision"], base_dir)
        lora_dir = build_release("lora", config, lora_sha, review, base_hf_revision=base["revision"])
        lora = upload(api, "lora", lora_dir)
        status("uploads_verified", base=base, lora=lora)
        if args.verify_inference:
            status("verifying_published_inference")
            # HF checks server etags and can reuse identical local weight files.
            snapshot_download(REPOS["base"], revision=base["revision"], token=True, local_dir=base_dir)
            snapshot_download(REPOS["lora"], revision=lora["revision"], token=True, local_dir=lora_dir)
            call_with_flags(Path(__file__).with_name("verify_published.ipynb"), ["--base", str(base_dir), "--lora", str(lora_dir), "--output", str(ARTIFACTS)])
        status("complete", base=base, lora=lora, inference_verified=args.verify_inference)
        path = ARTIFACTS / "run-status.json"
        run = json.loads(path.read_text())
        atomic_json(path, {**run, "publish": True, "publication": "published_private", "published_revisions": {
            "base": base["revision"], "lora": lora["revision"]}, "updated_at": time.time()})



## publish.publish_spatial

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_e3a86f080f4edb4f':
    """Package and publish the completed spatial candidate to its authorized HF repo."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import hashlib


    import json


    import math


    import os


    from pathlib import Path, PurePosixPath


    import shutil


    ROOT = Path(__file__).resolve().parents[1]


    PROJECT = ROOT.parent


    ARTIFACTS = ROOT / "artifacts/spatial"


    REPO = "ethanzxv/Sharm-Image-2.1-VL4B-FP8"


    def sha(path):
        value = hashlib.sha256()
        with Path(path).open("rb") as stream:
            for chunk in iter(lambda: stream.read(4*1024*1024), b""):
                value.update(chunk)
        return value.hexdigest()


    def read(path):
        return json.loads(Path(path).read_text(encoding="utf-8"))


    def write(path, value):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix+".tmp")
        temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
        temporary.replace(path)


    def contained(root, name, suffix=None):
        if not isinstance(name, str) or not name:
            raise ValueError("Chemin relatif non vide requis.")
        relative = PurePosixPath(name)
        if ("\\" in name or ":" in name or relative.is_absolute()
                or any(part in ("", ".", "..") for part in name.split("/"))):
            raise ValueError(f"Chemin relatif invalide : {name}")
        path = (Path(root) / name).resolve()
        if not path.is_relative_to(Path(root).resolve()) or (suffix and path.suffix != suffix):
            raise ValueError(f"Chemin externe ou format incorrect : {name}")
        return path


    def payload_path_allowed(name):
        """Only model artifacts, shipped Python and auditable comparison images."""
        path = PurePosixPath(name)
        if name in {"README.md", "LICENSE", "LICENSE.encoder", "NOTICE", "requirements-inference.txt",
                    "model_index.json", "sharm_config.json", "training_manifest.json", "visual-review.json"}:
            return True
        if len(path.parts) == 2:
            component = path.parts[0]
            if component in ("transformer", "vae", "scheduler", "text_encoder"):
                return path.suffix in (".json", ".safetensors")
            if component == "processor":
                return path.suffix in (".json", ".txt", ".jinja") and path.name.startswith(
                    ("tokenizer", "preprocessor", "video_preprocessor", "chat_template", "vocab", "merges"))
            if component == "projector":
                return path.name in ("config.json", "model.safetensors")
            if component == "lora":
                return path.name == "pytorch_lora_weights.safetensors"
            if component == "sharm_image21":
                return path.suffix == ".py"
        return (name == "comparisons/spatial/predictions.json"
                or len(path.parts) == 4 and path.parts[:2] == ("comparisons", "spatial")
                and path.parts[2] in ("images", "previews") and path.suffix == ".png")


    def link_file(source, destination):
        source, destination = Path(source), Path(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists():
            if sha(source) != sha(destination):
                raise ValueError(f"Fichier différent déjà présent : {destination}")
            return
        try:
            os.link(source, destination)
        except OSError:
            if shutil.disk_usage(destination.parent).free < source.stat().st_size+512*2**20:
                raise RuntimeError("Espace insuffisant pour copier les poids ; utiliser un volume avec hardlinks.")
            shutil.copy2(source, destination)


    def prerequisites(run, dataset):
        candidate_dir = run / "flow/best"
        candidate = read(candidate_dir / "candidate.json")
        complete = read(run / "flow/training-complete.json")
        warmup = read(run / "warmup/training-complete.json")
        if read(run / "run-status.json").get("state") != "complete":
            raise ValueError("La chaîne d'entraînement et sa galerie doivent être terminées.")
        if complete["signature"] != candidate["signature"] or complete["steps_completed"] < candidate["step"]:
            raise ValueError("Le candidat ne correspond pas à l'entraînement terminé.")
        if candidate["config"]["phase"] != "flow" or not math.isfinite(candidate["numerical_validation_loss"]):
            raise ValueError("Candidat flow avec validation numérique finie requis.")
        if sha(candidate_dir / "projector/model.safetensors") != candidate["projector_sha256"]:
            raise ValueError("Projecteur modifié depuis la validation.")
        if candidate["lora_sha256"] and sha(candidate_dir / "lora/pytorch_lora_weights.safetensors") != candidate["lora_sha256"]:
            raise ValueError("LoRA modifié depuis la validation.")
        manifest = read(dataset / "manifest.json")
        if manifest["dataset_sha256"] != candidate["identity"]["dataset_sha256"]:
            raise ValueError("Dataset incompatible avec le candidat.")
        predictions = read(run / "validation/predictions.json")
        if predictions["config"]["candidate_sha256"] != sha(candidate_dir / "candidate.json"):
            raise ValueError("Galerie d'une autre révision.")
        review = read(ARTIFACTS / "visual-review.json")
        if (review.get("passed") is not True or not review.get("reviewer")
                or review.get("method") not in ("agent_visual_review", "human_visual_review")
                or review.get("prediction_signature") != predictions["signature"]
                or review.get("projector_sha256") != candidate["projector_sha256"]):
            raise ValueError("Revue visuelle de ce candidat et de sa galerie requise.")
        reviewed_ids = set(review.get("reviewed_ids", []))
        required = {row["id"] for row in predictions["rows"] if row["method"].startswith("candidate")}
        if not required or not required <= reviewed_ids:
            raise ValueError("La revue doit couvrir toutes les générations du candidat de cette galerie.")
        return candidate_dir, candidate, complete, warmup, manifest, predictions, review


    def card(config, manifest, predictions):
        text = f"""---
    license: other
    license_name: qwen-research
    license_link: LICENSE
    pipeline_tag: text-to-image
    base_model:
    - Qwen/Qwen-Image-2.1
    - Qwen/Qwen3-VL-4B-Instruct-FP8
    language: [fr, en]
    ---

    # Sharm Image 2.1 VL4B FP8 — spatial

    Built with Qwen. Encodeur Qwen3-VL 4B FP8 gelé, projecteur contextuel BF16 et contrôle optionnel par boîtes `xyxy` normalisées. Le diffuseur est chargé en NF4 avec calcul BF16. Les PNG téléchargés conservent leur alpha ; le damier est un aperçu séparé.

    ## Entraînement et portée

    Le corpus local comprend {sum(manifest['counts'].values())} compositions RGBA à {manifest['resolution']} px, à partir de 1 540 sprites et 384 espèces, palettes normales et shiny. Les variantes d'une paire d'espèces restent dans le même split. Le test porte sur de nouvelles combinaisons d'assets connus. Les sources PNG et caches d'entraînement ne sont pas distribués dans ce dépôt.

    Les paramètres et résultats numériques figurent dans `training_manifest.json`. Le meilleur checkpoint numérique a été retenu sur validation, puis ses générations ont reçu la revue décrite dans `visual-review.json`. Une revue par agent est identifiée comme telle ; elle ne constitue pas le protocole humain avec intervalle de Wilson. **La parité générale avec le 8B et des positions parfaites ne sont pas établies.**

    ## Utilisation

    Installer `requirements-inference.txt` dans un environnement CUDA compatible, avec un compilateur C pour Triton. Télécharger ce dépôt, puis ajouter son dossier au `PYTHONPATH`. Le kernel officiel `kernels-community/finegrained-fp8` version 4 doit être disponible dans le cache HF ; son téléchargement initial nécessite Internet.

    ```python
    import torch
    from sharm_image21.pipeline import SharmImage21Pipeline

    model = SharmImage21Pipeline.from_pretrained("{REPO}")
    model.generate_png(
        "Exactly two distinct objects. Bulbasaur to the left of Charmander. Transparent background.",
        "result.png", preview_path="result-preview.png",
        layout=[{{"label": "Bulbasaur", "box": [0.1, 0.3, 0.4, 0.7]}},
                {{"label": "Charmander", "box": [0.6, 0.3, 0.9, 0.7]}}],
        width={manifest['resolution']}, height={manifest['resolution']}, output_resolution={manifest['resolution']},
        num_inference_steps=30, true_cfg_scale=1.0, use_kv_cache=False,
        generator=torch.Generator(device="cuda").manual_seed(1000),
    )
    ```

    Les labels et boîtes suivent l'ordre des objets. Les boîtes sont un contrôle appris, sans contrainte mathématique sur les pixels générés. Pour placer des PNG existants dans des allocations imposées, utiliser `sharm_image21.spatial_data.compose`, qui conserve le ratio et retourne les boîtes peintes. L'édition à partir d'images d'entrée n'est pas validée.

    ## Comparaisons

    Même prompt et mêmes graines pour le 8B et le candidat, avec le même diffuseur NF4. Les modes naturels et les modes avec coordonnées sont séparés dans `comparisons/spatial/predictions.json`. Les aperçus ci-dessous sont les premiers exemples sélectionnés avant leur génération.

    """
        for method in ("teacher_text", "candidate_text", "teacher_layout", "candidate_layout"):
            row = next((r for r in predictions["rows"] if r["method"] == method), None)
            if row:
                text += f"### {method}\n\n![{method}](comparisons/spatial/{row['preview']})\n\n[PNG RGBA](comparisons/spatial/{row['file_name']})\n\n"
        text += "## Licence et vérification\n\nLa licence Qwen Research et les notices sont conservées ; l'encodeur conserve ses éléments Apache-2.0. Les empreintes des fichiers sont dans `release_manifest.json`. `export-validation.json` vérifie le loader livré et une génération RGBA réelle ; sa portée est technique.\n"
        return text


    def prepare(run, dataset, output):
        candidate_dir, candidate, complete, warmup, manifest, predictions, review = prerequisites(run, dataset)
        original = PROJECT / "1-local-generation/weights/qwen-image-2.1"
        student = PROJECT / "1-local-generation/weights/sharm-vl4b-fp8"
        if (output / "sharm_config.json").exists() and read(output / "sharm_config.json")["projector_sha256"] != candidate["projector_sha256"]:
            raise ValueError("Un autre candidat occupe le dossier d'export.")
        for component in ("transformer", "vae", "scheduler"):
            for source in (original / component).iterdir():
                if source.is_file() and source.suffix in (".json", ".safetensors"):
                    link_file(source, output / component / source.name)
        for source in student.iterdir():
            if source.suffix == ".safetensors" or source.name in ("config.json", "generation_config.json", "model.safetensors.index.json"):
                link_file(source, output / "text_encoder" / source.name)
            elif source.is_file() and source.name.startswith(("tokenizer", "preprocessor", "video_preprocessor", "chat_template", "vocab", "merges")):
                link_file(source, output / "processor" / source.name)
        link_file(candidate_dir / "projector/model.safetensors", output / "projector/model.safetensors")
        settings = read(candidate_dir / "projector/config.json")
        keys = ("input_dim", "output_dim", "bottleneck", "context_layers", "context_heads", "layout_enabled", "convention")
        write(output / "projector/config.json", {k: settings[k] for k in keys if k in settings})
        if candidate["lora_sha256"]:
            link_file(candidate_dir / "lora/pytorch_lora_weights.safetensors", output / "lora/pytorch_lora_weights.safetensors")
        export_runtime(output / "sharm_image21")
        config = {"name": "Sharm Image 2.1 VL4B FP8 spatial", "pipeline_format": "spatial-v1",
                  "encoder_format": "fp8-e4m3", "encoder_frozen": True, "training_complete": True,
                  "student_identity": candidate["identity"]["student"]["encoder"],
                  "base_revision": candidate["identity"]["teacher"]["encoder"]["revision"],
                  "projector_sha256": candidate["projector_sha256"], "spatial_lora_sha256": candidate["lora_sha256"],
                  "base_validation_passed": False, "spatial_quality_validated": False,
                  "scope": "experimental-text-to-image-with-optional-layout", "image_editing_validated": False}
        write(output / "sharm_config.json", config)
        write(output / "model_index.json", {"_class_name": "SharmImage21Pipeline", "_module": "sharm_image21.pipeline"})
        write(output / "visual-review.json", review)
        def summarize(record):
            return {k: record[k] for k in ("signature", "steps_completed", "best_numerical_validation_loss", "reason")}
        write(output / "training_manifest.json", {"warmup": summarize(warmup), "flow": summarize(complete),
              "selected_step": candidate["step"], "selected_validation_loss": candidate["numerical_validation_loss"],
              "identity": candidate["identity"], "dataset": {k: manifest[k] for k in ("dataset_sha256", "counts", "resolution", "seed")}})
        for name, source in (("LICENSE", original / "LICENSE"), ("LICENSE.encoder", ROOT / "Sharm-Image-2.1-VL4B/licenses/Apache-2.0.txt"),
                             ("requirements-inference.txt", ROOT / "Sharm-Image-2.1-VL4B/requirements-inference.txt")):
            shutil.copy2(source, output / name)
        (output / "NOTICE").write_text("Built with Qwen. Qwen Research license applies to Qwen-Image-2.1.\n"
                                       "Qwen3-VL-4B-Instruct-FP8: Qwen Team, Apache-2.0.\n"
                                       "Sharm adds a trained contextual projector, optional layout tokens and optional diffuser LoRA.\n", encoding="utf-8")
        gallery = output / "comparisons/spatial"
        for row in predictions["rows"]:
            for key in ("file_name", "preview"):
                link_file(contained(run / "validation", row[key], ".png"), contained(gallery, row[key], ".png"))
            if sha(gallery / row["file_name"]) != row["image_sha256"]:
                raise ValueError("Image de comparaison modifiée.")
        write(gallery / "predictions.json", predictions)
        (output / "README.md").write_text(card(config, manifest, predictions), encoding="utf-8")
        files = {p.relative_to(output).as_posix(): sha(p) for p in output.rglob("*") if p.is_file()
                 and payload_path_allowed(p.relative_to(output).as_posix())}
        write(output / "release_manifest.json", {"schema": 1, "repo_id": REPO, "projector_sha256": candidate["projector_sha256"],
              "dataset_sha256": manifest["dataset_sha256"], "files": files})
        print(json.dumps({"prepared": str(output), "files": len(files)}))


    def verified_payload(output):
        manifest, verification = read(output / "release_manifest.json"), read(output / "export-validation.json")
        if manifest.get("repo_id") != REPO:
            raise ValueError("Le manifeste vise un autre dépôt.")
        if verification.get("passed") is not True or verification["release_manifest_sha256"] != sha(output / "release_manifest.json"):
            raise ValueError("Vérification du loader exporté requise pour ces fichiers exacts.")
        if verification.get("projector_sha256") != manifest["projector_sha256"]:
            raise ValueError("Vérification d'un autre projecteur.")
        for name in manifest["files"]:
            if not payload_path_allowed(name):
                raise ValueError(f"Fichier hors périmètre de publication : {name}")
        allowed_images = {f"validation/{method}{suffix}.png" for method in ("candidate_text", "candidate_layout")
                          for suffix in ("", "-preview")}
        if set(verification["images"]) != allowed_images:
            raise ValueError("Les deux vérifications RGBA et leurs aperçus sont requis.")
        files = {**manifest["files"], "release_manifest.json": sha(output / "release_manifest.json"),
                 "export-validation.json": sha(output / "export-validation.json"), **verification["images"]}
        for name, expected in files.items():
            path = contained(output, name)
            if sha(path) != expected:
                raise ValueError(f"Export modifié ou chemin externe : {name}")
        return manifest, files


    def push(output):
        from huggingface_hub import HfApi, hf_hub_download
        request = read(ARTIFACTS / "publication-request.json")
        if not request.get("authorized") or request.get("account") != "ethanzxv" or request.get("repo_id") != REPO:
            raise ValueError("Autorisation de publication manquante pour ce dépôt précis.")
        manifest, files = verified_payload(output)
        api = HfApi()
        if api.whoami()["name"] != "ethanzxv":
            raise ValueError("Compte Hugging Face incorrect.")
        info = api.model_info(REPO)
        expected_private = request["visibility_at_authorization"] == "private"
        if info.private != expected_private:
            raise ValueError("La visibilité du dépôt a changé depuis la demande ; vérifier avant publication.")
        if info.sha != request["base_hf_revision"]:
            # A retried upload can verify an already published, identical release.
            remote = read(hf_hub_download(REPO, "release_manifest.json", revision=info.sha))
            if remote != manifest:
                raise ValueError("Le dépôt a reçu d'autres modifications depuis l'autorisation.")
            revision = info.sha
        else:
            commit = api.upload_folder(repo_id=REPO, repo_type="model", folder_path=output,
                    allow_patterns=list(files), parent_commit=info.sha,
                    commit_message="Publish trained spatial projector, RGBA loader and paired validation gallery")
            revision = commit.oid
        # Verify the returned immutable revision, rather than trusting the upload response.
        remote = read(hf_hub_download(REPO, "release_manifest.json", revision=revision))
        if remote != manifest:
            raise ValueError("Le manifeste distant diffère de l'export local.")
        info = api.model_info(REPO, revision=revision, files_metadata=True)
        remote_files = {r.rfilename: r for r in info.siblings}
        for name, expected in files.items():
            remote_file = remote_files.get(name)
            if remote_file is None or remote_file.size != (output / name).stat().st_size:
                raise ValueError(f"Fichier distant absent ou tronqué : {name}")
            if remote_file.lfs and remote_file.lfs.sha256 != expected:
                raise ValueError(f"Empreinte distante incorrecte : {name}")
            if not remote_file.lfs and sha(hf_hub_download(REPO, name, revision=revision)) != expected:
                raise ValueError(f"Contenu distant incorrect : {name}")
        write(ARTIFACTS / "publication-status.json", {"state": "complete", "repo_id": REPO,
              "revision": revision, "private": info.private, "files": len(files), "projector_sha256": manifest["projector_sha256"]})
        print(json.dumps({"published": REPO, "revision": revision}))


    def run(parameters=None):
        args = parameter_namespace({'action': None, 'run': ARTIFACTS / 'run', 'dataset': ARTIFACTS / 'dataset', 'output': ARTIFACTS / 'release'}, parameters, required=['action'], converters={'run': Path, 'dataset': Path, 'output': Path}, choices={'action': ('prepare', 'push')}, groups=[])
        if args.action == "prepare":
            prepare(args.run, args.dataset, args.output)
        else:
            push(args.output)



## publish.verify_export

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_c5a6553bd004717a':
    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    from pathlib import Path


    import json


    import sys


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    import torch


    from sharm_image21.core import ARTIFACTS, CACHE, WEIGHTS, FeatureCache, Projector, atomic_json, read_jsonl


    from sharm_image21.pipeline import SharmImage21Pipeline


    @torch.inference_mode()
    def run(parameters=None):
        args = parameter_namespace({}, parameters, required=[], converters={}, choices={}, groups=[])
        base = WEIGHTS / "sharm-image-2.1-vl4b-fp8"
        pipeline = SharmImage21Pipeline.from_pretrained(base)
        row = next(row for row in read_jsonl(ARTIFACTS / "corpus.jsonl") if row["split"] == "validation")
        expected_source = FeatureCache(CACHE / "student").get(row["id"])
        projector = Projector.load(base / "projector", "cuda").eval()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            expected = projector(expected_source.cuda()).to(torch.bfloat16).cpu()
        del projector
        torch.cuda.empty_cache()
        actual, mask = pipeline.encode_prompt(row["text"])
        torch.testing.assert_close(actual[0], expected, rtol=0.02, atol=0.1)
        if not mask.all() or actual.shape[-1] != 4096:
            raise ValueError("Embeddings du modèle exporté incompatibles.")
        atomic_json(ARTIFACTS / "export-validation.json", {"passed": True, "private": True,
                    "base": str(base), "projector_sha256": pipeline.config["projector_sha256"],
                    "scope": "exported 4B FP8 and projector reload, no 8B required"})
        print("Export Sharm rechargé et vérifié sans encodeur 8B.", flush=True)



## publish.verify_monster

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_005d5670372ba375':
    import json
    from pathlib import Path
    from sharm_notebook_runtime import ARTIFACTS, read_json, sha256, shipped_package

    def run(parameters=None):
        p = dict(parameters or {})
        base = Path(p.get("base", ARTIFACTS / "spatial/release"))
        lora = Path(p["lora"])
        output = Path(p.get("output", ARTIFACTS / "monster/loader-validation"))
        config = read_json(lora / "sharm_lora_config.json")
        comparison = read_json(lora / "comparisons/lora/comparison.json")
        output.mkdir(parents=True, exist_ok=True)
        files = {}
        with shipped_package(base):
            from sharm_image21.pipeline import SharmImage21Pipeline
            import torch
            from PIL import Image
            model = SharmImage21Pipeline.from_pretrained(base)
            case = comparison["pairs"][0]
            for label, adapter in (("base", None), ("monster", lora)):
                image = model.generate_png(case["prompt"], output / (label + "-512.png"),
                    preview_path=output / (label + "-512-preview.png"), transparent_background=True,
                    lora_dir=adapter, width=512, height=512, output_resolution=512, num_inference_steps=30,
                    true_cfg_scale=1.0, use_kv_cache=False,
                    attention_kwargs={"scale": config.get("recommended_weight", 0.5)},
                    generator=torch.Generator(device="cuda").manual_seed(case["seed"]))
                model.save_png(image.resize((128, 128), Image.Resampling.NEAREST), output / (label + "-128.png"),
                               preview_path=output / (label + "-128-preview.png"), require_transparency=True)
            files = {f.name: sha256(f) for f in output.glob("*.png")}
        report = {"passed": True, "projector_sha256": config["projector_sha256"], "lora_sha256": config["lora_sha256"],
                  "comparison_sha256": sha256(lora / "comparisons/lora/comparison.json"), "images": files,
                  "scope": "Real paired RGBA inference using the shipped Base loader and exact Monster adapter"}
        (output / "export-validation.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print(json.dumps(report, indent=2))
        return report



## publish.verify_published

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_2726e43500ad75a9':
    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    """Run the shipped loader in a clean process using the verified HF mirrors."""


    import argparse


    import hashlib


    import json


    from pathlib import Path


    import sys


    import time


    def run(parameters=None):
        args = parameter_namespace({'base': None, 'lora': None, 'output': None}, parameters, required=['base', 'lora', 'output'], converters={'base': Path, 'lora': Path, 'output': Path}, choices={}, groups=[])
        started = time.monotonic()
        # No import from the development package: exercise exactly the shipped code.
        sys.path.insert(0, str(args.base.resolve()))
        import torch
        import numpy as np
        from PIL import Image
        from sharm_image21 import pipeline as shipped

        if not Path(shipped.__file__).resolve().is_relative_to(args.base.resolve()):
            raise RuntimeError("Le loader utilisé ne provient pas du modèle publié.")
        case = json.loads((args.lora / "comparisons/lora/comparison.json").read_text())["pairs"][0]
        metadata = json.loads((args.lora / "sharm_lora_config.json").read_text())
        pipeline = shipped.SharmImage21Pipeline.from_pretrained(args.base)
        args.output.mkdir(parents=True, exist_ok=True)
        image = pipeline.generate_png(
            case["prompt"], args.output / "published-generation-512.png",
            preview_path=args.output / "published-generation-512-preview.png",
            transparent_background=True, lora_dir=args.lora, width=512, height=512,
            output_resolution=512, num_inference_steps=30, true_cfg_scale=1.0,
            attention_kwargs={"scale": metadata.get("recommended_weight", 1.0)},
            generator=torch.Generator(device="cuda").manual_seed(case["seed"]))
        pixels = np.asarray(image)
        if (image.mode != "RGBA" or image.size != (512, 512) or pixels[..., :3].std() < 1
                or image.getchannel("A").getextrema()[1] == 0):
            raise RuntimeError("La génération après relecture HF est vide ou invalide.")
        output = args.output / "published-generation-512.png"
        pipeline.save_png(image.resize((128, 128), Image.Resampling.NEAREST),
                          args.output / "published-generation-128.png",
                          preview_path=args.output / "published-generation-128-preview.png",
                          require_transparency=True)
        report = {"passed": True, "prompt": case["prompt"], "seed": case["seed"],
                  "lora_weight": metadata.get("recommended_weight", 1.0),
                  "sha256": hashlib.sha256(output.read_bytes()).hexdigest(),
                  "base": str(args.base), "lora": str(args.lora), "loader": str(shipped.__file__),
                  "seconds": time.monotonic() - started, "gpu": torch.cuda.get_device_name(),
                  "gpu_peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
                  "gpu_peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
                  "scope": "shipped loader and verified private HF mirrors, 4B FP8, projector and LoRA"}
        (args.output / "published-inference.json").write_text(json.dumps(report, indent=2) + "\n")
        print(json.dumps(report), flush=True)



## publish.verify_spatial_export

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_2e8cf4063fdc9393':
    """Validate the shipped package and execute real natural/layout RGBA inference."""




    from sharm_notebook_runtime import parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import argparse


    import hashlib


    import json


    from pathlib import Path


    import sys


    ROOT = Path(__file__).resolve().parents[1]


    def sha(path):
        value = hashlib.sha256()
        with Path(path).open("rb") as stream:
            for chunk in iter(lambda: stream.read(4*1024*1024), b""):
                value.update(chunk)
        return value.hexdigest()


    def run(parameters=None):
        args = parameter_namespace({'release': ROOT / 'artifacts/spatial/release', 'run': ROOT / 'artifacts/spatial/run', 'dataset': ROOT / 'artifacts/spatial/dataset'}, parameters, required=[], converters={'release': Path, 'run': Path, 'dataset': Path}, choices={}, groups=[])
        if json.loads((args.run / "run-status.json").read_text())["state"] != "complete":
            raise ValueError("Terminer la chaîne avant de mobiliser le GPU pour l'export.")
        lock = (args.run / "run.lock").open("a")
        if sys.platform != "win32":
            import fcntl
            fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        directory = args.release.resolve()
        manifest_path = directory / "release_manifest.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        for name, expected in manifest["files"].items():
            path = (directory / name).resolve()
            if not path.is_relative_to(directory) or sha(path) != expected:
                raise ValueError(f"Export modifié : {name}")
        # Import the exact code copied into the release, with no workspace package cached.
        sys.path.insert(0, str(directory))
        import torch
        import sharm_image21.pipeline as shipped
        from sharm_image21.core import Projector
        from sharm_image21.image_io import save_png, transparent_fraction
        if not Path(shipped.__file__).resolve().is_relative_to(directory):
            raise ValueError("Le loader importé ne provient pas de l'export.")
        candidate = Projector.load(args.run / "flow/best/projector")
        exported = Projector.load(directory / "projector")
        for name, value in candidate.state_dict().items():
            torch.testing.assert_close(value, exported.state_dict()[name], rtol=0, atol=0)
        del candidate, exported
        model = shipped.SharmImage21Pipeline.from_pretrained(directory)
        predictions = json.loads((args.run / "validation/predictions.json").read_text())
        scenes = {r["id"]: r for r in (json.loads(line) for line in (args.dataset / "scenes.jsonl").read_text(encoding="utf-8").splitlines())}
        images, results = {}, []
        for method in ("candidate_text", "candidate_layout"):
            example = next(r for r in predictions["rows"] if r["method"] == method)
            scene = scenes[example["scene_id"]]
            layout = [{"label": o["label"], "box": o["box"]} for o in scene["objects"]] if method.endswith("layout") else None
            resolution = predictions["config"]["resolution"]
            output = model(scene["text_plain"], layout=layout, width=resolution, height=resolution, output_resolution=resolution,
                           num_inference_steps=predictions["config"]["steps"], true_cfg_scale=1.0, use_kv_cache=False,
                           generator=torch.Generator(device="cuda").manual_seed(example["seed"]))
            image = output.images[0]
            if image.mode != "RGBA":
                raise ValueError("RGBA natif requis à la sortie du modèle exporté.")
            name, preview = f"validation/{method}.png", f"validation/{method}-preview.png"
            save_png(image, directory / name, preview_path=directory / preview, require_transparency=True)
            images.update({name: sha(directory / name), preview: sha(directory / preview)})
            results.append({"method": method, "scene_id": scene["id"], "seed": example["seed"],
                            "transparent_fraction": transparent_fraction(image)})
        report = {"passed": True, "release_manifest_sha256": sha(manifest_path), "images": images,
                  "projector_sha256": manifest["projector_sha256"], "projector_roundtrip_exact": True,
                  "loader_from_release": True, "cases": results,
                  "scope": "Export integrity and real natural/layout RGBA inference; spatial quality requires separate review."}
        (directory / "export-validation.json").write_text(json.dumps(report, indent=2)+"\n", encoding="utf-8")
        print(json.dumps(report), flush=True)



## tests.test_diffusion_dtype

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_0027eb10967168ae':
    from sharm_notebook_runtime import notebook_completed, parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    """Regression checks for FP32 NF4 outputs promoting the next Euler state."""


    from pathlib import Path


    import sys


    from types import SimpleNamespace


    import unittest


    import torch


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    from sharm_image21.diffusion import conditioning_output_dtype


    class FrozenDiffuserDtypeChecks(unittest.TestCase):
        def test_tuple_output_follows_conditioning_dtype_and_preserves_auxiliary_values(self):
            values = torch.randn(1, 16, 64)
            result = conditioning_output_dtype(None, (), {"encoder_hidden_states": torch.ones(1, 3, 16, dtype=torch.bfloat16)}, (values, "aux"))
            self.assertEqual(result[0].dtype, torch.bfloat16)
            self.assertEqual(result[1], "aux")
            torch.testing.assert_close(result[0], values.to(torch.bfloat16), rtol=0, atol=0)

        def test_model_output_and_float32_conditioning_supported(self):
            result = SimpleNamespace(sample=torch.ones(2, dtype=torch.bfloat16))
            actual = conditioning_output_dtype(None, (), {"encoder_hidden_states": torch.ones(1)}, result)
            self.assertIs(actual, result)
            self.assertEqual(actual.sample.dtype, torch.float32)
            self.assertIs(conditioning_output_dtype(None, (), {}, result), result)

        def test_multiple_euler_steps_keep_bf16_state(self):
            # A CPU surrogate exercises the output boundary; a real 20-step GPU
            # trajectory must additionally be run before accepting the runtime fix.
            class Model(torch.nn.Module):
                def forward(self, hidden_states, encoder_hidden_states):
                    if hidden_states.dtype != encoder_hidden_states.dtype:
                        raise ValueError("mixed image/text projection dtype")
                    return (hidden_states.float()*0.1,)

            model = Model()
            model.register_forward_hook(conditioning_output_dtype, with_kwargs=True)
            source = torch.ones(1, 8, 4, dtype=torch.bfloat16)
            conditioning = torch.ones(1, 3, 4, dtype=torch.bfloat16)
            for _ in range(20):
                target = model(hidden_states=source, encoder_hidden_states=conditioning)[0]
                source = (source.float()-0.05*target.float()).to(target.dtype)
                self.assertEqual(source.dtype, torch.bfloat16)
                self.assertTrue(torch.isfinite(source).all())


    def run(parameters=None):
        import unittest
        suite = unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])
        result = unittest.TextTestRunner(verbosity=2).run(suite)
        if not result.wasSuccessful():
            raise RuntimeError("La suite contient des échecs ; consulter le rapport.")
        return result



## tests.test_image_io

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_0a0aeeecae98e89c':
    from sharm_notebook_runtime import notebook_completed, parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    """PNG alpha and separate checkerboard preview regression checks."""


    from pathlib import Path


    import sys


    import tempfile


    import unittest


    from PIL import Image, ImageDraw


    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "common"))


    from sharm_image21.image_io import checkerboard_preview, require_transparent_background, save_png


    class PngExportChecks(unittest.TestCase):
        def sprite(self):
            image = Image.new("RGBA", (32, 32), (1, 2, 3, 0))
            ImageDraw.Draw(image).rectangle((10, 10, 20, 20), fill=(255, 0, 0, 255))
            image.putpixel((10, 10), (255, 0, 0, 128))
            return image

        def test_png_roundtrip_preserves_alpha_and_preview_is_separate(self):
            image = self.sprite()
            original = image.tobytes()
            with tempfile.TemporaryDirectory() as folder:
                output, preview = (Path(folder) / name for name in ("output.png", "preview.png"))
                save_png(image, output, preview_path=preview, require_transparency=True)
                with Image.open(output) as saved:
                    self.assertEqual(saved.format, "PNG")
                    self.assertEqual(saved.mode, "RGBA")
                    self.assertEqual(saved.tobytes(), original)
                with Image.open(preview) as saved:
                    self.assertEqual(saved.mode, "RGB")
                    self.assertEqual(saved.getpixel((0, 0)), (232, 232, 232))
                    self.assertEqual(saved.getpixel((16, 0)), (184, 184, 184))
                self.assertEqual(image.tobytes(), original)

        def test_transparency_request_rejects_rgb_opaque_and_empty_images(self):
            with tempfile.TemporaryDirectory() as folder:
                output = Path(folder) / "rejected.png"
                for image in (Image.new("RGB", (32, 32), "red"),
                              Image.new("RGBA", (32, 32), (255, 0, 0, 255)),
                              Image.new("RGBA", (32, 32), (0, 0, 0, 0))):
                    with self.subTest(mode=image.mode, extrema=image.getextrema()):
                        with self.assertRaises(ValueError):
                            save_png(image, output, require_transparency=True)
                        self.assertFalse(output.exists())

        def test_preview_cannot_overwrite_native_png(self):
            with tempfile.TemporaryDirectory() as folder:
                output = Path(folder) / "native.png"
                save_png(self.sprite(), output)
                original = output.read_bytes()
                with self.assertRaises(ValueError):
                    save_png(Image.new("RGB", (32, 32), "blue"), output, preview_path=output)
                self.assertEqual(output.read_bytes(), original)
                with self.assertRaises(ValueError):
                    save_png(self.sprite(), Path(folder) / "wrong.jpg")

        def test_checkerboard_preserves_input_and_checks_tile_size(self):
            image = self.sprite()
            original = image.tobytes()
            self.assertGreater(require_transparent_background(image), 0.8)
            self.assertEqual(checkerboard_preview(image).mode, "RGB")
            self.assertEqual(image.tobytes(), original)
            with self.assertRaises(ValueError):
                checkerboard_preview(image, tile_size=0)


    def run(parameters=None):
        import unittest
        suite = unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])
        result = unittest.TextTestRunner(verbosity=2).run(suite)
        if not result.wasSuccessful():
            raise RuntimeError("La suite contient des échecs ; consulter le rapport.")
        return result



## tests.test_remote_collection

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_6bd7f4b1dcf034b5':
    from sharm_notebook_runtime import notebook_completed, parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    """CPU checks for exact release collection without downloading duplicate base weights."""


    import importlib.util


    from pathlib import Path


    import sys


    import tempfile


    import unittest


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "publish"))


    import publish_spatial as publisher


    collector = load_notebook(ROOT / "publish/collect_remote_release.ipynb", name="collect_remote")


    class CollectionChecks(unittest.TestCase):
        def setUp(self):
            self.temporary = tempfile.TemporaryDirectory()
            self.addCleanup(self.temporary.cleanup)
            self.root = Path(self.temporary.name)
            self.incoming = self.root / "incoming"
            self.output = self.root / "release"
            self.incoming.mkdir()
            self.base = self.root / "1-local-generation/weights/sharm-vl4b-fp8/model.safetensors"
            self.base.parent.mkdir(parents=True)
            self.base.write_bytes(b"public frozen weights")
            self.projector = self.incoming / "projector/model.safetensors"
            self.projector.parent.mkdir()
            self.projector.write_bytes(b"trained projector")
            (self.incoming / "README.md").write_bytes(b"Exact remote card\n")
            self.manifest = {"repo_id": publisher.REPO, "projector_sha256": publisher.sha(self.projector),
                             "files": {"README.md": publisher.sha(self.incoming / "README.md"),
                                       "projector/model.safetensors": publisher.sha(self.projector),
                                       "text_encoder/model.safetensors": publisher.sha(self.base)}}
            publisher.write(self.incoming / "release_manifest.json", self.manifest)
            images = {}
            for method in ("candidate_text", "candidate_layout"):
                for suffix in ("", "-preview"):
                    name = f"validation/{method}{suffix}.png"
                    path = self.incoming / name
                    path.parent.mkdir(exist_ok=True)
                    path.write_bytes(b"checked image fixture")
                    images[name] = publisher.sha(path)
            self.verification = {"passed": True, "release_manifest_sha256": publisher.sha(self.incoming / "release_manifest.json"),
                                 "projector_sha256": self.manifest["projector_sha256"], "images": images}
            publisher.write(self.incoming / "export-validation.json", self.verification)

        def test_reuses_identical_local_base_and_preserves_exact_remote_bytes(self):
            report = collector.collect(self.incoming, self.output, self.root)
            self.assertTrue(report["passed"])
            self.assertEqual(report["reused_base_files"], 1)
            self.assertTrue((self.output / "text_encoder/model.safetensors").samefile(self.base))
            for name in ("README.md", "release_manifest.json", "export-validation.json", "projector/model.safetensors"):
                self.assertEqual((self.output / name).read_bytes(), (self.incoming / name).read_bytes())
            publisher.verified_payload(self.output)

        def test_altered_local_base_rejected_before_output_created(self):
            self.base.write_bytes(b"wrong revision")
            with self.assertRaisesRegex(ValueError, "incompatible"):
                collector.collect(self.incoming, self.output, self.root)
            self.assertFalse(self.output.exists())

        def test_altered_report_unverified_export_and_unsafe_paths_rejected(self):
            self.verification["passed"] = False
            publisher.write(self.incoming / "export-validation.json", self.verification)
            with self.assertRaises(ValueError):
                collector.collect(self.incoming, self.output, self.root)
            self.verification["passed"] = True
            publisher.write(self.incoming / "export-validation.json", self.verification)
            with self.assertRaises(ValueError):
                collector.collect(self.incoming, self.incoming / "nested", self.root)
            with self.assertRaises(ValueError):
                collector.base_source("text_encoder/../outside.safetensors", self.root)

        def test_trained_weights_never_substituted_with_public_base(self):
            self.assertIsNone(collector.base_source("projector/model.safetensors", self.root))
            self.assertIsNone(collector.base_source("lora/pytorch_lora_weights.safetensors", self.root))
            self.projector.write_bytes(b"altered")
            with self.assertRaisesRegex(ValueError, "incompatible"):
                collector.collect(self.incoming, self.output, self.root)
            self.assertFalse(self.output.exists())


    def run(parameters=None):
        import unittest
        suite = unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])
        result = unittest.TextTestRunner(verbosity=2).run(suite)
        if not result.wasSuccessful():
            raise RuntimeError("La suite contient des échecs ; consulter le rapport.")
        return result



## tests.test_remote_staging

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_05d1b97a078e0a3b':
    from sharm_notebook_runtime import notebook_completed, parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    """CPU guards for staged downloads and releasing owned temporary teacher weights."""


    import importlib.util


    import json


    from pathlib import Path


    import sys


    import tempfile


    from types import SimpleNamespace


    import unittest


    from unittest.mock import patch


    import torch


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    from sharm_image21.core import FeatureCache, atomic_json, digest


    def module(name, path):
        result = load_notebook(path, name=name)
        return result


    with patch.dict(sys.modules, {} if sys.platform != "win32" else {"fcntl": SimpleNamespace()}):
        remote = module("remote_staging", ROOT / "03-spatial-distillation/run_remote.ipynb")


    models = module("staged_models", ROOT / "01-encoder-distillation/prepare_models.ipynb")


    class DownloadStageChecks(unittest.TestCase):
        def patterns(self, stage):
            with patch.object(sys, "argv", ["prepare_models", "--stage", stage]), patch.object(models, "snapshot_download") as download:
                models.run({"stage": stage})
            return [call.kwargs["allow_patterns"] for call in download.call_args_list]

        def test_teacher_download_avoids_diffuser_and_student_weight_files(self):
            original, student = self.patterns("teacher")
            self.assertIn("text_encoder/*", original)
            self.assertNotIn("transformer/*", original)
            self.assertNotIn("vae/*", original)
            self.assertNotIn("*.safetensors", student)

        def test_inference_does_not_redownload_released_teacher_weights(self):
            original, student = self.patterns("inference")
            self.assertNotIn("text_encoder/*", original)
            self.assertIn("text_encoder/*.json", original)
            self.assertIn("transformer/*", original)
            self.assertIn("*.safetensors", student)

        def test_same_experiment_with_reduced_activation_recomputation(self):
            small = json.loads((ROOT / "Sharm-Image-2.1-VL4B/configs/8gb.json").read_text())
            server = json.loads((ROOT / "Sharm-Image-2.1-VL4B/configs/remote16gb-256.json").read_text())
            self.assertEqual(server["flow"].pop("checkpointing-mode"), "alternating")
            small["flow"].pop("checkpointing-mode")
            self.assertEqual(server, small)


    class TemporaryTeacherChecks(unittest.TestCase):
        def setUp(self):
            self.temporary = tempfile.TemporaryDirectory()
            self.addCleanup(self.temporary.cleanup)
            self.project = Path(self.temporary.name)
            self.root = self.project / "2-LoRA-qwen"
            self.artifacts = self.root / "artifacts/spatial"
            self.original = self.project / "1-local-generation/weights/qwen-image-2.1"
            self.directory = self.original / "text_encoder"
            self.directory.mkdir(parents=True)
            self.weight = self.directory / "model-00001-of-00001.safetensors"
            self.weight.write_bytes(b"temporary downloaded teacher")
            atomic_json(self.directory / "model.safetensors.index.json", {"weight_map": {"a": self.weight.name}})
            atomic_json(self.artifacts / "remote-workspace.json", {"root": str(self.project.resolve()), "temporary_teacher_weights": True})
            self.rows = [{"id": "a", "text": "fixture", "split": "train"}]
            corpus = self.artifacts / "dataset/corpus.jsonl"
            corpus.parent.mkdir(parents=True)
            corpus.write_text(json.dumps(self.rows[0])+"\n", encoding="utf-8")
            self.cache = FeatureCache(self.root / "cache/spatial/teacher", {"encoder": {"fixture": True}, "corpus": digest(self.rows)})
            self.cache.write([("a", torch.ones(3, 4, dtype=torch.bfloat16), [1, 2, 3])])
            for name, value in {"ROOT": self.root, "PROJECT": self.project, "ARTIFACTS": self.artifacts,
                                "ORIGINAL": self.original, "model_identity": lambda kind: {"fixture": True}}.items():
                active = patch.object(remote, name, value)
                active.start()
                self.addCleanup(active.stop)

        def test_only_owned_weights_released_after_complete_valid_cache(self):
            remote.release_teacher_weights()
            self.assertFalse(self.weight.exists())
            self.assertTrue((self.directory / "model.safetensors.index.json").exists())
            self.assertTrue((self.cache.directory / "shard-000000.safetensors").exists())
            report = json.loads((self.artifacts / "temporary-teacher-release.json").read_text())
            self.assertEqual(report["captions"], 1)
            remote.release_teacher_weights()  # Resuming does not need the removed weights.

        def test_unmarked_workspace_and_incomplete_cache_reject_deletion(self):
            atomic_json(self.artifacts / "remote-workspace.json", {"root": "wrong", "temporary_teacher_weights": True})
            with self.assertRaises(ValueError):
                remote.release_teacher_weights()
            self.assertTrue(self.weight.exists())
            atomic_json(self.artifacts / "remote-workspace.json", {"root": str(self.project.resolve()), "temporary_teacher_weights": True})
            self.cache.index["entries"] = {}
            atomic_json(self.cache.path, self.cache.index)
            with self.assertRaises(ValueError):
                remote.release_teacher_weights()
            self.assertTrue(self.weight.exists())

        def test_truncated_cache_and_external_weight_paths_rejected(self):
            shard = self.cache.directory / "shard-000000.safetensors"
            original = shard.read_bytes()
            shard.write_bytes(b"truncated")
            with self.assertRaises(Exception):
                remote.release_teacher_weights()
            self.assertTrue(self.weight.exists())
            shard.write_bytes(original)
            atomic_json(self.directory / "model.safetensors.index.json", {"weight_map": {"a": "../outside.safetensors"}})
            with self.assertRaises(ValueError):
                remote.release_teacher_weights()
            self.assertTrue(self.weight.exists())


    def run(parameters=None):
        import unittest
        suite = unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])
        result = unittest.TextTestRunner(verbosity=2).run(suite)
        if not result.wasSuccessful():
            raise RuntimeError("La suite contient des échecs ; consulter le rapport.")
        return result



## tests.test_spatial

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_7cb9c8de45f646f9':
    """CPU checks for ordered conditioning, RGBA ground truth and honest scoring."""




    from sharm_notebook_runtime import notebook_completed, parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import copy


    import importlib.util


    import json


    import os


    from pathlib import Path


    import subprocess


    import sys


    import tempfile


    import unittest


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    import torch


    from PIL import Image, ImageDraw


    from sharm_image21.core import FeatureCache, Projector, atomic_json, digest


    from sharm_image21.spatial_data import compose, fingerprint, read_rows, relation_holds, sha256, wilson


    evaluation = load_notebook(ROOT / "03-spatial-distillation/evaluate.ipynb", name="spatial_evaluation")


    class ProjectorChecks(unittest.TestCase):
        def setUp(self):
            torch.manual_seed(12)

        def test_legacy_arithmetic_and_serialization(self):
            model = Projector(8, 16, 8)
            source = torch.randn(7, 8)
            expected = model.skip(source)+model.down(torch.nn.functional.gelu(model.up(model.norm(source))))
            torch.testing.assert_close(model(source), expected, rtol=0, atol=0)
            with tempfile.TemporaryDirectory() as folder:
                model.save(Path(folder))
                loaded = Projector.load(Path(folder))
                torch.testing.assert_close(loaded(source), expected, rtol=0, atol=0)
                self.assertNotIn("context_layers", loaded.settings)

        def test_context_mask_order_layout_and_bf16_roundtrip(self):
            model = Projector(8, 16, 8, context_layers=1, context_heads=2, layout_enabled=True).eval()
            torch.nn.init.normal_(model.down.weight, std=0.1)
            source = torch.randn(5, 8)
            padding = torch.cat((source, torch.randn(3, 8)*100))[None]
            with torch.no_grad():
                result = model(source)
                padded = model(padding, attention_mask=torch.tensor([[1, 1, 1, 1, 1, 0, 0, 0]]))
                torch.testing.assert_close(result, padded[0, :5], atol=1e-5, rtol=1e-5)
                permutation = torch.tensor([4, 3, 2, 1, 0])
                self.assertFalse(torch.allclose(model(source[permutation]), result[permutation], atol=1e-5))
            boxes = torch.tensor([[0.1, 0.2, 0.4, 0.8], [0.6, 0.2, 0.9, 0.8]])
            output = model(source, layout=boxes)
            self.assertEqual(output.shape, (7, 16))
            output.square().mean().backward()
            for prefix in ("context.", "layout_box.", "layout_object."):
                self.assertTrue(any(p.grad is not None and p.grad.abs().sum() > 0 for n, p in model.named_parameters() if n.startswith(prefix)))
            with tempfile.TemporaryDirectory() as folder:
                model.save(Path(folder), storage_dtype=torch.bfloat16)
                loaded = Projector.load(Path(folder)).float()
                with torch.no_grad():
                    expected = model(source, layout=boxes)
                    torch.testing.assert_close(loaded(source, layout=boxes), expected, rtol=0.02, atol=0.01)
            with self.assertRaises(ValueError):
                model(source, layout=torch.tensor([[0.8, 0.1, 0.2, 0.4]]))
            with self.assertRaises(ValueError):
                Projector(8, 16, 8)(source, layout=boxes)

        def test_gradient_through_frozen_surrogate(self):
            # This checks graph construction, not the real NF4 CUDA kernel.
            model = Projector(8, 16, 8, context_layers=1, context_heads=2, layout_enabled=True)
            decoder = torch.nn.Linear(16, 4).requires_grad_(False)
            source, target = torch.randn(6, 8), torch.randn(4)
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
            for _ in range(2):
                optimizer.zero_grad()
                output = decoder(model(source).mean(0))
                torch.nn.functional.mse_loss(output, target).backward()
                self.assertTrue(model.skip.weight.grad.abs().sum() > 0)
                self.assertTrue(all(p.grad is None for p in decoder.parameters()))
                optimizer.step()


    class DataAndScoringChecks(unittest.TestCase):
        def setUp(self):
            self.temporary = tempfile.TemporaryDirectory()
            self.root = Path(self.temporary.name)
            self.assets = []
            for i, color in enumerate(("red", "blue", "green", "yellow", "purple", "orange")):
                image = Image.new("RGBA", (48, 64), (0, 0, 0, 0))
                ImageDraw.Draw(image).rectangle((6, 8, 40, 56), fill=color)
                name = f"{i}.png"
                image.save(self.root / name)
                self.assets.append({"id": str(i), "group_id": str(i), "label": color+" rectangle", "file_name": name})
            self.source = self.root / "assets.jsonl"
            self.source.write_text("".join(json.dumps(a)+"\n" for a in self.assets), encoding="utf-8")

        def tearDown(self):
            self.temporary.cleanup()

        def test_dataset_reproducibility_and_pair_holdout(self):
            paths = [self.root / "dataset-a", self.root / "dataset-b"]
            for output in paths:
                notebook_completed([sys.executable, str(ROOT / "03-spatial-distillation/prepare_dataset.ipynb"),
                                "--assets", str(self.source), "--output", str(output), "--count", "120", "--resolution", "128"],
                               check=True, capture_output=True, text=True)
            a, b = (read_rows(p / "scenes.jsonl") for p in paths)
            self.assertEqual(a, b)
            splits = {}
            self.assertEqual({len(r["objects"]) for r in a}, {2, 3})
            for row in a:
                self.assertNotIn("Normalized object boxes", row["text_plain"])
                self.assertIn("Normalized object boxes", row["text"])
                pair = tuple(row["pair_group"])
                self.assertEqual(splits.setdefault(pair, row["split"]), row["split"])
                self.assertEqual(sha256(paths[0] / row["file_name"]), row["image_sha256"])
                self.assertTrue(relation_holds(row["objects"][0]["box"], row["objects"][1]["box"], row["relation"]))
                with Image.open(paths[0] / row["file_name"]) as image:
                    self.assertEqual(image.mode, "RGBA")
                    self.assertEqual(image.getchannel("A").getextrema(), (0, 255))
            manifest = json.loads((paths[0] / "manifest.json").read_text())
            self.assertEqual(manifest["dataset_sha256"], fingerprint(a))

        def fixture(self):
            objects = [{**self.assets[0], "box": [0.1, 0.3, 0.35, 0.7]},
                       {**self.assets[1], "box": [0.65, 0.3, 0.9, 0.7]}]
            image, objects = compose(128, objects, self.root)
            image.save(self.root / "result.png")
            record = {"id": "prediction", "scene_id": "scene", "seed": 1, "method": "candidate_layout",
                      "pair_group": ["a", "b"], "relation": "left", "file_name": "result.png",
                      "image_sha256": sha256(self.root / "result.png"), "objects": objects}
            annotation = {"id": record["id"], "image_sha256": record["image_sha256"], "reviewed": True,
                          "reviewer": "unit-check", "method": "human_identity_and_boxes", "distinct": True, "extra_objects": 0,
                          "objects": [{"object_id": o["object_id"], "box": o["box"], "identity_ok": True} for o in objects]}
            return record, annotation

        def test_honest_success_and_failure_scores(self):
            record, annotation = self.fixture()
            score = lambda a: evaluation.score_record(record, a, self.root, 0.5, 0.05)
            self.assertTrue(score(annotation)["passed"])
            self.assertFalse(score(None)["passed"])
            for field, value in (("distinct", False), ("extra_objects", 1), ("reviewed", False), ("image_sha256", "stale")):
                changed = copy.deepcopy(annotation)
                changed[field] = value
                self.assertFalse(score(changed)["passed"])
            changed = copy.deepcopy(annotation)
            changed["objects"][0]["identity_ok"] = False
            self.assertFalse(score(changed)["passed"])
            shifted = copy.deepcopy(annotation)
            for obj in shifted["objects"]:
                obj["box"] = [v-0.1 if i in (1, 3) else v for i, v in enumerate(obj["box"])]
            self.assertFalse(score(shifted)["passed"])
            self.assertTrue(evaluation.score_record(record, shifted, self.root, 0.5, 0.05, require_placement=False)["passed"])
            changed = copy.deepcopy(annotation)
            changed["objects"][0]["box"], changed["objects"][1]["box"] = changed["objects"][1]["box"], changed["objects"][0]["box"]
            self.assertFalse(score(changed)["passed"])
            manifest = {"signature": "test", "config": {"rows": ["scene"], "seeds": [1]}, "rows": [record]}
            report = evaluation.evaluate(manifest, [annotation], self.root, ["candidate_layout"], min_groups=1, lower_bound=0)
            self.assertTrue(report["methods"]["candidate_layout"]["passed"])
            self.assertFalse(evaluation.evaluate(manifest, [], self.root, ["candidate_layout"], min_groups=1, lower_bound=0.9)["methods"]["candidate_layout"]["passed"])
            manifest["config"]["seeds"].append(2)
            with self.assertRaises(ValueError):
                evaluation.evaluate(manifest, [annotation], self.root, ["candidate_layout"])

        def test_composition_rejects_overlap_and_invalid_boxes(self):
            objects = [{**self.assets[0], "box": [0.1, 0.1, 0.8, 0.8]}, {**self.assets[1], "box": [0.1, 0.1, 0.8, 0.8]}]
            with self.assertRaises(ValueError):
                compose(128, objects, self.root)
            self.assertLess(wilson(4, 4)[0], 0.6)
            self.assertGreater(wilson(300, 300)[0], 0.98)

        def test_data_runner_reuse_and_profile_conflict(self):
            profile = self.root / "profile.json"
            profile.write_text(json.dumps({"dataset": {"count": 30, "resolution": 128, "seed": 42}}), encoding="utf-8")
            command = [sys.executable, str(ROOT / "03-spatial-distillation/run.ipynb"), "--stage", "data",
                       "--assets", str(self.source), "--dataset", str(self.root / "dataset"),
                       "--output", str(self.root / "run"), "--profile", str(profile)]
            for _ in range(2):
                result = notebook_completed(command, capture_output=True, text=True)
                self.assertEqual(result.returncode, 0, result.stdout+result.stderr)
            profile.write_text(json.dumps({"dataset": {"count": 31, "resolution": 128, "seed": 42}}), encoding="utf-8")
            result = notebook_completed(command, capture_output=True, text=True)
            self.assertNotEqual(result.returncode, 0)
            self.assertIn("ne correspond pas", result.stderr)


    class TrainingChecks(unittest.TestCase):
        def test_cpu_warmup_checkpoint_and_configuration_guard(self):
            with tempfile.TemporaryDirectory() as folder:
                root = Path(folder)
                dataset, cache, initial, output = (root / k for k in ("data", "cache", "initial", "run"))
                dataset.mkdir()
                rows = [{"id": str(i), "caption_id": str(i), "split": "validation" if i >= 4 else "train", "objects": []}
                        for i in range(6)]
                corpus = [{"id": str(i), "text": f"scene {i}", "split": r["split"]} for i, r in enumerate(rows)]
                for name, values in (("scenes.jsonl", rows), ("corpus.jsonl", corpus)):
                    (dataset / name).write_text("".join(json.dumps(r)+"\n" for r in values), encoding="utf-8")
                atomic_json(dataset / "manifest.json", {"dataset_sha256": digest(rows)})
                matrix = torch.randn(8, 16)
                student = FeatureCache(cache / "student", {"corpus": digest(corpus), "encoder": {"kind": "synthetic-student"}})
                teacher = FeatureCache(cache / "teacher", {"corpus": digest(corpus), "encoder": {"kind": "synthetic-teacher"}})
                for row in rows:
                    source = torch.randn(5, 8)
                    student.write([(row["id"], source, list(range(5)))])
                    teacher.write([(row["id"], source @ matrix, list(range(5)))])
                Projector(8, 16, 8).save(initial)
                command = [sys.executable, str(ROOT / "03-spatial-distillation/train.ipynb"), "warmup", "--device", "cpu",
                           "--dataset", str(dataset), "--cache", str(cache), "--output", str(output),
                           "--init-projector", str(initial), "--steps", "4", "--accumulation", "1", "--eval-every", "2",
                           "--context-layers", "1", "--context-heads", "2"]
                environment = {**os.environ, "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
                result = notebook_completed(command, capture_output=True, text=True, env=environment)
                self.assertEqual(result.returncode, 0, result.stdout+result.stderr)
                complete = json.loads((output / "training-complete.json").read_text())
                self.assertEqual(complete["steps_completed"], 4)
                self.assertFalse(complete["spatial_quality_validated"])
                checkpoint = torch.load(output / "checkpoint.pt", map_location="cpu", weights_only=True)
                self.assertEqual(checkpoint["step"], 4)
                self.assertIn("optimizer", checkpoint)
                self.assertIn("python_rng", checkpoint)
                self.assertIn("torch_rng", checkpoint)
                weights = output / "best/projector/model.safetensors"
                original_hash = sha256(weights)
                Projector.load(weights.parent)
                result = notebook_completed(command, capture_output=True, text=True, env=environment)
                self.assertEqual(result.returncode, 0, result.stdout+result.stderr)
                self.assertEqual(sha256(weights), original_hash)
                result = notebook_completed(command+["--learning-rate", "0.02"], capture_output=True, text=True, env=environment)
                self.assertNotEqual(result.returncode, 0)
                self.assertIn("autre expérience", result.stderr)


    def run(parameters=None):
        import unittest
        suite = unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])
        result = unittest.TextTestRunner(verbosity=2).run(suite)
        if not result.wasSuccessful():
            raise RuntimeError("La suite contient des échecs ; consulter le rapport.")
        return result



## tests.test_spatial_publication

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_8ac2a5f0c9655100':
    """CPU regression checks for spatial release guards; no GPU or network calls."""




    from sharm_notebook_runtime import notebook_completed, parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    from contextlib import nullcontext


    import copy


    import importlib.util


    import json


    from pathlib import Path


    import sys


    import tempfile


    from types import SimpleNamespace


    import unittest


    from unittest.mock import patch


    import torch


    ROOT = Path(__file__).resolve().parents[1]


    sys.path.insert(0, str(ROOT / "common"))


    from sharm_image21.core import file_hash


    from sharm_image21.pipeline import SharmImage21Pipeline


    from sharm_image21.spatial_data import layout_prompt, read_rows


    publisher = load_notebook(ROOT / "publish/publish_spatial.ipynb", name="publish_spatial")


    class LayoutPromptChecks(unittest.TestCase):
        objects = [{"label": "Bulbasaur", "box": [0.1, 0.2, 0.4, 0.8]},
                   {"label": "shiny Castform, rainy form", "box": [0.6, 0.2, 0.9, 0.8]}]

        def test_bindings_english_french_and_added_labels(self):
            for prompt in ("Object 1: Bulbasaur. Object 2: shiny Castform, rainy form.",
                           "Objet 1 : Bulbasaur. Objet 2 : shiny Castform, rainy form.",
                           "Bulbasaur to the left of shiny Castform, rainy form."):
                result, boxes = layout_prompt(prompt, self.objects)
                self.assertTrue(result.startswith(prompt))
                self.assertEqual(boxes, [o["box"] for o in self.objects])
                self.assertEqual(result.count("Normalized object boxes xyxy:"), 1)
                if prompt.startswith(("Object 1", "Objet 1")):
                    self.assertEqual(result.count("Bulbasaur"), 1)
                else:
                    self.assertIn("Object 1: Bulbasaur.", result)

        def test_mismatched_numbering_and_duplicate_coordinates_rejected(self):
            for prompt in ("Object 1: Charmander. Object 2: Bulbasaur.",
                           "Object 1: Bulbasaur.", "Normalized object boxes xyxy: []",
                           "Object 1: Bulbasaur. Object 2: shiny Castform, rainy form. Object 3: Charmander.",
                           "Object 1: Bulbasaur. Object 1: Bulbasaur. Object 2: shiny Castform, rainy form."):
                with self.subTest(prompt=prompt), self.assertRaises(ValueError):
                    layout_prompt(prompt, self.objects)

        def test_invalid_objects_and_boxes_rejected(self):
            for objects in ([], self.objects*9, [{"label": "", "box": [0, 0, 1, 1]}],
                            [{"label": "a", "box": [0.5, 0, 0.2, 1]}],
                            [{"label": "a", "box": [0, 0, 1.1, 1]}],
                            [{"label": "a", "box": [0, 0, float("nan"), 1]}],
                            [{"label": "a", "box": None}]):
                with self.subTest(objects=objects), self.assertRaises(ValueError):
                    layout_prompt("test", objects)

        def test_exact_training_prompt_convention(self):
            dataset = ROOT / "artifacts/spatial/dataset/scenes.jsonl"
            if not dataset.exists():
                self.skipTest("Local training dataset absent")
            for row in read_rows(dataset):
                text, boxes = layout_prompt(row["text_plain"], row["objects"])
                self.assertEqual(text, row["text"])
                self.assertEqual(boxes, [o["box"] for o in row["objects"]])


    class PipelineReleaseChecks(unittest.TestCase):
        def setUp(self):
            self.temporary = tempfile.TemporaryDirectory()
            self.addCleanup(self.temporary.cleanup)
            self.directory = Path(self.temporary.name)
            (self.directory / "projector").mkdir()
            (self.directory / "projector/model.safetensors").write_bytes(b"fixture")
            self.config = {"encoder_format": "fp8-e4m3", "encoder_frozen": True,
                           "projector_sha256": file_hash(self.directory / "projector/model.safetensors"),
                           "pipeline_format": "spatial-v1", "training_complete": True,
                           "student_identity": {"kind": "fixture"}, "spatial_lora_sha256": None}

        def pipeline(self):
            (self.directory / "sharm_config.json").write_text(json.dumps(self.config), encoding="utf-8")
            return SharmImage21Pipeline(self.directory)

        def test_spatial_completion_legacy_validation_and_weight_guards(self):
            self.pipeline()
            self.config["training_complete"] = False
            with self.assertRaisesRegex(ValueError, "incomplet"):
                self.pipeline()
            self.config.pop("pipeline_format")
            with self.assertRaisesRegex(ValueError, "non valid"):
                self.pipeline()
            self.config["base_validation_passed"] = True
            self.pipeline()
            self.config["projector_sha256"] = "wrong"
            with self.assertRaisesRegex(ValueError, "Empreinte"):
                self.pipeline()

        def test_spatial_lora_integrity(self):
            folder = self.directory / "lora"
            folder.mkdir()
            (folder / "pytorch_lora_weights.safetensors").write_bytes(b"adapter")
            self.config["spatial_lora_sha256"] = file_hash(folder / "pytorch_lora_weights.safetensors")
            self.pipeline()
            (folder / "pytorch_lora_weights.safetensors").write_bytes(b"modified")
            with self.assertRaisesRegex(ValueError, "LoRA spatial"):
                self.pipeline()

        def test_sequential_encoding_frees_encoder_before_projector(self):
            model = self.pipeline()
            events, prompts = [], []

            class Encoder:
                identity = {"kind": "fixture"}

                def __init__(self, *args, **kwargs):
                    events.append("encoder")

                def encode(self, texts):
                    prompts.extend(texts)
                    return [(torch.zeros(i+4, 8), []) for i in range(len(texts))]

                def __del__(self):
                    events.append("freed")

            class Projector:
                def eval(self):
                    return self

                def __call__(self, values, layout=None):
                    return torch.ones(len(values), 16)

            def load(*args):
                self.assertEqual(events, ["encoder", "freed"])
                events.append("projector")
                return Projector()

            prefix = "sharm_image21.pipeline."
            with patch(prefix+"FrozenEncoder", Encoder), patch(prefix+"Projector.load", load), \
                    patch(prefix+"torch.autocast", lambda *a, **k: nullcontext()), \
                    patch.object(torch.Tensor, "cuda", lambda value: value):
                embeddings, mask = model.encode_prompt(["a", "b"])
            self.assertEqual(prompts, ["a", "b"])
            self.assertEqual(embeddings.shape, (2, 5, 16))
            self.assertEqual(embeddings.dtype, torch.bfloat16)
            self.assertEqual(mask.sum(1).tolist(), [4, 5])
            with self.assertRaises(ValueError):
                model.encode_prompt(["a", "b"], layout=LayoutPromptChecks.objects)

        def test_call_forwards_layout_and_loads_matching_spatial_lora(self):
            (self.directory / "lora").mkdir()
            lora = self.directory / "lora/pytorch_lora_weights.safetensors"
            lora.write_bytes(b"lora")
            self.config["spatial_lora_sha256"] = file_hash(lora)
            model = self.pipeline()
            calls = []

            class Diffuser:
                def load_lora_weights(self, path, **kwargs):
                    calls.append((path, kwargs))

                def __call__(self, **kwargs):
                    self.last = kwargs
                    return SimpleNamespace(images=["output"])

            with patch.object(model, "encode_prompt", return_value=(torch.zeros(1, 4, 16), torch.ones(1, 4))) as encode, \
                    patch("sharm_image21.pipeline.load_diffuser_pipeline", return_value=Diffuser()), \
                    patch.object(torch.Tensor, "cuda", lambda value: value):
                result = model("prompt", layout=LayoutPromptChecks.objects, width=256)
                self.assertEqual(result.images, ["output"])
                encode.assert_called_once_with("prompt", layout=LayoutPromptChecks.objects)
                self.assertEqual(calls, [(self.directory / "lora", {"adapter_name": "spatial"})])


    class PublicationGuardChecks(unittest.TestCase):
        def setUp(self):
            self.temporary = tempfile.TemporaryDirectory()
            self.addCleanup(self.temporary.cleanup)
            self.root = Path(self.temporary.name)
            self.run, self.dataset = self.root / "run", self.root / "dataset"
            self.weights = self.run / "flow/best/projector/model.safetensors"
            self.weights.parent.mkdir(parents=True)
            self.weights.write_bytes(b"trained")
            self.candidate = {"signature": "training", "step": 10, "config": {"phase": "flow"},
                              "numerical_validation_loss": 0.1, "projector_sha256": publisher.sha(self.weights),
                              "lora_sha256": None, "identity": {"dataset_sha256": "dataset"}}
            publisher.write(self.run / "flow/best/candidate.json", self.candidate)
            complete = {"signature": "training", "steps_completed": 10}
            publisher.write(self.run / "flow/training-complete.json", complete)
            publisher.write(self.run / "warmup/training-complete.json", complete)
            publisher.write(self.run / "run-status.json", {"state": "complete"})
            publisher.write(self.dataset / "manifest.json", {"dataset_sha256": "dataset"})
            self.predictions = {"signature": "gallery", "config": {"candidate_sha256": publisher.sha(self.run / "flow/best/candidate.json")},
                                "rows": [{"id": "one", "method": "candidate_text"}, {"id": "two", "method": "candidate_layout"}]}
            publisher.write(self.run / "validation/predictions.json", self.predictions)
            self.review = {"passed": True, "method": "agent_visual_review", "reviewer": "test-fixture",
                           "prediction_signature": "gallery", "projector_sha256": publisher.sha(self.weights),
                           "reviewed_ids": ["one", "two"]}
            publisher.write(self.root / "visual-review.json", self.review)
            self.patch = patch.object(publisher, "ARTIFACTS", self.root)
            self.patch.start()
            self.addCleanup(self.patch.stop)

        def test_completed_matching_and_fully_reviewed_candidate_accepted(self):
            self.assertEqual(publisher.prerequisites(self.run, self.dataset)[1], self.candidate)

        def test_unfinished_stale_gallery_and_altered_weights_rejected(self):
            publisher.write(self.run / "run-status.json", {"state": "running"})
            with self.assertRaisesRegex(ValueError, "termin"):
                publisher.prerequisites(self.run, self.dataset)
            publisher.write(self.run / "run-status.json", {"state": "complete"})
            self.predictions["config"]["candidate_sha256"] = "stale"
            publisher.write(self.run / "validation/predictions.json", self.predictions)
            with self.assertRaisesRegex(ValueError, "autre révision"):
                publisher.prerequisites(self.run, self.dataset)
            self.weights.write_bytes(b"changed")
            with self.assertRaisesRegex(ValueError, "Projecteur modifi"):
                publisher.prerequisites(self.run, self.dataset)

        def test_review_requires_exact_gallery_all_images_and_identified_reviewer(self):
            for updates in ({"passed": False}, {"reviewer": ""}, {"method": "automated_mse"},
                            {"prediction_signature": "stale"}, {"projector_sha256": "stale"},
                            {"reviewed_ids": ["one"]}):
                with self.subTest(updates=updates):
                    publisher.write(self.root / "visual-review.json", {**self.review, **updates})
                    with self.assertRaises(ValueError):
                        publisher.prerequisites(self.run, self.dataset)

        def test_path_and_payload_scope(self):
            self.assertEqual(publisher.contained(self.root, "images/a.png", ".png"), self.root / "images/a.png")
            for name in ("../outside.png", "C:/outside.png", "/outside.png", "images/../../a.png", "images\\a.png", "images//a.png"):
                with self.subTest(name=name), self.assertRaises(ValueError):
                    publisher.contained(self.root, name, ".png")
            for name in (".env", "token", "publication-request.json", "checkpoint.pt", "cache/a.safetensors",
                         "dataset/images/a.png", "sharm_image21/__pycache__/a.pyc", "processor/tokenizer.secret"):
                self.assertFalse(publisher.payload_path_allowed(name), name)
            for name in ("README.md", "projector/model.safetensors", "text_encoder/model-00001-of-00002.safetensors",
                         "sharm_image21/pipeline.py", "comparisons/spatial/images/a.png"):
                self.assertTrue(publisher.payload_path_allowed(name), name)

        def test_verified_payload_detects_changes_and_excludes_unlisted_files(self):
            output = self.root / "release"
            output.mkdir()
            (output / "README.md").write_text("fixture", encoding="utf-8")
            (output / ".env").write_text("must never upload", encoding="utf-8")
            manifest = {"repo_id": publisher.REPO, "projector_sha256": "projector", "files": {"README.md": publisher.sha(output / "README.md")}}
            publisher.write(output / "release_manifest.json", manifest)
            images = {}
            for method in ("candidate_text", "candidate_layout"):
                for suffix in ("", "-preview"):
                    name = f"validation/{method}{suffix}.png"
                    path = output / name
                    path.parent.mkdir(exist_ok=True)
                    path.write_bytes(b"fixture")
                    images[name] = publisher.sha(path)
            verification = {"passed": True, "release_manifest_sha256": publisher.sha(output / "release_manifest.json"),
                            "projector_sha256": "projector", "images": images}
            publisher.write(output / "export-validation.json", verification)
            _, files = publisher.verified_payload(output)
            self.assertNotIn(".env", files)
            self.assertIn("validation/candidate_layout.png", files)
            (output / "README.md").write_text("altered", encoding="utf-8")
            with self.assertRaisesRegex(ValueError, "Export modifi"):
                publisher.verified_payload(output)
            changed = copy.deepcopy(manifest)
            changed["files"][".env"] = publisher.sha(output / ".env")
            publisher.write(output / "release_manifest.json", changed)
            verification["release_manifest_sha256"] = publisher.sha(output / "release_manifest.json")
            publisher.write(output / "export-validation.json", verification)
            with self.assertRaisesRegex(ValueError, "hors périmètre"):
                publisher.verified_payload(output)


    def run(parameters=None):
        import unittest
        suite = unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])
        result = unittest.TextTestRunner(verbosity=2).run(suite)
        if not result.wasSuccessful():
            raise RuntimeError("La suite contient des échecs ; consulter le rapport.")
        return result



## tests.test_sprite_dataset

Implémentation de cette étape. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_step_881d8103dffd4110':
    """Regressions for the six audited normal/shiny sprite sheets."""




    from sharm_notebook_runtime import notebook_completed, parameter_namespace, load_notebook, export_runtime, call_with_flags, sha256 as notebook_hash


    import contextlib


    import csv


    import importlib.util


    import io


    import json


    from pathlib import Path


    import sys


    import tempfile


    import unittest


    from unittest.mock import patch


    from PIL import Image, ImageDraw


    ROOT = Path(__file__).resolve().parents[1]


    sprites = load_notebook(ROOT / "02-sprite-lora/prepare_dataset.ipynb", name="sprite_preparation")


    class SpriteDatasetChecks(unittest.TestCase):
        def test_missing_species_and_extra_castform_cells(self):
            kanto = sprites.sheet_cells("Kanto / Gen 1")
            johto = sprites.sheet_cells("Johto / Gen 2")
            hoenn = sprites.sheet_cells("Hoenn / Gen 3")
            self.assertEqual((len(kanto), len(johto), len(hoenn)), (302, 198, 270))
            self.assertEqual(kanto[-2:], [(151, ""), (151, "")])
            self.assertEqual(johto[96:100], [(200, ""), (200, ""), (202, ""), (202, "")])
            self.assertEqual(johto[-2:], [(251, ""), (251, "")])
            self.assertEqual(hoenn[148:152], [(326, ""), (326, ""), (328, ""), (328, "")])
            self.assertEqual(hoenn[190], (348, ""))
            self.assertEqual(hoenn[196:200], [(351, form) for form in
                             ("normal form", "sunny form", "rainy form", "snowy form")])
            self.assertEqual(hoenn[200], (352, ""))
            self.assertEqual(hoenn[-2:], [(386, ""), (386, "")])
            ids = {species for cells in (kanto, johto, hoenn) for species, _ in cells}
            self.assertEqual(ids, set(range(1, 387)) - {201, 327})
            self.assertEqual(len(sprites.SHEETS), 6)

        def test_shiny_and_weather_captions_do_not_use_normal_color(self):
            species = {"name": "Castform", "genus": "Weather Pokémon", "color": "gray",
                       "shape": "ball", "pokedex_note": "Its form changes with the weather."}
            self.assertIn("gray coloration", sprites.make_caption(species))
            for shiny in (False, True):
                caption = sprites.make_caption(species, shiny, "snowy form")
                self.assertNotIn("gray coloration", caption)
                self.assertIn("snowy form", caption)
                self.assertEqual("shiny Castform" in caption, shiny)
            self.assertNotIn("gray coloration", sprites.make_caption(species, True))

        def test_alpha_removes_enclosed_background_and_preserves_nearby_color(self):
            tile = Image.new("RGBA", (16, 16), (*sprites.BACKGROUND, 255))
            draw = ImageDraw.Draw(tile)
            draw.rectangle((2, 2, 13, 13), fill=(200, 10, 20, 255))
            draw.rectangle((6, 6, 9, 9), fill=(*sprites.BACKGROUND, 255))
            tile.putpixel((0, 0), (165, 235, 254, 255))
            self.assertEqual(sprites.remove_background_color(tile, sprites.BACKGROUND), 129)
            self.assertEqual(tile.getpixel((7, 7)), (0, 0, 0, 0))
            self.assertEqual(tile.getpixel((15, 15)), (0, 0, 0, 0))
            self.assertEqual(tile.getpixel((0, 0)), (165, 235, 254, 255))
            self.assertEqual(tile.getpixel((3, 3)), (200, 10, 20, 255))

        def test_cli_writes_correct_labels_groups_and_source_hashes_without_network(self):
            with tempfile.TemporaryDirectory() as folder:
                root = Path(folder)
                output = root / "dataset"
                output.mkdir()
                traits = {str(i): {"name": f"Species {i}", "genus": "Test Pokémon", "color": "gray",
                                  "shape": "ball", "pokedex_note": "Fixture."} for i in range(1, 387)}
                (output / "pokemon_traits.json").write_text(json.dumps(traits), encoding="utf-8")
                marked = {"21229.png": 98, "21228.png": 196, "21650.png": 199}
                for name, generation, _ in sprites.SHEETS:
                    count = len(sprites.sheet_cells(generation))
                    sheet = Image.new("RGB", (695, 5+69*((count+9)//10)), sprites.BACKGROUND)
                    if name in marked:
                        row, col = divmod(marked[name], 10)
                        x, y = 5+col*69, 5+row*69
                        ImageDraw.Draw(sheet).rectangle((x+10, y+10, x+15, y+15), fill="red")
                    sheet.save(root / name)
                with patch.object(sys, "argv", ["prepare_dataset.ipynb", "--sheet-dir", str(root),
                                               "--output-dir", str(output), "--training-size", "64"]), \
                        patch.object(sprites, "urlopen", side_effect=AssertionError("Unexpected network")), \
                        contextlib.redirect_stdout(io.StringIO()):
                    call_with_flags(ROOT / "02-sprite-lora/prepare_dataset.ipynb", sys.argv[1:])
                with (output / "manifest.csv").open(encoding="utf-8") as stream:
                    rows = {row["sheet"]: row for row in csv.DictReader(stream)}
                self.assertEqual(len(rows), 3)
                self.assertEqual(rows["21229.png"]["species_id"], "202")
                self.assertEqual(rows["21650.png"]["label"], "shiny Species 351, snowy form")
                self.assertEqual(rows["21228.png"]["species_id"], "351")
                self.assertTrue(all(len(row["source_sha256"]) == 64 for row in rows.values()))
                metadata = [json.loads(line) for line in (output / "train_64/metadata.jsonl").read_text().splitlines()]
                self.assertEqual(sorted(row["group_id"] for row in metadata), ["202", "351", "351"])
                sources = json.loads((output / "sources.json").read_text())
                self.assertEqual(len(sources["sheets"]), 6)
                self.assertEqual(sources["sprite_count"], 3)


    def run(parameters=None):
        import unittest
        suite = unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])
        result = unittest.TextTestRunner(verbosity=2).run(suite)
        if not result.wasSuccessful():
            raise RuntimeError("La suite contient des échecs ; consulter le rapport.")
        return result



## library.core

Bibliothèque partagée. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_image21.core':


    import sys
    import hashlib


    import json


    import os


    import random


    import time


    from pathlib import Path


    import torch


    from safetensors import safe_open


    from safetensors.torch import load_file, save_file


    from torch import nn


    from torch.nn import functional as F


    ROOT = Path(__file__).resolve().parents[2]


    PROJECT = ROOT.parent


    WEIGHTS = PROJECT / "1-local-generation" / "weights"


    ORIGINAL = Path(os.environ["SHARM_BASE_DIR"]) if os.environ.get("SHARM_BASE_DIR") else WEIGHTS / "qwen-image-2.1"


    STUDENT = ORIGINAL / "text_encoder" if os.environ.get("SHARM_BASE_DIR") else WEIGHTS / "sharm-vl4b-fp8"


    ARTIFACTS = ROOT / "artifacts"


    CACHE = ROOT / "cache"


    FEATURE_CONVENTION = "qwenimage21-raw-template-leftpad-pre-final-rmsnorm-v1"


    TEACHER_REVISION = "d26bb61231c349cf6b7896fa83353113880e1ba3"


    STUDENT_REVISION = "fefbb44cbcce8d1bb7e20b920b94f77432b3446d"


    SYS_PROMPT = "Comprehend and analyze the provided prompt."


    TEMPLATE = (
        f"<|im_start|>system\n{SYS_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n{}<|im_end|>\n<|im_start|>assistant\n"
    )


    def digest(value) -> str:
        return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


    def file_hash(path: Path) -> str:
        path = Path(path)
        if path.suffix == ".ipynb":
            return sys.modules["sharm_notebook_runtime"].sha256(path)
        sha = hashlib.sha256()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):
                sha.update(chunk)
        return sha.hexdigest()


    def atomic_json(path: Path, value) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + ".tmp")
        temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        os.replace(temporary, path)


    def read_jsonl(path: Path) -> list[dict]:
        return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


    def seed_all(seed: int = 42) -> None:
        random.seed(seed)
        torch.manual_seed(seed)


    def model_identity(kind: str, model_dir: Path | None = None) -> dict:
        model_dir = model_dir or (STUDENT if kind == "student" else ORIGINAL / "text_encoder")
        return {
            "kind": kind,
            "revision": STUDENT_REVISION if kind == "student" else TEACHER_REVISION,
            "config_sha256": file_hash(model_dir / "config.json"),
            "index_sha256": file_hash(model_dir / "model.safetensors.index.json"),
            "convention": FEATURE_CONVENTION,
            "prefill_policy": "single-prompt-fp8" if kind == "student" else "left-padded-bf16",
        }


    class Projector(nn.Module):
        def __init__(self, input_dim: int = 2560, output_dim: int = 4096, bottleneck: int = 1024,
                     context_layers: int = 0, context_heads: int = 8, layout_enabled: bool = False):
            super().__init__()
            self.settings = dict(input_dim=input_dim, output_dim=output_dim, bottleneck=bottleneck)
            if context_layers < 0 or context_heads < 1 or (context_layers and bottleneck % context_heads):
                raise ValueError("Largeur d'attention divisible par le nombre de têtes requise.")
            # Preserve the v1 config and state keys when no spatial extension is used.
            if context_layers or layout_enabled:
                self.settings.update(context_layers=context_layers, context_heads=context_heads,
                                     layout_enabled=layout_enabled)
            self.skip = nn.Linear(input_dim, output_dim)
            self.norm = nn.RMSNorm(input_dim, eps=1e-6)
            self.up = nn.Linear(input_dim, bottleneck)
            self.down = nn.Linear(bottleneck, output_dim)
            nn.init.zeros_(self.down.weight)
            nn.init.zeros_(self.down.bias)
            self.context = nn.ModuleList([
                nn.TransformerEncoderLayer(bottleneck, context_heads, 4*bottleneck, dropout=0.0,
                                           activation="gelu", batch_first=True, norm_first=True)
                for _ in range(context_layers)])
            if layout_enabled:
                self.layout_box = nn.Sequential(nn.Linear(4, bottleneck), nn.GELU(), nn.Linear(bottleneck, bottleneck))
                self.layout_object = nn.Embedding(16, bottleneck)

        def forward(self, hidden, attention_mask=None, layout=None):
            """Project ordered tokens; optional normalized xyxy boxes follow object order.

            The legacy token-wise path keeps identical arithmetic. In the contextual
            path, valid tokens must stay in order (no random token subsampling).
            """
            if not self.context and layout is None:
                return self.skip(hidden) + self.down(F.gelu(self.up(self.norm(hidden))))
            unbatched = hidden.ndim == 2
            hidden = hidden.unsqueeze(0) if unbatched else hidden
            if hidden.ndim != 3:
                raise ValueError("États attendus : [tokens, dimensions] ou [lot, tokens, dimensions].")
            values = self.up(self.norm(hidden))
            text_length = values.shape[1]
            padding = None
            if attention_mask is not None:
                mask = attention_mask.unsqueeze(0) if attention_mask.ndim == 1 else attention_mask
                if mask.shape != hidden.shape[:2] or not mask.bool().any(dim=1).all():
                    raise ValueError("Masque de tokens invalide ou séquence entièrement masquée.")
                padding = ~mask.to(device=hidden.device, dtype=torch.bool)
            if layout is not None:
                if not self.settings.get("layout_enabled"):
                    raise ValueError("Ce projecteur n'a pas été entraîné avec des boîtes spatiales.")
                boxes = layout.unsqueeze(0) if layout.ndim == 2 else layout
                if boxes.ndim != 3 or boxes.shape[0] != hidden.shape[0] or boxes.shape[-1] != 4 or not 1 <= boxes.shape[1] <= 16:
                    raise ValueError("De 1 à 16 boîtes xyxy par prompt sont requises.")
                if not torch.isfinite(boxes).all() or (boxes < 0).any() or (boxes > 1).any() or (boxes[..., 2:] <= boxes[..., :2]).any():
                    raise ValueError("Boîtes xyxy normalisées non vides requises.")
                slots = torch.arange(boxes.shape[1], device=hidden.device)
                layout_values = self.layout_box(boxes.to(device=hidden.device, dtype=values.dtype)) + self.layout_object(slots)[None].to(values.dtype)
                values = torch.cat((values, layout_values), dim=1)
                if padding is not None:
                    padding = F.pad(padding, (0, boxes.shape[1]), value=False)
            width = values.shape[-1]
            positions = torch.arange(values.shape[1], device=values.device).float()[:, None]
            frequencies = torch.exp(torch.arange(0, width, 2, device=values.device).float()*(-9.210340372/width))
            angles = positions*frequencies
            positional = torch.zeros(values.shape[1], width, device=values.device)
            positional[:, 0::2], positional[:, 1::2] = angles.sin(), angles[:, :width//2].cos()
            values = values + positional.to(values.dtype)
            for layer in self.context:
                values = layer(values, src_key_padding_mask=padding)
            text = self.skip(hidden) + self.down(F.gelu(values[:, :text_length]))
            if values.shape[1] > text_length:
                text = torch.cat((text, self.down(F.gelu(values[:, text_length:]))), dim=1)
            return text[0] if unbatched else text

        def save(self, path: Path, metadata: dict | None = None, storage_dtype=None):
            path.mkdir(parents=True, exist_ok=True)
            atomic_json(path / "config.json", {**self.settings, "convention": FEATURE_CONVENTION, **(metadata or {})})
            temporary = path / "model.tmp.safetensors"
            save_file({key: val.detach().to(device="cpu", dtype=storage_dtype or val.dtype).contiguous()
                       for key, val in self.state_dict().items()}, temporary)
            os.replace(temporary, path / "model.safetensors")

        @classmethod
        def load(cls, path: Path, device="cpu"):
            config = json.loads((path / "config.json").read_text())
            if config.get("convention") != FEATURE_CONVENTION:
                raise ValueError("Convention d'embeddings du projecteur incompatible.")
            state = load_file(path / "model.safetensors")
            keys = ("input_dim", "output_dim", "bottleneck", "context_layers", "context_heads", "layout_enabled")
            model = cls(**{key: config[key] for key in keys if key in config}).to(state["skip.weight"].dtype)
            model.load_state_dict(state)
            return model.to(device)


    class FrozenEncoder:
        """Extract the norm INPUT without LM logits, past-key-values or all-layer captures."""

        def __init__(self, kind: str, gpu_memory: str = "auto", model_dir: Path | None = None,
                     processor_dir: Path | None = None):
            from transformers import Qwen3VLForConditionalGeneration, Qwen3VLProcessor

            if kind not in ("teacher", "student"):
                raise ValueError(kind)
            self.kind = kind
            self.model_dir = model_dir or (STUDENT if kind == "student" else ORIGINAL / "text_encoder")
            processor_dir = processor_dir or (ORIGINAL / "processor" if (ORIGINAL / "sharm_config.json").exists() else self.model_dir if kind == "student" else ORIGINAL / "processor")
            self.processor = Qwen3VLProcessor.from_pretrained(processor_dir, local_files_only=True)
            self.identity = model_identity(kind, self.model_dir)
            sys_message = [{"role": "system", "content": [{"type": "text", "text": SYS_PROMPT}]}]
            self.drop_idx = len(self.processor.apply_chat_template(sys_message, tokenize=True, return_dict=False)[0])
            kwargs = dict(dtype=torch.bfloat16, attn_implementation="sdpa", local_files_only=True)
            if kind == "student":
                os.environ.setdefault("TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR", "1")
                # This extractor calls model.model with text only: visual and lm_head
                # are never executed. Keep their weights in RAM to leave room for
                # FP8 prefill on an 8 GB GPU; all language layers stay on CUDA.
                kwargs["device_map"] = {"model.language_model": "cuda", "model.visual": "cpu", "lm_head": "cpu"}
            else:
                import psutil
                free_gpu, _ = torch.cuda.mem_get_info()
                gpu_budget = max(512*2**20, free_gpu-2*2**30) if gpu_memory == "auto" else gpu_memory
                cpu_budget = max(1*2**30, psutil.virtual_memory().available-2*2**30)
                offload = CACHE / "teacher-offload"
                offload.mkdir(parents=True, exist_ok=True)
                kwargs.update(device_map="auto", max_memory={0: gpu_budget, "cpu": cpu_budget},
                              offload_folder=str(offload), offload_state_dict=True)
            started = time.monotonic()
            self.model = Qwen3VLForConditionalGeneration.from_pretrained(self.model_dir, **kwargs)
            self.model.eval().requires_grad_(False)
            self.hidden_size = self.model.config.text_config.hidden_size
            self.device = torch.device("cuda:0")
            self.quantized_linears = 0
            if kind == "student":
                for name, module in self.model.named_modules():
                    if type(module).__name__ == "FP8Linear":
                        if module.weight.dtype != torch.float8_e4m3fn:
                            raise RuntimeError(f"Déquantification inattendue : {name} / {module.weight.dtype}")
                        if name.startswith("model.language_model.") and module.weight.device.type != "cuda":
                            raise RuntimeError(f"Couche FP8 textuelle hors CUDA : {name}")
                        self.quantized_linears += 1
                if not self.quantized_linears or getattr(self.model.config.quantization_config, "dequantize", False):
                    raise RuntimeError("Le décodeur étudiant n'est pas chargé en FP8 effectif.")
            print(json.dumps({"encoder": kind, "load_seconds": round(time.monotonic()-started, 2),
                              "hidden_size": self.hidden_size, "fp8_linears": self.quantized_linears,
                              "gpu_allocated_gib": torch.cuda.memory_allocated()/2**30}), flush=True)

        @torch.inference_mode()
        def encode(self, prompts: list[str]) -> list[tuple[torch.Tensor, list[int]]]:
            # On SM120, mixed-length FP8 prefill changed these features by 8.4% relative
            # to single-prompt inference. Keep the same prefill in cache and deployment.
            if self.kind == "student" and len(prompts) > 1:
                return [self.encode([text])[0] for text in prompts]
            inputs = self.processor(text=[TEMPLATE.format(text or " ") for text in prompts],
                                    padding=True, padding_side="left", return_tensors="pt").to(self.device)
            captured = []

            def take_norm_input(module, args):
                captured.append(args[0].detach())

            handle = self.model.model.language_model.norm.register_forward_pre_hook(take_norm_input)
            try:
                self.model.model(**inputs, use_cache=False, output_hidden_states=False, return_dict=True)
            finally:
                handle.remove()
            if len(captured) != 1:
                raise RuntimeError("Impossible d'isoler la dernière représentation pré-normalisation.")
            hidden = captured.pop()
            result = []
            for row, mask, ids in zip(hidden, inputs.attention_mask.bool(), inputs.input_ids):
                # With CPU/disk offload, the final hidden states can live on a
                # different device from the token IDs and their attention mask.
                values = row[mask.to(row.device)][self.drop_idx:].to(device="cpu", dtype=torch.bfloat16).contiguous()
                token_ids = ids[mask][self.drop_idx:].cpu().tolist()
                if not len(values) or values.shape[-1] != self.hidden_size or not torch.isfinite(values).all():
                    raise RuntimeError("Embeddings vides, de mauvaise taille ou non finis.")
                result.append((values, token_ids))
            return result


    class FeatureCache:
        def __init__(self, directory: Path, expected: dict | None = None):
            self.directory = directory
            self.path = directory / "index.json"
            directory.mkdir(parents=True, exist_ok=True)
            if self.path.exists():
                self.index = json.loads(self.path.read_text())
                if expected is not None and self.index["identity"] != expected:
                    raise ValueError(f"Cache incompatible : {directory}. Choisir un nouveau dossier.")
            else:
                if expected is None:
                    raise FileNotFoundError(self.path)
                self.index = {"identity": expected, "entries": {}, "shards": 0}
                atomic_json(self.path, self.index)

        def __contains__(self, sample_id):
            return sample_id in self.index["entries"]

        def write(self, rows: list[tuple[str, torch.Tensor, list[int]]]):
            if not rows:
                return
            shard_name = f"shard-{self.index['shards']:06d}.safetensors"
            target = self.directory / shard_name
            temporary = target.with_suffix(".tmp")
            save_file({sample_id: tensor.cpu().contiguous() for sample_id, tensor, _ in rows}, temporary)
            os.replace(temporary, target)
            for sample_id, tensor, token_ids in rows:
                self.index["entries"][sample_id] = {"shard": shard_name, "shape": list(tensor.shape),
                                                       "token_ids": token_ids}
            self.index["shards"] += 1
            atomic_json(self.path, self.index)

        def get(self, sample_id: str) -> torch.Tensor:
            entry = self.index["entries"][sample_id]
            with safe_open(self.directory / entry["shard"], framework="pt", device="cpu") as handle:
                return handle.get_tensor(sample_id)

        def require(self, rows: list[dict]):
            missing = [row["id"] for row in rows if row["id"] not in self]
            if missing:
                raise ValueError(f"Cache incomplet : {self.directory}, {len(missing)} entrées manquantes.")


    def alignment_metrics(predicted: torch.Tensor, target: torch.Tensor) -> dict[str, float]:
        pred, gold = predicted.float(), target.float()
        return {
            "relative_mse": ((pred-gold).square().mean()/gold.square().mean().clamp_min(1e-8)).item(),
            "cosine": F.cosine_similarity(pred, gold, dim=-1).mean().item(),
            "norm_ratio": (pred.norm(dim=-1).mean()/gold.norm(dim=-1).mean().clamp_min(1e-8)).item(),
        }



## library.diffusion

Bibliothèque partagée. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_image21.diffusion':
    """Diffuser-only loading: never constructs the 8B text encoder."""


    from pathlib import Path


    import torch


    from .core import ORIGINAL, STUDENT


    def conditioning_output_dtype(module, args, kwargs, output):
        """NF4 may return FP32; keep Euler's next latent in conditioning precision."""
        conditioning = kwargs.get("encoder_hidden_states")
        if conditioning is None:
            return output
        if isinstance(output, tuple):
            return (output[0].to(conditioning.dtype), *output[1:])
        output.sample = output.sample.to(conditioning.dtype)
        return output


    def load_transformer(base: Path = ORIGINAL, training: bool = False, gradient_checkpointing: bool | str = True):
        from diffusers import BitsAndBytesConfig, QwenImage21Transformer2DModel
        quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                          bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
        model = QwenImage21Transformer2DModel.from_pretrained(
            base, subfolder="transformer", quantization_config=quantization,
            torch_dtype=torch.bfloat16, local_files_only=True)
        model.requires_grad_(False)
        model.to("cuda")
        if training:
            from peft import prepare_model_for_kbit_training
            model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
            mode = "all" if gradient_checkpointing is True else "none" if gradient_checkpointing is False else gradient_checkpointing
            if mode not in ("all", "alternating", "none"):
                raise ValueError(f"Mode de checkpointing inconnu : {mode}")
            if mode != "none":
                model.enable_gradient_checkpointing()
            if mode == "alternating":
                default_checkpoint = model._gradient_checkpointing_func
                selected = {id(block) for index, block in enumerate(model.transformer_blocks) if index % 2 == 0}

                def checkpoint_some_blocks(module, *args):
                    if id(module) in selected:
                        return default_checkpoint(module, *args)
                    return module(*args)

                model.enable_gradient_checkpointing(gradient_checkpointing_func=checkpoint_some_blocks)
        else:
            model.eval()
            # bitsandbytes returns the input dtype from the final projection, whose
            # normalization can upcast to FP32. Euler otherwise keeps FP32 latents
            # and the next NF4 image projection disagrees with the BF16 text branch.
            model.register_forward_hook(conditioning_output_dtype, with_kwargs=True)
        return model


    def load_diffuser_pipeline(base: Path = ORIGINAL, transformer=None):
        from diffusers import AutoencoderKLQwenImage21, FlowMatchEulerDiscreteScheduler, QwenImage21Pipeline
        from transformers import Qwen3VLProcessor
        if transformer is None:
            transformer = load_transformer(base)
        processor_path = base / "processor" if (base / "sharm_config.json").exists() else STUDENT
        processor = Qwen3VLProcessor.from_pretrained(processor_path, local_files_only=True)
        vae = AutoencoderKLQwenImage21.from_pretrained(base, subfolder="vae", torch_dtype=torch.bfloat16,
                                                      local_files_only=True)
        scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(base, subfolder="scheduler", local_files_only=True)
        pipeline = QwenImage21Pipeline(vae=vae, text_encoder=None, processor=processor,
                                      transformer=transformer, scheduler=scheduler)
        pipeline.to("cuda")
        return pipeline


    def predict_flow(transformer, latents, embeddings, timestep):
        from diffusers import QwenImage21Pipeline
        batch, channels, frames, height, width = latents.shape
        if frames != 1 or embeddings.shape[0] != batch:
            raise ValueError("Batch ou nombre de frames incompatible avec le trainer sprites.")
        packed = QwenImage21Pipeline._pack_latents(latents, batch, channels, height, width).to(embeddings.dtype)
        image_mask = torch.cat([
            torch.zeros(batch, embeddings.shape[1], dtype=torch.bool, device=latents.device),
            torch.ones(batch, height*width//4, dtype=torch.bool, device=latents.device)], dim=1)
        result = transformer(hidden_states=packed, encoder_hidden_states=embeddings,
                             encoder_hidden_states_mask=None, timestep=timestep,
                             img_shapes=[[(1, height, width)]]*batch, img_mask=image_mask, return_dict=False)[0]
        result = result[:, -packed.shape[1]:]
        return QwenImage21Pipeline._unpack_latents(result, height*16, width*16, 16)



## library.image_io

Bibliothèque partagée. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_image21.image_io':
    """PNG export helpers that keep alpha intact and make it visible in previews."""




    from pathlib import Path


    from PIL import Image, ImageDraw


    def transparent_fraction(image: Image.Image, threshold: int = 16) -> float:
        """Return the share of pixels with a clearly transparent alpha value."""
        if image.mode != "RGBA":
            raise ValueError("Une image RGBA est requise pour mesurer l’alpha.")
        alpha = image.getchannel("A").histogram()
        return sum(alpha[:threshold + 1]) / (image.width * image.height)


    def require_transparent_background(image: Image.Image) -> float:
        """Reject opaque/empty results when transparent output was requested."""
        fraction = transparent_fraction(image)
        alpha = image.getchannel("A").histogram()
        visible_fraction = sum(alpha[240:]) / (image.width * image.height)
        if fraction < 0.005 or visible_fraction < 0.01:
            raise ValueError(
                "Le rendu n’a pas de fond transparent exploitable "
                f"(pixels transparents: {fraction:.1%}, pixels visibles: {visible_fraction:.1%})."
            )
        return fraction


    def checkerboard_preview(image: Image.Image, tile_size: int = 16) -> Image.Image:
        """Composite RGBA onto a neutral checkerboard for clear display in model cards."""
        if tile_size < 1:
            raise ValueError("La taille des cases du damier doit être positive.")
        rgba = image.convert("RGBA")
        background = Image.new("RGBA", rgba.size, (232, 232, 232, 255))
        draw = ImageDraw.Draw(background)
        for y in range(0, rgba.height, tile_size):
            for x in range(0, rgba.width, tile_size):
                if (x // tile_size + y // tile_size) % 2:
                    draw.rectangle((x, y, min(x + tile_size - 1, rgba.width - 1),
                                    min(y + tile_size - 1, rgba.height - 1)), fill=(184, 184, 184, 255))
        return Image.alpha_composite(background, rgba).convert("RGB")


    def save_png(image: Image.Image, path: str | Path, *, preview_path: str | Path | None = None,
                 require_transparency: bool = False) -> Image.Image:
        """Save an image as RGBA PNG and optionally save a checkerboard preview."""
        if not isinstance(image, Image.Image):
            raise TypeError("L’image à enregistrer doit être une image PIL.")
        output = Path(path)
        if output.suffix.lower() != ".png":
            raise ValueError("Le fichier de sortie doit porter l’extension .png.")
        rgba = image.convert("RGBA")
        if require_transparency:
            require_transparent_background(rgba)
        preview = Path(preview_path) if preview_path is not None else None
        if preview is not None:
            if preview.suffix.lower() != ".png" or preview.resolve() == output.resolve():
                raise ValueError("L’aperçu doit être un autre fichier .png.")
        output.parent.mkdir(parents=True, exist_ok=True)
        rgba.save(output, format="PNG")
        if preview is not None:
            preview.parent.mkdir(parents=True, exist_ok=True)
            checkerboard_preview(rgba).save(preview, format="PNG")
        return rgba



## library.model_cards

Bibliothèque partagée. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_image21.model_cards':
    """Package real paired generations and reproducible Hugging Face model cards."""




    import sys
    import hashlib


    import importlib.metadata


    import json


    from pathlib import Path, PurePosixPath


    import re


    import shutil


    import subprocess


    import time


    from urllib.parse import quote


    from .image_io import checkerboard_preview


    COMPARISON_SCHEMA = 1


    LABELS = {
        "encoder": ("VL 8B d’origine — BF16", "Sharm VL 4B gelé — FP8 + projecteur BF16"),
        "lora": ("Base Sharm sans LoRA", "Même base Sharm avec LoRA sprites"),
    }


    def sha256(path):
        path = Path(path)
        if path.suffix == ".ipynb":
            return sys.modules["sharm_notebook_runtime"].sha256(path)
        result = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
                result.update(block)
        return result.hexdigest()


    def generation_kwargs(steps: int) -> dict:
        """One parameter set is passed unchanged to both members of every pair."""
        if steps < 1:
            raise ValueError("Le nombre de pas de comparaison doit être positif.")
        return dict(width=512, height=512, output_resolution=512, num_inference_steps=steps,
                    true_cfg_scale=1.0, negative_prompt=None, num_images_per_prompt=1,
                    use_kv_cache=True, output_type="pil")


    def comparison_manifest(kind: str, pipeline, *, steps: int, projector_sha256: str,
                            controls: dict, selection: dict, script: Path, project: Path) -> dict:
        import diffusers
        import torch

        if kind not in LABELS:
            raise ValueError(kind)
        versions = {name: importlib.metadata.version(name) for name in
                    ("torch", "diffusers", "transformers", "bitsandbytes", "safetensors", "Pillow", "kernels")}
        source_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=project, text=True).strip()
        diffusers_root = Path(diffusers.__file__).resolve().parents[2]
        diffusers_commit = None
        if (diffusers_root / ".git").exists():
            revision = subprocess.run(["git", "rev-parse", "HEAD"], cwd=diffusers_root,
                                      text=True, capture_output=True, check=False)
            if revision.returncode == 0:
                diffusers_commit = revision.stdout.strip()
        scheduler = json.loads(json.dumps(dict(pipeline.scheduler.config), default=str))
        return {
            "schema_version": COMPARISON_SCHEMA, "kind": kind,
            "projector_sha256": projector_sha256,
            "settings": {**generation_kwargs(steps), "scheduler_class": type(pipeline.scheduler).__name__,
                         "scheduler_config": scheduler, "generator_device": "cuda",
                         "initial_noise": "fresh generator reset to the pair seed for each member",
                         "diffusion_precision": "NF4, double quantization, BF16 compute",
                         "vae_precision": "BF16", "postprocess": "none" if kind == "encoder" else "512 to 128, nearest-neighbor"},
            "controls": controls, "selection": selection,
            "provenance": {"created_at": time.time(), "source_commit": source_commit,
                           "script_sha256": sha256(script), "model_cards_sha256": sha256(Path(__file__)),
                           "versions": versions, "cuda_version": torch.version.cuda,
                           "diffusers_source_commit": diffusers_commit,
                           "gpu": torch.cuda.get_device_name(), "capability": list(torch.cuda.get_device_capability())},
            "pairs": [],
        }


    def save_member(image, directory: Path, stem: str, label: str, final_size: int | None = None) -> dict:
        """Keep alpha in downloads; composite a separate checkerboard for the card."""
        from PIL import Image

        if image.mode != "RGBA" or image.size != (512, 512):
            raise ValueError("La comparaison attend une génération RGBA 512 × 512.")
        if not re.fullmatch(r"[a-zA-Z0-9_-]+", stem):
            raise ValueError("Nom de fichier de comparaison invalide.")
        directory.mkdir(parents=True, exist_ok=True)

        def save_asset(value, filename):
            path = directory / filename
            value.save(path)
            return {"file": filename, "sha256": sha256(path), "width": value.width,
                    "height": value.height, "mode": value.mode}

        native = save_asset(image, f"{stem}-512.png")
        if final_size is not None:
            if final_size != 128:
                raise ValueError("Seule la réduction documentée 512 vers 128 est prise en charge.")
            image = image.resize((final_size, final_size), Image.Resampling.NEAREST)
            final = save_asset(image, f"{stem}-128.png")
        else:
            final = native
        preview = checkerboard_preview(image, tile_size=8)
        return {"label": label, "image": final, "source": native,
                "preview": save_asset(preview, f"{stem}-preview.png")}


    def _asset_path(directory: Path, filename: str) -> Path:
        relative = PurePosixPath(filename)
        if not filename or relative.is_absolute() or ".." in relative.parts or "\\" in filename:
            raise ValueError("Chemin d’image de comparaison invalide.")
        path = directory.joinpath(*relative.parts)
        if not path.resolve().is_relative_to(directory.resolve()) or path.suffix != ".png":
            raise ValueError("Image située hors du dossier de comparaison.")
        return path


    def load_comparisons(directory: Path, kind: str, projector_sha256: str,
                         lora_sha256: str | None = None) -> dict:
        """Refuse incomplete, stale or unbound image galleries during export."""
        from PIL import Image

        manifest = json.loads((directory / "comparison.json").read_text(encoding="utf-8"))
        if (manifest["schema_version"] != COMPARISON_SCHEMA or manifest["kind"] != kind
                or manifest["projector_sha256"] != projector_sha256):
            raise ValueError("Galerie incompatible avec cette base Sharm.")
        if lora_sha256 is not None and manifest["controls"].get("lora_sha256") != lora_sha256:
            raise ValueError("Galerie incompatible avec ces poids LoRA.")
        settings = manifest["settings"]
        if any(settings.get(key) != value for key, value in generation_kwargs(settings["num_inference_steps"]).items()):
            raise ValueError("Paramètres de génération de comparaison incompatibles.")
        if not manifest["pairs"]:
            raise ValueError("Une galerie vide ne peut pas être publiée.")
        seen = set()
        for pair in manifest["pairs"]:
            if pair["id"] in seen or not pair["prompt"].strip() or not isinstance(pair["seed"], int):
                raise ValueError("Paire de comparaison vide ou dupliquée.")
            seen.add(pair["id"])
            for side, label in zip(("left", "right"), LABELS[kind]):
                member = pair[side]
                if member["label"] != label:
                    raise ValueError("Les côtés de la comparaison sont ambigus.")
                for key in ("image", "source", "preview"):
                    asset = member[key]
                    path = _asset_path(directory, asset["file"])
                    if sha256(path) != asset["sha256"]:
                        raise ValueError(f"Image de comparaison modifiée : {path.name}")
                    expected_size = (128, 128) if kind == "lora" and key != "source" else (512, 512)
                    with Image.open(path) as image:
                        if (image.size != expected_size or image.size != (asset["width"], asset["height"])
                                or image.mode != asset["mode"] or image.mode != ("RGB" if key == "preview" else "RGBA")):
                            raise ValueError("Format ou taille d’image de comparaison incorrects.")
        return manifest


    def package_comparisons(source: Path, destination: Path, kind: str, projector_sha256: str,
                            lora_sha256: str | None = None) -> dict:
        manifest = load_comparisons(source, kind, projector_sha256, lora_sha256)
        destination.mkdir(parents=True, exist_ok=True)
        filenames = {asset["file"] for pair in manifest["pairs"] for side in ("left", "right")
                     for asset in (pair[side]["image"], pair[side]["source"], pair[side]["preview"])}
        for filename in sorted(filenames):
            target = _asset_path(destination, filename)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(_asset_path(source, filename), target)
        shutil.copy2(source / "comparison.json", destination / "comparison.json")
        return load_comparisons(destination, kind, projector_sha256, lora_sha256)


    def measure_footprint(original: Path, student: Path, projector: Path) -> dict:
        def tensor_bytes(directory):
            paths = list(directory.glob("*.safetensors"))
            if not paths:
                raise ValueError(f"Aucun poids safetensors dans {directory}")
            return sum(path.stat().st_size for path in paths)

        teacher_bytes = tensor_bytes(original / "text_encoder")
        student_bytes = tensor_bytes(student)
        projector_bytes = (projector / "model.safetensors").stat().st_size
        shared_bytes = tensor_bytes(original / "transformer") + tensor_bytes(original / "vae")
        return {"measurement": "sum of checkpoint safetensors file sizes, headers included; excludes configs, caches and optimizer states",
                "original_encoder_bytes": teacher_bytes, "student_encoder_bytes": student_bytes,
                "projector_bytes": projector_bytes, "shared_diffusion_vae_bytes": shared_bytes,
                "original_total_bytes": teacher_bytes + shared_bytes,
                "sharm_total_bytes": student_bytes + projector_bytes + shared_bytes,
                "encoder_reduction_percent": 100 * (1 - (student_bytes + projector_bytes) / teacher_bytes),
                "total_reduction_percent": 100 * (1 - (student_bytes + projector_bytes + shared_bytes) / (teacher_bytes + shared_bytes))}


    def _prompt_block(prompt: str) -> str:
        fence = "`" * max(3, 1 + max((len(run) for run in re.findall(r"`+", prompt)), default=0))
        return f"{fence}text\n{prompt}\n{fence}"


    def _hf_file_url(repo_id: str, prefix: str, filename: str, *, download: bool = False) -> str:
        path = "/".join(quote(part, safe="-_.") for part in (*prefix.split("/"), *filename.split("/")))
        url = f"https://huggingface.co/{quote(repo_id, safe='/-_.')}/resolve/main/{path}"
        return f"{url}?download=true" if download else url


    def gallery_markdown(manifest: dict, prefix: str, repo_id: str) -> str:
        lines = []
        left, right = LABELS[manifest["kind"]]
        for index, pair in enumerate(manifest["pairs"], 1):
            lines.extend((f"### Exemple {index} — graine `{pair['seed']}`", "", _prompt_block(pair["prompt"]), "",
                          f"| {left} | {right} |", "| --- | --- |"))
            cells, downloads = [], []
            for side, label in zip(("left", "right"), (left, right)):
                member = pair[side]
                preview_url = _hf_file_url(repo_id, prefix, member["preview"]["file"])
                cells.append(f'<img src="{preview_url}" alt="{label}" width="256" />')
                final, native = member["image"], member["source"]
                final_url = _hf_file_url(repo_id, prefix, final["file"], download=True)
                links = f"[Télécharger le PNG RGBA {final['width']} px]({final_url})"
                if native["file"] != final["file"]:
                    native_url = _hf_file_url(repo_id, prefix, native["file"], download=True)
                    links += f" · [source PNG 512 px]({native_url})"
                downloads.append(links)
            lines.extend(("| " + " | ".join(cells) + " |", "| " + " | ".join(downloads) + " |", ""))
        settings = manifest["settings"]
        lines.extend((f"Réglages communs : {settings['width']} × {settings['height']}, "
                      f"{settings['num_inference_steps']} pas, `{settings['scheduler_class']}`, "
                      f"CFG {settings['true_cfg_scale']:g}, aucun prompt négatif. "
                      "Le générateur CUDA est réinitialisé à la même graine pour chaque côté.", "",
                      f"[Prompts, graines, réglages complets, versions et empreintes des fichiers]({prefix}/comparison.json).", ""))
        return "\n".join(lines) + "\n"


    def compact_objective(footprint: dict) -> str:
        def gib(value):
            return f"{value / 2**30:.2f} Gio"

        return (
            "## Objectif : compacter le modèle de base\n\n"
            "Sharm remplace l’encodeur VL d’environ 8B fourni avec Qwen-Image-2.1 par "
            "Qwen3-VL-4B-Instruct-FP8 gelé et un petit projecteur MLP entraîné, de largeur 2560 → 4096. "
            "Le but est de réduire la taille des poids du modèle de base tout en mesurant la qualité conservée. "
            "Le projecteur est exporté en BF16. Le checkpoint 4B utilise FP8 E4M3 pour ses couches quantifiées ; "
            "ses exceptions déclarées, dont la branche vision, conservent leur précision BF16.\n\n"
            "| Poids sur disque | Base d’origine | Base Sharm |\n"
            "| --- | --- | --- |\n"
            f"| Encodeur et éventuel projecteur | {gib(footprint['original_encoder_bytes'])} | "
            f"{gib(footprint['student_encoder_bytes'] + footprint['projector_bytes'])} |\n"
            f"| Diffuseur + VAE, communs aux deux bases | {gib(footprint['shared_diffusion_vae_bytes'])} | "
            f"{gib(footprint['shared_diffusion_vae_bytes'])} |\n"
            f"| Total des poids safetensors | {gib(footprint['original_total_bytes'])} | "
            f"{gib(footprint['sharm_total_bytes'])} |\n\n"
            f"Réduction mesurée : **{footprint['encoder_reduction_percent']:.1f} %** pour l’encodeur avec son "
            f"projecteur, **{footprint['total_reduction_percent']:.1f} %** sur les poids de la base complète. "
            "Ces mesures additionnent les tailles des fichiers safetensors ; configurations, caches, optimiseur "
            "et LoRA sont exclus. [Mesures en octets](footprint.json). La VRAM maximale et la vitesse "
            "d’inférence demandent des mesures séparées.\n\n"
            "Les poids du diffuseur et du VAE exportés sont ceux de la base d’origine. Pour les galeries, "
            "le même diffuseur est chargé en NF4 avec calcul BF16 des deux côtés, et le VAE en BF16. "
            "La comparaison VL mesure donc conjointement le remplacement 8B → 4B, FP8 et le projecteur, "
            "dans ce runtime commun.\n\n"
        )


    def loading_example(with_lora: bool = False) -> str:
        setup = (
            "## Charger le modèle\n\n"
            "Le runtime de référence est Linux/WSL, CUDA 13.0 et une RTX 5060 Ti de 16 Gio. "
            "Les versions et la révision Diffusers sont figées dans "
            "[requirements-inference.txt](requirements-inference.txt). Installer ces dépendances dans "
            "un environnement Python séparé, puis authentifier localement un compte ayant accès "
            "aux dépôts privés avec `hf auth login`.\n\n"
            "```python\nimport json\nimport sys\nfrom pathlib import Path\nimport torch\n"
            "from PIL import Image\nfrom huggingface_hub import snapshot_download\n\n"
        )
        if with_lora:
            setup += (
                'lora = Path(snapshot_download("ethanzxv/Sharm-Image-2.1-Sprites-128-LoRA", token=True))\n'
                'metadata = json.loads((lora / "sharm_lora_config.json").read_text())\n'
                'base = Path(snapshot_download(metadata["base_model"],\n'
                '                              revision=metadata.get("base_hf_revision"), token=True))\n'
            )
        else:
            setup += 'base = Path(snapshot_download("ethanzxv/Sharm-Image-2.1-VL4B-FP8", token=True))\n'
        setup += (
            'sys.path.insert(0, str(base))\n'
            'from sharm_image21.pipeline import SharmImage21Pipeline\n\n'
            'pipeline = SharmImage21Pipeline.from_pretrained(base)\n'
        )
        if with_lora:
            setup += (
                'case = json.loads((lora / "comparisons/lora/comparison.json").read_text())["pairs"][0]\n'
                'image = pipeline.generate_png(case["prompt"], "sharm-sprite-512.png",\n'
                '    preview_path="sharm-sprite-512-preview.png", transparent_background=True,\n'
                '    lora_dir=lora, width=512, height=512, output_resolution=512,\n'
                '    num_inference_steps=30, true_cfg_scale=1.0,\n'
                '    attention_kwargs={"scale": metadata.get("recommended_weight", 1.0)},\n'
                '    generator=torch.Generator("cuda").manual_seed(case["seed"]))\n'
                'pipeline.save_png(image.resize((128, 128), Image.Resampling.NEAREST),\n'
                '    "sharm-sprite-128.png", preview_path="sharm-sprite-128-preview.png",\n'
                '    require_transparency=True)\n'
            )
        else:
            setup += (
                'case = json.loads((base / "comparisons/encoder/comparison.json").read_text())["pairs"][0]\n'
                'pipeline.generate_png(case["prompt"], "sharm-image.png",\n'
                '    preview_path="sharm-image-preview.png", transparent_background=True,\n'
                '    width=512, height=512, output_resolution=512, num_inference_steps=20,\n'
                '    true_cfg_scale=1.0,\n'
                '    generator=torch.Generator("cuda").manual_seed(case["seed"]))\n'
            )
        return setup + "```\n\n"


    def base_card(manifest: dict, footprint: dict, validation: dict) -> str:
        metrics = validation["embedding_metrics"]
        return (
            "---\nlicense: other\nlicense_name: qwen-research\nlicense_link: LICENSE\n"
            "pipeline_tag: text-to-image\nbase_model:\n- Qwen/Qwen-Image-2.1\n"
            "- Qwen/Qwen3-VL-4B-Instruct-FP8\nlanguage:\n- fr\n- en\n---\n\n"
            "# Sharm Image 2.1 VL4B FP8\n\nBuilt with Qwen. Checkpoint de recherche destiné au dépôt privé "
            "`ethanzxv/Sharm-Image-2.1-VL4B-FP8`.\n\n"
            + compact_objective(footprint)
            + "## Comparaison du VL 8B et du VL 4B FP8 + projecteur\n\n"
            "Même prompt, même graine et mêmes réglages pour chaque paire, sans LoRA. "
            "Les prompts sont issus de la validation du projecteur. Les exemples sont sélectionnés avant "
            "génération, sans sélection selon leur apparence. Les aperçus compositent l’alpha sur un damier ; "
            "les PNG téléchargeables conservent le canal alpha.\n\n"
            + gallery_markdown(manifest, "comparisons/encoder", "ethanzxv/Sharm-Image-2.1-VL4B-FP8")
            + "## Validation et utilisation\n\n"
            f"Validation des embeddings : erreur quadratique relative `{metrics['relative_mse']:.4f}`, "
            f"cosinus `{metrics['cosine']:.4f}`, ratio de normes `{metrics['norm_ratio']:.4f}`. "
            f"Erreur relative moyenne des prédictions du diffuseur : `{validation['denoiser_relative_mse']:.4f}`. "
            "[Rapport complet](base-validation.json). Ces contrôles numériques et les quelques exemples "
            "visuels demandent une revue humaine pour juger le suivi des consignes et la qualité.\n\n"
            "Utiliser le loader fourni `sharm_image21.pipeline.SharmImage21Pipeline.from_pretrained`. "
            "Il encode le texte avec le 4B FP8 et le projecteur, puis libère l’encodeur avant de charger "
            "le diffuseur en NF4. L’encodeur 8B sert à la distillation et aux références en cache. "
            "Cette version est expérimentale en texte vers image ; l’édition par image reste à valider.\n\n"
            + loading_example()
            + "## Sources et licence\n\n"
            "Base : [Qwen-Image-2.1](https://huggingface.co/Qwen/Qwen-Image-2.1). "
            "Encodeur : [Qwen3-VL-4B-Instruct-FP8](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct-FP8). "
            "Le corpus d’alignement comprend des prompts "
            "[DiffusionDB](https://huggingface.co/datasets/poloclub/diffusiondb) sous CC0-1.0, "
            "des compositions rédigées par templates et des légendes sprites. "
            f"[Code d’entraînement et d’évaluation](https://github.com/EthanCarollo/poc-malaga/tree/"
            f"{manifest['provenance']['source_commit']}/2-LoRA-qwen). "
            "Les révisions sources et le projecteur sont identifiés dans `sharm_config.json` "
            "et `training_manifest.json`. Qwen Research s’applique au modèle combiné ; "
            "les notices et la licence Apache-2.0 de l’encodeur sont conservées.\n"
        )


    def lora_card(lora: dict, encoder: dict, footprint: dict) -> str:
        training = lora["controls"]["training"]
        return (
            "---\nlicense: other\nlicense_name: qwen-research\nlicense_link: LICENSE\n"
            "pipeline_tag: text-to-image\nbase_model: ethanzxv/Sharm-Image-2.1-VL4B-FP8\nbase_model_relation: adapter\n"
            "tags:\n- lora\n- pixel-art\nlanguage:\n- fr\n- en\n---\n\n"
            "# Sharm Image 2.1 Sprites 128 LoRA\n\nBuilt with Qwen. Checkpoint de recherche destiné au dépôt privé "
            "`ethanzxv/Sharm-Image-2.1-Sprites-128-LoRA`.\n\n"
            "La base Sharm vise à compacter Qwen-Image-2.1 grâce à un VL 4B FP8 gelé et son projecteur. "
            "Ce LoRA adapte le diffuseur de la base Sharm aux sprites de monstres. "
            "Le 4B FP8 et le projecteur restent gelés pendant cet entraînement. "
            "Les latents d’apprentissage sont en 512 × 512 RGBA ; le livrable sprite est un PNG "
            "128 × 128 après réduction nearest-neighbor.\n\n"
            f"Entraînement : {training['steps']} pas d’optimiseur, rang/alpha {training['rank']}, "
            f"batch 1 et accumulation {training['accumulation']}, taux d’apprentissage `{training['learning_rate']:g}`. "
            f"{lora['controls']['training_image_count']} images sont utilisées pour l’apprentissage du LoRA "
            f"et {lora['controls']['validation_image_count']} pour sa validation.\n\n"
            "## Avant / après LoRA — même prompt\n\n"
            "Chaque paire utilise exactement le même prompt, la même graine, les mêmes embeddings "
            "du VL 4B FP8 + projecteur et les mêmes réglages du diffuseur. Seul l’adaptateur LoRA "
            f"change, avec un poids de {lora['controls']['lora_weight']:g}. Les espèces montrées sont réservées à la validation du LoRA ; "
            "leurs légendes peuvent être présentes dans le corpus d’alignement du projecteur. "
            "Les six premières espèces distinctes de validation sont retenues avant génération.\n\n"
            "Les deux côtés subissent la même réduction 512 → 128 en nearest-neighbor, sans détourage "
            "ni retouche après génération. Le damier des aperçus permet de voir l’alpha ; les liens "
            "fournissent les PNG RGBA 128 px et leurs sources 512 px.\n\n"
            + gallery_markdown(lora, "comparisons/lora", "ethanzxv/Sharm-Image-2.1-Sprites-128-LoRA")
            + compact_objective(footprint)
            + "## VL 8B d’origine / VL 4B FP8 + projecteur — même prompt\n\n"
            "Cette seconde comparaison concerne la compaction de la base avant le LoRA sprites. "
            "Le LoRA est désactivé des deux côtés. Le même diffuseur NF4, le même VAE BF16, "
            "le prompt, la graine et les réglages sont conservés ; le conditionnement VL change. "
            "Ces images sont les références de validation de la base Sharm, également présentes "
            "sur sa fiche.\n\n"
            + gallery_markdown(encoder, "comparisons/encoder", "ethanzxv/Sharm-Image-2.1-Sprites-128-LoRA")
            + "## Compatibilité et portée\n\n"
            "Charger ce LoRA avec la base `ethanzxv/Sharm-Image-2.1-VL4B-FP8` et son loader fourni. "
            "`sharm_lora_config.json` lie les poids LoRA à l’empreinte du projecteur exact. "
            "Les versions du runtime, révisions du code, réglages et empreintes des images sont "
            "enregistrés dans les manifestes de comparaison. La reproductibilité dépend aussi du matériel "
            "et des kernels utilisés. La revue visuelle de l’identité, du pixel art et de la transparence "
            "reste nécessaire ; ces exemples ne constituent pas une garantie de qualité générale.\n\n"
            + loading_example(with_lora=True)
            + "Le dataset local comprend 770 sprites RGBA issus de 385 espèces. Le fond exact "
            "`#A5EBFF` a été retiré du dataset, y compris les îlots déconnectés. Les deux variantes "
            "d’une espèce restent dans le même ensemble. Qwen Research et les attributions de la base "
            "sont conservées dans `LICENSE`, `LICENSE.encoder` et `NOTICE`.\n"
            f"[Code d’entraînement et d’évaluation](https://github.com/EthanCarollo/poc-malaga/tree/"
            f"{lora['provenance']['source_commit']}/2-LoRA-qwen).\n"
        )



## library.pipeline

Bibliothèque partagée. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_image21.pipeline':


    import gc


    import json


    from pathlib import Path


    import torch


    from torch.nn.utils.rnn import pad_sequence


    from .core import FrozenEncoder, Projector, file_hash


    from .diffusion import load_diffuser_pipeline


    from .image_io import save_png as _save_png


    from .spatial_data import layout_prompt


    class SharmImage21Pipeline:
        """Sequential FP8 conditioning and NF4 diffusion, with optional spatial control.

        This first training run validates text-to-image only. Reference-image editing
        must receive its own multimodal alignment and evaluation before enabling it.
        """

        def __init__(self, directory: Path):
            self.directory = directory
            self.config = json.loads((directory / "sharm_config.json").read_text())
            if self.config["encoder_format"] != "fp8-e4m3" or not self.config["encoder_frozen"]:
                raise ValueError("Encodeur Sharm gelé en FP8 requis.")
            if file_hash(directory / "projector/model.safetensors") != self.config["projector_sha256"]:
                raise ValueError("Empreinte du projecteur incorrecte.")
            if self.config.get("pipeline_format") == "spatial-v1":
                if not self.config.get("training_complete"):
                    raise ValueError("Entraînement spatial incomplet.")
                lora_sha = self.config.get("spatial_lora_sha256")
                if lora_sha and file_hash(directory / "lora/pytorch_lora_weights.safetensors") != lora_sha:
                    raise ValueError("Empreinte du LoRA spatial incorrecte.")
            elif not self.config.get("base_validation_passed"):
                raise ValueError("Base Sharm non validée.")

        @classmethod
        def from_pretrained(cls, path_or_repo, revision=None, token=None):
            directory = Path(path_or_repo)
            if not directory.is_dir():
                from huggingface_hub import snapshot_download
                directory = Path(snapshot_download(str(path_or_repo), revision=revision, token=token))
            return cls(directory)

        @torch.inference_mode()
        def encode_prompt(self, prompt, layout=None):
            prompts = [prompt] if isinstance(prompt, str) else prompt
            if not isinstance(prompts, (list, tuple)) or not prompts or not all(isinstance(p, str) for p in prompts):
                raise ValueError("Un prompt ou une liste non vide de prompts est requis.")
            boxes = None
            if layout is not None:
                if len(prompts) != 1:
                    raise ValueError("Le contrôle par boîtes attend un seul prompt.")
                text, boxes = layout_prompt(prompts[0], layout)
                prompts = [text]
            encoder = FrozenEncoder("student", model_dir=self.directory / "text_encoder",
                                    processor_dir=self.directory / "processor")
            if encoder.identity != self.config["student_identity"]:
                raise ValueError("Encodeur différent de celui utilisé pour entraîner le projecteur.")
            features = encoder.encode(prompts)
            del encoder
            gc.collect()
            torch.cuda.empty_cache()
            model = Projector.load(self.directory / "projector", "cuda").eval()
            if boxes is not None:
                boxes = torch.tensor(boxes, device="cuda", dtype=torch.bfloat16)
            output = []
            for hidden, _ in features:
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    output.append(model(hidden.cuda(), layout=boxes).to(torch.bfloat16).cpu())
            del model, features
            gc.collect()
            torch.cuda.empty_cache()
            embeddings = pad_sequence(output, batch_first=True)
            mask = torch.arange(embeddings.shape[1])[None, :] < torch.tensor([len(x) for x in output])[:, None]
            return embeddings, mask

        def __call__(self, prompt, *, layout=None, lora_dir=None, image=None, **kwargs):
            if image is not None:
                raise ValueError("L'édition par image n'est pas validée pour cette première base Sharm.")
            embeddings, mask = self.encode_prompt(prompt, layout=layout)
            negative = kwargs.pop("negative_prompt", None)
            if negative is not None and kwargs.get("true_cfg_scale", 1.0) > 1:
                negative_embeddings, negative_mask = self.encode_prompt(negative)
                kwargs.update(negative_prompt_embeds=negative_embeddings.cuda(), negative_prompt_embeds_mask=negative_mask.cuda())
            pipeline = load_diffuser_pipeline(self.directory)
            if self.config.get("pipeline_format") == "spatial-v1" and self.config.get("spatial_lora_sha256"):
                if lora_dir is not None:
                    raise ValueError("Le candidat spatial utilise déjà son LoRA entraîné.")
                pipeline.load_lora_weights(self.directory / "lora", adapter_name="spatial")
            elif lora_dir is not None:
                lora_dir = Path(lora_dir)
                metadata = json.loads((lora_dir / "sharm_lora_config.json").read_text())
                if metadata["projector_sha256"] != self.config["projector_sha256"]:
                    raise ValueError("Ce LoRA a été entraîné avec un autre projecteur.")
                if file_hash(lora_dir / "pytorch_lora_weights.safetensors") != metadata["lora_sha256"]:
                    raise ValueError("Empreinte des poids LoRA incorrecte.")
                if metadata.get("base_diffusion_revision", self.config["base_revision"]) != self.config["base_revision"]:
                    raise ValueError("Ce LoRA vise une autre révision du diffuseur.")
                pipeline.load_lora_weights(lora_dir, adapter_name="sharm_sprites")
            try:
                return pipeline(prompt_embeds=embeddings.cuda(), prompt_embeds_mask=mask.cuda(), **kwargs)
            finally:
                del pipeline
                gc.collect()
                torch.cuda.empty_cache()

        def generate_png(self, prompt, path, *, preview_path=None, transparent_background=True,
                         **kwargs):
            """Generate one RGBA PNG, optionally with a checkerboard preview beside it.

            Transparent output is requested in the prompt and checked before the PNG
            is written. The regular ``__call__`` API remains available for batched or
            non-PNG inference.
            """
            if kwargs.get("output_type", "pil") != "pil":
                raise ValueError("generate_png requiert output_type='pil'.")
            if kwargs.get("num_images_per_prompt", 1) != 1:
                raise ValueError("generate_png attend une seule image par prompt.")
            if transparent_background:
                instruction = (
                    "The background is fully transparent; only the foreground subject "
                    "is visible, with transparent pixels outside the subject."
                )
                if isinstance(prompt, str):
                    prompt = f"{prompt.rstrip()} {instruction}"
                elif isinstance(prompt, (list, tuple)) and all(isinstance(item, str) for item in prompt):
                    if len(prompt) != 1:
                        raise ValueError("generate_png attend un seul prompt.")
                    prompt = [f"{item.rstrip()} {instruction}" for item in prompt]
                else:
                    raise TypeError("Le prompt doit être une chaîne ou une liste de chaînes.")
            elif isinstance(prompt, (list, tuple)) and len(prompt) != 1:
                raise ValueError("generate_png attend un seul prompt.")
            elif not isinstance(prompt, str) and not isinstance(prompt, (list, tuple)):
                raise TypeError("Le prompt doit être une chaîne ou une liste de chaînes.")

            output = self(prompt, **kwargs)
            images = getattr(output, "images", None)
            if images is None or len(images) != 1:
                raise ValueError("generate_png attend une seule image PIL (output_type='pil').")
            return _save_png(images[0], path, preview_path=preview_path,
                             require_transparency=transparent_background)

        @staticmethod
        def save_png(image, path, *, preview_path=None, require_transparency=False):
            """Save an existing result as PNG and optionally create its card preview."""
            return _save_png(image, path, preview_path=preview_path,
                             require_transparency=require_transparency)



## library.spatial_data

Bibliothèque partagée. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_image21.spatial_data':
    """Geometry and dataset helpers. No model or CUDA dependency."""




    import sys
    import hashlib


    import json


    import math


    import re


    from pathlib import Path


    RELATIONS = ("left", "right", "above", "below")


    def fingerprint(value):
        return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


    def sha256(path):
        path = Path(path)
        if path.suffix == ".ipynb":
            return sys.modules["sharm_notebook_runtime"].sha256(path)
        result = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
                result.update(block)
        return result.hexdigest()


    def read_rows(path):
        return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


    def write_json(path, value):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + ".tmp")
        temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        temporary.replace(path)


    def checked_box(box):
        if len(box) != 4 or not all(math.isfinite(float(v)) and 0 <= v <= 1 for v in box):
            raise ValueError("Boîte attendue : [x0, y0, x1, y1] entre 0 et 1.")
        if box[2] <= box[0] or box[3] <= box[1]:
            raise ValueError("Une boîte doit avoir une surface positive.")
        return list(map(float, box))


    def layout_prompt(prompt, objects):
        """Bind ordered labels and normalized boxes using the training convention."""
        if not isinstance(prompt, str) or not isinstance(objects, list) or not 1 <= len(objects) <= 16:
            raise ValueError("Un prompt et une liste de 1 à 16 objets sont requis.")
        boxes, labels = [], []
        for obj in objects:
            if not isinstance(obj, dict) or not isinstance(obj.get("label"), str) or not obj["label"].strip():
                raise ValueError("Chaque objet doit avoir un label explicite.")
            try:
                boxes.append(checked_box(obj.get("box", [])))
            except (TypeError, ValueError) as error:
                raise ValueError("Chaque objet requiert une boîte xyxy normalisée valide.") from error
            labels.append(obj["label"].strip())
        if "normalized object boxes xyxy:" in prompt.lower():
            raise ValueError("Fournir le prompt naturel et les boîtes séparément.")
        bound = [bool(re.search(rf"(?:Object|Objet)\s+{i+1}\s*:\s*{re.escape(label)}\.", prompt, re.IGNORECASE))
                 for i, label in enumerate(labels)]
        numbered = [int(n) for n in re.findall(r"(?:Object|Objet)\s+(\d+)\s*:", prompt, re.IGNORECASE)]
        text = prompt.rstrip()
        if numbered:
            if sorted(numbered) != list(range(1, len(labels)+1)) or not all(bound):
                raise ValueError("Les labels numérotés du prompt et du layout doivent correspondre.")
        else:
            text += " "+" ".join(f"Object {i+1}: {label}." for i, label in enumerate(labels))
        text += " Normalized object boxes xyxy: "+json.dumps(boxes, separators=(",", ":"))+"."
        return text, boxes


    def compose(size, objects, asset_root):
        """Place existing alpha crops inside requested boxes without overlap or clipping.

        Actual visible alpha boxes are returned: aspect ratio is preserved, so an
        object's painted box can be smaller than its requested allocation.
        """
        from PIL import Image, ImageChops
        if size < 32 or not objects:
            raise ValueError("Une résolution d'au moins 32 pixels est requise.")
        canvas = Image.new("RGBA", (size, size), (0, 0, 0, 0))
        records = []
        occupied = Image.new("L", canvas.size, 0)
        for index, obj in enumerate(objects):
            slot = checked_box(obj["box"])
            path = Path(asset_root) / obj["file_name"]
            with Image.open(path) as source:
                if source.mode != "RGBA":
                    raise ValueError(f"Asset RGBA requis : {path}")
                bounds = source.getchannel("A").getbbox()
                if bounds is None:
                    raise ValueError(f"Asset vide : {path}")
                tile = source.crop(bounds)
            x0, y0 = math.ceil(slot[0]*size), math.ceil(slot[1]*size)
            x1, y1 = math.floor(slot[2]*size), math.floor(slot[3]*size)
            if min(x1-x0, y1-y0) < 1:
                raise ValueError("Boîte trop petite à cette résolution.")
            scale = min((x1-x0)/tile.width, (y1-y0)/tile.height)
            tile = tile.resize((max(1, int(tile.width*scale)), max(1, int(tile.height*scale))), Image.Resampling.NEAREST)
            x, y = x0+(x1-x0-tile.width)//2, y0+(y1-y0-tile.height)//2
            layer = Image.new("RGBA", canvas.size, (0, 0, 0, 0))
            layer.paste(tile, (x, y))
            alpha = layer.getchannel("A")
            if ImageChops.multiply(occupied, alpha).getbbox() is not None:
                raise ValueError("Les objets se chevauchent dans cette composition.")
            occupied = ImageChops.lighter(occupied, alpha)
            canvas = Image.alpha_composite(canvas, layer)
            actual = [v/size for v in alpha.getbbox()]
            records.append({**obj, "object_id": str(index+1), "box": actual, "allocation": slot})
        return canvas, records


    def relation_holds(a, b, relation, margin=0.0):
        a, b = checked_box(a), checked_box(b)
        if relation == "left":
            return a[2]+margin <= b[0]
        if relation == "right":
            return b[2]+margin <= a[0]
        if relation == "above":
            return a[3]+margin <= b[1]
        if relation == "below":
            return b[3]+margin <= a[1]
        raise ValueError(f"Relation inconnue : {relation}")


    def box_iou(a, b):
        a, b = checked_box(a), checked_box(b)
        intersection = max(0, min(a[2], b[2])-max(a[0], b[0]))*max(0, min(a[3], b[3])-max(a[1], b[1]))
        area = lambda x: (x[2]-x[0])*(x[3]-x[1])
        return intersection/(area(a)+area(b)-intersection)


    def center_error(a, b):
        return math.hypot((a[0]+a[2]-b[0]-b[2])/2, (a[1]+a[3]-b[1]-b[3])/2)


    def wilson(successes, count, z=1.959963984540054):
        if count < 1:
            raise ValueError("Un score sans exemple ne peut pas être validé.")
        p = successes/count
        denominator = 1+z*z/count
        center = (p+z*z/(2*count))/denominator
        radius = z*math.sqrt(p*(1-p)/count+z*z/(4*count*count))/denominator
        return [max(0, center-radius), min(1, center+radius)]



## library.__init__

Bibliothèque partagée. Le code est contenu dans ce fichier.

In [ ]:
if __name__ == 'sharm_image21':
    """Frozen FP8 conditioning and projector training for Sharm Image 2.1."""


    __version__ = "0.1.0"

